<a href="https://colab.research.google.com/github/BirasaDivine/Scam-Detection/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ScamGuard: comparing machine learning models for SMS and chat scam detection

This notebook holds the experiments behind **ScamGuard**, a capstone project by Birasa Laura Divine (African Leadership University). It compares 16 approaches from six model families on the same data, tests how well each one catches **scam types it never saw in training**, and exports the best model to the ScamGuard website.

**Data**
- [UCI SMS Spam Collection](https://huggingface.co/datasets/ucirvine/sms_spam): 5,574 real SMS.
- [BothBosu scam dialogues](https://huggingface.co/datasets/BothBosu/scam-dialogue): synthetic phone-call chats labelled by topic (scam: reward, refund, SSN, support; normal: delivery, insurance, telemarketing, wrong number).
- Chats are cut into 45-word pieces, scams and normal messages are balanced within each source, and names, emails, links, amounts and numbers are masked.

**Models:** Naive Bayes, Logistic Regression, Linear SVM, calibrated SVM, XGBoost, CNN, BiLSTM, DistilBERT, Qwen2.5-1.5B (zero-shot and LoRA fine-tuned), four ensembles, Isolation Forest and One-Class SVM.

**Main result:** the plain Linear SVM caught 97.6% of hidden scam topics at a 5% false-alarm cut-off (F1 0.966), as well as the best ensemble and better than the larger models. It is the model used by the website.

**How to read this notebook**
- Cells are kept in the order they were run, so early sections are exploration and later sections refine them. The final numbers come from sections 10 to 13.
- Some cells reload saved results from Google Drive (folder `ScamGuard_masked_v2_seed42`), which is not public. Those cells need your own copy of the results.
- The language-model cells need a GPU (a free Colab T4 works).

**After a runtime restart, run in this order**
1. The GPU restart setup cell (section 3)
2. The masking_model cell (section 5)
3. The cell you need

**Sections** (use the Table of contents on the left to jump)
1. Early exploration and classical baselines (Models 1 to 4)
2. Fair comparison: same false-alarm budget, 3 seeds, bigger false-alarm test
3. GPU restart setup
4. Neural, transformer and language models (Models 5 to 10)
5. Masked rerun v1
6. Masked rerun v2 (main masked results)
7. Website handoff (first version)
8. Extra checks
9. Per-message scores and Google Drive
10. Hidden scam topics at the 5% cut-off (main test)
11. Chat cut-offs and the call-level calibration split
12. Website export (final model)
13. Final results
14. Comparison with existing systems

# 1. Early exploration and classical baselines
## 1.1 First data load and dialogue pieces
Install, first look at BothBosu/scam-dialogue, make_pieces() and the first Naive Bayes run.

### Load the chat data
Installs the libraries and loads the BothBosu scam-dialogue dataset: synthetic phone-call chats labelled scam or normal, each with a topic.

In [ ]:
!pip -q install datasets scikit-learn
from datasets import load_dataset

ds = load_dataset("BothBosu/scam-dialogue")
print(ds)

README.md:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

scam-dialogue_train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

scam-dialogue_test.csv:   0%|          | 0.00/453k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1280 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/320 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 1280
    })
    test: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 320
    })
})


### Shared test harness
Defines the TF-IDF features (words and word pairs), a few hand-written test messages, and `evaluate()`, which trains a model and reports three things: the standard test split, the hand-written messages, and recall when one scam type is hidden from training.

In [ ]:
import pandas as pd, numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

# ---- same data for every model ----
train = ds["train"].to_pandas()
test  = ds["test"].to_pandas()
full  = pd.concat([train, test], ignore_index=True)
SCAM_TYPES   = sorted(full.loc[full["label"] == 1, "type"].unique())
NORMAL_TYPES = sorted(full.loc[full["label"] == 0, "type"].unique())

# ---- same hand-written messages for every model ----
HAND = {
 "scam": [
  "Hi, I sent you money by mistake. Please send it back to this number as soon as possible.",
  "Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 to restore it.",
  "Congratulations! You won a scholarship. Pay the 20,000 processing fee to claim it today.",
  "This is your bank. We noticed unusual activity. Confirm your card number and OTP to avoid account closure.",
  "Your package is held at customs. Pay a 5,000 fee at this link to release it."],
 "normal": [
  "Hey, are we still meeting at 5 for the group project?",
  "Reminder: your appointment is tomorrow at 10am. Reply C to confirm.",
  "Your package has been delivered. Thanks for shopping with us."],
 "normal_but_scam_words": [
  "I called Social Security today and they said my refund is on the way.",
  "My computer got a virus last week. The shop fixed it and gave me a refund."],
}

def tfidf():
    """Same text-to-numbers step for every classical model."""
    return TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)

RESULTS = []

def evaluate(name, make_model, verbose=True):
    """make_model() must return a NEW untrained object with .fit(texts, labels) and .predict(texts)."""
    row = {"model": name}

    # 1. Standard split
    m = make_model().fit(train["dialogue"], train["label"])
    p = m.predict(test["dialogue"])
    row["std_accuracy"] = accuracy_score(test["label"], p)
    row["std_macro_f1"] = f1_score(test["label"], p, average="macro")
    row["std_scam_recall"] = recall_score(test["label"], p, pos_label=1)
    row["std_scam_precision"] = precision_score(test["label"], p, pos_label=1)

    # 2. Hand-written messages
    pred = {g: list(m.predict(msgs)) for g, msgs in HAND.items()}
    row["hand_scams_caught_of_5"] = int(sum(pred["scam"]))
    row["hand_normal_ok_of_3"] = int(3 - sum(pred["normal"]))
    row["hand_scamwords_wrongly_flagged_of_2"] = int(sum(pred["normal_but_scam_words"]))
    if verbose:
        for g, msgs in HAND.items():
            print("\n" + g.upper())
            for msg, y in zip(msgs, pred[g]):
                print(f"  {'SCAM' if y == 1 else 'not scam':9} | {msg[:80]}")

    # 3. Hide one scam type: how many of it are still caught?
    rec = {}
    for t in SCAM_TYPES:
        tr = full[full["type"] != t]
        te = full[(full["type"] == t) & (full["label"] == 1)]
        mm = make_model().fit(tr["dialogue"], tr["label"])
        rec[t] = float((np.array(mm.predict(te["dialogue"])) == 1).mean())
    row["unseen_scam_recall_mean"] = float(np.mean(list(rec.values())))
    row["unseen_scam_recall_min"] = float(min(rec.values()))

    # 4. Hide one normal type: how many are wrongly flagged as scam?
    fpr = {}
    for t in NORMAL_TYPES:
        tr = full[full["type"] != t]
        te = full[(full["type"] == t) & (full["label"] == 0)]
        mm = make_model().fit(tr["dialogue"], tr["label"])
        fpr[t] = float((np.array(mm.predict(te["dialogue"])) == 1).mean())
    row["unseen_normal_false_alarm_mean"] = float(np.mean(list(fpr.values())))
    row["unseen_normal_false_alarm_max"] = float(max(fpr.values()))

    if verbose:
        print("\nHide one scam type, share still caught:", {k: round(v, 2) for k, v in rec.items()})
        print("Hide one normal type, share wrongly flagged:", {k: round(v, 2) for k, v in fpr.items()})

    RESULTS[:] = [r for r in RESULTS if r["model"] != name] + [row]
    return row

def show_results():
    df = pd.DataFrame(RESULTS).set_index("model").round(3)
    print(df.T.to_string())
    return df

### Model 1: Naive Bayes (baseline)
The classic spam filter and the simplest model. Every other model is judged against it.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

evaluate("1. Naive Bayes", lambda: make_pipeline(tfidf(), MultinomialNB()))
show_results()


SCAM
  SCAM      | Hi, I sent you money by mistake. Please send it back to this number as soon as p
  SCAM      | Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 to restore
  SCAM      | Congratulations! You won a scholarship. Pay the 20,000 processing fee to claim i
  SCAM      | This is your bank. We noticed unusual activity. Confirm your card number and OTP
  SCAM      | Your package is held at customs. Pay a 5,000 fee at this link to release it.

NORMAL
  not scam  | Hey, are we still meeting at 5 for the group project?
  not scam  | Reminder: your appointment is tomorrow at 10am. Reply C to confirm.
  not scam  | Your package has been delivered. Thanks for shopping with us.

NORMAL_BUT_SCAM_WORDS
  SCAM      | I called Social Security today and they said my refund is on the way.
  SCAM      | My computer got a virus last week. The shop fixed it and gave me a refund.

Hide one scam type, share still caught: {'refund': 1.0, 'reward': 0.99, 'ssn': 1.0, 'support

,std_accuracy,std_macro_f1,std_scam_recall,std_scam_precision,hand_scams_caught_of_5,hand_normal_ok_of_3,hand_scamwords_wrongly_flagged_of_2,unseen_scam_recall_mean,unseen_scam_recall_min,unseen_normal_false_alarm_mean,unseen_normal_false_alarm_max
model,,,,,,,,,,,
1. Naive Bayes,1.0,1.0,1.0,1.0,5,3,2,0.999,0.995,0.402,0.85


### Model 2: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

evaluate("2. Logistic Regression", lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42)))
show_results()


SCAM
  not scam  | Hi, I sent you money by mistake. Please send it back to this number as soon as p
  not scam  | Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 to restore
  not scam  | Congratulations! You won a scholarship. Pay the 20,000 processing fee to claim i
  not scam  | This is your bank. We noticed unusual activity. Confirm your card number and OTP
  not scam  | Your package is held at customs. Pay a 5,000 fee at this link to release it.

NORMAL
  not scam  | Hey, are we still meeting at 5 for the group project?
  not scam  | Reminder: your appointment is tomorrow at 10am. Reply C to confirm.
  not scam  | Your package has been delivered. Thanks for shopping with us.

NORMAL_BUT_SCAM_WORDS
  SCAM      | I called Social Security today and they said my refund is on the way.
  not scam  | My computer got a virus last week. The shop fixed it and gave me a refund.

Hide one scam type, share still caught: {'refund': 0.95, 'reward': 0.8, 'ssn': 1.0, 'support

,std_accuracy,std_macro_f1,std_scam_recall,std_scam_precision,hand_scams_caught_of_5,hand_normal_ok_of_3,hand_scamwords_wrongly_flagged_of_2,unseen_scam_recall_mean,unseen_scam_recall_min,unseen_normal_false_alarm_mean,unseen_normal_false_alarm_max
model,,,,,,,,,,,
1. Naive Bayes,1.0,1.0,1.0,1.0,5,3,2,0.999,0.995,0.402,0.85
2. Logistic Regression,1.0,1.0,1.0,1.0,0,3,1,0.911,0.795,0.159,0.45


Why whole chats are a problem: they average about 244 words, while the hand-written scams average 16, so scores on short messages are unreliable.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42))
lr.fit(train["dialogue"], train["label"])

print("Average words per training dialogue:", round(train["dialogue"].str.split().str.len().mean()))
print("Average words per hand-written scam:", round(np.mean([len(m.split()) for m in HAND["scam"]])))

print("\nLogistic Regression scam probability for each hand-written scam:")
for msg, p in zip(HAND["scam"], lr.predict_proba(HAND["scam"])[:, 1]):
    print(f"  {p:.2f} | {msg[:70]}")

Average words per training dialogue: 244
Average words per hand-written scam: 16

Logistic Regression scam probability for each hand-written scam:
  0.29 | Hi, I sent you money by mistake. Please send it back to this number as
  0.47 | Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 
  0.42 | Congratulations! You won a scholarship. Pay the 20,000 processing fee 
  0.49 | This is your bank. We noticed unusual activity. Confirm your card numb
  0.25 | Your package is held at customs. Pay a 5,000 fee at this link to relea


### Cut chats into 45-word pieces
Chats are cut into pieces of at most 45 words, and leftovers under 10 words are dropped. Each piece keeps its chat's label and topic, so chats and SMS look alike to the model.

In [ ]:
def make_pieces(df, max_words=45, min_words=10):
    """Cut each dialogue into pieces of at most max_words words. Each piece keeps its dialogue's label and type."""
    rows = []
    for text, typ, lab in zip(df["dialogue"], df["type"], df["label"]):
        words = text.split()
        for i in range(0, len(words), max_words):
            chunk = words[i:i + max_words]
            if len(chunk) >= min_words:          # drop tiny leftovers
                rows.append({"dialogue": " ".join(chunk), "type": typ, "label": lab})
    return pd.DataFrame(rows)

# keep the originals, then switch every test to the short pieces
train_long, test_long = train, test
train = make_pieces(train_long)
test  = make_pieces(test_long)
full  = pd.concat([train, test], ignore_index=True)

print("Training pieces:", len(train), "| test pieces:", len(test))
print("Average words per piece:", round(train["dialogue"].str.split().str.len().mean()))
print(train["label"].value_counts().to_string())

print("\nFirst three pieces of one scam dialogue:")
for t in make_pieces(train_long[train_long["label"] == 1].head(1))["dialogue"].head(3):
    print(" -", t[:150], "...")

Training pieces: 7313 | test pieces: 1817
Average words per piece: 42
label
1    4405
0    2908

First three pieces of one scam dialogue:
 - caller: Hello, is this John? receiver: Yes, it is. Who's calling? caller: My name is Officer Johnson from the Social Security Administration. How are  ...
 - a very important matter. Your social security number has been compromised and we need to take immediate action to protect your identity. receiver: Oh  ...
 - ensure your benefits aren't interrupted. receiver: Okay, what do you need to know? caller: Can you please confirm your social security number for me?  ...


In [ ]:
evaluate("1. Naive Bayes, pieces", lambda: make_pipeline(tfidf(), MultinomialNB()), verbose=False)
evaluate("2. Logistic Regression, pieces", lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42)), verbose=False)
show_results()

model                                1. Naive Bayes  2. Logistic Regression  1. Naive Bayes, pieces  2. Logistic Regression, pieces
std_accuracy                                  1.000                   1.000                   0.977                           0.981
std_macro_f1                                  1.000                   1.000                   0.976                           0.980
std_scam_recall                               1.000                   1.000                   0.990                           0.993
std_scam_precision                            1.000                   1.000                   0.973                           0.976
hand_scams_caught_of_5                        5.000                   0.000                   5.000                           5.000
hand_normal_ok_of_3                           3.000                   3.000                   3.000                           1.000
hand_scamwords_wrongly_flagged_of_2           2.000                   1.000 

,std_accuracy,std_macro_f1,std_scam_recall,std_scam_precision,hand_scams_caught_of_5,hand_normal_ok_of_3,hand_scamwords_wrongly_flagged_of_2,unseen_scam_recall_mean,unseen_scam_recall_min,unseen_normal_false_alarm_mean,unseen_normal_false_alarm_max
model,,,,,,,,,,,
1. Naive Bayes,1.000,1.000,1.000,1.000,5,3,2,0.999,0.995,0.402,0.850
2. Logistic Regression,1.000,1.000,1.000,1.000,0,3,1,0.911,0.795,0.159,0.450
"1. Naive Bayes, pieces",0.977,0.976,0.990,0.973,5,3,2,0.931,0.866,0.375,0.658
"2. Logistic Regression, pieces",0.981,0.980,0.993,0.976,5,1,2,0.913,0.828,0.402,0.686


## 1.2 Adding real SMS (combined data)
ucirvine/sms_spam joined to the dialogue pieces, cell_report() and the combined-data runs.

A first look at another public scam/spam dataset. It was explored here but not used in the final benchmark.

In [ ]:
from datasets import load_dataset

d2 = load_dataset("FredZhang7/all-scam-spam")
print(d2)
df2 = d2["train"].to_pandas()
print(df2.head(10).to_string())
print("\nColumns:", list(df2.columns))
print(df2.iloc[:, -1].value_counts().head(10) if df2.shape[1] > 1 else "")

README.md:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

junkmail_dataset.csv: reconstructing file:   0%|          |  0.00B / 67.2MB            

junkmail_dataset.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/42619 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'is_spam'],
        num_rows: 42619
    })
})
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

### Load the real SMS data
Loads the UCI SMS Spam Collection and checks labels, message length and duplicates.

In [ ]:
from datasets import load_dataset

try:
    sms = load_dataset("ucirvine/sms_spam")
except Exception as e:
    print("First name failed:", e)
    sms = load_dataset("sms_spam")
print(sms)

s = sms["train"].to_pandas()
print("\nColumns:", s.columns.tolist())
print(s["label"].value_counts().to_string())

text_col = [c for c in s.columns if c != "label"][0]
print("\nAverage words per message:", round(s[text_col].str.split().str.len().mean()))

print("\nExamples labelled 1 (spam):")
for t in s[s["label"] == 1][text_col].head(5): print(" -", t[:140])
print("\nExamples labelled 0 (normal):")
for t in s[s["label"] == 0][text_col].head(5): print(" -", t[:140])
print("\nDuplicate messages:", s[text_col].duplicated().sum())

README.md:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  359kB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

Columns: ['sms', 'label']
label
0    4827
1     747

Average words per message: 16

Examples labelled 1 (spam):
 - Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 0845
 - FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50
 - WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Val
 - Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE 
 - SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info


Examples labelled 0 (normal):
 - Go until jurong point, crazy.. A

### Combine SMS and chats
Removes duplicate SMS, splits them 80/20 (stratified, seed 42) and joins them with the chat pieces. The `source` column marks each row as real (SMS) or synthetic (chat).

In [ ]:
from sklearn.model_selection import train_test_split

# --- 1. real SMS: clean, label, split ---
sms_df = s.rename(columns={"sms": "dialogue"})[["dialogue", "label"]].copy()
sms_df["dialogue"] = sms_df["dialogue"].str.strip()

conflicts = (sms_df.groupby("dialogue")["label"].nunique() > 1).sum()
print("Messages that appear with both labels:", conflicts)
sms_df = sms_df.drop_duplicates(subset="dialogue")           # no copy can land in both train and test
sms_df["type"] = np.where(sms_df["label"] == 1, "sms_spam", "sms_ham")
sms_df["source"] = "real"
sms_tr, sms_te = train_test_split(sms_df, test_size=0.2, stratify=sms_df["label"], random_state=42)

# --- 2. synthetic dialogue pieces ---
syn_tr = make_pieces(train_long); syn_tr["source"] = "synthetic"
syn_te = make_pieces(test_long);  syn_te["source"] = "synthetic"

# --- 3. one combined dataset; every test now reads these three tables ---
train = pd.concat([syn_tr, sms_tr], ignore_index=True)
test  = pd.concat([syn_te, sms_te], ignore_index=True)
full  = pd.concat([train, test], ignore_index=True)
SCAM_TYPES   = sorted(full.loc[full["label"] == 1, "type"].unique())
NORMAL_TYPES = sorted(full.loc[full["label"] == 0, "type"].unique())

print("\nTraining rows by source and label (the 2 x 2 table):")
print(pd.crosstab(train["source"], train["label"]).rename(columns={0: "normal", 1: "scam/spam"}).to_string())
print("\nTest rows by source and label:")
print(pd.crosstab(test["source"], test["label"]).rename(columns={0: "normal", 1: "scam/spam"}).to_string())
print("\nAverage words:", train.groupby("source")["dialogue"].apply(lambda x: round(x.str.split().str.len().mean())).to_dict())
print("Scam types:", SCAM_TYPES, "\nNormal types:", NORMAL_TYPES)

Messages that appear with both labels: 0

Training rows by source and label (the 2 x 2 table):
label      normal  scam/spam
source                      
real         3614        514
synthetic    2908       4405

Test rows by source and label:
label      normal  scam/spam
source                      
real          904        128
synthetic     707       1110

Average words: {'real': 15, 'synthetic': 42}
Scam types: ['refund', 'reward', 'sms_spam', 'ssn', 'support'] 
Normal types: ['delivery', 'insurance', 'sms_ham', 'telemarketing', 'wrong']


`cell_report()` shows accuracy separately for each of the four groups: real or synthetic, scam or normal.

In [ ]:
def cell_report(make_model):
    """Accuracy in each of the four cells of the 2 x 2 table, on the test split."""
    m = make_model().fit(train["dialogue"], train["label"])
    t = test.copy()
    t["correct"] = (np.array(m.predict(t["dialogue"])) == t["label"].values)
    out = t.groupby(["source", "label"])["correct"].agg(["mean", "size"])
    out.index = out.index.set_levels(["normal", "scam/spam"], level="label")
    return out.rename(columns={"mean": "share correct", "size": "rows"}).round(3)

print("Naive Bayes:\n", cell_report(lambda: make_pipeline(tfidf(), MultinomialNB())), "\n")
print("Logistic Regression:\n", cell_report(lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42))))

Naive Bayes:
                      share correct  rows
source    label                         
real      normal             0.971   904
          scam/spam          0.781   128
synthetic normal             0.958   707
          scam/spam          0.987  1110 

Logistic Regression:
                      share correct  rows
source    label                         
real      normal             0.998   904
          scam/spam          0.727   128
synthetic normal             0.963   707
          scam/spam          0.982  1110


In [ ]:
evaluate("1. Naive Bayes, combined data", lambda: make_pipeline(tfidf(), MultinomialNB()), verbose=False)
evaluate("2. Logistic Regression, combined data", lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42)), verbose=False)
show_results()

model                                1. Naive Bayes  2. Logistic Regression  1. Naive Bayes, pieces  2. Logistic Regression, pieces  1. Naive Bayes, combined data  2. Logistic Regression, combined data
std_accuracy                                  1.000                   1.000                   0.977                           0.981                          0.966                                  0.971
std_macro_f1                                  1.000                   1.000                   0.976                           0.980                          0.965                                  0.970
std_scam_recall                               1.000                   1.000                   0.990                           0.993                          0.966                                  0.956
std_scam_precision                            1.000                   1.000                   0.973                           0.976                          0.955                              

,std_accuracy,std_macro_f1,std_scam_recall,std_scam_precision,hand_scams_caught_of_5,hand_normal_ok_of_3,hand_scamwords_wrongly_flagged_of_2,unseen_scam_recall_mean,unseen_scam_recall_min,unseen_normal_false_alarm_mean,unseen_normal_false_alarm_max
model,,,,,,,,,,,
1. Naive Bayes,1.000,1.000,1.000,1.000,5,3,2,0.999,0.995,0.402,0.850
2. Logistic Regression,1.000,1.000,1.000,1.000,0,3,1,0.911,0.795,0.159,0.450
"1. Naive Bayes, pieces",0.977,0.976,0.990,0.973,5,3,2,0.931,0.866,0.375,0.658
"2. Logistic Regression, pieces",0.981,0.980,0.993,0.976,5,1,2,0.913,0.828,0.402,0.686
"1. Naive Bayes, combined data",0.966,0.965,0.966,0.955,3,3,2,0.754,0.162,0.408,0.787
"2. Logistic Regression, combined data",0.971,0.970,0.956,0.977,3,3,2,0.696,0.000,0.449,0.939


## 1.3 Balanced benchmark and Models 1 to 4
balance(), evaluate() version 2, then Naive Bayes, Logistic Regression, Linear SVM and XGBoost.

### Balance each source
Inside each source, the larger class is down-sampled to match the smaller one. Otherwise a model could tell scam from normal just by recognising which dataset a message came from.

In [ ]:
def balance(df, seed=42):
    """Inside each source, keep equal numbers of scam and normal rows."""
    parts = []
    for src, g in df.groupby("source"):
        n = g["label"].value_counts().min()
        for lab, gg in g.groupby("label"):
            parts.append(gg.sample(n, random_state=seed))
    return pd.concat(parts, ignore_index=True)

train = balance(pd.concat([syn_tr, sms_tr], ignore_index=True))
test  = balance(pd.concat([syn_te, sms_te], ignore_index=True))
full  = pd.concat([train, test], ignore_index=True)

print("Training rows:")
print(pd.crosstab(train["source"], train["label"]).rename(columns={0: "normal", 1: "scam/spam"}).to_string())
print("\nTest rows:")
print(pd.crosstab(test["source"], test["label"]).rename(columns={0: "normal", 1: "scam/spam"}).to_string())

Training rows:
label      normal  scam/spam
source                      
real          514        514
synthetic    2908       2908

Test rows:
label      normal  scam/spam
source                      
real          128        128
synthetic     707        707


In [ ]:
mk_nb = lambda: make_pipeline(tfidf(), MultinomialNB())
mk_lr = lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42))

print("Naive Bayes, balanced:\n", cell_report(mk_nb), "\n")
print("Logistic Regression, balanced:\n", cell_report(mk_lr), "\n")

evaluate("1. Naive Bayes, balanced", mk_nb, verbose=False)
evaluate("2. Logistic Regression, balanced", mk_lr, verbose=False)
show_results().tail(2).T

Naive Bayes, balanced:
                      share correct  rows
source    label                         
real      normal             0.789   128
          scam/spam          0.961   128
synthetic normal             0.972   707
          scam/spam          0.986   707 

Logistic Regression, balanced:
                      share correct  rows
source    label                         
real      normal             0.906   128
          scam/spam          0.953   128
synthetic normal             0.969   707
          scam/spam          0.987   707 

model                                1. Naive Bayes  2. Logistic Regression  1. Naive Bayes, pieces  2. Logistic Regression, pieces  1. Naive Bayes, combined data  2. Logistic Regression, combined data  1. Naive Bayes, balanced  2. Logistic Regression, balanced
std_accuracy                                  1.000                   1.000                   0.977                           0.981                          0.966                        

model,"1. Naive Bayes, balanced","2. Logistic Regression, balanced"
std_accuracy,0.963,0.971
std_macro_f1,0.963,0.971
std_scam_recall,0.982,0.982
std_scam_precision,0.946,0.960
hand_scams_caught_of_5,4.000,5.000
hand_normal_ok_of_3,2.000,2.000
hand_scamwords_wrongly_flagged_of_2,2.000,2.000
unseen_scam_recall_mean,0.762,0.698
unseen_scam_recall_min,0.315,0.065
unseen_normal_false_alarm_mean,0.380,0.440


`evaluate()` version 2 adds accuracy per source and splits the hide-one-type test into synthetic chat topics and real SMS.

In [ ]:
def evaluate(name, make_model, verbose=True):
    """Version 2. Needs the combined data from Cells 7 and 11 (columns: dialogue, label, type, source)."""
    row = {"model": name}
    SYN_TYPES = set(full.loc[full["source"] == "synthetic", "type"])

    # 1. Standard split, plus accuracy in each of the four cells
    m = make_model().fit(train["dialogue"], train["label"])
    p = np.array(m.predict(test["dialogue"]))
    row["std_macro_f1"] = f1_score(test["label"], p, average="macro")
    row["std_scam_recall"] = recall_score(test["label"], p, pos_label=1)
    row["std_scam_precision"] = precision_score(test["label"], p, pos_label=1)
    cell = test.assign(ok=(p == test["label"].values)).groupby(["source", "label"])["ok"].mean()
    row["real_normal_correct"] = cell[("real", 0)]
    row["real_spam_caught"] = cell[("real", 1)]
    row["synthetic_normal_correct"] = cell[("synthetic", 0)]
    row["synthetic_scam_caught"] = cell[("synthetic", 1)]

    # 2. Hand-written messages
    pred = {g: list(m.predict(msgs)) for g, msgs in HAND.items()}
    row["hand_scams_caught_of_5"] = int(sum(pred["scam"]))
    row["hand_normal_ok_of_3"] = int(3 - sum(pred["normal"]))
    row["hand_scamwords_wrongly_flagged_of_2"] = int(sum(pred["normal_but_scam_words"]))
    if verbose:
        for g, msgs in HAND.items():
            print("\n" + g.upper())
            for msg, y in zip(msgs, pred[g]):
                print(f"  {'SCAM' if y == 1 else 'not scam':9} | {msg[:80]}")

    # 3. Hide one type at a time
    def hide(types, lab):
        out = {}
        for t in types:
            tr = full[full["type"] != t]
            te = full[(full["type"] == t) & (full["label"] == lab)]
            mm = make_model().fit(tr["dialogue"], tr["label"])
            out[t] = float((np.array(mm.predict(te["dialogue"])) == 1).mean())
        return out
    rec = hide(SCAM_TYPES, 1)      # share of the hidden scam type still caught
    fpr = hide(NORMAL_TYPES, 0)    # share of the hidden normal type wrongly flagged

    # 3a. Within the synthetic data: a new scam type, a new normal type
    rs = [v for k, v in rec.items() if k in SYN_TYPES]
    fs = [v for k, v in fpr.items() if k in SYN_TYPES]
    row["new_synthetic_scam_type_recall_mean"] = float(np.mean(rs))
    row["new_synthetic_scam_type_recall_min"] = float(min(rs))
    row["new_synthetic_normal_type_false_alarm_mean"] = float(np.mean(fs))
    row["new_synthetic_normal_type_false_alarm_max"] = float(max(fs))
    # 3b. Real data never seen in training (synthetic only -> real)
    row["no_real_training_spam_caught"] = rec["sms_spam"]
    row["no_real_training_normal_false_alarm"] = fpr["sms_ham"]

    if verbose:
        print("\nHide one scam type, share still caught:", {k: round(v, 2) for k, v in rec.items()})
        print("Hide one normal type, share wrongly flagged:", {k: round(v, 2) for k, v in fpr.items()})

    RESULTS[:] = [r for r in RESULTS if r["model"] != name] + [row]
    return row

def show_results(last=None):
    df = pd.DataFrame(RESULTS).set_index("model").round(3)
    if last: df = df.tail(last)
    print(df.T.to_string())

RESULTS.clear()   # start a clean table for the new metrics
evaluate("1. Naive Bayes", mk_nb, verbose=True)
evaluate("2. Logistic Regression", mk_lr, verbose=True)
show_results()


SCAM
  SCAM      | Hi, I sent you money by mistake. Please send it back to this number as soon as p
  SCAM      | Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 to restore
  SCAM      | Congratulations! You won a scholarship. Pay the 20,000 processing fee to claim i
  SCAM      | This is your bank. We noticed unusual activity. Confirm your card number and OTP
  not scam  | Your package is held at customs. Pay a 5,000 fee at this link to release it.

NORMAL
  not scam  | Hey, are we still meeting at 5 for the group project?
  SCAM      | Reminder: your appointment is tomorrow at 10am. Reply C to confirm.
  not scam  | Your package has been delivered. Thanks for shopping with us.

NORMAL_BUT_SCAM_WORDS
  SCAM      | I called Social Security today and they said my refund is on the way.
  SCAM      | My computer got a virus last week. The shop fixed it and gave me a refund.

Hide one scam type, share still caught: {'refund': 0.8, 'reward': 0.82, 'sms_spam': 0.31, 's

### Model 3: Linear SVM
TF-IDF features with a linear support vector machine.

In [ ]:
from sklearn.svm import LinearSVC

mk_svm = lambda: make_pipeline(tfidf(), LinearSVC(random_state=42))
evaluate("3. Linear SVM", mk_svm, verbose=True)
show_results()


SCAM
  SCAM      | Hi, I sent you money by mistake. Please send it back to this number as soon as p
  SCAM      | Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 to restore
  SCAM      | Congratulations! You won a scholarship. Pay the 20,000 processing fee to claim i
  SCAM      | This is your bank. We noticed unusual activity. Confirm your card number and OTP
  SCAM      | Your package is held at customs. Pay a 5,000 fee at this link to release it.

NORMAL
  not scam  | Hey, are we still meeting at 5 for the group project?
  SCAM      | Reminder: your appointment is tomorrow at 10am. Reply C to confirm.
  not scam  | Your package has been delivered. Thanks for shopping with us.

NORMAL_BUT_SCAM_WORDS
  SCAM      | I called Social Security today and they said my refund is on the way.
  SCAM      | My computer got a virus last week. The shop fixed it and gave me a refund.

Hide one scam type, share still caught: {'refund': 0.83, 'reward': 0.88, 'sms_spam': 0.07, '

### Model 4: XGBoost
Gradient-boosted trees on the same TF-IDF features (300 trees, depth 6).

In [ ]:
from xgboost import XGBClassifier
import time

mk_xgb = lambda: make_pipeline(tfidf(), XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                                      tree_method="hist", n_jobs=-1, random_state=42))

t0 = time.time()
evaluate("4. XGBoost", mk_xgb, verbose=True)
print(f"\nTime for full evaluate(): {time.time() - t0:.0f} s")
show_results()


SCAM
  SCAM      | Hi, I sent you money by mistake. Please send it back to this number as soon as p
  SCAM      | Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 to restore
  SCAM      | Congratulations! You won a scholarship. Pay the 20,000 processing fee to claim i
  SCAM      | This is your bank. We noticed unusual activity. Confirm your card number and OTP
  not scam  | Your package is held at customs. Pay a 5,000 fee at this link to release it.

NORMAL
  not scam  | Hey, are we still meeting at 5 for the group project?
  SCAM      | Reminder: your appointment is tomorrow at 10am. Reply C to confirm.
  not scam  | Your package has been delivered. Thanks for shopping with us.

NORMAL_BUT_SCAM_WORDS
  SCAM      | I called Social Security today and they said my refund is on the way.
  SCAM      | My computer got a virus last week. The shop fixed it and gave me a refund.

Hide one scam type, share still caught: {'refund': 0.76, 'reward': 0.75, 'sms_spam': 0.04, '

# 2. Fair comparison
Same 5% false-alarm budget for every model, repeated over 3 seeds (42, 7, 2026), and the bigger false-alarm test on leftover real normal SMS.

### Same false-alarm budget for every model
Default thresholds differ between models, so each model's cut-off is moved until it flags 5% of normal validation SMS. Then we compare what each model still catches.

In [ ]:
# Fair-comparison check: give every model the SAME false-alarm budget (5% on real normal), then compare what it still catches.
# The cut-off is chosen on a validation slice of the TRAINING data, never on the test set.
from sklearn.metrics import roc_auc_score
import time

key = train["source"] + "_" + train["label"].astype(str)
fit, val = train_test_split(train, test_size=0.2, stratify=key, random_state=42)
print("fit rows:", len(fit), "| val rows:", len(val), "| val real normal rows:", int(((val.source == "real") & (val.label == 0)).sum()))

def scores(m, texts):
    # a number per message: higher = more scam-like
    if hasattr(m, "predict_proba"):
        return m.predict_proba(texts)[:, 1]
    return m.decision_function(texts)

def cell_rates(df, flagged):
    d = df.assign(flag=flagged)
    rn = d[(d.source == "real") & (d.label == 0)].flag.mean()
    rs = d[(d.source == "real") & (d.label == 1)].flag.mean()
    sn = d[(d.source == "synthetic") & (d.label == 0)].flag.mean()
    ss = d[(d.source == "synthetic") & (d.label == 1)].flag.mean()
    return rn, rs, sn, ss

rows = []
for name, mk in [("NB", mk_nb), ("LR", mk_lr), ("SVM", mk_svm), ("XGB", mk_xgb)]:
    t0 = time.time()
    m = mk().fit(fit["dialogue"], fit["label"])
    sv, st = scores(m, val["dialogue"]), scores(m, test["dialogue"])
    default_cut = 0.5 if hasattr(m, "predict_proba") else 0.0
    # cut-off that flags 5% of the validation real-normal messages
    cut = np.quantile(sv[((val.source == "real") & (val.label == 0)).values], 0.95)
    print(f"{name}: default cut-off {default_cut:.2f} -> 5% cut-off {cut:.3f}  ({time.time() - t0:.0f} s)")
    for label, c in [("default", default_cut), ("5% budget", cut)]:
        flagged = (st > c).astype(int)
        rn, rs, sn, ss = cell_rates(test, flagged)
        rows.append({"model": name, "cut-off": label,
                     "AUC (no cut-off)": roc_auc_score(test["label"], st),
                     "macro-F1": f1_score(test["label"], flagged, average="macro"),
                     "real normal flagged": rn, "real spam caught": rs,
                     "synth normal flagged": sn, "synth scam caught": ss})

print()
print(pd.DataFrame(rows).set_index(["model", "cut-off"]).round(3).to_string())

fit rows: 5475 | val rows: 1369 | val real normal rows: 103
NB: default cut-off 0.50 -> 5% cut-off 0.658  (1 s)
LR: default cut-off 0.50 -> 5% cut-off 0.513  (1 s)
SVM: default cut-off 0.00 -> 5% cut-off 0.068  (1 s)
XGB: default cut-off 0.50 -> 5% cut-off 0.335  (53 s)

                 AUC (no cut-off)  macro-F1  real normal flagged  real spam caught  synth normal flagged  synth scam caught
model cut-off                                                                                                              
NB    default               0.996     0.959                0.234             0.945                 0.030              0.986
      5% budget             0.996     0.969                0.047             0.898                 0.018              0.973
LR    default               0.996     0.969                0.094             0.945                 0.033              0.987
      5% budget             0.996     0.971                0.086             0.938                 0.028    

### Repeat over three seeds (42, 7, 2026)
Checks that the ranking is not down to one lucky split. Takes about 30 to 40 minutes, mostly XGBoost.

In [ ]:
# Step 2: repeat the fair comparison over 3 seeds, all 4 models.
# Each seed changes the SMS split, the balancing sample, the validation slice and the model's own seed.
# About 30 to 40 minutes (XGBoost is most of it). Progress prints as it goes.
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import time

SEEDS = [42, 7, 2026]
SYN_SCAM   = ["refund", "reward", "ssn", "support"]
SYN_NORMAL = ["delivery", "insurance", "telemarketing", "wrong"]

def makers(seed):
    return {
        "NB":  lambda: make_pipeline(tfidf(), MultinomialNB()),
        "LR":  lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=seed)),
        "SVM": lambda: make_pipeline(tfidf(), LinearSVC(random_state=seed)),
        "XGB": lambda: make_pipeline(tfidf(), XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                                            tree_method="hist", n_jobs=-1, random_state=seed)),
    }

def build_data(seed):
    s_tr, s_te = train_test_split(sms_df, test_size=0.2, stratify=sms_df["label"], random_state=seed)
    tr = balance(pd.concat([syn_tr, s_tr], ignore_index=True), seed=seed)
    te = balance(pd.concat([syn_te, s_te], ignore_index=True), seed=seed)
    return tr, te

def scores(m, texts):
    if hasattr(m, "predict_proba"):
        return m.predict_proba(texts)[:, 1]
    return m.decision_function(texts)

def rates(df, flagged):
    d = df.assign(flag=flagged)
    g = lambda src, lab: d[(d.source == src) & (d.label == lab)].flag.mean()
    return {"real normal flagged": g("real", 0), "real spam caught": g("real", 1),
            "synth normal flagged": g("synthetic", 0), "synth scam caught": g("synthetic", 1)}

SEED_ROWS = []
t_all = time.time()
for seed in SEEDS:
    tr, te = build_data(seed)
    key = tr["source"] + "_" + tr["label"].astype(str)
    fit, val = train_test_split(tr, test_size=0.2, stratify=key, random_state=seed)
    full_s = pd.concat([tr, te], ignore_index=True)
    vmask = ((val.source == "real") & (val.label == 0)).values
    print(f"\n=== seed {seed}: train {len(tr)}, test {len(te)}, val real normal {vmask.sum()} ===")
    for name, mk in makers(seed).items():
        t0 = time.time()
        m = mk().fit(fit["dialogue"], fit["label"])
        sv, st = scores(m, val["dialogue"]), scores(m, te["dialogue"])
        default_cut = 0.5 if hasattr(m, "predict_proba") else 0.0
        cut = np.quantile(sv[vmask], 0.95)
        row = {"seed": seed, "model": name, "AUC": roc_auc_score(te["label"], st)}
        for tag, c in [("default", default_cut), ("5%", cut)]:
            flagged = (st > c).astype(int)
            row[f"F1 ({tag})"] = f1_score(te["label"], flagged, average="macro")
            for k, v in rates(te, flagged).items():
                row[f"{k} ({tag})"] = v

        # hide one synthetic type at a time (default cut-off)
        def hide(t, lab):
            keep = full_s[full_s.type != t]
            mm = mk().fit(keep["dialogue"], keep["label"])
            x = full_s[(full_s.type == t) & (full_s.label == lab)]
            return float((np.array(mm.predict(x["dialogue"])) == 1).mean())
        row["new scam type recall"] = np.mean([hide(t, 1) for t in SYN_SCAM])
        row["new normal type false alarm"] = np.mean([hide(t, 0) for t in SYN_NORMAL])
        SEED_ROWS.append(row)
        print(f"  {name}: AUC {row['AUC']:.3f} | real normal flagged: default {row['real normal flagged (default)']:.3f}, "
              f"5% budget {row['real normal flagged (5%)']:.3f} | new scam type {row['new scam type recall']:.3f} | {time.time() - t0:.0f} s")

print(f"\nTotal time: {(time.time() - t_all) / 60:.1f} min")
seed_df = pd.DataFrame(SEED_ROWS).drop(columns="seed")
means = seed_df.groupby("model").mean().reindex(["NB", "LR", "SVM", "XGB"])
stds  = seed_df.groupby("model").std().reindex(["NB", "LR", "SVM", "XGB"])
print("\nMean ± spread (std) over 3 seeds")
print((means.round(3).astype(str) + " ± " + stds.round(3).astype(str)).T.to_string())


=== seed 42: train 6844, test 1670, val real normal 103 ===
  NB: AUC 0.996 | real normal flagged: default 0.234, 5% budget 0.047 | new scam type 0.873 | 7 s
  LR: AUC 0.996 | real normal flagged: default 0.094, 5% budget 0.086 | new scam type 0.856 | 10 s
  SVM: AUC 0.998 | real normal flagged: default 0.070, 5% budget 0.047 | new scam type 0.897 | 7 s
  XGB: AUC 0.993 | real normal flagged: default 0.023, 5% budget 0.094 | new scam type 0.771 | 600 s

=== seed 7: train 6844, test 1670, val real normal 103 ===
  NB: AUC 0.995 | real normal flagged: default 0.281, 5% budget 0.047 | new scam type 0.872 | 7 s
  LR: AUC 0.996 | real normal flagged: default 0.133, 5% budget 0.023 | new scam type 0.852 | 10 s
  SVM: AUC 0.998 | real normal flagged: default 0.078, 5% budget 0.047 | new scam type 0.896 | 7 s
  XGB: AUC 0.994 | real normal flagged: default 0.031, 5% budget 0.023 | new scam type 0.781 | 592 s

=== seed 2026: train 6844, test 1670, val real normal 103 ===
  NB: AUC 0.995 | real

### Bigger false-alarm test
The ~3,100 normal SMS left out by balancing set the cut-off, and all ~900 normal test SMS measure false alarms.

In [ ]:
# Bigger false-alarm check: reuse the real normal SMS that balancing left out.
# Training data is unchanged (same balanced train set as before, all of it).
# Leftover TRAIN ham (never trained on) picks the 5% cut-off; ALL real normal TEST SMS measure false alarms.
# Needs makers, build_data, scores, rates, SEEDS from the 3-seed cell.
BIG_ROWS = []
t_all = time.time()
for seed in SEEDS:
    s_tr, s_te = train_test_split(sms_df, test_size=0.2, stratify=sms_df["label"], random_state=seed)
    tr, te = build_data(seed)
    cut_ham  = s_tr[(s_tr.label == 0) & ~s_tr.dialogue.isin(tr.dialogue)]
    test_ham = s_te[s_te.label == 0]
    assert not cut_ham.dialogue.isin(tr.dialogue).any() and not test_ham.dialogue.isin(tr.dialogue).any()
    print(f"\n=== seed {seed}: cut-off slice {len(cut_ham)} real normal | false-alarm test {len(test_ham)} real normal (was 128) ===")
    for name, mk in makers(seed).items():
        t0 = time.time()
        m = mk().fit(tr["dialogue"], tr["label"])
        default_cut = 0.5 if hasattr(m, "predict_proba") else 0.0
        cut = np.quantile(scores(m, cut_ham["dialogue"]), 0.95)
        s_big, st = scores(m, test_ham["dialogue"]), scores(m, te["dialogue"])
        row = {"seed": seed, "model": name}
        for tag, c in [("default", default_cut), ("5%", cut)]:
            flagged = (st > c).astype(int)
            r = rates(te, flagged)
            row[f"real normal flagged, big test ({tag})"] = float((s_big > c).mean())
            row[f"real normal flagged, old 128 ({tag})"] = r["real normal flagged"]
            row[f"real spam caught ({tag})"] = r["real spam caught"]
            row[f"synth normal flagged ({tag})"] = r["synth normal flagged"]
            row[f"synth scam caught ({tag})"] = r["synth scam caught"]
            row[f"F1 ({tag})"] = f1_score(te["label"], flagged, average="macro")
        BIG_ROWS.append(row)
        print(f"  {name}: cut-off {default_cut:.2f} -> {cut:.3f} | real normal flagged at 5%: big test {row['real normal flagged, big test (5%)']:.3f}, "
              f"old 128 {row['real normal flagged, old 128 (5%)']:.3f} | real spam caught {row['real spam caught (5%)']:.3f} | {time.time() - t0:.0f} s")

print(f"\nTotal time: {(time.time() - t_all) / 60:.1f} min")
big_df = pd.DataFrame(BIG_ROWS).drop(columns="seed")
bm = big_df.groupby("model").mean().reindex(["NB", "LR", "SVM", "XGB"])
bs = big_df.groupby("model").std().reindex(["NB", "LR", "SVM", "XGB"])
print("\nMean ± spread (std) over 3 seeds")
print((bm.round(3).astype(str) + " ± " + bs.round(3).astype(str)).T.to_string())


=== seed 42: cut-off slice 3100 real normal | false-alarm test 904 real normal (was 128) ===
  NB: cut-off 0.50 -> 0.686 | real normal flagged at 5%: big test 0.050, old 128 0.031 | real spam caught 0.891 | 1 s
  LR: cut-off 0.50 -> 0.568 | real normal flagged at 5%: big test 0.055, old 128 0.055 | real spam caught 0.914 | 2 s
  SVM: cut-off 0.00 -> 0.067 | real normal flagged at 5%: big test 0.055, old 128 0.047 | real spam caught 0.945 | 1 s
  XGB: cut-off 0.50 -> 0.450 | real normal flagged at 5%: big test 0.052, old 128 0.039 | real spam caught 0.906 | 60 s

=== seed 7: cut-off slice 3100 real normal | false-alarm test 904 real normal (was 128) ===
  NB: cut-off 0.50 -> 0.685 | real normal flagged at 5%: big test 0.056, old 128 0.055 | real spam caught 0.875 | 1 s
  LR: cut-off 0.50 -> 0.572 | real normal flagged at 5%: big test 0.059, old 128 0.031 | real spam caught 0.922 | 1 s
  SVM: cut-off 0.00 -> 0.048 | real normal flagged at 5%: big test 0.064, old 128 0.047 | real spam ca

# 3. GPU restart setup
**Run this cell first after any runtime restart.** It rebuilds the data, splits, tfidf(), build_data() and the shared helpers.

**Run this first after any restart.** It rebuilds the data, splits and shared helpers (`evaluate`, `seed_eval`, `build_data`).

In [ ]:
# GPU RESTART SETUP: run this after any runtime restart (replaces Cells A to D and the seed helpers)
!pip -q install datasets
import time, numpy as np, pandas as pd
from datasets import load_dataset
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (check Runtime > Change runtime type)")

ds  = load_dataset("BothBosu/scam-dialogue")
sms = load_dataset("ucirvine/sms_spam")["train"].to_pandas()

HAND = {
 "scam": [
  "Hi, I sent you money by mistake. Please send it back to this number as soon as possible.",
  "Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 to restore it.",
  "Congratulations! You won a scholarship. Pay the 20,000 processing fee to claim it today.",
  "This is your bank. We noticed unusual activity. Confirm your card number and OTP to avoid account closure.",
  "Your package is held at customs. Pay a 5,000 fee at this link to release it."],
 "normal": [
  "Hey, are we still meeting at 5 for the group project?",
  "Reminder: your appointment is tomorrow at 10am. Reply C to confirm.",
  "Your package has been delivered. Thanks for shopping with us."],
 "normal_but_scam_words": [
  "I called Social Security today and they said my refund is on the way.",
  "My computer got a virus last week. The shop fixed it and gave me a refund."],
}
def tfidf():
    return TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)
RESULTS = []

def make_pieces(df, max_words=45, min_words=10):
    rows = []
    for text, typ, lab in zip(df["dialogue"], df["type"], df["label"]):
        words = text.split()
        for i in range(0, len(words), max_words):
            chunk = words[i:i + max_words]
            if len(chunk) >= min_words:
                rows.append({"dialogue": " ".join(chunk), "type": typ, "label": lab})
    return pd.DataFrame(rows)

def balance(df, seed=42):
    parts = []
    for src, g in df.groupby("source"):
        n = g["label"].value_counts().min()
        for lab, gg in g.groupby("label"):
            parts.append(gg.sample(n, random_state=seed))
    return pd.concat(parts, ignore_index=True)

syn_tr = make_pieces(ds["train"].to_pandas()); syn_tr["source"] = "synthetic"
syn_te = make_pieces(ds["test"].to_pandas());  syn_te["source"] = "synthetic"
sms_df = sms.rename(columns={"sms": "dialogue"})[["dialogue", "label"]].copy()
sms_df["dialogue"] = sms_df["dialogue"].str.strip()
sms_df = sms_df.drop_duplicates(subset="dialogue")
sms_df["type"] = np.where(sms_df["label"] == 1, "sms_spam", "sms_ham")
sms_df["source"] = "real"
sms_tr, sms_te = train_test_split(sms_df, test_size=0.2, stratify=sms_df["label"], random_state=42)
train = balance(pd.concat([syn_tr, sms_tr], ignore_index=True))
test  = balance(pd.concat([syn_te, sms_te], ignore_index=True))
full  = pd.concat([train, test], ignore_index=True)
SCAM_TYPES   = sorted(full.loc[full["label"] == 1, "type"].unique())
NORMAL_TYPES = sorted(full.loc[full["label"] == 0, "type"].unique())
print(pd.crosstab(train["source"], train["label"]).rename(columns={0: "normal", 1: "scam/spam"}))

def evaluate(name, make_model, verbose=True):
    row = {"model": name}
    SYN_TYPES = set(full.loc[full["source"] == "synthetic", "type"])
    m = make_model().fit(train["dialogue"], train["label"])
    p = np.array(m.predict(test["dialogue"]))
    row["std_macro_f1"] = f1_score(test["label"], p, average="macro")
    row["std_scam_recall"] = recall_score(test["label"], p, pos_label=1)
    row["std_scam_precision"] = precision_score(test["label"], p, pos_label=1)
    cell = test.assign(ok=(p == test["label"].values)).groupby(["source", "label"])["ok"].mean()
    row["real_normal_correct"] = cell[("real", 0)]
    row["real_spam_caught"] = cell[("real", 1)]
    row["synthetic_normal_correct"] = cell[("synthetic", 0)]
    row["synthetic_scam_caught"] = cell[("synthetic", 1)]
    pred = {g: list(m.predict(msgs)) for g, msgs in HAND.items()}
    row["hand_scams_caught_of_5"] = int(sum(pred["scam"]))
    row["hand_normal_ok_of_3"] = int(3 - sum(pred["normal"]))
    row["hand_scamwords_wrongly_flagged_of_2"] = int(sum(pred["normal_but_scam_words"]))
    if verbose:
        for g, msgs in HAND.items():
            print("\n" + g.upper())
            for msg, y in zip(msgs, pred[g]):
                print(f"  {'SCAM' if y == 1 else 'not scam':9} | {msg[:80]}")
    def hide(types, lab):
        out = {}
        for t in types:
            tr = full[full["type"] != t]
            te = full[(full["type"] == t) & (full["label"] == lab)]
            mm = make_model().fit(tr["dialogue"], tr["label"])
            out[t] = float((np.array(mm.predict(te["dialogue"])) == 1).mean())
        return out
    rec = hide(SCAM_TYPES, 1); fpr = hide(NORMAL_TYPES, 0)
    rs = [v for k, v in rec.items() if k in SYN_TYPES]; fs = [v for k, v in fpr.items() if k in SYN_TYPES]
    row["new_synthetic_scam_type_recall_mean"] = float(np.mean(rs))
    row["new_synthetic_scam_type_recall_min"] = float(min(rs))
    row["new_synthetic_normal_type_false_alarm_mean"] = float(np.mean(fs))
    row["new_synthetic_normal_type_false_alarm_max"] = float(max(fs))
    row["no_real_training_spam_caught"] = rec["sms_spam"]
    row["no_real_training_normal_false_alarm"] = fpr["sms_ham"]
    if verbose:
        print("\nHide one scam type, share still caught:", {k: round(v, 2) for k, v in rec.items()})
        print("Hide one normal type, share wrongly flagged:", {k: round(v, 2) for k, v in fpr.items()})
    RESULTS[:] = [r for r in RESULTS if r["model"] != name] + [row]
    return row

def show_results(last=None):
    df = pd.DataFrame(RESULTS).set_index("model").round(3)
    if last: df = df.tail(last)
    print(df.T.to_string())

mk_nb  = lambda: make_pipeline(tfidf(), MultinomialNB())
mk_lr  = lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42))
mk_svm = lambda: make_pipeline(tfidf(), LinearSVC(random_state=42))

# ----- seed + big false-alarm harness (same as the classical runs) -----
SEEDS = [42, 7, 2026]
SYN_SCAM   = ["refund", "reward", "ssn", "support"]
SYN_NORMAL = ["delivery", "insurance", "telemarketing", "wrong"]

def build_data(seed):
    s_tr, s_te = train_test_split(sms_df, test_size=0.2, stratify=sms_df["label"], random_state=seed)
    tr = balance(pd.concat([syn_tr, s_tr], ignore_index=True), seed=seed)
    te = balance(pd.concat([syn_te, s_te], ignore_index=True), seed=seed)
    return tr, te, s_tr, s_te

def scores(m, texts):
    if hasattr(m, "predict_proba"):
        return m.predict_proba(texts)[:, 1]
    return m.decision_function(texts)

def rates(df, flagged):
    d = df.assign(flag=flagged)
    g = lambda src, lab: d[(d.source == src) & (d.label == lab)].flag.mean()
    return {"real normal flagged": g("real", 0), "real spam caught": g("real", 1),
            "synth normal flagged": g("synthetic", 0), "synth scam caught": g("synthetic", 1)}

def seed_eval(name, make_for_seed):
    """3 seeds. Train on the full balanced train set; 5% cut-off from ~3,100 leftover real normal TRAIN SMS;
    false alarms on all ~900 real normal TEST SMS; plus hide-one-synthetic-type tests (default cut-off)."""
    rows = []
    for seed in SEEDS:
        tr, te, s_tr, s_te = build_data(seed)
        cut_ham  = s_tr[(s_tr.label == 0) & ~s_tr.dialogue.isin(tr.dialogue)]
        test_ham = s_te[s_te.label == 0]
        full_s = pd.concat([tr, te], ignore_index=True)
        t0 = time.time()
        mk = lambda: make_for_seed(seed)
        m = mk().fit(tr["dialogue"], tr["label"])
        default_cut = 0.5 if hasattr(m, "predict_proba") else 0.0
        cut = np.quantile(scores(m, cut_ham["dialogue"]), 0.95)
        s_big, st = scores(m, test_ham["dialogue"]), scores(m, te["dialogue"])
        row = {"seed": seed, "model": name, "AUC": roc_auc_score(te["label"], st)}
        for tag, c in [("default", default_cut), ("5%", cut)]:
            flagged = (st > c).astype(int)
            r = rates(te, flagged)
            row[f"real normal flagged, big test ({tag})"] = float((s_big > c).mean())
            row[f"real spam caught ({tag})"] = r["real spam caught"]
            row[f"synth normal flagged ({tag})"] = r["synth normal flagged"]
            row[f"synth scam caught ({tag})"] = r["synth scam caught"]
            row[f"F1 ({tag})"] = f1_score(te["label"], flagged, average="macro")
        def hide(t, lab):
            keep = full_s[full_s.type != t]
            mm = mk().fit(keep["dialogue"], keep["label"])
            x = full_s[(full_s.type == t) & (full_s.label == lab)]
            return float((np.array(mm.predict(x["dialogue"])) == 1).mean())
        row["new scam type recall"] = np.mean([hide(t, 1) for t in SYN_SCAM])
        row["new normal type false alarm"] = np.mean([hide(t, 0) for t in SYN_NORMAL])
        rows.append(row)
        print(f"  seed {seed}: AUC {row['AUC']:.3f} | cut-off {default_cut:.2f} -> {cut:.3f} | real normal flagged at 5% {row['real normal flagged, big test (5%)']:.3f} "
              f"| real spam caught {row['real spam caught (5%)']:.3f} | new scam type {row['new scam type recall']:.3f} | {time.time() - t0:.0f} s")
    d = pd.DataFrame(rows).drop(columns=["seed", "model"])
    print(f"\n{name}: mean ± spread over 3 seeds")
    print((d.mean().round(3).astype(str) + " ± " + d.std().round(3).astype(str)).to_string())
    return rows
print("Setup done.")

GPU: Tesla T4


README.md:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

scam-dialogue_train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

scam-dialogue_test.csv:   0%|          | 0.00/453k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1280 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/320 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  359kB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

label      normal  scam/spam
source                      
real          514        514
synthetic    2908       2908
Setup done.


# 4. Neural, transformer and LLM models
## Model 5: CNN
Also the unknown-word test, the real-data share test and the every-model both-ways run.

### Model 5: CNN
A convolutional network that learns from 3-, 4- and 5-word windows, trained on the GPU with early stopping.

In [ ]:
# Model 5: CNN for text (PyTorch, runs on the GPU)
# Words -> numbers -> embedding (128 numbers per word) -> filters that look at 3, 4 and 5 words at a time
# -> keep each filter's strongest match anywhere in the message -> one scam score.
# Same fit/predict/predict_proba interface as the sklearn models, so evaluate() and seed_eval() work unchanged.
import re, torch, torch.nn as nn
from collections import Counter
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
TOKEN = re.compile(r"[a-z0-9£$]+|[!?]")
def tokenize(t): return TOKEN.findall(str(t).lower())

class TextCNN(nn.Module):
    def __init__(self, vocab, dim=128, filters=100, sizes=(3, 4, 5), drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(dim, filters, k) for k in sizes])
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(filters * len(sizes), 1)
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        h = torch.cat([torch.relu(c(e)).max(dim=2).values for c in self.convs], dim=1)
        return self.out(self.drop(h)).squeeze(1)

class CNNClassifier:
    def __init__(self, seed=42, max_len=64, max_vocab=20000, epochs=8, patience=2, batch=64, lr=1e-3, verbose=False):
        self.seed, self.max_len, self.max_vocab = seed, max_len, max_vocab
        self.epochs, self.patience, self.batch, self.lr, self.verbose = epochs, patience, batch, lr, verbose
    def _encode(self, texts):
        X = np.zeros((len(texts), self.max_len), dtype=np.int64)
        for i, t in enumerate(texts):
            ids = [self.vocab.get(w, 1) for w in tokenize(t)][:self.max_len]
            X[i, :len(ids)] = ids
        return torch.tensor(X)
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = np.asarray(labels, dtype=np.float32)
        # 10% of the training rows are held back to decide when to stop training (early stopping)
        tr_i, va_i = train_test_split(np.arange(len(texts)), test_size=0.1, stratify=y, random_state=self.seed)
        cnt = Counter(w for i in tr_i for w in tokenize(texts[i]))
        words = [w for w, c in cnt.most_common(self.max_vocab - 2) if c >= 2]
        self.vocab = {w: i + 2 for i, w in enumerate(words)}          # 0 = padding, 1 = unknown word
        X, Y = self._encode(texts), torch.tensor(y)
        self.model = TextCNN(len(self.vocab) + 2).to(DEVICE)
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.BCEWithLogitsLoss()
        g = torch.Generator().manual_seed(self.seed)
        Xva, Yva = X[va_i].to(DEVICE), Y[va_i].to(DEVICE)
        best, best_state, bad = 1e9, None, 0
        for ep in range(self.epochs):
            self.model.train()
            perm = torch.tensor(tr_i)[torch.randperm(len(tr_i), generator=g)]
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                opt.zero_grad()
                loss = loss_fn(self.model(X[idx].to(DEVICE)), Y[idx].to(DEVICE))
                loss.backward(); opt.step()
            self.model.eval()
            with torch.no_grad():
                vl = loss_fn(self.model(Xva), Yva).item()
            if self.verbose: print(f"    epoch {ep + 1}: validation loss {vl:.4f}")
            if vl < best - 1e-4:
                best, bad = vl, 0
                best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            else:
                bad += 1
                if bad >= self.patience: break
        self.model.load_state_dict(best_state)
        self.epochs_run = ep + 1
        return self
    def predict_proba(self, texts):
        self.model.eval(); X = self._encode(list(texts)); out = []
        with torch.no_grad():
            for b in range(0, len(X), 512):
                out.append(torch.sigmoid(self.model(X[b:b + 512].to(DEVICE))).cpu().numpy())
        p = np.concatenate(out)
        return np.column_stack([1 - p, p])
    def predict(self, texts):
        return (self.predict_proba(texts)[:, 1] > 0.5).astype(int)

# --- Step 1: one training run with everything printed ---
t0 = time.time()
cnn = CNNClassifier(seed=42, verbose=True).fit(train["dialogue"], train["label"])
print(f"vocabulary: {len(cnn.vocab)} words | epochs run: {cnn.epochs_run} | {time.time() - t0:.0f} s")
print("example tokens:", tokenize(HAND["scam"][1]))
for g, msgs in HAND.items():
    print("\n" + g.upper())
    for msg, p in zip(msgs, cnn.predict_proba(msgs)[:, 1]):
        print(f"  score {p:.2f} {'SCAM' if p > 0.5 else 'not scam':9}| {msg[:75]}")

# --- Step 2: same tests as models 1 to 4 ---
t0 = time.time()
evaluate("5. CNN", lambda: CNNClassifier(seed=42), verbose=False)
print(f"\nevaluate(): {time.time() - t0:.0f} s")
show_results(last=1)

# --- Step 3: 3 seeds + big false-alarm test ---
CNN_SEED_ROWS = seed_eval("5. CNN", lambda s: CNNClassifier(seed=s))

    epoch 1: validation loss 0.2101
    epoch 2: validation loss 0.1624
    epoch 3: validation loss 0.1399
    epoch 4: validation loss 0.1401
    epoch 5: validation loss 0.1353
    epoch 6: validation loss 0.1326
    epoch 7: validation loss 0.1398
    epoch 8: validation loss 0.1425
vocabulary: 3256 words | epochs run: 8 | 13 s
example tokens: ['your', 'm', 'pesa', 'account', 'has', 'been', 'suspended', 'send', 'your', 'pin', 'to', '0700', '000', '000', 'to', 'restore', 'it']

SCAM
  score 0.34 not scam | Hi, I sent you money by mistake. Please send it back to this number as soon
  score 0.99 SCAM     | Your M-Pesa account has been suspended. Send your PIN to 0700 000 000 to re
  score 1.00 SCAM     | Congratulations! You won a scholarship. Pay the 20,000 processing fee to cl
  score 0.94 SCAM     | This is your bank. We noticed unusual activity. Confirm your card number an
  score 0.64 SCAM     | Your package is held at customs. Pay a 5,000 fee at this link to release it

NORMAL
 

**Diagnostic:** does the CNN miss real SMS because their words are unknown to it?

In [ ]:
# Test the hypothesis: does the CNN fail on real SMS because their words are unknown to it?
# Uses cnn (seed 42, trained on train) from the CNN cell.
def unk_share(t):
    toks = tokenize(t)
    return float(np.mean([w not in cnn.vocab for w in toks])) if toks else 0.0

te = test.copy()
te["unk"] = te["dialogue"].map(unk_share)
te["p"] = cnn.predict_proba(te["dialogue"])[:, 1]
te["ok"] = ((te["p"] > 0.5).astype(int) == te["label"])

print("1. Share of words unknown to the CNN, by cell (0 = normal, 1 = scam/spam):")
print(te.groupby(["source", "label"])["unk"].mean().round(3).to_string())

real = te[te.source == "real"].copy()
real["unknown words"] = pd.cut(real["unk"], [-0.01, 0.10, 0.25, 1.0], labels=["0-10%", "10-25%", "over 25%"])
print("\n2. Real SMS: share the CNN gets right, by how many words are unknown:")
print(real.groupby(["label", "unknown words"], observed=False)["ok"].agg(["mean", "size"]).round(3).to_string())

v = tfidf().fit(train["dialogue"])
single = [f for f in v.get_feature_names_out() if " " not in f]
print(f"\n3. Vocabulary size: CNN {len(cnn.vocab)} words | TF-IDF (used by NB, LR, SVM) {len(single)} single words + {len(v.vocabulary_) - len(single)} two-word phrases")

print("\n4. The missed scam, word by word (True = known to the CNN):")
print([(w, w in cnn.vocab) for w in tokenize(HAND["scam"][0])])
print("\n5. Which training types use these words most (number of pieces):")
for w in ["mistake", "back", "number", "sent"]:
    hits = train[train["dialogue"].str.lower().str.contains(r"\b" + w + r"\b", regex=True)]
    print(f"  {w:8}", hits.groupby("type").size().sort_values(ascending=False).head(4).to_dict())

1. Share of words unknown to the CNN, by cell (0 = normal, 1 = scam/spam):
source     label
real       0        0.143
           1        0.136
synthetic  0        0.007
           1        0.006

2. Real SMS: share the CNN gets right, by how many words are unknown:
                      mean  size
label unknown words             
0     0-10%          0.879    58
      10-25%         0.918    49
      over 25%       0.857    21
1     0-10%          0.918    49
      10-25%         0.881    67
      over 25%       0.750    12

3. Vocabulary size: CNN 3256 words | TF-IDF (used by NB, LR, SVM) 3274 single words + 15712 two-word phrases

4. The missed scam, word by word (True = known to the CNN):
[('hi', True), ('i', True), ('sent', True), ('you', True), ('money', True), ('by', True), ('mistake', True), ('please', True), ('send', True), ('it', True), ('back', True), ('to', True), ('this', True), ('number', True), ('as', True), ('soon', True), ('as', True), ('possible', True)]

5. Which tra

**Diagnostic:** does the CNN do worse on real SMS because they are only about 15% of its training data? The SVM runs alongside as a control.

In [ ]:
# Test: does the CNN do worse on real SMS because real rows are only ~15% of its training data?
# Only the TRAINING data changes. The test set stays exactly the same. SVM runs alongside as a control.
syn_part  = train[train.source == "synthetic"]
real_part = train[train.source == "real"]
real_test = test[test.source == "real"]
syn_test  = test[test.source == "synthetic"]
print(f"training rows: synthetic {len(syn_part)} | real {len(real_part)} ({len(real_part) / len(train):.0%} of training)")
print(f"test rows: real {len(real_test)} | synthetic {len(syn_test)}  (unchanged in every run)\n")

PLANS = [("25% of real", 0.25, 1), ("50% of real", 0.50, 1), ("all real (current)", 1.0, 1),
         ("all real, repeated 3x", 1.0, 3), ("all real, repeated 5x", 1.0, 5)]
rows = []
t0 = time.time()
for seed in SEEDS:
    for plan, frac, rep in PLANS:
        rp = real_part.groupby("label", group_keys=False).sample(frac=frac, random_state=seed)   # keeps scam/normal 50/50
        tr = pd.concat([syn_part] + [rp] * rep, ignore_index=True)
        share = (tr.source == "real").mean()
        for name, mk in [("SVM", lambda: make_pipeline(tfidf(), LinearSVC(random_state=seed))),
                         ("CNN", lambda: CNNClassifier(seed=seed))]:
            m = mk().fit(tr["dialogue"], tr["label"])
            pr, ps = np.array(m.predict(real_test["dialogue"])), np.array(m.predict(syn_test["dialogue"]))
            rows.append({"seed": seed, "plan": plan, "model": name, "real share of training": share,
                         "real AUC": roc_auc_score(real_test["label"], scores(m, real_test["dialogue"])),
                         "real normal correct": (pr[real_test.label.values == 0] == 0).mean(),
                         "real spam caught": (pr[real_test.label.values == 1] == 1).mean(),
                         "synth normal correct": (ps[syn_test.label.values == 0] == 0).mean(),
                         "synth scam caught": (ps[syn_test.label.values == 1] == 1).mean()})
        print(f"  seed {seed} | {plan:22} | real share {share:.0%} | CNN real AUC {rows[-1]['real AUC']:.3f} | SVM real AUC {rows[-2]['real AUC']:.3f} | {time.time() - t0:.0f} s")

res = pd.DataFrame(rows)
order = [p[0] for p in PLANS]
for metric in ["real AUC", "real normal correct", "real spam caught", "synth normal correct", "synth scam caught"]:
    mean = res.pivot_table(index="plan", columns="model", values=metric, aggfunc="mean").reindex(order)
    std  = res.pivot_table(index="plan", columns="model", values=metric, aggfunc="std").reindex(order)
    print(f"\n{metric} (mean ± spread over 3 seeds)")
    print((mean.round(3).astype(str) + " ± " + std.round(3).astype(str)).to_string())

training rows: synthetic 5816 | real 1028 (15% of training)
test rows: real 256 | synthetic 1414  (unchanged in every run)

  seed 42 | 25% of real            | real share 4% | CNN real AUC 0.896 | SVM real AUC 0.952 | 4 s
  seed 42 | 50% of real            | real share 8% | CNN real AUC 0.950 | SVM real AUC 0.974 | 9 s
  seed 42 | all real (current)     | real share 15% | CNN real AUC 0.951 | SVM real AUC 0.985 | 13 s
  seed 42 | all real, repeated 3x  | real share 35% | CNN real AUC 0.976 | SVM real AUC 0.987 | 19 s
  seed 42 | all real, repeated 5x  | real share 47% | CNN real AUC 0.981 | SVM real AUC 0.986 | 27 s
  seed 7 | 25% of real            | real share 4% | CNN real AUC 0.910 | SVM real AUC 0.955 | 31 s
  seed 7 | 50% of real            | real share 8% | CNN real AUC 0.938 | SVM real AUC 0.979 | 35 s
  seed 7 | all real (current)     | real share 15% | CNN real AUC 0.971 | SVM real AUC 0.985 | 40 s
  seed 7 | all real, repeated 3x  | real share 35% | CNN real AUC 0.980 | SVM

**Every model, both ways:** (a) the current data and (b) real SMS repeated three times in training, each with 3 seeds and the big false-alarm test.

In [ ]:
# Every model, both ways: (a) current data, (b) real SMS rows repeated 3x in training.
# Same harness for all: 3 seeds, big false-alarm test, AUC, hide-one-synthetic-type tests.
# The repeat is applied inside fit(), so it also applies to every hide-one-type training run. Test sets never change.
from xgboost import XGBClassifier
REAL_TEXTS = set(sms_df["dialogue"])

class RepeatReal:
    """Wraps any model: repeats real-SMS rows k times before training. Prediction is unchanged."""
    def __init__(self, base, k):
        self.base, self.k = base, k
    def fit(self, X, y):
        X = pd.Series(list(X)); y = pd.Series(np.asarray(y))
        real = X.isin(REAL_TEXTS).values
        self.base.fit(pd.concat([X] + [X[real]] * (self.k - 1), ignore_index=True),
                      pd.concat([y] + [y[real]] * (self.k - 1), ignore_index=True))
        return self
    def predict(self, X):
        return self.base.predict(X)
    def __getattr__(self, name):          # passes predict_proba / decision_function through, if the model has them
        return getattr(self.base, name)

MODELS = {
    "NB":  lambda s: make_pipeline(tfidf(), MultinomialNB()),
    "LR":  lambda s: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=s)),
    "SVM": lambda s: make_pipeline(tfidf(), LinearSVC(random_state=s)),
    "XGB": lambda s: make_pipeline(tfidf(), XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                                          tree_method="hist", device="cuda", random_state=s)),
    "CNN": lambda s: CNNClassifier(seed=s),
}
# quick check that the wrapper really repeats real rows
chk = train.sample(200, random_state=0)
print("check: real rows in sample", int(chk.dialogue.isin(REAL_TEXTS).sum()), "-> rows after 3x repeat",
      len(chk) + 2 * int(chk.dialogue.isin(REAL_TEXTS).sum()))

BOTH_ROWS = []
t_all = time.time()
for name, mk in MODELS.items():
    for k in (1, 3):
        way = "current" if k == 1 else "real x3"
        print(f"\n### {name}, {way}")
        rows = seed_eval(f"{name} ({way})", (lambda s, mk=mk, k=k: mk(s) if k == 1 else RepeatReal(mk(s), k)))
        for r in rows:
            r["way"] = way; r["model"] = name
        BOTH_ROWS += rows
        print(f"(elapsed {(time.time() - t_all) / 60:.1f} min)")

both = pd.DataFrame(BOTH_ROWS).drop(columns="seed")
m = both.groupby(["model", "way"]).mean().T
s = both.groupby(["model", "way"]).std().T
cols = [(n, w) for n in MODELS for w in ("current", "real x3")]
print("\n\n===== SUMMARY: mean ± spread over 3 seeds =====")
print((m[cols].round(3).astype(str) + " ± " + s[cols].round(3).astype(str)).to_string())

check: real rows in sample 30 -> rows after 3x repeat 260

### NB, current
  seed 42: AUC 0.996 | cut-off 0.50 -> 0.686 | real normal flagged at 5% 0.050 | real spam caught 0.891 | new scam type 0.873 | 13 s
  seed 7: AUC 0.995 | cut-off 0.50 -> 0.685 | real normal flagged at 5% 0.056 | real spam caught 0.875 | new scam type 0.872 | 6 s
  seed 2026: AUC 0.995 | cut-off 0.50 -> 0.680 | real normal flagged at 5% 0.069 | real spam caught 0.906 | new scam type 0.874 | 6 s

NB (current): mean ± spread over 3 seeds
AUC                                        0.996 ± 0.001
real normal flagged, big test (default)    0.196 ± 0.012
real spam caught (default)                 0.951 ± 0.009
synth normal flagged (default)             0.027 ± 0.001
synth scam caught (default)                0.986 ± 0.001
F1 (default)                               0.961 ± 0.002
real normal flagged, big test (5%)          0.058 ± 0.01
real spam caught (5%)                      0.891 ± 0.016
synth normal flagged (5%)    

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [21:08:37] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  seed 42: AUC 0.995 | cut-off 0.50 -> 0.455 | real normal flagged at 5% 0.054 | real spam caught 0.906 | new scam type 0.773 | 34 s
  seed 7: AUC 0.994 | cut-off 0.50 -> 0.385 | real normal flagged at 5% 0.064 | real spam caught 0.922 | new scam type 0.782 | 35 s
  seed 2026: AUC 0.993 | cut-off 0.50 -> 0.416 | real normal flagged at 5% 0.049 | real spam caught 0.883 | new scam type 0.775 | 34 s

XGB (current): mean ± spread over 3 seeds
AUC                                        0.994 ± 0.001
real normal flagged, big test (default)     0.042 ± 0.01
real spam caught (default)                 0.885 ± 0.023
synth normal flagged (default)             0.035 ± 0.005
synth scam caught (default)                0.973 ± 0.003
F1 (default)                               0.962 ± 0.002
real normal flagged, big test (5%)         0.056 ± 0.008
real spam caught (5%)                       0.904 ± 0.02
synth normal flagged (5%)                  0.046 ± 0.007
synth scam caught (5%)                     0

## Model 6: BiLSTM

In [ ]:
# Model 6: BiLSTM (PyTorch, GPU)
# Reads the message word by word, left to right AND right to left, carrying a memory along the way.
# The final memory of each direction is joined and turned into one scam score.
# Same tokens, vocabulary rule, early stopping and interface as the CNN, so the only change is the network.
class BiLSTMNet(nn.Module):
    def __init__(self, vocab, dim=128, hidden=128, drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.lstm = nn.LSTM(dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(2 * hidden, 1)
    def forward(self, x):
        lengths = (x != 0).sum(1).clamp(min=1).cpu()        # real length of each message, so padding is skipped
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(x), lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.out(self.drop(torch.cat([h[0], h[1]], dim=1))).squeeze(1)

class BiLSTMClassifier(CNNClassifier):
    """Same training loop as CNNClassifier; only the network is swapped."""
    def fit(self, texts, labels):
        global TextCNN
        saved, TextCNN = TextCNN, BiLSTMNet      # CNNClassifier.fit builds TextCNN(vocab); point it at the BiLSTM for this call
        try:
            return super().fit(texts, labels)
        finally:
            TextCNN = saved

# --- Step 1: one training run with everything printed ---
t0 = time.time()
lstm = BiLSTMClassifier(seed=42, verbose=True).fit(train["dialogue"], train["label"])
print(f"network: {type(lstm.model).__name__} | vocabulary: {len(lstm.vocab)} | epochs run: {lstm.epochs_run} | {time.time() - t0:.0f} s")
for g, msgs in HAND.items():
    print("\n" + g.upper())
    for msg, p, pc in zip(msgs, lstm.predict_proba(msgs)[:, 1], cnn.predict_proba(msgs)[:, 1]):
        print(f"  BiLSTM {p:.2f} {'SCAM' if p > 0.5 else 'not scam':9}| CNN {pc:.2f} | {msg[:65]}")

# --- Step 2: same tests as models 1 to 5 (for the Results sheet) ---
t0 = time.time()
evaluate("6. BiLSTM", lambda: BiLSTMClassifier(seed=42), verbose=False)
print(f"\nevaluate(): {time.time() - t0:.0f} s")
show_results(last=1)

# --- Step 3: both ways (current data, real SMS x3), 3 seeds, big false-alarm test ---
BOTH_ROWS = [r for r in BOTH_ROWS if r["model"] != "BiLSTM"]
for k in (1, 3):
    way = "current" if k == 1 else "real x3"
    print(f"\n### BiLSTM, {way}")
    rows = seed_eval(f"BiLSTM ({way})", (lambda s, k=k: BiLSTMClassifier(seed=s) if k == 1 else RepeatReal(BiLSTMClassifier(seed=s), k)))
    for r in rows:
        r["way"] = way; r["model"] = "BiLSTM"
    BOTH_ROWS += rows

both = pd.DataFrame(BOTH_ROWS).drop(columns="seed")
m, s = both.groupby(["model", "way"]).mean().T, both.groupby(["model", "way"]).std().T
cols = [(n, w) for n in ["SVM", "CNN", "BiLSTM"] for w in ("current", "real x3")]
print("\n\n===== SUMMARY (SVM and CNN for comparison): mean ± spread over 3 seeds =====")
print((m[cols].round(3).astype(str) + " ± " + s[cols].round(3).astype(str)).to_string())

    epoch 1: validation loss 0.2621
    epoch 2: validation loss 0.1914
    epoch 3: validation loss 0.1555
    epoch 4: validation loss 0.1389
    epoch 5: validation loss 0.1412
    epoch 6: validation loss 0.1367
    epoch 7: validation loss 0.1367
    epoch 8: validation loss 0.1453
network: BiLSTMNet | vocabulary: 3256 | epochs run: 8 | 14 s

SCAM
  BiLSTM 0.10 not scam | CNN 0.34 | Hi, I sent you money by mistake. Please send it back to this numb
  BiLSTM 1.00 SCAM     | CNN 0.99 | Your M-Pesa account has been suspended. Send your PIN to 0700 000
  BiLSTM 1.00 SCAM     | CNN 1.00 | Congratulations! You won a scholarship. Pay the 20,000 processing
  BiLSTM 1.00 SCAM     | CNN 0.94 | This is your bank. We noticed unusual activity. Confirm your card
  BiLSTM 0.87 SCAM     | CNN 0.64 | Your package is held at customs. Pay a 5,000 fee at this link to 

NORMAL
  BiLSTM 0.02 not scam | CNN 0.06 | Hey, are we still meeting at 5 for the group project?
  BiLSTM 1.00 SCAM     | CNN 0.93 | R

## Model 7: DistilBERT (fine-tuned)

In [ ]:
# Model 7: DistilBERT, fine-tuned (Hugging Face, GPU)
# Starts from a model already trained on a large amount of English (Wikipedia + books), so it arrives knowing
# what words mean. We add a 2-way output (scam / not scam) and fine-tune the whole network on our training data.
# Settings: max 64 tokens, 2 epochs, batch 32, learning rate 5e-5 with warm-up, mixed precision (fp16) for speed.
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
BERT_NAME = "distilbert-base-uncased"
bert_tok = AutoTokenizer.from_pretrained(BERT_NAME)

class DistilBERTClassifier:
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5, verbose=False):
        self.seed, self.max_len, self.epochs, self.batch, self.lr, self.verbose = seed, max_len, epochs, batch, lr, verbose
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long)
        enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        scaler = torch.cuda.amp.GradScaler()
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); total = 0.0
            perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE),
                                     attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad()
                scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
                total += out.loss.item() * len(idx)
            if self.verbose: print(f"    epoch {ep + 1}: training loss {total / len(texts):.4f}")
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                logits = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE),
                                    attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        p = np.concatenate(out)
        return np.column_stack([1 - p, p])
    def predict(self, texts):
        return (self.predict_proba(texts)[:, 1] > 0.5).astype(int)

print("how DistilBERT splits a message:", bert_tok.tokenize(HAND["scam"][1]))

# --- Step 1: one training run with everything printed ---
t0 = time.time()
bert = DistilBERTClassifier(seed=42, verbose=True).fit(train["dialogue"], train["label"])
print(f"one fine-tuning run: {time.time() - t0:.0f} s")
for g_, msgs in HAND.items():
    print("\n" + g_.upper())
    for msg, p, pl, pc in zip(msgs, bert.predict_proba(msgs)[:, 1], lstm.predict_proba(msgs)[:, 1], cnn.predict_proba(msgs)[:, 1]):
        print(f"  DistilBERT {p:.2f} {'SCAM' if p > 0.5 else 'not scam':9}| BiLSTM {pl:.2f} | CNN {pc:.2f} | {msg[:55]}")

# --- Step 2: same tests as models 1 to 6 (for the Results sheet) ---
t0 = time.time()
evaluate("7. DistilBERT", lambda: DistilBERTClassifier(seed=42), verbose=False)
print(f"\nevaluate(): {time.time() - t0:.0f} s")
show_results(last=1)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

how DistilBERT splits a message: ['your', 'm', '-', 'pe', '##sa', 'account', 'has', 'been', 'suspended', '.', 'send', 'your', 'pin', 'to', '07', '##00', '000', '000', 'to', 'restore', 'it', '.']


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

... (13 outputs left out)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


evaluate(): 395 s
model                                       7. DistilBERT
std_macro_f1                                        0.981
std_scam_recall                                     0.981
std_scam_precision                                  0.982
real_normal_correct                                 0.992
real_spam_caught                                    0.969
synthetic_normal_correct                            0.980
synthetic_scam_caught                               0.983
hand_scams_caught_of_5                              4.000
hand_normal_ok_of_3                                 2.000
hand_scamwords_wrongly_flagged_of_2                 2.000
new_synthetic_scam_type_recall_mean                 0.813
new_synthetic_scam_type_recall_min                  0.584
new_synthetic_normal_type_false_alarm_mean          0.412
new_synthetic_normal_type_false_alarm_max           0.725
no_real_training_spam_caught                        0.103
no_real_training_normal_false_alarm                 0

In [ ]:
hf_logging.disable_progress_bar()   # keeps the output plain text
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
# DistilBERT both ways (current data, real SMS x3), 3 seeds, big false-alarm test
BOTH_ROWS = [r for r in BOTH_ROWS if r["model"] != "DistilBERT"]
t_all = time.time()
for k in (1, 3):
    way = "current" if k == 1 else "real x3"
    print(f"\n### DistilBERT, {way}")
    rows = seed_eval(f"DistilBERT ({way})", (lambda s, k=k: DistilBERTClassifier(seed=s) if k == 1 else RepeatReal(DistilBERTClassifier(seed=s), k)))
    for r in rows:
        r["way"] = way; r["model"] = "DistilBERT"
    BOTH_ROWS += rows
    print(f"(elapsed {(time.time() - t_all) / 60:.1f} min)")

both = pd.DataFrame(BOTH_ROWS).drop(columns="seed")
m, s = both.groupby(["model", "way"]).mean().T, both.groupby(["model", "way"]).std().T
cols = [(n, w) for n in ["SVM", "LR", "CNN", "BiLSTM", "DistilBERT"] for w in ("current", "real x3")]
print("\n\n===== SUMMARY: mean ± spread over 3 seeds =====")
print((m[cols].round(3).astype(str) + " ± " + s[cols].round(3).astype(str)).to_string())


### DistilBERT, current
  seed 42: AUC 0.998 | cut-off 0.50 -> 0.439 | real normal flagged at 5% 0.045 | real spam caught 0.969 | new scam type 0.813 | 320 s
  seed 7: AUC 0.998 | cut-off 0.50 -> 0.297 | real normal flagged at 5% 0.054 | real spam caught 0.969 | new scam type 0.840 | 319 s
  seed 2026: AUC 0.997 | cut-off 0.50 -> 0.312 | real normal flagged at 5% 0.056 | real spam caught 0.969 | new scam type 0.869 | 319 s

DistilBERT (current): mean ± spread over 3 seeds
AUC                                          0.997 ± 0.0
real normal flagged, big test (default)    0.042 ± 0.003
real spam caught (default)                 0.964 ± 0.005
synth normal flagged (default)             0.022 ± 0.002
synth scam caught (default)                0.984 ± 0.001
F1 (default)                               0.979 ± 0.002
real normal flagged, big test (5%)         0.052 ± 0.006
real spam caught (5%)                        0.969 ± 0.0
synth normal flagged (5%)                  0.025 ± 0.005
synth sca

## Model 8: Qwen2.5-1.5B zero-shot
Also the check on why it misses about half of the dialogue scams.

In [ ]:
# Model 8: small open LLM, zero-shot (Qwen2.5-1.5B-Instruct, GPU)
# No training at all. We ask the model a Yes/No question and read how strongly it leans to "Yes" vs "No"
# for its next word. Score = P(Yes) / (P(Yes) + P(No)), a number from 0 to 1, so AUC and the 5% budget work as before.
# fit() does nothing, so every hide-one-type test is just its score on that type (everything is "unseen" for it).
from huggingface_hub.utils import disable_progress_bars; disable_progress_bars()
from transformers import AutoModelForCausalLM
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE).eval()
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]

SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

LLM_CACHE = {}
def llm_scores(texts, batch=32):
    texts = [str(t) for t in texts]
    todo = sorted(set(t for t in texts if t not in LLM_CACHE), key=len)    # similar lengths together = faster
    t0 = time.time()
    for b in range(0, len(todo), batch):
        chunk = todo[b:b + batch]
        enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            last = llm(**enc).logits[:, -1, :].float()
        p = torch.softmax(torch.stack([last[:, YES], last[:, NO]], dim=1), dim=1)[:, 0].cpu().numpy()
        LLM_CACHE.update(zip(chunk, p))
        if (b // batch) % 60 == 0 and len(todo) > 500:
            print(f"    scored {b + len(chunk)} / {len(todo)} new messages ({time.time() - t0:.0f} s)")
    return np.array([LLM_CACHE[t] for t in texts])

class ZeroShotLLM:
    def fit(self, X, y): return self                      # no training
    def predict_proba(self, X):
        p = llm_scores(X); return np.column_stack([1 - p, p])
    def predict(self, X):
        return (llm_scores(X) > 0.5).astype(int)

print("Yes/No token ids:", YES, NO)
print("\nWhat the model actually sees (one example):\n" + build(HAND["scam"][0]))

# --- Step 1: hand-written messages, side by side with DistilBERT ---
for g_, msgs in HAND.items():
    print("\n" + g_.upper())
    for msg, p, pb in zip(msgs, llm_scores(msgs), bert.predict_proba(msgs)[:, 1]):
        print(f"  LLM {p:.2f} {'SCAM' if p > 0.5 else 'not scam':9}| DistilBERT {pb:.2f} | {msg[:60]}")

# --- Step 2: same tests as models 1 to 7 ---
t0 = time.time()
evaluate("8. LLM zero-shot (Qwen2.5-1.5B)", lambda: ZeroShotLLM(), verbose=False)
print(f"\nevaluate(): {time.time() - t0:.0f} s")
show_results(last=1)
# per-topic view: average score and share flagged for each type (everything is unseen for this model)
sc = full.assign(score=llm_scores(full["dialogue"]))
print("\nPer topic: mean score and share flagged at 0.5")
print(sc.groupby(["label", "type"])["score"].agg(mean="mean", flagged=lambda s: (s > 0.5).mean(), n="size").round(3).to_string())

# --- Step 3: 3 seeds + big false-alarm test (only the test sets change between seeds) ---
BOTH_ROWS = [r for r in BOTH_ROWS if r["model"] != "LLM zero-shot"]
rows = seed_eval("LLM zero-shot", lambda s: ZeroShotLLM())
for r in rows:
    r["way"] = "no training"; r["model"] = "LLM zero-shot"
BOTH_ROWS += rows
both = pd.DataFrame(BOTH_ROWS).drop(columns="seed")
m, s = both.groupby(["model", "way"]).mean().T, both.groupby(["model", "way"]).std().T
cols = [("SVM", "current"), ("DistilBERT", "current"), ("LLM zero-shot", "no training")]
print("\n\n===== SUMMARY: mean ± spread over 3 seeds =====")
print((m[cols].round(3).astype(str) + " ± " + s[cols].round(3).astype(str)).to_string())

Yes/No token ids: 9454 2753

What the model actually sees (one example):
<|im_start|>system
You are a careful assistant that checks text messages and call transcripts for fraud.<|im_end|>
<|im_start|>user
Is the following message a scam or spam, meaning it tries to trick the reader into giving money or personal information, or pushes an unwanted offer? Answer with only Yes or No.

Message: "Hi, I sent you money by mistake. Please send it back to this number as soon as possible."<|im_end|>
<|im_start|>assistant


SCAM
  LLM 0.51 SCAM     | DistilBERT 0.02 | Hi, I sent you money by mistake. Please send it back to this
  LLM 0.83 SCAM     | DistilBERT 1.00 | Your M-Pesa account has been suspended. Send your PIN to 070
  LLM 0.86 SCAM     | DistilBERT 0.99 | Congratulations! You won a scholarship. Pay the 20,000 proce
  LLM 0.87 SCAM     | DistilBERT 1.00 | This is your bank. We noticed unusual activity. Confirm your
  LLM 0.92 SCAM     | DistilBERT 1.00 | Your package is held at customs. 

**Diagnostic:** why the untrained language model misses about half of the scam chat pieces.

In [ ]:
# Why does the zero-shot LLM catch only about half of the synthetic scam pieces?
# Look at pieces from SCAM dialogues that it scored as clearly not scam (below 0.2). Uses cached scores, so it is instant.
miss = sc[(sc.source == "synthetic") & (sc.label == 1) & (sc.score < 0.2)]
print(f"synthetic scam pieces scored below 0.2: {len(miss)} of {int(((sc.source == 'synthetic') & (sc.label == 1)).sum())}\n")
for _, r in miss.sample(8, random_state=1).iterrows():
    print(f"[{r['type']}, score {r['score']:.2f}] {r['dialogue']}\n")

synthetic scam pieces scored below 0.2: 775 of 3615

[support, score 0.07] caller: Well, our system monitors all Windows computers and we detected some suspicious activity coming from your IP address. I can help you fix the issue if you allow me to remote access your computer. receiver: That's interesting. I didn't know Windows had a system

[refund, score 0.12] I apologize, but our system requires us to mail out a physical check for security reasons. receiver: That's weird. I've never heard of that before. What's the name of your supervisor? caller: Let me check on that. Ah yes, his name is John Smith. receiver:

[refund, score 0.06] this lawsuit? Which corporation is it against? caller: Uh, let me check... Ah yes, it's against Microsoft. receiver: That's interesting. I've never heard of a class-action lawsuit against Microsoft that resulted in a refund. Do you have a case number or any documentation I can

[refund, score 0.05] Can you please confirm your account number so I can look 

## Model 9: Qwen2.5-1.5B with LoRA
Training, the resumed hide-one-type run and the LoRA definitions.

In [ ]:
# Model 9: the same LLM (Qwen2.5-1.5B-Instruct), fine-tuned with LoRA
# Self-contained: needs only the GPU RESTART SETUP cell above (run that first after any restart).
# Same prompt and Yes/No score as Model 8. LoRA adds small trainable matrices (rank 8) to the attention layers;
# the 1.5B original weights stay frozen. Training target: "Yes" for scam/spam, "No" for normal. 1 epoch, batch 16, lr 2e-4.
!pip -q install peft
!pip -q uninstall -y torchao   # Colab's old torchao (0.10) clashes with peft; LoRA does not need it
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); disable_progress_bars()
DEVICE = "cuda"

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)
ZERO_SHOT_HAND = [0.51, 0.83, 0.86, 0.87, 0.92, 0.00, 0.26, 0.00, 0.11, 0.04]   # Model 8 scores, from its run

class LoRALLM:
    _last = None                                   # only one fine-tuned copy in GPU memory at a time
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4, verbose=False):
        self.seed, self.r, self.epochs, self.batch, self.lr, self.verbose = seed, r, epochs, batch, lr, verbose
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]      # only the last position: saves memory
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()      # class 0 = No, class 1 = Yes
    def fit(self, X, y):
        if LoRALLM._last is not None and hasattr(LoRALLM._last, "model"):
            del LoRALLM._last.model
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        base = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
        base.gradient_checkpointing_enable(); base.enable_input_require_grads()   # trade a little speed for memory
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(base, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()                  # trainable LoRA weights in fp32
        if self.verbose: self.model.print_trainable_parameters()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed); t0 = time.time(); step = 0
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
                step += 1
                if self.verbose and step % 50 == 0:
                    print(f"    step {step}/{steps}: loss {loss.item():.4f} ({time.time() - t0:.0f} s)")
        self.model.eval(); LoRALLM._last = self
        return self
    def predict_proba(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts))
        order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return np.column_stack([1 - out, out])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

print(f"GPU memory free before training: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")
# --- Step 1: one fine-tuning run, timed, with everything printed ---
t0 = time.time()
lora = LoRALLM(seed=42, verbose=True).fit(train["dialogue"], train["label"])
fit_time = time.time() - t0
print(f"\none LoRA fine-tuning run: {fit_time:.0f} s")
zs = iter(ZERO_SHOT_HAND)
for g_, msgs in HAND.items():
    print("\n" + g_.upper())
    for msg, p in zip(msgs, lora.predict_proba(msgs)[:, 1]):
        print(f"  LoRA {p:.2f} {'SCAM' if p > 0.5 else 'not scam':9}| zero-shot {next(zs):.2f} | {msg[:60]}")
t0 = time.time()
pt = lora.predict(test["dialogue"])
cell_ok = test.assign(ok=(pt == test["label"].values)).groupby(["source", "label"])["ok"].mean()
print(f"\nstandard split: macro-F1 {f1_score(test['label'], pt, average='macro'):.3f} ({time.time() - t0:.0f} s to score the test set)")
print(cell_ok.round(3).to_string())
print(f"\nEstimated time for the full evaluate() (11 training runs): about {11 * fit_time / 60:.0f} min")

GPU memory free before training: 15.5 GB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
    step 50/428: loss 0.4357 (64 s)
    step 100/428: loss 0.1005 (129 s)
    step 150/428: loss 0.0450 (197 s)
    step 200/428: loss 0.1315 (267 s)
    step 250/428: loss 0.1391 (338 s)
    step 300/428: loss 0.0798 (408 s)
    step 350/428: loss 0.0619 (479 s)
    step 400/428: loss 0.0595 (549 s)

one LoRA fine-tuning run: 608 s

SCAM
  LoRA 0.99 SCAM     | zero-shot 0.51 | Hi, I sent you money by mistake. Please send it back to this
  LoRA 1.00 SCAM     | zero-shot 0.83 | Your M-Pesa account has been suspended. Send your PIN to 070
  LoRA 1.00 SCAM     | zero-shot 0.86 | Congratulations! You won a scholarship. Pay the 20,000 proce
  LoRA 1.00 SCAM     | zero-shot 0.87 | This is your bank. We noticed unusual activity. Confirm your
  LoRA 1.00 SCAM     | zero-shot 0.92 | Your package is held at customs. Pay a 5,000 fee at this lin

NORMAL
  LoRA 0.00 not scam | zero-shot 0.00 | Hey, are we still meeting a

The LoRA hide-one-type run was resumed here after the session ran out of memory. The first four results come from the earlier run's output.

In [ ]:
# Model 9 (LoRA): RESUME the hide-one-type tests after the RAM crash
# Note: an earlier run ran out of memory; the first four hidden scam-type results below were recorded from that run.
# Fix: load the base model ONCE, and for each run add a fresh LoRA adapter, train it, then remove it (unload).
# Needs only the GPU RESTART SETUP cell above.
!pip -q install peft
!pip -q uninstall -y torchao
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc, psutil
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); disable_progress_bars()
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

BASE = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
BASE.gradient_checkpointing_enable(); BASE.enable_input_require_grads()

class LoRALLM:
    _active = None
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4):
        self.seed, self.r, self.epochs, self.batch, self.lr = seed, r, epochs, batch, lr
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()
    def fit(self, X, y):
        global BASE
        if LoRALLM._active is not None:                     # remove the previous adapter, keep the base
            BASE = LoRALLM._active.model.unload(); LoRALLM._active.model = None
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(BASE, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        self.model.eval(); LoRALLM._active = self
        return self
    def predict(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return (out > 0.5).astype(int)

SYN_TYPES = set(full.loc[full["source"] == "synthetic", "type"])
HIDE = {"refund": 0.836, "reward": 0.738, "ssn": 0.981, "support": 0.883}      # finished before the crash
order = [(t, 0) for t in NORMAL_TYPES if t in SYN_TYPES] + [("sms_spam", 1), ("sms_ham", 0)]
print("already done:", HIDE)
t_all = time.time()
for i, (t, lab) in enumerate(order, 5):
    t0 = time.time()
    tr = full[full["type"] != t]
    te = full[(full["type"] == t) & (full["label"] == lab)]
    mm = LoRALLM(seed=42).fit(tr["dialogue"], tr["label"])
    HIDE[t] = float((mm.predict(te["dialogue"]) == 1).mean())
    what = "still caught" if lab == 1 else "wrongly flagged"
    print(f"  [{i}/10] hide {t:14} ({'scam' if lab else 'normal'}): {what} {HIDE[t]:.3f} | {time.time() - t0:.0f} s | "
          f"total {(time.time() - t_all) / 60:.0f} min | RAM used {psutil.virtual_memory().percent:.0f}%", flush=True)

rs = [HIDE[t] for t in ["refund", "reward", "ssn", "support"]]
fs = [HIDE[t] for t, lab in order if lab == 0 and t in SYN_TYPES]
print(f"\nnew synthetic scam type recall: mean {np.mean(rs):.3f}, lowest {min(rs):.3f}")
print(f"new synthetic normal type false alarms: mean {np.mean(fs):.3f}, highest {max(fs):.3f}")
print(f"no real data in training: real spam caught {HIDE['sms_spam']:.3f}, real normal wrongly flagged {HIDE['sms_ham']:.3f}")

already done: {'refund': 0.836, 'reward': 0.738, 'ssn': 0.981, 'support': 0.883}
  [5/10] hide delivery       (normal): wrongly flagged 0.440 | 691 s | total 12 min | RAM used 35%
  [6/10] hide insurance      (normal): wrongly flagged 0.048 | 684 s | total 23 min | RAM used 35%
  [7/10] hide telemarketing  (normal): wrongly flagged 0.070 | 686 s | total 34 min | RAM used 35%
  [8/10] hide wrong          (normal): wrongly flagged 0.346 | 720 s | total 46 min | RAM used 35%
  [9/10] hide sms_spam       (scam): still caught 0.276 | 716 s | total 58 min | RAM used 35%
  [10/10] hide sms_ham        (normal): wrongly flagged 0.782 | 713 s | total 70 min | RAM used 21%

new synthetic scam type recall: mean 0.859, lowest 0.738
new synthetic normal type false alarms: mean 0.226, highest 0.440
no real data in training: real spam caught 0.276, real normal wrongly flagged 0.782


In [ ]:
# Fix: load the base model ONCE, and for each run add a fresh LoRA adapter, train it, then remove it (unload).
# Needs only the GPU RESTART SETUP cell above.
!pip -q install peft
!pip -q uninstall -y torchao
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc, psutil
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); disable_progress_bars()
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

BASE = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
BASE.gradient_checkpointing_enable(); BASE.enable_input_require_grads()

class LoRALLM:
    _active = None
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4):
        self.seed, self.r, self.epochs, self.batch, self.lr = seed, r, epochs, batch, lr
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()
    def fit(self, X, y):
        global BASE
        if LoRALLM._active is not None:                     # remove the previous adapter, keep the base
            BASE = LoRALLM._active.model.unload(); LoRALLM._active.model = None
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(BASE, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        self.model.eval(); LoRALLM._active = self
        return self
    def predict(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return (out > 0.5).astype(int)


# Model 9 (LoRA): 3 seeds + big false-alarm test (same as seed_eval, without the hide-one-type part)
# Hide-one-type would add 27 training runs (about 5.5 hours), so those stay single-seed for LoRA.
# Each seed: new SMS split, new balancing sample, new LoRA seed; cut-off chosen on ~3,100 leftover real normal TRAIN SMS.
def _lora_pp(self, X, batch=32):
    texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(texts), batch):
            ids = order[b:b + batch]
            enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
            out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
    return np.column_stack([1 - out, out])
LoRALLM.predict_proba = _lora_pp

LORA_SEED_ROWS = []
t_all = time.time()
for seed in SEEDS:
    t0 = time.time()
    tr, te, s_tr, s_te = build_data(seed)
    cut_ham  = s_tr[(s_tr.label == 0) & ~s_tr.dialogue.isin(tr.dialogue)]
    test_ham = s_te[s_te.label == 0]
    m = LoRALLM(seed=seed).fit(tr["dialogue"], tr["label"])
    t_fit = time.time() - t0
    cut = np.quantile(m.predict_proba(cut_ham["dialogue"])[:, 1], 0.95)
    s_big, st = m.predict_proba(test_ham["dialogue"])[:, 1], m.predict_proba(te["dialogue"])[:, 1]
    row = {"seed": seed, "AUC": roc_auc_score(te["label"], st)}
    for tag, c in [("default", 0.5), ("5%", cut)]:
        flagged = (st > c).astype(int); r = rates(te, flagged)
        row[f"real normal flagged, big test ({tag})"] = float((s_big > c).mean())
        row[f"real spam caught ({tag})"] = r["real spam caught"]
        row[f"synth normal flagged ({tag})"] = r["synth normal flagged"]
        row[f"synth scam caught ({tag})"] = r["synth scam caught"]
        row[f"F1 ({tag})"] = f1_score(te["label"], flagged, average="macro")
    LORA_SEED_ROWS.append(row)
    print(f"  seed {seed}: AUC {row['AUC']:.3f} | cut-off 0.50 -> {cut:.3f} | real normal flagged at 5% {row['real normal flagged, big test (5%)']:.3f} "
          f"| real spam caught {row['real spam caught (5%)']:.3f} | F1 {row['F1 (default)']:.3f} | train {t_fit:.0f} s, total {(time.time() - t0) / 60:.1f} min", flush=True)

d = pd.DataFrame(LORA_SEED_ROWS).drop(columns="seed")
print(f"\nLoRA: mean ± spread over 3 seeds ({(time.time() - t_all) / 60:.0f} min)")
print((d.mean().round(3).astype(str) + " ± " + d.std().round(3).astype(str)).to_string())

  seed 42: AUC 0.996 | cut-off 0.50 -> 0.500 | real normal flagged at 5% 0.058 | real spam caught 0.961 | F1 0.972 | train 501 s, total 10.3 min
  seed 7: AUC 0.995 | cut-off 0.50 -> 0.317 | real normal flagged at 5% 0.040 | real spam caught 0.977 | F1 0.970 | train 523 s, total 10.7 min
  seed 2026: AUC 0.994 | cut-off 0.50 -> 0.356 | real normal flagged at 5% 0.063 | real spam caught 0.938 | F1 0.967 | train 520 s, total 10.7 min

LoRA: mean ± spread over 3 seeds (32 min)
AUC                                        0.995 ± 0.001
real normal flagged, big test (default)    0.045 ± 0.014
real spam caught (default)                  0.948 ± 0.03
synth normal flagged (default)             0.029 ± 0.006
synth scam caught (default)                0.972 ± 0.008
F1 (default)                                0.97 ± 0.002
real normal flagged, big test (5%)         0.053 ± 0.012
real spam caught (5%)                       0.958 ± 0.02
synth normal flagged (5%)                  0.041 ± 0.005
synth sc

## Model 10: ensembles E1 to E4
Plain average of member scores (calibrated SVM, DistilBERT, LoRA, zero-shot).

In [ ]:
# Model 10: ensembles (average of member scores)
# Members, retrained on each seed's data: SVM (calibrated to give probabilities), DistilBERT, LoRA LLM, and the
# zero-shot LLM (same Qwen weights with the LoRA adapter switched off, so no second copy in memory).
# Each ensemble score = plain average of its members' 0-to-1 scores. Same harness: 3 seeds, big false-alarm test, 5% cut-offs.
# Needs: GPU RESTART SETUP cell + the LoRA cell above (LoRALLM, BASE, build, llm_tok) in this runtime.
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])

ZS_CACHE = {}
def zs_scores(texts, batch=32):
    """Zero-shot scores from the base Qwen model (LoRA adapter switched off). Cached, so each text is scored once."""
    texts = [str(t) for t in texts]; todo = sorted(set(t for t in texts if t not in ZS_CACHE), key=len)
    act = LoRALLM._active
    with act.model.disable_adapter(), torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(todo), batch):
            chunk = todo[b:b + batch]
            enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
            ZS_CACHE.update(zip(chunk, torch.softmax(act._last_logits(enc), dim=1)[:, 1].cpu().numpy()))
    return np.array([ZS_CACHE[t] for t in texts])

ENSEMBLES = {"E1 SVM+LoRA": ["SVM", "LoRA"], "E2 SVM+LoRA+zero-shot": ["SVM", "LoRA", "zero-shot"],
             "E3 LoRA+zero-shot": ["LoRA", "zero-shot"], "E4 all four": ["SVM", "DistilBERT", "LoRA", "zero-shot"]}
HAND_TEXTS = sum(HAND.values(), [])
ENS_ROWS = []
t_all = time.time()
for seed in SEEDS:
    t0 = time.time()
    tr, te, s_tr, s_te = build_data(seed)
    cut_ham = s_tr[(s_tr.label == 0) & ~s_tr.dialogue.isin(tr.dialogue)]; test_ham = s_te[s_te.label == 0]
    parts = {"te": list(te["dialogue"]), "cut": list(cut_ham["dialogue"]), "big": list(test_ham["dialogue"]), "hand": HAND_TEXTS}
    allx = sum(parts.values(), [])
    S = {}
    svm = make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=seed), cv=3)).fit(tr["dialogue"], tr["label"])
    S["SVM"] = svm.predict_proba(allx)[:, 1]
    bm = DistilBERTClassifier(seed=seed).fit(tr["dialogue"], tr["label"]); S["DistilBERT"] = bm.predict_proba(allx)[:, 1]
    del bm.model, bm; gc.collect(); torch.cuda.empty_cache()
    lm = LoRALLM(seed=seed).fit(tr["dialogue"], tr["label"]); S["LoRA"] = lm.predict_proba(allx)[:, 1]
    S["zero-shot"] = zs_scores(allx)
    for name, members in ENSEMBLES.items():
        S[name] = np.mean([S[m] for m in members], axis=0)
    cuts = np.cumsum([0] + [len(v) for v in parts.values()])
    for name, sc in S.items():
        st, s_cut, s_big, s_hand = (sc[cuts[i]:cuts[i + 1]] for i in range(4))
        cut = np.quantile(s_cut, 0.95)
        row = {"seed": seed, "model": name, "AUC": roc_auc_score(te["label"], st)}
        for tag, c in [("default", 0.5), ("5%", cut)]:
            flagged = (st > c).astype(int); r = rates(te, flagged)
            row[f"real normal flagged, big test ({tag})"] = float((s_big > c).mean())
            row[f"real spam caught ({tag})"] = r["real spam caught"]
            row[f"synth normal flagged ({tag})"] = r["synth normal flagged"]
            row[f"synth scam caught ({tag})"] = r["synth scam caught"]
            row[f"F1 ({tag})"] = f1_score(te["label"], flagged, average="macro")
        ENS_ROWS.append(row)
    if seed == 42:
        print("Hand-written messages, seed 42 (score; above 0.5 = SCAM):")
        hs = {k: v[cuts[3]:cuts[4]] for k, v in S.items()}
        print(pd.DataFrame(hs, index=[m[:45] for m in HAND_TEXTS]).round(2).to_string())
        print("check: zero-shot scores should match Model 8 (0.51, 0.83, 0.86, 0.87, 0.92, 0.00, 0.26, 0.00, 0.11, 0.04)\n")
    e2 = [r for r in ENS_ROWS if r["seed"] == seed and r["model"] == "E2 SVM+LoRA+zero-shot"][0]
    print(f"seed {seed} done in {(time.time() - t0) / 60:.1f} min | E2 real spam caught at 5% {e2['real spam caught (5%)']:.3f}, real normal flagged {e2['real normal flagged, big test (5%)']:.3f}", flush=True)

ens = pd.DataFrame(ENS_ROWS).drop(columns="seed")
order = ["SVM", "DistilBERT", "LoRA", "zero-shot"] + list(ENSEMBLES)
mm, ss = ens.groupby("model").mean().reindex(order).T, ens.groupby("model").std().reindex(order).T
print(f"\n===== SUMMARY: mean ± spread over 3 seeds ({(time.time() - t_all) / 60:.0f} min) =====")
print((mm.round(3).astype(str) + " ± " + ss.round(3).astype(str)).to_string())

Hand-written messages, seed 42 (score; above 0.5 = SCAM):
                                                SVM  DistilBERT  LoRA  zero-shot  E1 SVM+LoRA  E2 SVM+LoRA+zero-shot  E3 LoRA+zero-shot  E4 all four
Hi, I sent you money by mistake. Please send   0.78        0.02  0.99       0.51         0.89                   0.76               0.75         0.58
Your M-Pesa account has been suspended. Send   0.93        1.00  1.00       0.83         0.97                   0.92               0.92         0.94
Congratulations! You won a scholarship. Pay t  0.98        0.99  1.00       0.86         0.99                   0.94               0.93         0.96
This is your bank. We noticed unusual activit  0.94        1.00  1.00       0.86         0.97                   0.93               0.93         0.95
Your package is held at customs. Pay a 5,000   0.63        1.00  1.00       0.92         0.81                   0.85               0.96         0.89
Hey, are we still meeting at 5 for the group   0

In [ ]:
# Fix: load the base model ONCE, and for each run add a fresh LoRA adapter, train it, then remove it (unload).
# Needs only the GPU RESTART SETUP cell above.
!pip -q install peft
!pip -q uninstall -y torchao
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc, psutil
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); disable_progress_bars()
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

BASE = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
BASE.gradient_checkpointing_enable(); BASE.enable_input_require_grads()

class LoRALLM:
    _active = None
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4):
        self.seed, self.r, self.epochs, self.batch, self.lr = seed, r, epochs, batch, lr
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()
    def fit(self, X, y):
        global BASE
        if LoRALLM._active is not None:                     # remove the previous adapter, keep the base
            BASE = LoRALLM._active.model.unload(); LoRALLM._active.model = None
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(BASE, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        self.model.eval(); LoRALLM._active = self
        return self
    def predict(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return (out > 0.5).astype(int)


# Model 9 (LoRA): 3 seeds + big false-alarm test (same as seed_eval, without the hide-one-type part)
# Hide-one-type would add 27 training runs (about 5.5 hours), so those stay single-seed for LoRA.
# Each seed: new SMS split, new balancing sample, new LoRA seed; cut-off chosen on ~3,100 leftover real normal TRAIN SMS.
def _lora_pp(self, X, batch=32):
    texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(texts), batch):
            ids = order[b:b + batch]
            enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
            out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
    return np.column_stack([1 - out, out])
LoRALLM.predict_proba = _lora_pp


# Model 10: ensembles (average of member scores)
# Members, retrained on each seed's data: SVM (calibrated to give probabilities), DistilBERT, LoRA LLM, and the
# zero-shot LLM (same Qwen weights with the LoRA adapter switched off, so no second copy in memory).
# Each ensemble score = plain average of its members' 0-to-1 scores. Same harness: 3 seeds, big false-alarm test, 5% cut-offs.
# Needs: GPU RESTART SETUP cell + the LoRA cell above (LoRALLM, BASE, build, llm_tok) in this runtime.
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])

ZS_CACHE = {}
def zs_scores(texts, batch=32):
    """Zero-shot scores from the base Qwen model (LoRA adapter switched off). Cached, so each text is scored once."""
    texts = [str(t) for t in texts]; todo = sorted(set(t for t in texts if t not in ZS_CACHE), key=len)
    act = LoRALLM._active
    with act.model.disable_adapter(), torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(todo), batch):
            chunk = todo[b:b + batch]
            enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
            ZS_CACHE.update(zip(chunk, torch.softmax(act._last_logits(enc), dim=1)[:, 1].cpu().numpy()))
    return np.array([ZS_CACHE[t] for t in texts])

ENSEMBLES = {"E1 SVM+LoRA": ["SVM", "LoRA"], "E2 SVM+LoRA+zero-shot": ["SVM", "LoRA", "zero-shot"],
             "E3 LoRA+zero-shot": ["LoRA", "zero-shot"], "E4 all four": ["SVM", "DistilBERT", "LoRA", "zero-shot"]}

# Model 10: hide-one-type tests for the ensembles (seed 42, same design as evaluate())
# For each hidden topic: retrain SVM, DistilBERT and LoRA without it; zero-shot needs no training (cached scores).
# Then average the members' scores on the hidden topic's messages and count the share above 0.5.
# Each result prints as soon as it is ready and is also saved to /content/hide_ensembles.json.
import json
SYN_TYPES = set(full.loc[full["source"] == "synthetic", "type"])
order = [(t, 1) for t in SCAM_TYPES if t in SYN_TYPES] + [(t, 0) for t in NORMAL_TYPES if t in SYN_TYPES] + [("sms_spam", 1), ("sms_ham", 0)]
names = ["SVM", "DistilBERT", "LoRA", "zero-shot"] + list(ENSEMBLES)
HIDE_ENS = {   # runs 1 to 6 finished before Colab stopped; copied from the printed output
    "refund": {"label": 1, "SVM": 0.844, "DistilBERT": 0.791, "LoRA": 0.838, "zero-shot": 0.435, "E1 SVM+LoRA": 0.846, "E2 SVM+LoRA+zero-shot": 0.825, "E3 LoRA+zero-shot": 0.787, "E4 all four": 0.827},
    "reward": {"label": 1, "SVM": 0.887, "DistilBERT": 0.584, "LoRA": 0.74, "zero-shot": 0.518, "E1 SVM+LoRA": 0.852, "E2 SVM+LoRA+zero-shot": 0.809, "E3 LoRA+zero-shot": 0.719, "E4 all four": 0.777},
    "ssn": {"label": 1, "SVM": 0.976, "DistilBERT": 0.983, "LoRA": 0.982, "zero-shot": 0.533, "E1 SVM+LoRA": 0.989, "E2 SVM+LoRA+zero-shot": 0.975, "E3 LoRA+zero-shot": 0.945, "E4 all four": 0.988},
    "support": {"label": 1, "SVM": 0.92, "DistilBERT": 0.895, "LoRA": 0.883, "zero-shot": 0.472, "E1 SVM+LoRA": 0.913, "E2 SVM+LoRA+zero-shot": 0.87, "E3 LoRA+zero-shot": 0.799, "E4 all four": 0.896},
    "delivery": {"label": 0, "SVM": 0.641, "DistilBERT": 0.725, "LoRA": 0.44, "zero-shot": 0.15, "E1 SVM+LoRA": 0.596, "E2 SVM+LoRA+zero-shot": 0.477, "E3 LoRA+zero-shot": 0.328, "E4 all four": 0.591},
    "insurance": {"label": 0, "SVM": 0.019, "DistilBERT": 0.027, "LoRA": 0.048, "zero-shot": 0.299, "E1 SVM+LoRA": 0.024, "E2 SVM+LoRA+zero-shot": 0.03, "E3 LoRA+zero-shot": 0.069, "E4 all four": 0.025},
}
t_all = time.time()
for i, (t, lab) in enumerate(order, 1):
    if t in HIDE_ENS: continue                 # already done
    t0 = time.time()
    tr = full[full["type"] != t]
    x = list(full[(full["type"] == t) & (full["label"] == lab)]["dialogue"])
    S = {}
    svm = make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=42), cv=3)).fit(tr["dialogue"], tr["label"])
    S["SVM"] = svm.predict_proba(x)[:, 1]
    bm = DistilBERTClassifier(seed=42).fit(tr["dialogue"], tr["label"]); S["DistilBERT"] = bm.predict_proba(x)[:, 1]
    del bm.model, bm; gc.collect(); torch.cuda.empty_cache()
    lm = LoRALLM(seed=42).fit(tr["dialogue"], tr["label"]); S["LoRA"] = lm.predict_proba(x)[:, 1]
    S["zero-shot"] = zs_scores(x)
    for name, members in ENSEMBLES.items():
        S[name] = np.mean([S[m] for m in members], axis=0)
    HIDE_ENS[t] = {"label": lab, **{k: float((v > 0.5).mean()) for k, v in S.items()}}
    json.dump(HIDE_ENS, open("/content/hide_ensembles.json", "w"))
    what = "still caught" if lab == 1 else "wrongly flagged"
    print(f"[{i}/{len(order)}] hide {t} ({'scam' if lab else 'normal'}), share {what}: " +
          " | ".join(f"{k} {HIDE_ENS[t][k]:.3f}" for k in names) + f" | {(time.time() - t0) / 60:.1f} min, total {(time.time() - t_all) / 60:.0f} min", flush=True)

H = pd.DataFrame(HIDE_ENS).T
syn = H.index.isin(list(SYN_TYPES))
summary = pd.DataFrame({
    "new scam type caught (mean)":   H[syn & (H.label == 1)][names].mean(),
    "new scam type caught (lowest)": H[syn & (H.label == 1)][names].min(),
    "new normal topic flagged (mean)":    H[syn & (H.label == 0)][names].mean(),
    "new normal topic flagged (highest)": H[syn & (H.label == 0)][names].max(),
    "no real data: real spam caught":         H.loc["sms_spam", names],
    "no real data: real normal flagged":      H.loc["sms_ham", names]}).T
print("\n===== HIDE-ONE-TYPE SUMMARY (seed 42) =====")
print(summary.astype(float).round(3).to_string())

[7/10] hide telemarketing (normal), share wrongly flagged: SVM 0.093 | DistilBERT 0.297 | LoRA 0.070 | zero-shot 0.228 | E1 SVM+LoRA 0.074 | E2 SVM+LoRA+zero-shot 0.055 | E3 LoRA+zero-shot 0.061 | E4 all four 0.096 | 14.7 min, total 15 min
[8/10] hide wrong (normal), share wrongly flagged: SVM 0.499 | DistilBERT 0.600 | LoRA 0.346 | zero-shot 0.018 | E1 SVM+LoRA 0.410 | E2 SVM+LoRA+zero-shot 0.252 | E3 LoRA+zero-shot 0.144 | E4 all four 0.421 | 15.1 min, total 30 min
[9/10] hide sms_spam (scam), share still caught: SVM 0.092 | DistilBERT 0.103 | LoRA 0.274 | zero-shot 0.914 | E1 SVM+LoRA 0.192 | E2 SVM+LoRA+zero-shot 0.283 | E3 LoRA+zero-shot 0.375 | E4 all four 0.218 | 15.2 min, total 45 min
[10/10] hide sms_ham (normal), share wrongly flagged: SVM 0.875 | DistilBERT 0.872 | LoRA 0.782 | zero-shot 0.053 | E1 SVM+LoRA 0.871 | E2 SVM+LoRA+zero-shot 0.693 | E3 LoRA+zero-shot 0.363 | E4 all four 0.793 | 15.1 min, total 60 min

===== HIDE-ONE-TYPE SUMMARY (seed 42) =====
                  

# 5. Masked rerun v1
Names, emails, links, amounts and numbers replaced by placeholders, plus audits and evidence checks. **The masking_model cell (now the v2 code) is in this section: run it right after the setup cell.**

**Masked rerun:** the same experiments repeated on masked text, so models cannot rely on names, numbers or links. This first cell checks what is still in memory.

In [ ]:
# MASKED RERUN: probe (what is in memory right now)
import sys
names = ["train", "test", "full", "sms_df", "syn_tr", "syn_te", "RESULTS", "BOTH_ROWS", "ENS_ROWS", "HIDE_ENS", "LORA_SEED_ROWS", "build_data", "seed_eval", "RepeatReal", "LoRALLM", "DistilBERTClassifier", "CNNClassifier"]
print({n: (n in globals()) for n in names})
try:
    import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
except Exception as e:
    print("torch error:", e)
print(sys.version)

{'train': True, 'test': True, 'full': True, 'sms_df': True, 'syn_tr': True, 'syn_te': True, 'RESULTS': True, 'BOTH_ROWS': False, 'ENS_ROWS': False, 'HIDE_ENS': True, 'LORA_SEED_ROWS': False, 'build_data': True, 'seed_eval': True, 'RepeatReal': False, 'LoRALLM': True, 'DistilBERTClassifier': True, 'CNNClassifier': False}
GPU: Tesla T4
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


### masking_model
Replaces names, emails, links, money amounts and phone numbers with fixed tokens (nametoken, emailtoken, linktoken, amounttoken, numtoken), using regular expressions and spaCy name tags. **Run this after every restart.**

In [ ]:
# Masking_model
import re, spacy

EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
LINK = re.compile(r"(?:https?://|www\.)\S+|\b[\w-]+\.(?:com|net|org|co|io|ly|me|rw|ke|ug|tz|ng|gh|za)(?:/\S*)?\b", re.IGNORECASE)
CURRENCY = r"(?:rwf|frw|kes|ksh|ugx|tzs|ngn|ghs|zar|usd|shillings?|francs?|naira|cedis?|rand|dollars?|birr)"
AMOUNT = re.compile(rf"\b{CURRENCY}\.?\s?\d[\d,.]*|[$€£]\s?\d[\d,.]*|\d[\d,.]*\s?{CURRENCY}\b", re.IGNORECASE)
NUMBER = re.compile(r"\+?\d[\d\s\-()]{5,}\d")

GREETING = r"(?i:hi|hello|hey|dear|thanks|thank you)"
TITLE = r"(?i:mr|mrs|ms|miss|dr|prof)"
# A space only (no comma or full stop) between the greeting and the name: "Hi Chidi", "Dear Dr. Habimana"
AFTER_GREETING = re.compile(rf"\b{GREETING}\s+(?:{TITLE}\.?\s+)*([A-Z][a-z\-]{{2,}})")
AFTER_TITLE = re.compile(rf"\b{TITLE}\.?\s+([A-Z][a-z\-]{{2,}})")
NOT_NAMES = {"customer", "client", "user", "friend", "winner", "beneficiary", "sir", "madam",
             "everyone", "all", "team", "there", "what", "when", "where", "who", "why", "how",
             "this", "that", "please", "just", "yes", "well", "now", "today", "and", "but",
             "so", "the", "you", "your", "our", "for", "from", "with", "hmm"}

# Filters for spaCy's PERSON tags
INTERJECTIONS = {"hmm", "hm", "ya", "yeah", "yep", "wat", "ok", "okay", "lol", "haha", "oh", "ah",
                 "wow", "hey", "hi", "hello", "thanks", "pls", "plz", "sorry", "yes", "no"}
LEADING = {"this", "the", "a", "an", "my", "your", "our", "his", "her", "their", "dear", "hello",
           "hi", "hey", "thanks", "mr", "mrs", "ms", "miss", "dr", "prof"}
BAD_WORDS = {"st", "street", "rd", "road", "ave", "avenue", "usa", "uk", "ltd", "inc", "limited", "delivery", "bank"}

TOK = dict(name="nametoken", email="emailtoken", link="linktoken", amount="amounttoken", number="numtoken")
DISPLAY = {v: f"<{k.upper()}>" for k, v in TOK.items()}

_nlp = None
def _get_nlp():
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("en_core_web_sm")
    return _nlp

def _person_span(ent):
    toks = list(ent)
    while toks and toks[0].lower_.rstrip(".") in LEADING:   # "this John Smith" -> "John Smith"
        toks = toks[1:]
    if not toks or len(toks) > 3:
        return None
    for t in toks:
        w = t.text
        if not w.replace("-", "").replace("'", "").isalpha() or not w[0].isupper() or len(w) < 2 or w.isupper():
            return None
        if w.lower() in INTERJECTIONS or w.lower() in BAD_WORDS:
            return None
    return toks[0].idx, toks[-1].idx + len(toks[-1])

def _greeted(m):
    if m.group(1).lower() in NOT_NAMES:
        return m.group(0)
    return m.group(0)[: m.start(1) - m.start(0)] + TOK["name"]

def _rest(text, known_names):
    for n in known_names:
        text = re.sub(rf"(?<!\w){re.escape(n)}(?!\w)", TOK["name"], text, flags=re.IGNORECASE)
    for pattern in (AFTER_GREETING, AFTER_TITLE):
        text = pattern.sub(_greeted, text)
    text = EMAIL.sub(TOK["email"], text)
    text = LINK.sub(TOK["link"], text)
    text = AMOUNT.sub(TOK["amount"], text)   # amounts before long numbers
    text = NUMBER.sub(TOK["number"], text)
    return text

def mask_texts(texts, known_names=()):
    texts = [str(t) for t in texts]
    out = []
    for doc, text in zip(_get_nlp().pipe(texts, batch_size=64), texts):
        spans = [s for s in (_person_span(e) for e in doc.ents if e.label_ == "PERSON") if s]
        for start, end in reversed(spans):
            text = text[:start] + TOK["name"] + text[end:]
        out.append(_rest(text, known_names))
    return out

def to_display(text):
    for tok, shown in DISPLAY.items():
        text = text.replace(tok, shown)
    return text

Masks every text. Splits and balancing are chosen on the raw text exactly as before; only the text is swapped for its masked version.

In [ ]:
# MASKED RERUN, step 2: mask every text (new names only; raw data untouched)
# Splits, balancing and held-out sets are chosen on the RAW text exactly as before; only the text is then swapped
# for its masked version through one lookup table (MASK). Labels and types are never touched.
import time, collections
print("spaCy check:", nlp("Hello John, call Mary Smith at 0788123456.").ents)

t0 = time.time()
all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
MASK = dict(zip(all_raw, mask_texts(all_raw)))
print(f"masked {len(MASK)} unique texts in {time.time() - t0:.0f} s")

def m(series_or_list):
    return [MASK[str(t)] for t in series_or_list]

syn_tr_m = syn_tr.assign(dialogue=m(syn_tr["dialogue"]))
syn_te_m = syn_te.assign(dialogue=m(syn_te["dialogue"]))
sms_df_m = sms_df.assign(dialogue=m(sms_df["dialogue"]))
HAND_m = {g: m(v) for g, v in HAND.items()}
train_m, test_m, full_m = (d.assign(dialogue=m(d["dialogue"])) for d in (train, test, full))

# 1. 10 random before/after pairs per source (drawn from texts that masking changed)
rng = np.random.default_rng(0)
for src, raw in [("synthetic", list(syn_tr["dialogue"]) + list(syn_te["dialogue"])), ("real SMS", list(sms_df["dialogue"]))]:
    changed = [t for t in raw if MASK[t] != t]
    print(f"\n--- {src}: {len(changed)} of {len(raw)} texts changed ({len(changed) / len(raw):.1%}); 10 random changed ones ---")
    for t in rng.choice(changed, size=min(10, len(changed)), replace=False):
        print("  BEFORE:", t[:160]); print("  AFTER: ", MASK[t][:160])
print("\n--- hand-written messages ---")
for g in HAND:
    for a, b in zip(HAND[g], HAND_m[g]):
        if a != b: print("  BEFORE:", a); print("  AFTER: ", b)

# 2. placeholder counts per source and label (texts containing each placeholder)
rows = []
for name, d in [("synthetic", pd.concat([syn_tr_m, syn_te_m])), ("real SMS", sms_df_m)]:
    for lab, g in d.groupby("label"):
        r = {"source": name, "label": "scam/spam" if lab else "normal", "texts": len(g)}
        for k, tok in TOK.items():
            r[tok] = int(g["dialogue"].str.count(tok).sum())
        rows.append(r)
print("\nPlaceholder counts (occurrences):")
print(pd.DataFrame(rows).to_string(index=False))

# 3. no run of 7+ digits remains anywhere
left = [t for t in MASK.values() if re.search(r"\d{7,}", t)]
print("\ntexts with a run of 7+ digits after masking:", len(left))
assert not left, left[:5]

# 4. side effects worth knowing: masking can make two different raw texts identical
raw_n, m_n = sms_df["dialogue"].nunique(), sms_df_m["dialogue"].nunique()
print(f"\nreal SMS: {raw_n} unique raw texts -> {m_n} unique masked texts ({raw_n - m_n} collisions)")
tr_set, te_set = set(train_m["dialogue"]), set(test_m["dialogue"])
print("seed-42 masked texts present in BOTH train and test:", len(tr_set & te_set),
      "| raw:", len(set(train["dialogue"]) & set(test["dialogue"])))
print("masked synthetic texts identical to a masked real SMS:", len(set(pd.concat([syn_tr_m, syn_te_m])["dialogue"]) & set(sms_df_m["dialogue"])))

spaCy check: (John, Mary Smith, 0788123456)
masked 14184 unique texts in 93 s

--- synthetic: 3649 of 9130 texts changed (40.0%); 10 random changed ones ---
  BEFORE: and a phone number where I can call back to verify your identity? caller: My employee ID is WTS1234. And you can call our main office at 1-800-WINDOWS-HELP. rec
  AFTER:  and a phone number where I can call back to verify your identity? caller: My employee ID is nametoken. And you can call our main office at 1-800-WINDOWS-HELP. r
  BEFORE: caller: Hello, my name is Officer Johnson from the Social Security Administration. We've received a report that your social security number has been compromised
  AFTER:  caller: Hello, my name is Officer nametoken from the Social Security Administration. We've received a report that your social security number has been compromis
  BEFORE: caller: Hello, my name is John and I'm calling from XYZ Company. We're offering a special deal on our new software that can help you manage your busi

Evidence checks on the raw text: phrase counts, the share of unknown words, and an audit of what the masking replaced.

In [ ]:
# MASKED RERUN, step 3: evidence checks on raw text (+ a masking audit)
from collections import Counter
TOKEN_RE = re.compile(r"[a-z0-9£$]+|[!?]")              # same tokenizer as the CNN / BiLSTM
tok = lambda t: TOKEN_RE.findall(str(t).lower())

# (a) phrase counts in the seed-42 TRAINING pieces, raw text
print("(a) pieces containing each phrase, seed-42 training set (raw text)")
rows = []
for (src, typ, lab), g in train.groupby(["source", "type", "label"]):
    low = g["dialogue"].str.lower()
    rows.append({"source": src, "type": typ, "label": "scam/spam" if lab else "normal", "pieces": len(g),
                 "by mistake": int(low.str.contains("by mistake").sum()), "wrong number": int(low.str.contains("wrong number").sum())})
print(pd.DataFrame(rows).to_string(index=False))

# (b) share of unknown words, vocabulary built like the CNN (words seen at least 2 times in training)
def unk_table(tr_texts, te_df, label):
    cnt = Counter(w for t in tr_texts for w in tok(t)); vocab = {w for w, c in cnt.items() if c >= 2}
    out = {}
    for src, g in te_df.groupby("source"):
        words = [w for t in g["dialogue"] for w in tok(t)]
        out[src] = sum(w not in vocab for w in words) / max(len(words), 1)
    return {"text": label, "vocab size": len(vocab), **{f"unknown share, {k} test": round(v, 3) for k, v in out.items()}}
print("\n(b) unknown-word share in the seed-42 test set (vocab from the full training set; the CNN used 90% of it)")
print(pd.DataFrame([unk_table(train["dialogue"], test, "raw"), unk_table(train_m["dialogue"], test_m, "masked")]).to_string(index=False))

# masking audit: which words were replaced by nametoken?
greet_words = Counter(mt.group(1) for t in all_raw for mt in GREETED.finditer(t) if mt.group(1).lower() not in NOT_NAMES)
print("\nAudit 1: top 20 words the greeting rule turns into nametoken:")
print(greet_words.most_common(20))
person = Counter(e.text for d in nlp.pipe(all_raw, batch_size=64) for e in d.ents if e.label_ == "PERSON")
print("\nAudit 2: top 30 spaCy PERSON entities replaced by nametoken:")
print(person.most_common(30))
common = {"What", "How", "Waiting", "Reply", "Thank", "Please", "Sorry", "Ok", "Yes", "No", "Well", "Good", "Can", "Is", "Are", "Do", "I"}
print("\ngreeting-rule hits on these common words:", {w: greet_words[w] for w in sorted(common) if greet_words[w]})

(a) pieces containing each phrase, seed-42 training set (raw text)
   source          type     label  pieces  by mistake  wrong number
     real       sms_ham    normal     514           0             0
     real      sms_spam scam/spam     514           1             0
synthetic      delivery    normal     656           0             0
synthetic     insurance    normal     967           0             0
synthetic        refund scam/spam     763           1             0
synthetic        reward scam/spam     668           0             0
synthetic           ssn scam/spam     771           0             0
synthetic       support scam/spam     706           0             0
synthetic telemarketing    normal     934           0             0
synthetic         wrong    normal     351           2           191

(b) unknown-word share in the seed-42 test set (vocab from the full training set; the CNN used 90% of it)
  text  vocab size  unknown share, real test  unknown share, synthetic test
  

In [ ]:
# Model 5: CNN for text (PyTorch, runs on the GPU)
# Words -> numbers -> embedding (128 numbers per word) -> filters that look at 3, 4 and 5 words at a time
# -> keep each filter's strongest match anywhere in the message -> one scam score.
# Same fit/predict/predict_proba interface as the sklearn models, so evaluate() and seed_eval() work unchanged.
import re, torch, torch.nn as nn
from collections import Counter
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
TOKEN = re.compile(r"[a-z0-9£$]+|[!?]")
def tokenize(t): return TOKEN.findall(str(t).lower())

class TextCNN(nn.Module):
    def __init__(self, vocab, dim=128, filters=100, sizes=(3, 4, 5), drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(dim, filters, k) for k in sizes])
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(filters * len(sizes), 1)
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        h = torch.cat([torch.relu(c(e)).max(dim=2).values for c in self.convs], dim=1)
        return self.out(self.drop(h)).squeeze(1)

class CNNClassifier:
    def __init__(self, seed=42, max_len=64, max_vocab=20000, epochs=8, patience=2, batch=64, lr=1e-3, verbose=False):
        self.seed, self.max_len, self.max_vocab = seed, max_len, max_vocab
        self.epochs, self.patience, self.batch, self.lr, self.verbose = epochs, patience, batch, lr, verbose
    def _encode(self, texts):
        X = np.zeros((len(texts), self.max_len), dtype=np.int64)
        for i, t in enumerate(texts):
            ids = [self.vocab.get(w, 1) for w in tokenize(t)][:self.max_len]
            X[i, :len(ids)] = ids
        return torch.tensor(X)
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = np.asarray(labels, dtype=np.float32)
        # 10% of the training rows are held back to decide when to stop training (early stopping)
        tr_i, va_i = train_test_split(np.arange(len(texts)), test_size=0.1, stratify=y, random_state=self.seed)
        cnt = Counter(w for i in tr_i for w in tokenize(texts[i]))
        words = [w for w, c in cnt.most_common(self.max_vocab - 2) if c >= 2]
        self.vocab = {w: i + 2 for i, w in enumerate(words)}          # 0 = padding, 1 = unknown word
        X, Y = self._encode(texts), torch.tensor(y)
        self.model = TextCNN(len(self.vocab) + 2).to(DEVICE)
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.BCEWithLogitsLoss()
        g = torch.Generator().manual_seed(self.seed)
        Xva, Yva = X[va_i].to(DEVICE), Y[va_i].to(DEVICE)
        best, best_state, bad = 1e9, None, 0
        for ep in range(self.epochs):
            self.model.train()
            perm = torch.tensor(tr_i)[torch.randperm(len(tr_i), generator=g)]
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                opt.zero_grad()
                loss = loss_fn(self.model(X[idx].to(DEVICE)), Y[idx].to(DEVICE))
                loss.backward(); opt.step()
            self.model.eval()
            with torch.no_grad():
                vl = loss_fn(self.model(Xva), Yva).item()
            if self.verbose: print(f"    epoch {ep + 1}: validation loss {vl:.4f}")
            if vl < best - 1e-4:
                best, bad = vl, 0
                best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            else:
                bad += 1
                if bad >= self.patience: break
        self.model.load_state_dict(best_state)
        self.epochs_run = ep + 1
        return self
    def predict_proba(self, texts):
        self.model.eval(); X = self._encode(list(texts)); out = []
        with torch.no_grad():
            for b in range(0, len(X), 512):
                out.append(torch.sigmoid(self.model(X[b:b + 512].to(DEVICE))).cpu().numpy())
        p = np.concatenate(out)
        return np.column_stack([1 - p, p])
    def predict(self, texts):
        return (self.predict_proba(texts)[:, 1] > 0.5).astype(int)


# Model 6: BiLSTM (PyTorch, GPU)
# Reads the message word by word, left to right AND right to left, carrying a memory along the way.
# The final memory of each direction is joined and turned into one scam score.
# Same tokens, vocabulary rule, early stopping and interface as the CNN, so the only change is the network.
class BiLSTMNet(nn.Module):
    def __init__(self, vocab, dim=128, hidden=128, drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.lstm = nn.LSTM(dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(2 * hidden, 1)
    def forward(self, x):
        lengths = (x != 0).sum(1).clamp(min=1).cpu()        # real length of each message, so padding is skipped
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(x), lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.out(self.drop(torch.cat([h[0], h[1]], dim=1))).squeeze(1)

class BiLSTMClassifier(CNNClassifier):
    """Same training loop as CNNClassifier; only the network is swapped."""
    def fit(self, texts, labels):
        global TextCNN
        saved, TextCNN = TextCNN, BiLSTMNet      # CNNClassifier.fit builds TextCNN(vocab); point it at the BiLSTM for this call
        try:
            return super().fit(texts, labels)
        finally:
            TextCNN = saved



# MASKED RERUN, step 4a: seed 42, six models, both training ways, raw AND masked
# Same splits, seed, cut-off rule and tests as seed_eval / evaluate; only the text fed to the model changes.
# The raw run is a check: it should reproduce the earlier seed-42 numbers. Hand-written count = right of 10.
from xgboost import XGBClassifier

class RepeatRealSet:
    """Repeats real-SMS rows k times before training (same as RepeatReal), using the real-SMS set of the text version in use."""
    def __init__(self, base, k, real_set):
        self.base, self.k, self.real_set = base, k, real_set
    def fit(self, X, y):
        X = pd.Series(list(X)); y = pd.Series(np.asarray(y)); real = X.isin(self.real_set).values
        self.base.fit(pd.concat([X] + [X[real]] * (self.k - 1), ignore_index=True),
                      pd.concat([y] + [y[real]] * (self.k - 1), ignore_index=True))
        return self
    def predict(self, X): return self.base.predict(X)
    def __getattr__(self, name): return getattr(self.base, name)

SIX = {
    "NB":     lambda: make_pipeline(tfidf(), MultinomialNB()),
    "LR":     lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42)),
    "SVM":    lambda: make_pipeline(tfidf(), LinearSVC(random_state=42)),
    "XGB":    lambda: make_pipeline(tfidf(), XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist", device="cuda", random_state=42)),
    "CNN":    lambda: CNNClassifier(seed=42),
    "BiLSTM": lambda: BiLSTMClassifier(seed=42),
}
HIDE_ORDER = [(t, 1) for t in SYN_SCAM] + [(t, 0) for t in SYN_NORMAL] + [("sms_spam", 1)]

def run42(name, mk, masked, k):
    T = m if masked else (lambda x: [str(t) for t in x])
    tr, te, s_tr, s_te = build_data(42)
    cut_ham = s_tr[(s_tr.label == 0) & ~s_tr.dialogue.isin(tr.dialogue)]; test_ham = s_te[s_te.label == 0]
    full_s = pd.concat([tr, te], ignore_index=True)
    real_set = set(T(sms_df["dialogue"]))
    make = mk if k == 1 else (lambda: RepeatRealSet(mk(), k, real_set))
    t0 = time.time()
    mod = make().fit(T(tr["dialogue"]), tr["label"].values)
    st, sc, sb = scores(mod, T(te["dialogue"])), scores(mod, T(cut_ham["dialogue"])), scores(mod, T(test_ham["dialogue"]))
    dcut = 0.5 if hasattr(mod, "predict_proba") else 0.0
    cut = np.quantile(sc, 0.95)
    f5, fd = (st > cut).astype(int), (st > dcut).astype(int)
    row = {"model": name, "text": "masked" if masked else "raw", "way": "current" if k == 1 else "real x3",
           "real spam caught (5%)": rates(te, f5)["real spam caught"],
           "real normal flagged, big (5%)": float((sb > cut).mean()), "real normal flagged, big (0.5)": float((sb > dcut).mean()),
           "macro-F1 (5%)": f1_score(te["label"], f5, average="macro"), "macro-F1 (0.5)": f1_score(te["label"], fd, average="macro")}
    H = {g: T(v) for g, v in HAND.items()}
    hp = {g: np.array(mod.predict(v)) for g, v in H.items()}
    row["hand right of 10"] = int(hp["scam"].sum() + (hp["normal"] == 0).sum() + (hp["normal_but_scam_words"] == 0).sum())
    hide = {}
    for t, lab in HIDE_ORDER:
        keep = full_s[full_s.type != t]
        mm = make().fit(T(keep["dialogue"]), keep["label"].values)
        x = full_s[(full_s.type == t) & (full_s.label == lab)]
        hide[t] = float((np.array(mm.predict(T(x["dialogue"]))) == 1).mean())
    row["new scam type caught (mean 4)"] = float(np.mean([hide[t] for t in SYN_SCAM]))
    row["unseen normal topic flagged (mean 4)"] = float(np.mean([hide[t] for t in SYN_NORMAL]))
    row["no real spam in training: spam caught"] = hide["sms_spam"]
    row["minutes"] = (time.time() - t0) / 60
    return row

MASKED_RERUN_ROWS = []
for name, mk in SIX.items():
    for k in (1, 3):
        for masked in (False, True):
            try:
                r = run42(name, mk, masked, k)
            except Exception as e:
                print(f"ERROR {name} {'masked' if masked else 'raw'} k={k}: {type(e).__name__}: {e}"); raise
            MASKED_RERUN_ROWS.append(r)
            print(f"{name:6} {r['text']:6} {r['way']:7} | spam@5% {r['real spam caught (5%)']:.3f} | normal@5% {r['real normal flagged, big (5%)']:.3f} "
                  f"| normal@0.5 {r['real normal flagged, big (0.5)']:.3f} | F1@5% {r['macro-F1 (5%)']:.3f} | new scam {r['new scam type caught (mean 4)']:.3f} "
                  f"| unseen normal {r['unseen normal topic flagged (mean 4)']:.3f} | no-real spam {r['no real spam in training: spam caught']:.3f} | hand {r['hand right of 10']} | {r['minutes']:.1f} min", flush=True)
print("\nDONE six models")

NB     raw    current | spam@5% 0.891 | normal@5% 0.050 | normal@0.5 0.183 | F1@5% 0.969 | new scam 0.873 | unseen normal 0.311 | no-real spam 0.315 | hand 6 | 0.2 min
NB     masked current | spam@5% 0.914 | normal@5% 0.052 | normal@0.5 0.178 | F1@5% 0.969 | new scam 0.874 | unseen normal 0.317 | no-real spam 0.315 | hand 7 | 0.1 min
NB     raw    real x3 | spam@5% 0.938 | normal@5% 0.055 | normal@0.5 0.118 | F1@5% 0.975 | new scam 0.835 | unseen normal 0.315 | no-real spam 0.226 | hand 6 | 0.1 min
NB     masked real x3 | spam@5% 0.930 | normal@5% 0.054 | normal@0.5 0.124 | F1@5% 0.972 | new scam 0.844 | unseen normal 0.323 | no-real spam 0.221 | hand 6 | 0.1 min
LR     raw    current | spam@5% 0.914 | normal@5% 0.055 | normal@0.5 0.096 | F1@5% 0.969 | new scam 0.856 | unseen normal 0.323 | no-real spam 0.065 | hand 7 | 0.1 min
LR     masked current | spam@5% 0.914 | normal@5% 0.056 | normal@0.5 0.082 | F1@5% 0.968 | new scam 0.855 | unseen normal 0.339 | no-real spam 0.062 | hand 7 | 

Word counts in the training pieces, to see which words could act as shortcuts.

In [ ]:
# MASKED RERUN follow-up, step 4: word counts in the seed-42 TRAINING pieces (raw text)
# Pieces containing each word as a whole word (case-insensitive).
words = ["mistake", "sorry", "wrong", "back"]
rows = []
for (src, typ, lab), g in train.groupby(["source", "type", "label"]):
    low = g["dialogue"].str.lower()
    r = {"source": src, "type": typ, "label": "scam/spam" if lab else "normal", "pieces": len(g)}
    for w in words:
        r[w] = int(low.str.contains(rf"\b{w}\b", regex=True).sum())
    rows.append(r)
wc = pd.DataFrame(rows)
print(wc.to_string(index=False))
print("\nshare of pieces per type containing each word:")
print((wc.set_index(["source", "type", "label"])[words].div(wc.set_index(["source", "type", "label"])["pieces"], axis=0)).round(3).to_string())

   source          type     label  pieces  mistake  sorry  wrong  back
     real       sms_ham    normal     514        1     11      0     6
     real      sms_spam scam/spam     514        1      1      0    15
synthetic      delivery    normal     656       26     43      3    12
synthetic     insurance    normal     967        0      6      1    43
synthetic        refund scam/spam     763       28     12      4   106
synthetic        reward scam/spam     668        1      6      1    57
synthetic           ssn scam/spam     771        8      7      4   102
synthetic       support scam/spam     706        1      6      3    45
synthetic telemarketing    normal     934        0     15      0    35
synthetic         wrong    normal     351       38    203    200     2

share of pieces per type containing each word:
                                   mistake  sorry  wrong   back
source    type          label                                  
real      sms_ham       normal       0.002 

# 6. Masked rerun v2 (main masked results)
Re-mask and audit, six CPU models, summary tables, DistilBERT and zero-shot, then LoRA and ensembles. Seed 42; NB, LR and SVM also seeds 7 and 2026.

Re-masks everything with the fixed masking_model and audits what was replaced.

In [ ]:
# MASKED RERUN v2, step 2: re-mask with the fixed masking_model and re-audit (v1 results kept as MASK, *_m)
t0 = time.time()
MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
print(f"masked {len(MASK_V2)} unique texts in {time.time() - t0:.0f} s")
def m2(x): return [MASK_V2[str(t)] for t in x]
syn_tr_m2, syn_te_m2, sms_df_m2 = (d.assign(dialogue=m2(d["dialogue"])) for d in (syn_tr, syn_te, sms_df))
train_m2, test_m2, full_m2 = (d.assign(dialogue=m2(d["dialogue"])) for d in (train, test, full))
HAND_m2 = {g: m2(v) for g, v in HAND.items()}

# 1. were these words replaced? (whole-word count in all raw texts minus count after masking)
def wcount(texts, w): return sum(len(re.findall(rf"(?<!\w){re.escape(w)}(?!\w)", t)) for t in texts)
raw_vals, v1_vals, v2_vals = all_raw, [MASK[t] for t in all_raw], [MASK_V2[t] for t in all_raw]
print("\n1. times each word was replaced (raw count minus masked count):")
for w in ["I'm", "What", "Hmm", "It's", "We're"]:
    r = wcount(raw_vals, w)
    print(f"  {w:6} v1 {r - wcount(v1_vals, w):4} | v2 {r - wcount(v2_vals, w):4}")

# 2. the 30 most frequent masked name strings (same logic as mask_texts, recorded instead of replaced)
names = Counter()
for doc, text in zip(_get_nlp().pipe(all_raw, batch_size=64), all_raw):
    spans = [s for s in (_person_span(e) for e in doc.ents if e.label_ == "PERSON") if s]
    for a, b in spans: names[text[a:b]] += 1
    for a, b in reversed(spans): text = text[:a] + TOK["name"] + text[b:]
    for pat in (AFTER_GREETING, AFTER_TITLE):
        for mt in pat.finditer(text):
            if mt.group(1).lower() not in NOT_NAMES: names[mt.group(1)] += 1
print("\n2. 30 most frequent masked name strings:"); print(names.most_common(30))

# 3. 10 random changed pieces (synthetic and SMS together)
changed = [t for t in all_raw if MASK_V2[t] != t]
print(f"\n3. {len(changed)} of {len(all_raw)} texts changed (v1: {sum(MASK[t] != t for t in all_raw)}); 10 random:")
for t in np.random.default_rng(1).choice(changed, 10, replace=False):
    print("  BEFORE:", t[:150]); print("  AFTER: ", MASK_V2[t][:150])

# 4. hand-written check
print("\n4. 'Reply C to confirm' unchanged:", HAND_m2["normal"][1] == HAND["normal"][1], "|", HAND_m2["normal"][1])
for g in HAND:
    for a, b in zip(HAND[g], HAND_m2[g]):
        if a != b: print("  changed:", b)

# 5. placeholder counts per source and label
rows = []
for name, d in [("synthetic", pd.concat([syn_tr_m2, syn_te_m2])), ("real SMS", sms_df_m2)]:
    for lab, g in d.groupby("label"):
        r = {"source": name, "label": "scam/spam" if lab else "normal", "texts": len(g)}
        for k, tk in TOK.items(): r[tk] = int(g["dialogue"].str.count(tk).sum())
        rows.append(r)
print("\n5. placeholder counts (occurrences):"); print(pd.DataFrame(rows).to_string(index=False))

# 6. digits, collisions, overlap
left = [t for t in MASK_V2.values() if re.search(r"\d{7,}", t)]
print("\n6. texts with 7+ digits left:", len(left)); assert not left, left[:5]
print(f"   real SMS unique: raw {sms_df['dialogue'].nunique()} -> v2 {sms_df_m2['dialogue'].nunique()}")
print("   seed-42 texts in both train and test: raw", len(set(train['dialogue']) & set(test['dialogue'])),
      "| v1", len(set(train_m['dialogue']) & set(test_m['dialogue'])), "| v2", len(set(train_m2['dialogue']) & set(test_m2['dialogue'])))
print("   to_display example:", to_display(HAND_m2["scam"][1]))

masked 14184 unique texts in 67 s

1. times each word was replaced (raw count minus masked count):
  I'm    v1  236 | v2    0
  What   v1  124 | v2    0
  Hmm    v1  269 | v2    0
  It's   v1   40 | v2    0
  We're  v1   37 | v2    0

2. 30 most frequent masked name strings:
[('John', 1283), ('Johnson', 394), ('David', 259), ('Karen', 216), ('John Smith', 92), ('Rachel', 52), ('Michael', 50), ('Sarah', 21), ('Alex', 20), ('Thompson', 17), ('Michael Johnson', 16), ('Mike', 16), ('Trojan', 14), ('John Doe', 13), ('Yar', 13), ('Jus', 12), ('Officer James', 11), ('Michael Davis', 11), ('Hee', 11), ('Officer', 10), ('Smith', 10), ('James', 8), ('Nah', 8), ('Xmas', 8), ('Pete', 8), ('Kate', 7), ('Mark', 6), ('Jay', 6), ('Voucher', 6), ('Carlos', 6)]

3. 3735 of 14184 texts changed (v1: 4697); 10 random:
  BEFORE: my supervisor is busy with another call. He'll call you back within the hour. receiver: What's your supervisor's name and extension so I can reach him
  AFTER:  my supervisor is bus

Six models on masked v2 data, both ways, then Naive Bayes, Logistic Regression and SVM on seeds 7 and 2026.

In [ ]:
# MASKED RERUN v2, step 3: six models, masked v2, both ways; then NB/LR/SVM seeds 7 and 2026 (raw + v2)
# Same code as run42, with the seed and the text mapper as arguments. Raw seed-42 rows are reused from MASKED_RERUN_ROWS.
def run_v2(name, mk, seed, k, mapper, text_label):
    T = mapper
    tr, te, s_tr, s_te = build_data(seed)
    cut_ham = s_tr[(s_tr.label == 0) & ~s_tr.dialogue.isin(tr.dialogue)]; test_ham = s_te[s_te.label == 0]
    full_s = pd.concat([tr, te], ignore_index=True)
    real_set = set(T(sms_df["dialogue"]))
    make = mk if k == 1 else (lambda: RepeatRealSet(mk(), k, real_set))
    t0 = time.time()
    mod = make().fit(T(tr["dialogue"]), tr["label"].values)
    st, sc, sb = scores(mod, T(te["dialogue"])), scores(mod, T(cut_ham["dialogue"])), scores(mod, T(test_ham["dialogue"]))
    dcut = 0.5 if hasattr(mod, "predict_proba") else 0.0
    cut = np.quantile(sc, 0.95)
    f5, fd = (st > cut).astype(int), (st > dcut).astype(int)
    row = {"model": name, "seed": seed, "text": text_label, "way": "current" if k == 1 else "real x3",
           "real spam caught (5%)": rates(te, f5)["real spam caught"],
           "real normal flagged, big (5%)": float((sb > cut).mean()), "real normal flagged, big (0.5)": float((sb > dcut).mean()),
           "macro-F1 (5%)": f1_score(te["label"], f5, average="macro"), "macro-F1 (0.5)": f1_score(te["label"], fd, average="macro")}
    hp = {g: np.array(mod.predict(T(v))) for g, v in HAND.items()}
    row["hand right of 10"] = int(hp["scam"].sum() + (hp["normal"] == 0).sum() + (hp["normal_but_scam_words"] == 0).sum())
    hide = {}
    for t, lab in HIDE_ORDER:
        keep = full_s[full_s.type != t]
        mm = make().fit(T(keep["dialogue"]), keep["label"].values)
        x = full_s[(full_s.type == t) & (full_s.label == lab)]
        hide[t] = float((np.array(mm.predict(T(x["dialogue"]))) == 1).mean())
    row["new scam type caught (mean 4)"] = float(np.mean([hide[t] for t in SYN_SCAM]))
    row["unseen normal topic flagged (mean 4)"] = float(np.mean([hide[t] for t in SYN_NORMAL]))
    row["no real spam in training: spam caught"] = hide["sms_spam"]
    row["minutes"] = (time.time() - t0) / 60
    return row

def show(r):
    print(f"{r['model']:6} s{r['seed']:<4} {r['text']:9} {r['way']:7} | spam@5% {r['real spam caught (5%)']:.3f} | normal@5% {r['real normal flagged, big (5%)']:.3f} "
          f"| normal@0.5 {r['real normal flagged, big (0.5)']:.3f} | F1@5% {r['macro-F1 (5%)']:.3f} | new scam {r['new scam type caught (mean 4)']:.3f} "
          f"| unseen normal {r['unseen normal topic flagged (mean 4)']:.3f} | no-real spam {r['no real spam in training: spam caught']:.3f} | hand {r['hand right of 10']} | {r['minutes']:.1f} min", flush=True)

raw_T = lambda x: [str(t) for t in x]
MASKED_V2_ROWS = []
print("=== seed 42, six models, masked v2 ===")
for name, mk in SIX.items():
    for k in (1, 3):
        r = run_v2(name, mk, 42, k, m2, "masked v2"); MASKED_V2_ROWS.append(r); show(r)

print("\n=== NB, LR, SVM: seeds 7 and 2026, raw and masked v2 ===")
MK_SEED = {"NB": lambda s: (lambda: make_pipeline(tfidf(), MultinomialNB())),
           "LR": lambda s: (lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=s))),
           "SVM": lambda s: (lambda: make_pipeline(tfidf(), LinearSVC(random_state=s)))}
for seed in (7, 2026):
    for name, f in MK_SEED.items():
        for k in (1, 3):
            for T, lab in ((raw_T, "raw"), (m2, "masked v2")):
                r = run_v2(name, f(seed), seed, k, T, lab); MASKED_V2_ROWS.append(r); show(r)
print("\nDONE")

=== seed 42, six models, masked v2 ===
NB     s42   masked v2 current | spam@5% 0.914 | normal@5% 0.052 | normal@0.5 0.176 | F1@5% 0.968 | new scam 0.876 | unseen normal 0.316 | no-real spam 0.313 | hand 6 | 0.2 min
NB     s42   masked v2 real x3 | spam@5% 0.930 | normal@5% 0.053 | normal@0.5 0.119 | F1@5% 0.973 | new scam 0.846 | unseen normal 0.324 | no-real spam 0.221 | hand 6 | 0.1 min
LR     s42   masked v2 current | spam@5% 0.938 | normal@5% 0.055 | normal@0.5 0.081 | F1@5% 0.969 | new scam 0.855 | unseen normal 0.339 | no-real spam 0.062 | hand 6 | 0.1 min
LR     s42   masked v2 real x3 | spam@5% 0.969 | normal@5% 0.055 | normal@0.5 0.058 | F1@5% 0.972 | new scam 0.851 | unseen normal 0.343 | no-real spam 0.022 | hand 6 | 0.1 min
SVM    s42   masked v2 current | spam@5% 0.945 | normal@5% 0.059 | normal@0.5 0.065 | F1@5% 0.978 | new scam 0.901 | unseen normal 0.343 | no-real spam 0.067 | hand 7 | 0.1 min
SVM    s42   masked v2 real x3 | spam@5% 0.969 | normal@5% 0.062 | normal@0.

Summary tables for the masked v2 runs (no training).

In [ ]:
# MASKED RERUN v2: summary tables (no training)
raw42 = [dict(r, seed=42) for r in MASKED_RERUN_ROWS if r["text"] == "raw"]
v1_42 = [dict(r, seed=42, text="masked v1") for r in MASKED_RERUN_ROWS if r["text"] == "masked"]
allrows = pd.DataFrame(raw42 + v1_42 + MASKED_V2_ROWS)
cols = ["real spam caught (5%)", "real normal flagged, big (5%)", "real normal flagged, big (0.5)", "macro-F1 (5%)",
        "unseen normal topic flagged (mean 4)", "new scam type caught (mean 4)", "no real spam in training: spam caught", "hand right of 10"]
s42 = allrows[allrows.seed == 42]
print("=== seed 42, current way: raw / v1 / v2 ===")
print(s42[s42.way == "current"].pivot_table(index="model", columns="text", values=cols).round(3).T.to_string())
print("\n=== seed 42, real x3: raw / v2 ===")
print(s42[(s42.way == "real x3") & (s42.text != "masked v1")].pivot_table(index="model", columns="text", values=cols).round(3).T.to_string())
three = allrows[allrows.model.isin(["NB", "LR", "SVM"]) & allrows.text.isin(["raw", "masked v2"])]
print("\n=== NB, LR, SVM: mean over 3 seeds (42, 7, 2026), raw vs masked v2 ===")
for way in ("current", "real x3"):
    d = three[three.way == way]
    mean = d.groupby(["model", "text"])[cols].mean(); sd = d.groupby(["model", "text"])[cols].std()
    print(f"\n-- {way} -- (n seeds per cell: {d.groupby(['model','text']).size().unique()})")
    print((mean.round(3).astype(str) + " ± " + sd.round(3).astype(str)).T.to_string())

=== seed 42, current way: raw / v1 / v2 ===
model                                            BiLSTM    CNN     LR     NB    SVM    XGB
                                      text                                                
hand right of 10                      masked v1   6.000  6.000  7.000  7.000  7.000  7.000
                                      masked v2   6.000  6.000  6.000  6.000  7.000  6.000
                                      raw         6.000  6.000  7.000  6.000  7.000  6.000
macro-F1 (5%)                         masked v1   0.950  0.969  0.968  0.969  0.978  0.964
                                      masked v2   0.945  0.972  0.969  0.968  0.978  0.967
                                      raw         0.963  0.965  0.969  0.969  0.979  0.963
new scam type caught (mean 4)         masked v1   0.805  0.891  0.855  0.874  0.900  0.784
                                      masked v2   0.870  0.873  0.855  0.876  0.901  0.789
                                      raw     

DistilBERT and the zero-shot model on masked v2 data. The next cell retries DistilBERT only.

In [ ]:
# MASKED RERUN v2, step 5a: DistilBERT (both ways) and zero-shot, seed 42, raw and masked v2
# Raw is recomputed here too so every column (for example DistilBERT at 0.5) comes from the same harness.
# Same run_v2 harness; zero-shot "fit" does nothing, so its hide-one-type numbers are just its scores per topic.
class DistilBERTFree(DistilBERTClassifier):
    """DistilBERTClassifier unchanged, except it frees the previous copy from the GPU before training a new one."""
    _prev = None
    def fit(self, X, y):
        if DistilBERTFree._prev is not None and hasattr(DistilBERTFree._prev, "model"):
            del DistilBERTFree._prev.model
        gc.collect(); torch.cuda.empty_cache()
        out = super().fit(X, y); DistilBERTFree._prev = self
        return out

class ZeroShotV2:
    def fit(self, X, y): return self
    def predict_proba(self, X):
        p = zs_scores(list(X)); return np.column_stack([1 - p, p])
    def predict(self, X): return (zs_scores(list(X)) > 0.5).astype(int)

print("check: zero-shot raw hand scores (should match Model 8: 0.51 0.83 0.86 0.87 0.92 0.00 0.26 0.00 0.11 0.04):",
      np.round(zs_scores(sum(HAND.values(), [])), 2))
GPU_V2_ROWS = []
for name, mk, ways in [("zero-shot", lambda: ZeroShotV2(), (1,)), ("DistilBERT", lambda: DistilBERTFree(seed=42), (1, 3))]:
    for k in ways:
        for T, lab in ((raw_T, "raw"), (m2, "masked v2")):
            try:
                r = run_v2(name, mk, 42, k, T, lab)
            except Exception as e:
                print(f"ERROR {name} {lab} k={k}: {type(e).__name__}: {e}"); raise
            GPU_V2_ROWS.append(r); show(r)
print("\nDONE")

check: zero-shot raw hand scores (should match Model 8: 0.51 0.83 0.86 0.87 0.92 0.00 0.26 0.00 0.11 0.04): [0.5  0.83 0.86 0.86 0.92 0.   0.26 0.   0.11 0.04]
zero-shot s42   raw       current | spam@5% 0.906 | normal@5% 0.044 | normal@0.5 0.053 | F1@5% 0.688 | new scam 0.490 | unseen normal 0.173 | no-real spam 0.914 | hand 10 | 5.3 min
zero-shot s42   masked v2 current | spam@5% 0.906 | normal@5% 0.045 | normal@0.5 0.059 | F1@5% 0.688 | new scam 0.498 | unseen normal 0.189 | no-real spam 0.922 | hand 10 | 1.9 min
ERROR DistilBERT raw k=1: AttributeError: 'DistilBERTFree' object has no attribute 'predict'


AttributeError: 'DistilBERTFree' object has no attribute 'predict'

In [ ]:
# MASKED RERUN v2, step 5a (retry): DistilBERT only; zero-shot rows are already in GPU_V2_ROWS
# Fix: the DistilBERTClassifier in memory (redefined in the ensemble cell) had no predict(). Model 7's predict was
# "score > 0.5 = scam"; the same rule is added here. Nothing else changes.
DistilBERTFree.predict = lambda self, X: (self.predict_proba(X)[:, 1] > 0.5).astype(int)
print("zero-shot rows kept:", [(r["model"], r["text"]) for r in GPU_V2_ROWS])
for r in GPU_V2_ROWS: show(r)
for k in (1, 3):
    for T, lab in ((raw_T, "raw"), (m2, "masked v2")):
        try:
            r = run_v2("DistilBERT", lambda: DistilBERTFree(seed=42), 42, k, T, lab)
        except Exception as e:
            print(f"ERROR DistilBERT {lab} k={k}: {type(e).__name__}: {e}"); raise
        GPU_V2_ROWS.append(r); show(r)
print("\nDONE")

zero-shot rows kept: [('zero-shot', 'raw'), ('zero-shot', 'masked v2')]
zero-shot s42   raw       current | spam@5% 0.906 | normal@5% 0.044 | normal@0.5 0.053 | F1@5% 0.688 | new scam 0.490 | unseen normal 0.173 | no-real spam 0.914 | hand 10 | 5.3 min
zero-shot s42   masked v2 current | spam@5% 0.906 | normal@5% 0.045 | normal@0.5 0.059 | F1@5% 0.688 | new scam 0.498 | unseen normal 0.189 | no-real spam 0.922 | hand 10 | 1.9 min
DistilBERT s42   raw       current | spam@5% 0.969 | normal@5% 0.045 | normal@0.5 0.043 | F1@5% 0.981 | new scam 0.813 | unseen normal 0.412 | no-real spam 0.103 | hand 6 | 6.3 min
DistilBERT s42   masked v2 current | spam@5% 0.969 | normal@5% 0.058 | normal@0.5 0.050 | F1@5% 0.980 | new scam 0.800 | unseen normal 0.402 | no-real spam 0.097 | hand 6 | 6.3 min
DistilBERT s42   raw       real x3 | spam@5% 0.969 | normal@5% 0.062 | normal@0.5 0.022 | F1@5% 0.975 | new scam 0.823 | unseen normal 0.398 | no-real spam 0.156 | hand 6 | 8.1 min
DistilBERT s42   masked

LoRA and ensembles E1 to E4 on masked v2 data. Each training context is either the main split or one hidden scam type.

In [ ]:
# MASKED RERUN v2, step 5b: LoRA (current data) + ensembles E1-E4, seed 42, masked v2
# One training "context" = the main split or one hidden type. In each context SVM (calibrated, as in the ensembles),
# DistilBERT and LoRA are trained once on masked v2 text; every member's scores are kept, so LoRA and E1-E4 come
# from the same runs. Last context: the raw main split, to fill raw seed-42 numbers missing from the logs.
# Each context prints as it finishes and is saved to /content/masked_v2_scores.json.
import json
tr42, te42, s_tr42, s_te42 = build_data(42)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]; big42 = s_te42[s_te42.label == 0]
full42 = pd.concat([tr42, te42], ignore_index=True)
HAND_LIST = sum(HAND.values(), [])
CONTEXTS = [("main", None, "masked v2")] + [(t, lab, "masked v2") for t, lab in HIDE_ORDER] + [("main", None, "raw")]
MASKED_V2_SCORES = {}
t_all = time.time()
for i, (ctx, lab, textv) in enumerate(CONTEXTS, 1):
    t0 = time.time(); T = m2 if textv == "masked v2" else raw_T
    if ctx == "main":
        trn = tr42; sets = {"te": te42["dialogue"], "cut": cut42["dialogue"], "big": big42["dialogue"], "hand": HAND_LIST}
    else:
        trn = full42[full42.type != ctx]; sets = {"hidden": full42[(full42.type == ctx) & (full42.label == lab)]["dialogue"]}
    X, y = T(trn["dialogue"]), trn["label"].values
    svm = make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=42), cv=3)).fit(X, y)
    bm = DistilBERTFree(seed=42).fit(X, y)
    lm = LoRALLM(seed=42).fit(X, y)
    S = {}
    for sname, texts in sets.items():
        xs = T(texts)
        S[sname] = {"SVM": svm.predict_proba(xs)[:, 1], "DistilBERT": bm.predict_proba(xs)[:, 1],
                    "LoRA": lm.predict_proba(xs)[:, 1], "zero-shot": zs_scores(xs)}
        for en, mem in ENSEMBLES.items():
            S[sname][en] = np.mean([S[sname][x] for x in mem], axis=0)
    key = f"{textv}|{ctx}"
    MASKED_V2_SCORES[key] = S
    json.dump({k: {s: {mm: list(map(float, v)) for mm, v in d.items()} for s, d in vv.items()} for k, vv in MASKED_V2_SCORES.items()},
              open("/content/masked_v2_scores.json", "w"))
    if ctx == "main":
        summ = " | ".join(f"{k} hand {int((S['hand'][k][:5] > 0.5).sum() + (S['hand'][k][5:] <= 0.5).sum())}" for k in S["hand"])
    else:
        summ = " | ".join(f"{k} {float((v > 0.5).mean()):.3f}" for k, v in S["hidden"].items())
    print(f"[{i}/{len(CONTEXTS)}] {textv} {ctx}: {summ} | {(time.time() - t0) / 60:.1f} min, total {(time.time() - t_all) / 60:.0f} min", flush=True)
print("\nDONE all contexts")

[1/11] masked v2 main: SVM hand 7 | DistilBERT hand 6 | LoRA hand 9 | zero-shot hand 10 | E1 SVM+LoRA hand 7 | E2 SVM+LoRA+zero-shot hand 9 | E3 LoRA+zero-shot hand 10 | E4 all four hand 8 | 15.6 min, total 16 min
[2/11] masked v2 refund: SVM 0.855 | DistilBERT 0.782 | LoRA 0.822 | zero-shot 0.445 | E1 SVM+LoRA 0.840 | E2 SVM+LoRA+zero-shot 0.832 | E3 LoRA+zero-shot 0.775 | E4 all four 0.830 | 14.3 min, total 30 min
[3/11] masked v2 reward: SVM 0.882 | DistilBERT 0.515 | LoRA 0.701 | zero-shot 0.533 | E1 SVM+LoRA 0.809 | E2 SVM+LoRA+zero-shot 0.800 | E3 LoRA+zero-shot 0.696 | E4 all four 0.719 | 14.4 min, total 44 min
[4/11] masked v2 ssn: SVM 0.976 | DistilBERT 0.983 | LoRA 0.980 | zero-shot 0.544 | E1 SVM+LoRA 0.990 | E2 SVM+LoRA+zero-shot 0.977 | E3 LoRA+zero-shot 0.946 | E4 all four 0.986 | 14.3 min, total 59 min


In [ ]:
# Fix: load the base model ONCE, and for each run add a fresh LoRA adapter, train it, then remove it (unload).
# Needs only the GPU RESTART SETUP cell above.
!pip -q install peft
!pip -q uninstall -y torchao
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc, psutil
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); disable_progress_bars()
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

BASE = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
BASE.gradient_checkpointing_enable(); BASE.enable_input_require_grads()

class LoRALLM:
    _active = None
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4):
        self.seed, self.r, self.epochs, self.batch, self.lr = seed, r, epochs, batch, lr
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()
    def fit(self, X, y):
        global BASE
        if LoRALLM._active is not None:                     # remove the previous adapter, keep the base
            BASE = LoRALLM._active.model.unload(); LoRALLM._active.model = None
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(BASE, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        self.model.eval(); LoRALLM._active = self
        return self
    def predict(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return (out > 0.5).astype(int)


# Model 9 (LoRA): 3 seeds + big false-alarm test (same as seed_eval, without the hide-one-type part)
# Hide-one-type would add 27 training runs (about 5.5 hours), so those stay single-seed for LoRA.
# Each seed: new SMS split, new balancing sample, new LoRA seed; cut-off chosen on ~3,100 leftover real normal TRAIN SMS.
def _lora_pp(self, X, batch=32):
    texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(texts), batch):
            ids = order[b:b + batch]
            enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
            out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
    return np.column_stack([1 - out, out])
LoRALLM.predict_proba = _lora_pp


# Model 10: ensembles (average of member scores)
# Members, retrained on each seed's data: SVM (calibrated to give probabilities), DistilBERT, LoRA LLM, and the
# zero-shot LLM (same Qwen weights with the LoRA adapter switched off, so no second copy in memory).
# Each ensemble score = plain average of its members' 0-to-1 scores. Same harness: 3 seeds, big false-alarm test, 5% cut-offs.
# Needs: GPU RESTART SETUP cell + the LoRA cell above (LoRALLM, BASE, build, llm_tok) in this runtime.
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])

ZS_CACHE = {}
def zs_scores(texts, batch=32):
    """Zero-shot scores from the base Qwen model (LoRA adapter switched off). Cached, so each text is scored once."""
    texts = [str(t) for t in texts]; todo = sorted(set(t for t in texts if t not in ZS_CACHE), key=len)
    act = LoRALLM._active
    with act.model.disable_adapter(), torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(todo), batch):
            chunk = todo[b:b + batch]
            enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
            ZS_CACHE.update(zip(chunk, torch.softmax(act._last_logits(enc), dim=1)[:, 1].cpu().numpy()))
    return np.array([ZS_CACHE[t] for t in texts])

ENSEMBLES = {"E1 SVM+LoRA": ["SVM", "LoRA"], "E2 SVM+LoRA+zero-shot": ["SVM", "LoRA", "zero-shot"],
             "E3 LoRA+zero-shot": ["LoRA", "zero-shot"], "E4 all four": ["SVM", "DistilBERT", "LoRA", "zero-shot"]}



# MASKED RERUN v2, step 5b RESUME (runtime was reset; needs GPU RESTART SETUP + masking_model cells first)
from sklearn.calibration import CalibratedClassifierCV
t0 = time.time()
all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
raw_T = lambda x: [str(t) for t in x]
print(f"re-masked {len(MASK_V2)} texts in {time.time() - t0:.0f} s; 'Reply C' unchanged:", m2(HAND["normal"])[1] == HAND["normal"][1])

class DistilBERTFree(DistilBERTClassifier):
    _prev = None
    def fit(self, X, y):
        if DistilBERTFree._prev is not None and hasattr(DistilBERTFree._prev, "model"):
            del DistilBERTFree._prev.model
        gc.collect(); torch.cuda.empty_cache()
        out = super().fit(X, y); DistilBERTFree._prev = self
        return out
    def predict(self, X): return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

NAMES = ["SVM", "DistilBERT", "LoRA", "zero-shot"] + list(ENSEMBLES)
HIDDEN_V2 = {   # masked v2, copied from the printed output of the stopped run
    "refund": dict(zip(NAMES, [0.855, 0.782, 0.822, 0.445, 0.840, 0.832, 0.775, 0.830])),
    "reward": dict(zip(NAMES, [0.882, 0.515, 0.701, 0.533, 0.809, 0.800, 0.696, 0.719])),
    "ssn":    dict(zip(NAMES, [0.976, 0.983, 0.980, 0.544, 0.990, 0.977, 0.946, 0.986])),
}
MAIN_ROWS_V2 = {}
tr42, te42, s_tr42, s_te42 = build_data(42)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]; big42 = s_te42[s_te42.label == 0]
full42 = pd.concat([tr42, te42], ignore_index=True)
HAND_LIST = sum(HAND.values(), [])
CONTEXTS = [("main", None, "masked v2"), ("support", 1, "masked v2"), ("delivery", 0, "masked v2"), ("insurance", 0, "masked v2"),
            ("telemarketing", 0, "masked v2"), ("wrong", 0, "masked v2"), ("sms_spam", 1, "masked v2"), ("main", None, "raw")]
t_all = time.time()
for i, (ctx, lab, textv) in enumerate(CONTEXTS, 1):
    t0 = time.time(); T = m2 if textv == "masked v2" else raw_T
    trn = tr42 if ctx == "main" else full42[full42.type != ctx]
    X, y = T(trn["dialogue"]), trn["label"].values
    svm = make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=42), cv=3)).fit(X, y)
    bm = DistilBERTFree(seed=42).fit(X, y)
    lm = LoRALLM(seed=42).fit(X, y)
    def score_all(texts):
        xs = T(texts)
        S = {"SVM": svm.predict_proba(xs)[:, 1], "DistilBERT": bm.predict_proba(xs)[:, 1],
             "LoRA": lm.predict_proba(xs)[:, 1], "zero-shot": zs_scores(xs)}
        for en, mem in ENSEMBLES.items(): S[en] = np.mean([S[x] for x in mem], axis=0)
        return S
    if ctx == "main":
        St, Sc, Sb, Sh = score_all(te42["dialogue"]), score_all(cut42["dialogue"]), score_all(big42["dialogue"]), score_all(HAND_LIST)
        rows = {}
        for k in NAMES:
            cut = np.quantile(Sc[k], 0.95); f5, fd = (St[k] > cut).astype(int), (St[k] > 0.5).astype(int)
            rows[k] = {"real spam caught (5%)": rates(te42, f5)["real spam caught"],
                       "real normal flagged, big (5%)": float((Sb[k] > cut).mean()), "real normal flagged, big (0.5)": float((Sb[k] > 0.5).mean()),
                       "macro-F1 (5%)": f1_score(te42["label"], f5, average="macro"), "macro-F1 (0.5)": f1_score(te42["label"], fd, average="macro"),
                       "hand right of 10": int((Sh[k][:5] > 0.5).sum() + (Sh[k][5:] <= 0.5).sum())}
        MAIN_ROWS_V2[textv] = rows
        print(f"[{i}/{len(CONTEXTS)}] {textv} main ({(time.time() - t0) / 60:.1f} min):")
        print(pd.DataFrame(rows).round(3).to_string(), flush=True)
    else:
        x = full42[(full42.type == ctx) & (full42.label == lab)]["dialogue"]
        S = score_all(x)
        HIDDEN_V2[ctx] = {k: float((v > 0.5).mean()) for k, v in S.items()}
        print(f"[{i}/{len(CONTEXTS)}] {textv} {ctx}: " + " | ".join(f"{k} {v:.3f}" for k, v in HIDDEN_V2[ctx].items())
              + f" | {(time.time() - t0) / 60:.1f} min, total {(time.time() - t_all) / 60:.0f} min", flush=True)

H = pd.DataFrame(HIDDEN_V2).T
print("\n===== masked v2, hide-one-type summary (seed 42) =====")
print(pd.DataFrame({"new scam type caught (mean 4)": H.loc[SYN_SCAM, NAMES].mean(),
                    "unseen normal topic flagged (mean 4)": H.loc[SYN_NORMAL, NAMES].mean(),
                    "no real spam in training: spam caught": H.loc["sms_spam", NAMES]}).T.round(3).to_string())
print("\nDONE all contexts")

re-masked 14184 texts in 67 s; 'Reply C' unchanged: True
[1/8] masked v2 main (14.0 min):
                                  SVM  DistilBERT   LoRA  zero-shot  E1 SVM+LoRA  E2 SVM+LoRA+zero-shot  E3 LoRA+zero-shot  E4 all four
real spam caught (5%)           0.945       0.969  0.945      0.906        0.977                  0.961              0.953        0.977
real normal flagged, big (5%)   0.054       0.058  0.054      0.045        0.055                  0.049              0.049        0.046
real normal flagged, big (0.5)  0.086       0.050  0.064      0.059        0.043                  0.028              0.042        0.023
macro-F1 (5%)                   0.978       0.980  0.970      0.687        0.981                  0.975              0.954        0.982
macro-F1 (0.5)                  0.973       0.981  0.972      0.690        0.981                  0.978              0.952        0.984
hand right of 10                7.000       6.000  9.000     10.000        7.000              

# 7. Website handoff
Calibrated SVM saved as svm_masked_seed42.joblib with cutoff.json (cut-off 0.615383).

Calibrated SVM trained on masked v2 data and saved for the website. It was later replaced by the plain LinearSVC (section 12).

In [ ]:
# Website handoff: calibrated SVM (TF-IDF + CalibratedClassifierCV(LinearSVC)), masked v2, seed 42
all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
tr42, te42, s_tr42, s_te42 = build_data(42)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]; big42 = s_te42[s_te42.label == 0]
HAND_LIST = sum(HAND.values(), [])

import sys, os, json, joblib, numpy as np, sklearn, spacy
from sklearn.pipeline import make_pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
svm_web = make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=42), cv=3))
svm_web.fit(m2(tr42['dialogue']), tr42['label'].values)
print('trained on', len(tr42), 'masked v2 training pieces')
s_cut = svm_web.predict_proba(m2(cut42['dialogue']))[:, 1]
cutoff = float(np.quantile(s_cut, 0.95))
print('cut-off (95th pct of', len(s_cut), 'leftover real normal training SMS):', round(cutoff, 6))
joblib.dump(svm_web, 'svm_masked_seed42.joblib')
info = {'cutoff': cutoff, 'rule': 'scam if predict_proba(mask_texts([text]))[:,1] > cutoff',
        'budget': '5% false alarms on leftover real normal training SMS', 'seed': 42, 'masking': 'masking_model v2',
        'python': sys.version.split()[0], 'scikit-learn': sklearn.__version__, 'spacy': spacy.__version__}
try:
    import en_core_web_sm; info['en_core_web_sm'] = en_core_web_sm.__version__
except Exception:
    info['en_core_web_sm'] = spacy.load('en_core_web_sm').meta['version']
json.dump(info, open('cutoff.json', 'w'), indent=2)
back = joblib.load('svm_masked_seed42.joblib')
s_te = back.predict_proba(m2(te42['dialogue']))[:, 1]
s_big = back.predict_proba(m2(big42['dialogue']))[:, 1]
print('reloaded scores identical:', np.allclose(back.predict_proba(m2(cut42['dialogue']))[:, 1], s_cut))
print('big test normal flagged at cut-off:', round(float((s_big > cutoff).mean()), 3), '(main run said .054)')
print('macro-F1 at cut-off:', round(f1_score(te42['label'].values, (s_te > cutoff).astype(int), average='macro'), 3), '(main run said .978)')
print('hand messages:', [round(float(p), 2) for p in back.predict_proba(m2(HAND_LIST))[:, 1]])
print(json.dumps(info, indent=2))
print({f: os.path.getsize(f) for f in ['svm_masked_seed42.joblib', 'cutoff.json']})
from google.colab import files
files.download('svm_masked_seed42.joblib'); files.download('cutoff.json')
print('DONE handoff')

trained on 6844 masked v2 training pieces
cut-off (95th pct of 3100 leftover real normal training SMS): 0.615383
reloaded scores identical: True
big test normal flagged at cut-off: 0.054 (main run said .054)
macro-F1 at cut-off: 0.978 (main run said .978)
hand messages: [0.79, 0.93, 0.97, 0.93, 0.63, 0.17, 0.72, 0.02, 0.9, 0.99]
{
  "cutoff": 0.6153828434070271,
  "rule": "scam if predict_proba(mask_texts([text]))[:,1] > cutoff",
  "budget": "5% false alarms on leftover real normal training SMS",
  "seed": 42,
  "masking": "masking_model v2",
  "python": "3.13.15",
  "scikit-learn": "1.6.1",
  "spacy": "3.8.16",
  "en_core_web_sm": "3.8.0"
}
{'svm_masked_seed42.joblib': 898481, 'cutoff.json': 318}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DONE handoff


# 8. Extra checks
Calibration SMS vs training overlap, Isolation Forest and One-Class SVM, and the near-copy check of the real spam test (SVM and LoRA).

Do the calibration SMS, which set the cut-off, look like the training and test SMS? Checked on all 3 seeds.

In [ ]:
# Check: calibration SMS (cut-off set) vs training, test and big test, all 3 seeds
# needs the GPU RESTART SETUP and masking_model cells first
all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]

for seed in [42, 7, 2026]:
    tr, te, s_tr, s_te = build_data(seed)
    cut = s_tr[(s_tr.label == 0) & ~s_tr.dialogue.isin(tr.dialogue)]; big = s_te[s_te.label == 0]
    R = lambda d: set(d['dialogue'].astype(str)); M = lambda d: set(m2(d['dialogue']))
    tr_sms_normal = tr[tr.dialogue.isin(s_tr.dialogue) & (tr.label == 0)]
    print(f'seed {seed}: calibration SMS {len(cut)} (unique {len(R(cut))}) | normal real SMS in training {len(tr_sms_normal)}')
    print(f'   calibration vs training  raw {len(R(cut) & R(tr))}  masked {len(M(cut) & M(tr))}')
    print(f'   calibration vs test      raw {len(R(cut) & R(te))}  masked {len(M(cut) & M(te))}')
    print(f'   calibration vs big test  raw {len(R(cut) & R(big))}  masked {len(M(cut) & M(big))}')
    print(f'   big test vs training     raw {len(R(big) & R(tr))}  masked {len(M(big) & M(tr))}')
    print(f'   SMS train pool vs SMS test pool raw {len(R(s_tr) & R(s_te))}')
    if seed == 42:
        ex = sorted(M(cut) & M(tr))[:5]; print('   masked collisions, examples:', ex)
print('DONE check')

seed 42: calibration SMS 3100 (unique 3100) | normal real SMS in training 514
   calibration vs training  raw 0  masked 0
   calibration vs test      raw 0  masked 0
   calibration vs big test  raw 0  masked 0
   big test vs training     raw 0  masked 0
   SMS train pool vs SMS test pool raw 0
   masked collisions, examples: []
seed 7: calibration SMS 3100 (unique 3100) | normal real SMS in training 514
   calibration vs training  raw 0  masked 0
   calibration vs test      raw 0  masked 0
   calibration vs big test  raw 0  masked 0
   big test vs training     raw 0  masked 0
   SMS train pool vs SMS test pool raw 0
seed 2026: calibration SMS 3100 (unique 3100) | normal real SMS in training 514
   calibration vs training  raw 0  masked 0
   calibration vs test      raw 0  masked 0
   calibration vs big test  raw 0  masked 0
   big test vs training     raw 0  masked 0
   SMS train pool vs SMS test pool raw 0
DONE check


### One-class models
Isolation Forest and One-Class SVM learn only from normal messages and flag anything unusual. They need no scam labels.

In [ ]:
# One-class models (Isolation Forest, One-Class SVM), masked v2, seed 42
import os, json, time, numpy as np, pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import f1_score, roc_auc_score
t_start = time.time()
if "MASK_V2" not in globals():
    all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
    MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
tr42, te42, s_tr42, s_te42 = build_data(42)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]; big42 = s_te42[s_te42.label == 0]
full42 = pd.concat([tr42.assign(split="train"), te42.assign(split="test")], ignore_index=True)
HAND_LIST = sum(HAND.values(), [])
HAND_Y = np.array([1] * len(HAND["scam"]) + [0] * (len(HAND["normal"]) + len(HAND["normal_but_scam_words"])))
SCAM_T = sorted(full42.loc[(full42.source == "synthetic") & (full42.label == 1), "type"].unique())
NORMAL_T = sorted(full42.loc[(full42.source == "synthetic") & (full42.label == 0), "type"].unique())
OUT = "oneclass_seed42"; os.makedirs(OUT, exist_ok=True)

# ---- settings, fixed once before any result was seen (no tuning on any test set) ----
SET = dict(features="masked v2 text -> tfidf() (word 1-2 grams, min_df=2, same as supervised) fitted on normal training pieces only -> TruncatedSVD -> L2 normalise",
           svd_components=200, ocsvm_nu=0.05, ocsvm_kernel="rbf", ocsvm_gamma="scale", if_trees=300, if_max_samples="auto (256)", random_state=42,
           score="IF and OCSVM: minus score_samples, so higher = more unusual = more scam-like",
           cutoff="95th percentile of scores on the 3,100 unseen real normal training SMS (cut42)")
for k, v in SET.items(): print(f"{k}: {v}")
print("why: 200 SVD dims is a common LSA size for a few thousand short texts; nu=0.05 matches the 5% budget; 300 trees is above the default 100 for stabler scores")
print("scam topics:", SCAM_T, "| normal topics:", NORMAL_T)

# ---- data counts ----
norm_tr = tr42[tr42.label == 0]
print("\nnormal training pieces by source:", norm_tr.source.value_counts().to_dict(), "| total", len(norm_tr))
print("normal training pieces by type:", norm_tr.type.value_counts().to_dict())
print("calibration SMS:", len(cut42), "| big test real normal SMS:", len(big42))
print("\npieces per topic (seed 42, train and test splits):")
print(pd.crosstab([full42.source, full42.type, full42.label], full42.split, margins=True).to_string())

class OneClassText:
    def __init__(self, kind): self.kind = kind
    def fit(self, texts):
        self.vec = tfidf(); X = self.vec.fit_transform(texts)
        self.svd = TruncatedSVD(SET["svd_components"], random_state=42); Z = self.svd.fit_transform(X)
        self.nrm = Normalizer(); Z = self.nrm.fit_transform(Z)
        self.m = (IsolationForest(n_estimators=SET["if_trees"], random_state=42, n_jobs=-1) if self.kind == "IsolationForest"
                  else OneClassSVM(kernel="rbf", gamma="scale", nu=SET["ocsvm_nu"]))
        self.m.fit(Z)
        self.info = dict(n_train=len(texts), vocab=len(self.vec.vocabulary_), svd_var_kept=round(float(self.svd.explained_variance_ratio_.sum()), 3))
        return self
    def score(self, texts):
        X = self.vec.transform(texts)
        Z = self.nrm.transform(self.svd.transform(X))
        self.last_empty = int((X.getnnz(axis=1) == 0).sum())   # texts with no known word
        return -self.m.score_samples(Z)

REC = []   # per-message scores
def keep(model, context, setname, data, s, cut):
    if isinstance(data, pd.DataFrame):
        REC.extend(dict(model=model, context=context, set=setname, source=a, type=b, label=int(c), text_masked=t, score=float(x), cutoff=cut, flagged=int(x > cut))
                   for a, b, c, t, x in zip(data.source, data.type, data.label, m2(data.dialogue), s))
    else:
        kinds = sum([[k] * len(v) for k, v in HAND.items()], [])
        REC.extend(dict(model=model, context=context, set=setname, source="hand", type=k, label=int(y), text_masked=t, score=float(x), cutoff=cut, flagged=int(x > cut))
                   for k, y, t, x in zip(kinds, HAND_Y, m2(data), s))

def share(fl): return f"{int(fl.sum())}/{len(fl)} = {fl.mean():.3f}"
def trip(fl): return [int(fl.sum()), len(fl), round(float(fl.mean()), 4)]
SUMMARY = {}
for kind in ["IsolationForest", "OneClassSVM"]:
    t0 = time.time(); S = SUMMARY[kind] = {}
    mdl = OneClassText(kind).fit(m2(norm_tr.dialogue))
    print(f"\n========== {kind} (main: all {len(norm_tr)} normal training pieces) ==========")
    print("fit info:", mdl.info)
    sc = mdl.score(m2(cut42.dialogue)); e_c = mdl.last_empty; cut = float(np.quantile(sc, 0.95))
    st = mdl.score(m2(te42.dialogue)); e_t = mdl.last_empty
    sb = mdl.score(m2(big42.dialogue)); e_b = mdl.last_empty
    sh = mdl.score(m2(HAND_LIST))
    print(f"scores on calibration SMS: min {sc.min():.4f} median {np.median(sc):.4f} max {sc.max():.4f} | cut-off {cut:.4f}")
    print(f"texts with no known word: calibration {e_c}, test {e_t}, big test {e_b}")
    f_te = st > cut
    def g(src, lab): return f_te[((te42.source == src) & (te42.label == lab)).values]
    S["cutoff"] = cut; S["fit"] = mdl.info
    for name, fl in [("calibration SMS flagged (should be 5%)", sc > cut), ("big test real normal SMS flagged", sb > cut),
                     ("real spam test caught (real spam SMS in te42)", g("real", 1)), ("real normal SMS in te42 flagged", g("real", 0)),
                     ("dialogue scam test caught (BothBosu test-split scam pieces in te42)", g("synthetic", 1)),
                     ("dialogue normal test flagged (BothBosu test-split normal pieces in te42)", g("synthetic", 0))]:
        print(f"  {name}: {share(fl)}"); S[name] = trip(fl)
    S["macro-F1 on te42 (5% cut-off)"] = round(f1_score(te42.label, f_te.astype(int), average="macro"), 4)
    S["AUC on te42 (no cut-off)"] = round(roc_auc_score(te42.label, st), 4)
    S["hand right of 10"] = int(((sh > cut).astype(int) == HAND_Y).sum())
    print("  macro-F1 te42:", S["macro-F1 on te42 (5% cut-off)"], "| AUC te42:", S["AUC on te42 (no cut-off)"], "| hand right of 10:", S["hand right of 10"])
    print("  hand scores:", [round(float(x), 3) for x in sh], "flagged:", (sh > cut).astype(int).tolist())
    keep(kind, "main", "calibration_cut42", cut42, sc, cut); keep(kind, "main", "test_te42", te42, st, cut)
    keep(kind, "main", "bigtest_big42", big42, sb, cut); keep(kind, "main", "hand", HAND_LIST, sh, cut)
    # hidden scam topics: training has no scam at all, so the main model is reused; all pieces of the topic (train + test split)
    for t in SCAM_T:
        d = full42[(full42.type == t) & (full42.label == 1)]; s = mdl.score(m2(d.dialogue)); fl = s > cut
        print(f"  hidden scam topic {t}: caught {share(fl)}"); S[f"hidden scam {t} caught"] = trip(fl)
        keep(kind, "main", f"hidden_scam_{t}", d, s, cut)
    # hidden normal topics: retrain without the topic, new cut-off on the same 3,100 SMS, score all pieces of the topic
    for t in NORMAL_T:
        trn = norm_tr[norm_tr.type != t]; mh = OneClassText(kind).fit(m2(trn.dialogue))
        ch = float(np.quantile(mh.score(m2(cut42.dialogue)), 0.95))
        d = full42[(full42.type == t) & (full42.label == 0)]; s = mh.score(m2(d.dialogue)); fl = s > ch
        seen = st[((te42.type == t) & (te42.label == 0)).values] > cut
        print(f"  hidden normal topic {t}: trained on {len(trn)}, cut-off {ch:.4f}, flagged {share(fl)} | same topic when seen (main model, te42 pieces): {share(seen)}")
        S[f"hidden normal {t} flagged"] = trip(fl); S[f"seen normal {t} flagged (te42)"] = trip(seen)
        keep(kind, f"without {t}", f"hidden_normal_{t}", d, s, ch)
    # no real SMS in training: dialogue normal pieces only; scored on all real SMS rows (train + test split)
    trn = norm_tr[norm_tr.source == "synthetic"]; mn = OneClassText(kind).fit(m2(trn.dialogue))
    d = full42[full42.source == "real"]; s = mn.score(m2(d.dialogue))
    c_sms = float(np.quantile(mn.score(m2(cut42.dialogue)), 0.95)); c_dia = float(np.quantile(mn.score(m2(trn.dialogue)), 0.95))
    for cname, c in [("cut-off from 3,100 real normal SMS", c_sms), ("cut-off from dialogue normal training pieces, no real SMS used", c_dia)]:
        fs, fn = s[(d.label == 1).values] > c, s[(d.label == 0).values] > c
        print(f"  no real SMS in training [{cname}, {c:.4f}]: real spam caught {share(fs)}, real normal flagged {share(fn)}")
        S[f"no real SMS: real spam caught [{cname}]"] = trip(fs); S[f"no real SMS: real normal flagged [{cname}]"] = trip(fn)
    keep(kind, "no real SMS (cut-off from 3,100 SMS)", "no_real_sms_all_real_rows", d, s, c_sms)
    S["minutes"] = round((time.time() - t0) / 60, 1); print(f"  {kind} done in {S['minutes']} min")

pd.DataFrame(REC).to_csv(f"{OUT}/scores_oneclass_seed42.csv", index=False)
json.dump(dict(settings=SET, summary=SUMMARY), open(f"{OUT}/summary_oneclass_seed42.json", "w"), indent=2)
print("\nsaved", len(REC), "score rows to", f"{OUT}/scores_oneclass_seed42.csv", "| total", round((time.time() - t_start) / 60, 1), "min")
print("DONE one-class")

features: masked v2 text -> tfidf() (word 1-2 grams, min_df=2, same as supervised) fitted on normal training pieces only -> TruncatedSVD -> L2 normalise
svd_components: 200
ocsvm_nu: 0.05
ocsvm_kernel: rbf
ocsvm_gamma: scale
if_trees: 300
if_max_samples: auto (256)
random_state: 42
score: IF and OCSVM: minus score_samples, so higher = more unusual = more scam-like
cutoff: 95th percentile of scores on the 3,100 unseen real normal training SMS (cut42)
why: 200 SVD dims is a common LSA size for a few thousand short texts; nu=0.05 matches the 5% budget; 300 trees is above the default 100 for stabler scores
scam topics: ['refund', 'reward', 'ssn', 'support'] | normal topics: ['delivery', 'insurance', 'telemarketing', 'wrong']

normal training pieces by source: {'synthetic': 2908, 'real': 514} | total 3422
normal training pieces by type: {'insurance': 967, 'telemarketing': 934, 'delivery': 656, 'sms_ham': 514, 'wrong': 351}
calibration SMS: 3100 | big test real normal SMS: 904

pieces per to

**Diagnostic:** real spam test messages that nearly copy a training spam message, and how much they affect the results.

In [ ]:
# Separate check (no model changed): real spam test messages that nearly copy a training spam message
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
rs_te = te42[(te42.source == "real") & (te42.label == 1)]
sp_real = tr42[(tr42.source == "real") & (tr42.label == 1)]; sp_all = tr42[tr42.label == 1]
print("real spam test messages:", len(rs_te), "| training spam: real SMS", len(sp_real), ", all (incl. dialogue scam pieces)", len(sp_all))
def max_sim(T):
    vec = tfidf().fit(T(tr42["dialogue"]))      # same TF-IDF settings and training text as the SVM; rows are L2-normalised, so dot = cosine
    A = vec.transform(T(rs_te["dialogue"]))
    r = (A @ vec.transform(T(sp_real["dialogue"])).T).max(axis=1).toarray().ravel()
    a = (A @ vec.transform(T(sp_all["dialogue"])).T).max(axis=1).toarray().ravel()
    return r, a
sim_r, sim_a = max_sim(m2)
sim_r_raw, _ = max_sim(lambda x: [str(t) for t in x])
print("same answer against real training spam vs all training spam:", bool(np.allclose(sim_r, sim_a)))
bins = [0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0001]
print("highest cosine to a training spam SMS (masked v2), counts per band:", pd.cut(sim_r, bins, right=False).value_counts().sort_index().to_dict())
near = sim_r > 0.9
print(f"above 0.9: masked v2 {near.sum()}/{len(near)} = {near.mean():.3f} | raw text {int((sim_r_raw > 0.9).sum())}/{len(near)} | exactly 1.0 (masked): {int((sim_r > 0.9999).sum())}")
svm_chk = make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=42), cv=3)).fit(m2(tr42["dialogue"]), tr42["label"].values)
cut_svm = float(np.quantile(svm_chk.predict_proba(m2(cut42["dialogue"]))[:, 1], 0.95))
s_svm = svm_chk.predict_proba(m2(rs_te["dialogue"]))[:, 1]; f_svm = s_svm > cut_svm
print(f"calibrated SVM (same code as the website model): cut-off {cut_svm:.6f} (website cutoff.json 0.615383)")
for nm, m in [("all", np.ones(len(near), bool)), ("near-copies (>0.9)", near), ("remaining (<=0.9)", ~near)]:
    print(f"  SVM real spam caught, {nm}: {int(f_svm[m].sum())}/{int(m.sum())} = {f_svm[m].mean():.3f}")
SIM = rs_te[["dialogue"]].assign(text_masked=m2(rs_te["dialogue"]), max_cos_real_train_spam=sim_r, max_cos_all_train_spam=sim_a,
                                 max_cos_raw=sim_r_raw, near_copy=near.astype(int), svm_score=s_svm, svm_cutoff=cut_svm, svm_flagged=f_svm.astype(int))
SIM.drop(columns="dialogue").to_csv(f"{OUT}/near_copy_check_seed42.csv", index=False)
# diagnostic only (nothing tuned): one-class AUC inside each source, from the saved te42 scores
R = pd.DataFrame(REC); R = R[(R.context == "main") & (R.set == "test_te42")]
for (mname, src), d in R.groupby(["model", "source"]):
    print(f"  {mname} AUC inside {src} test pieces only: {roc_auc_score(d.label, d.score):.3f}  (scam mean score {d[d.label==1].score.mean():.4f}, normal mean {d[d.label==0].score.mean():.4f})")
print("DONE near-copy check (SVM)")

real spam test messages: 128 | training spam: real SMS 514 , all (incl. dialogue scam pieces) 3422
same answer against real training spam vs all training spam: False
highest cosine to a training spam SMS (masked v2), counts per band: {Interval(0.0, 0.5, closed='left'): 51, Interval(0.5, 0.7, closed='left'): 17, Interval(0.7, 0.8, closed='left'): 8, Interval(0.8, 0.9, closed='left'): 15, Interval(0.9, 0.95, closed='left'): 10, Interval(0.95, 1.0, closed='left'): 27}
above 0.9: masked v2 37/128 = 0.289 | raw text 41/128 | exactly 1.0 (masked): 12
calibrated SVM (same code as the website model): cut-off 0.615383 (website cutoff.json 0.615383)
  SVM real spam caught, all: 121/128 = 0.945
  SVM real spam caught, near-copies (>0.9): 37/37 = 1.000
  SVM real spam caught, remaining (<=0.9): 84/91 = 0.923
  IsolationForest AUC inside real test pieces only: 0.383  (scam mean score 0.4341, normal mean 0.4364)
  IsolationForest AUC inside synthetic test pieces only: 0.716  (scam mean score 0.4220,

In [ ]:
# Fix: load the base model ONCE, and for each run add a fresh LoRA adapter, train it, then remove it (unload).
# Needs only the GPU RESTART SETUP cell above.
!pip -q install peft
!pip -q uninstall -y torchao
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc, psutil
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); disable_progress_bars()
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

BASE = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
BASE.gradient_checkpointing_enable(); BASE.enable_input_require_grads()

class LoRALLM:
    _active = None
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4):
        self.seed, self.r, self.epochs, self.batch, self.lr = seed, r, epochs, batch, lr
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()
    def fit(self, X, y):
        global BASE
        if LoRALLM._active is not None:                     # remove the previous adapter, keep the base
            BASE = LoRALLM._active.model.unload(); LoRALLM._active.model = None
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(BASE, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        self.model.eval(); LoRALLM._active = self
        return self
    def predict(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return (out > 0.5).astype(int)


# Model 9 (LoRA): 3 seeds + big false-alarm test (same as seed_eval, without the hide-one-type part)
# Hide-one-type would add 27 training runs (about 5.5 hours), so those stay single-seed for LoRA.
# Each seed: new SMS split, new balancing sample, new LoRA seed; cut-off chosen on ~3,100 leftover real normal TRAIN SMS.
def _lora_pp(self, X, batch=32):
    texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(texts), batch):
            ids = order[b:b + batch]
            enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
            out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
    return np.column_stack([1 - out, out])
LoRALLM.predict_proba = _lora_pp


# Model 10: ensembles (average of member scores)
# Members, retrained on each seed's data: SVM (calibrated to give probabilities), DistilBERT, LoRA LLM, and the
# zero-shot LLM (same Qwen weights with the LoRA adapter switched off, so no second copy in memory).
# Each ensemble score = plain average of its members' 0-to-1 scores. Same harness: 3 seeds, big false-alarm test, 5% cut-offs.
# Needs: GPU RESTART SETUP cell + the LoRA cell above (LoRALLM, BASE, build, llm_tok) in this runtime.
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])

ZS_CACHE = {}
def zs_scores(texts, batch=32):
    """Zero-shot scores from the base Qwen model (LoRA adapter switched off). Cached, so each text is scored once."""
    texts = [str(t) for t in texts]; todo = sorted(set(t for t in texts if t not in ZS_CACHE), key=len)
    act = LoRALLM._active
    with act.model.disable_adapter(), torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(todo), batch):
            chunk = todo[b:b + batch]
            enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
            ZS_CACHE.update(zip(chunk, torch.softmax(act._last_logits(enc), dim=1)[:, 1].cpu().numpy()))
    return np.array([ZS_CACHE[t] for t in texts])

ENSEMBLES = {"E1 SVM+LoRA": ["SVM", "LoRA"], "E2 SVM+LoRA+zero-shot": ["SVM", "LoRA", "zero-shot"],
             "E3 LoRA+zero-shot": ["LoRA", "zero-shot"], "E4 all four": ["SVM", "DistilBERT", "LoRA", "zero-shot"]}



lm_chk = LoRALLM(seed=42).fit(m2(tr42["dialogue"]), tr42["label"].values)
cut_lora = float(np.quantile(lm_chk.predict_proba(m2(cut42["dialogue"]))[:, 1], 0.95))
s_lora_te = lm_chk.predict_proba(m2(te42["dialogue"]))[:, 1]
s_lora = s_lora_te[((te42.source == "real") & (te42.label == 1)).values]; f_lora = s_lora > cut_lora
print(f"LoRA cut-off {cut_lora:.4f} | real spam caught overall {f_lora.mean():.3f} (logged masked v2 seed-42 run: 0.945)")
for nm, m in [("all", np.ones(len(near), bool)), ("near-copies (>0.9)", near), ("remaining (<=0.9)", ~near)]:
    print(f"  LoRA real spam caught, {nm}: {int(f_lora[m].sum())}/{int(m.sum())} = {f_lora[m].mean():.3f}")
SIM = SIM.assign(lora_score=s_lora, lora_cutoff=cut_lora, lora_flagged=f_lora.astype(int))
SIM.drop(columns="dialogue").to_csv(f"{OUT}/near_copy_check_seed42.csv", index=False)
te42.assign(text_masked=m2(te42["dialogue"]), lora_score=s_lora_te, lora_cutoff=cut_lora).drop(columns="dialogue").to_csv(f"{OUT}/scores_lora_te42_seed42.csv", index=False)
import shutil; shutil.make_archive("oneclass_seed42", "zip", OUT); print(os.listdir(OUT))
from google.colab import files; files.download("oneclass_seed42.zip")
print("DONE LoRA check")

LoRA cut-off 0.5585 | real spam caught overall 0.945 (logged masked v2 seed-42 run: 0.945)
  LoRA real spam caught, all: 121/128 = 0.945
  LoRA real spam caught, near-copies (>0.9): 37/37 = 1.000
  LoRA real spam caught, remaining (<=0.9): 84/91 = 0.923
['scores_lora_te42_seed42.csv', 'near_copy_check_seed42.csv', 'scores_oneclass_seed42.csv', 'summary_oneclass_seed42.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DONE LoRA check


# 9. Per-message scores and Google Drive
Inventory, scores for the six quick models, Drive mount and copy, scores for calibrated SVM, DistilBERT, zero-shot and LoRA, then ensembles and catch rates at 1%, 5% and 10% false-alarm limits.

Lists what is in memory and on disk before saving scores (reads only).

In [ ]:
# Step 1 inventory (reads only, trains nothing)
import os, time, psutil
print("runtime up for", round((time.time() - psutil.boot_time()) / 60), "min | GPU memory in use:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
CHK = {"MASK_V2": "masking v2 cache", "tr42": "seed-42 split", "svm_chk": "calibrated SVM, masked v2, seed 42 (near-copy check)",
       "s_svm": "calibrated SVM scores, 128 real spam test", "lm_chk": "LoRA model, masked v2, seed 42 (near-copy check)",
       "s_lora_te": "LoRA scores on te42", "BASE": "Qwen base model (needed by zero-shot and LoRA)", "ZS_CACHE": "zero-shot score cache",
       "REC": "one-class per-message scores", "SIM": "near-copy table", "svm_web": "website SVM", "SIX": "six quick model makers",
       "MASKED_V2_ROWS": "six quick models summary rows (old run)", "HIDDEN_V2": "hidden-topic table (old run)"}
for k, v in CHK.items():
    ok = k in globals(); o = globals().get(k)
    n = len(o) if ok and hasattr(o, "__len__") and not isinstance(o, str) else ""
    print(f"  {k:15s} in memory: {str(ok):5s} {('size ' + str(n)) if n != '' else '':12s} {v}")
print("\nfiles under /content (excluding sample_data):")
for root, dirs, files in os.walk("/content"):
    if "sample_data" in root or "/." in root or root.startswith("/content/drive"): continue
    for f in files: print("  ", os.path.join(root, f), os.path.getsize(os.path.join(root, f)), "bytes")
print("Google Drive mounted:", os.path.isdir("/content/drive/MyDrive"))
print("DONE inventory")

runtime up for 88 min | GPU memory in use: 3.12 GB
  MASK_V2         in memory: True  size 14184   masking v2 cache
  tr42            in memory: True  size 6844    seed-42 split
  svm_chk         in memory: True  size 2       calibrated SVM, masked v2, seed 42 (near-copy check)
  s_svm           in memory: True  size 128     calibrated SVM scores, 128 real spam test
  lm_chk          in memory: True               LoRA model, masked v2, seed 42 (near-copy check)
  s_lora_te       in memory: True  size 1670    LoRA scores on te42
  BASE            in memory: True               Qwen base model (needed by zero-shot and LoRA)
  ZS_CACHE        in memory: True  size 0       zero-shot score cache
  REC             in memory: True  size 28396   one-class per-message scores
  SIM             in memory: True  size 128     near-copy table
  svm_web         in memory: False              website SVM
  SIX             in memory: False              six quick model makers
  MASKED_V2_ROWS  in memory: 

Saves a score for every test message for the six quick models, so later cells can compare models without retraining.

In [ ]:
# Step 2: per-message scores for the six quick models, masked v2, seed 42, current training data (k=1)
from xgboost import XGBClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
# Model 5: CNN for text (PyTorch, runs on the GPU)
# Words -> numbers -> embedding (128 numbers per word) -> filters that look at 3, 4 and 5 words at a time
# -> keep each filter's strongest match anywhere in the message -> one scam score.
# Same fit/predict/predict_proba interface as the sklearn models, so evaluate() and seed_eval() work unchanged.
import re, torch, torch.nn as nn
from collections import Counter
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
TOKEN = re.compile(r"[a-z0-9£$]+|[!?]")
def tokenize(t): return TOKEN.findall(str(t).lower())

class TextCNN(nn.Module):
    def __init__(self, vocab, dim=128, filters=100, sizes=(3, 4, 5), drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(dim, filters, k) for k in sizes])
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(filters * len(sizes), 1)
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        h = torch.cat([torch.relu(c(e)).max(dim=2).values for c in self.convs], dim=1)
        return self.out(self.drop(h)).squeeze(1)

class CNNClassifier:
    def __init__(self, seed=42, max_len=64, max_vocab=20000, epochs=8, patience=2, batch=64, lr=1e-3, verbose=False):
        self.seed, self.max_len, self.max_vocab = seed, max_len, max_vocab
        self.epochs, self.patience, self.batch, self.lr, self.verbose = epochs, patience, batch, lr, verbose
    def _encode(self, texts):
        X = np.zeros((len(texts), self.max_len), dtype=np.int64)
        for i, t in enumerate(texts):
            ids = [self.vocab.get(w, 1) for w in tokenize(t)][:self.max_len]
            X[i, :len(ids)] = ids
        return torch.tensor(X)
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = np.asarray(labels, dtype=np.float32)
        # 10% of the training rows are held back to decide when to stop training (early stopping)
        tr_i, va_i = train_test_split(np.arange(len(texts)), test_size=0.1, stratify=y, random_state=self.seed)
        cnt = Counter(w for i in tr_i for w in tokenize(texts[i]))
        words = [w for w, c in cnt.most_common(self.max_vocab - 2) if c >= 2]
        self.vocab = {w: i + 2 for i, w in enumerate(words)}          # 0 = padding, 1 = unknown word
        X, Y = self._encode(texts), torch.tensor(y)
        self.model = TextCNN(len(self.vocab) + 2).to(DEVICE)
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.BCEWithLogitsLoss()
        g = torch.Generator().manual_seed(self.seed)
        Xva, Yva = X[va_i].to(DEVICE), Y[va_i].to(DEVICE)
        best, best_state, bad = 1e9, None, 0
        for ep in range(self.epochs):
            self.model.train()
            perm = torch.tensor(tr_i)[torch.randperm(len(tr_i), generator=g)]
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                opt.zero_grad()
                loss = loss_fn(self.model(X[idx].to(DEVICE)), Y[idx].to(DEVICE))
                loss.backward(); opt.step()
            self.model.eval()
            with torch.no_grad():
                vl = loss_fn(self.model(Xva), Yva).item()
            if self.verbose: print(f"    epoch {ep + 1}: validation loss {vl:.4f}")
            if vl < best - 1e-4:
                best, bad = vl, 0
                best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            else:
                bad += 1
                if bad >= self.patience: break
        self.model.load_state_dict(best_state)
        self.epochs_run = ep + 1
        return self
    def predict_proba(self, texts):
        self.model.eval(); X = self._encode(list(texts)); out = []
        with torch.no_grad():
            for b in range(0, len(X), 512):
                out.append(torch.sigmoid(self.model(X[b:b + 512].to(DEVICE))).cpu().numpy())
        p = np.concatenate(out)
        return np.column_stack([1 - p, p])
    def predict(self, texts):
        return (self.predict_proba(texts)[:, 1] > 0.5).astype(int)


# Model 6: BiLSTM (PyTorch, GPU)
# Reads the message word by word, left to right AND right to left, carrying a memory along the way.
# The final memory of each direction is joined and turned into one scam score.
# Same tokens, vocabulary rule, early stopping and interface as the CNN, so the only change is the network.
class BiLSTMNet(nn.Module):
    def __init__(self, vocab, dim=128, hidden=128, drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.lstm = nn.LSTM(dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(2 * hidden, 1)
    def forward(self, x):
        lengths = (x != 0).sum(1).clamp(min=1).cpu()        # real length of each message, so padding is skipped
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(x), lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.out(self.drop(torch.cat([h[0], h[1]], dim=1))).squeeze(1)

class BiLSTMClassifier(CNNClassifier):
    """Same training loop as CNNClassifier; only the network is swapped."""
    def fit(self, texts, labels):
        global TextCNN
        saved, TextCNN = TextCNN, BiLSTMNet      # CNNClassifier.fit builds TextCNN(vocab); point it at the BiLSTM for this call
        try:
            return super().fit(texts, labels)
        finally:
            TextCNN = saved



# MASKED RERUN, step 4a: seed 42, six models, both training ways, raw AND masked
# Same splits, seed, cut-off rule and tests as seed_eval / evaluate; only the text fed to the model changes.
# The raw run is a check: it should reproduce the earlier seed-42 numbers. Hand-written count = right of 10.
from xgboost import XGBClassifier

class RepeatRealSet:
    """Repeats real-SMS rows k times before training (same as RepeatReal), using the real-SMS set of the text version in use."""
    def __init__(self, base, k, real_set):
        self.base, self.k, self.real_set = base, k, real_set
    def fit(self, X, y):
        X = pd.Series(list(X)); y = pd.Series(np.asarray(y)); real = X.isin(self.real_set).values
        self.base.fit(pd.concat([X] + [X[real]] * (self.k - 1), ignore_index=True),
                      pd.concat([y] + [y[real]] * (self.k - 1), ignore_index=True))
        return self
    def predict(self, X): return self.base.predict(X)
    def __getattr__(self, name): return getattr(self.base, name)

SIX = {
    "NB":     lambda: make_pipeline(tfidf(), MultinomialNB()),
    "LR":     lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42)),
    "SVM":    lambda: make_pipeline(tfidf(), LinearSVC(random_state=42)),
    "XGB":    lambda: make_pipeline(tfidf(), XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist", device="cuda", random_state=42)),
    "CNN":    lambda: CNNClassifier(seed=42),
    "BiLSTM": lambda: BiLSTMClassifier(seed=42),
}

# ---------- new code below: train each model once, score four sets, save ----------
import os, time, numpy as np, pandas as pd
OUT2 = "quick6_seed42"; os.makedirs(OUT2, exist_ok=True)
SETS = {"calibration_3100_sms": cut42, "fresh_normal_sms_big_test": big42,
        "real_spam_test": te42[(te42.source == "real") & (te42.label == 1)],
        "dialogue_scam_test": te42[(te42.source == "synthetic") & (te42.label == 1)]}
print({k: len(v) for k, v in SETS.items()})
LOGGED = {"NB": (.914, .052), "LR": (.938, .055), "SVM": (.945, .059), "XGB": (.961, .055), "CNN": (.891, .066), "BiLSTM": (.875, .044)}
ALL = []
for name, mk in SIX.items():
    t0 = time.time(); mod = mk().fit(m2(tr42["dialogue"]), tr42["label"].values)
    kind = "predict_proba[:,1]" if hasattr(mod, "predict_proba") else "decision_function"
    S = {k: scores(mod, m2(d["dialogue"])) for k, d in SETS.items()}
    cut = float(np.quantile(S["calibration_3100_sms"], 0.95))
    msg = []
    for k, d in SETS.items():
        fl = S[k] > cut; msg.append(f"{k} {int(fl.sum())}/{len(fl)} = {fl.mean():.3f}")
        ALL.append(pd.DataFrame(dict(model=name, set=k, source=d.source.values, type=d.type.values, label=d.label.values, text_masked=m2(d["dialogue"]),
                                     score=S[k], score_kind=kind, cutoff=cut, flagged=fl.astype(int))))
    print(f"{name}: score = {kind}, cut-off {cut:.4f} | " + " | ".join(msg) + f" | {time.time() - t0:.0f} s")
    print(f"     logged masked v2 seed 42: real spam caught {LOGGED[name][0]}, fresh normal flagged {LOGGED[name][1]}")
    pd.concat(ALL[-4:]).to_csv(f"{OUT2}/scores_{name}_masked_v2_seed42.csv", index=False)
pd.concat(ALL).to_csv(f"{OUT2}/scores_quick6_masked_v2_seed42.csv", index=False)
import shutil; shutil.make_archive("quick6_seed42", "zip", OUT2); print(sorted(os.listdir(OUT2)), "rows", sum(len(a) for a in ALL))
from google.colab import files; files.download("quick6_seed42.zip")
print("DONE step 2")

{'calibration_3100_sms': 3100, 'fresh_normal_sms_big_test': 904, 'real_spam_test': 128, 'dialogue_scam_test': 707}
NB: score = predict_proba[:,1], cut-off 0.6831 | calibration_3100_sms 155/3100 = 0.050 | fresh_normal_sms_big_test 47/904 = 0.052 | real_spam_test 117/128 = 0.914 | dialogue_scam_test 685/707 = 0.969 | 1 s
     logged masked v2 seed 42: real spam caught 0.914, fresh normal flagged 0.052
LR: score = predict_proba[:,1], cut-off 0.5542 | calibration_3100_sms 155/3100 = 0.050 | fresh_normal_sms_big_test 50/904 = 0.055 | real_spam_test 120/128 = 0.938 | dialogue_scam_test 690/707 = 0.976 | 1 s
     logged masked v2 seed 42: real spam caught 0.938, fresh normal flagged 0.055
SVM: score = decision_function, cut-off 0.0230 | calibration_3100_sms 155/3100 = 0.050 | fresh_normal_sms_big_test 53/904 = 0.059 | real_spam_test 121/128 = 0.945 | dialogue_scam_test 700/707 = 0.990 | 1 s
     logged masked v2 seed 42: real spam caught 0.945, fresh normal flagged 0.059
XGB: score = predict_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DONE step 2


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
import os; print("MyDrive visible:", os.path.isdir("/content/drive/MyDrive"))

Mounted at /content/drive
MyDrive visible: True


Copies the website model and result folders to Google Drive.

In [ ]:
# Copy website model + result folders to Google Drive (models unchanged)
import os, sys, json, shutil, joblib, numpy as np, sklearn, spacy
DRV = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42"; os.makedirs(DRV, exist_ok=True)
# svm_chk = calibrated SVM trained in this runtime with the same code as the website model (cell 50)
cut_now = float(np.quantile(svm_chk.predict_proba(m2(cut42["dialogue"]))[:, 1], 0.95))
print(f"cut-off now {cut_now:.10f} | website value 0.6153828434070271 | equal to 6 decimals: {round(cut_now, 6) == 0.615383} | exact: {cut_now == 0.6153828434070271}")
joblib.dump(svm_chk, f"{DRV}/svm_masked_seed42.joblib")
info = {"cutoff": cut_now, "rule": "scam if predict_proba(mask_texts([text]))[:,1] > cutoff",
        "budget": "5% false alarms on leftover real normal training SMS", "seed": 42, "masking": "masking_model v2",
        "python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "spacy": spacy.__version__}
try:
    import en_core_web_sm; info["en_core_web_sm"] = en_core_web_sm.__version__
except Exception:
    info["en_core_web_sm"] = spacy.load("en_core_web_sm").meta["version"]
json.dump(info, open(f"{DRV}/cutoff.json", "w"), indent=2); print(json.dumps(info, indent=2))
back = joblib.load(f"{DRV}/svm_masked_seed42.joblib")
print("reloaded from Drive, same calibration scores:", np.allclose(back.predict_proba(m2(cut42["dialogue"]))[:, 1], svm_chk.predict_proba(m2(cut42["dialogue"]))[:, 1]))
for d in ["quick6_seed42", "oneclass_seed42"]:
    shutil.copytree(d, f"{DRV}/{d}", dirs_exist_ok=True)
for root, dirs, files in os.walk(DRV):
    for f in files: print("  ", os.path.join(root, f).replace(DRV, "Drive/ScamGuard_masked_v2_seed42"), os.path.getsize(os.path.join(root, f)), "bytes")
print("DONE copy")

cut-off now 0.6153828434 | website value 0.6153828434070271 | equal to 6 decimals: True | exact: True
{
  "cutoff": 0.6153828434070271,
  "rule": "scam if predict_proba(mask_texts([text]))[:,1] > cutoff",
  "budget": "5% false alarms on leftover real normal training SMS",
  "seed": 42,
  "masking": "masking_model v2",
  "python": "3.13.15",
  "scikit-learn": "1.6.1",
  "spacy": "3.8.16",
  "en_core_web_sm": "3.8.0"
}
reloaded from Drive, same calibration scores: True
   Drive/ScamGuard_masked_v2_seed42/svm_masked_seed42.joblib 898481 bytes
   Drive/ScamGuard_masked_v2_seed42/cutoff.json 318 bytes
   Drive/ScamGuard_masked_v2_seed42/quick6_seed42/scores_CNN_masked_v2_seed42.csv 942011 bytes
   Drive/ScamGuard_masked_v2_seed42/quick6_seed42/scores_XGB_masked_v2_seed42.csv 940342 bytes
   Drive/ScamGuard_masked_v2_seed42/quick6_seed42/scores_SVM_masked_v2_seed42.csv 978238 bytes
   Drive/ScamGuard_masked_v2_seed42/quick6_seed42/scores_LR_masked_v2_seed42.csv 976051 bytes
   Drive/ScamGuar

Per-message scores for calibrated SVM, DistilBERT, zero-shot and LoRA.

In [ ]:
# Step 3: per-message scores for calibrated SVM, DistilBERT (retrained), zero-shot and LoRA, masked v2, seed 42
# Uses objects already in memory (svm_chk, lm_chk, BASE via zs_scores) and DistilBERTClassifier as defined (unchanged) in the LoRA check cell.
import time
OUT3 = f"{DRV}/gpu4_seed42"; os.makedirs(OUT3, exist_ok=True)
SETS = {"calibration_3100_sms": cut42, "fresh_normal_sms_big_test": big42,
        "real_spam_test": te42[(te42.source == "real") & (te42.label == 1)],
        "dialogue_scam_test": te42[(te42.source == "synthetic") & (te42.label == 1)]}
print({k: len(v) for k, v in SETS.items()})
t0 = time.time(); bm_chk = DistilBERTClassifier(seed=42).fit(m2(tr42["dialogue"]), tr42["label"].values); print(f"DistilBERT retrained in {time.time() - t0:.0f} s")
SCORERS = {"SVM_calibrated": lambda x: svm_chk.predict_proba(x)[:, 1], "DistilBERT": lambda x: bm_chk.predict_proba(x)[:, 1],
           "zero-shot": lambda x: zs_scores(x), "LoRA": lambda x: lm_chk.predict_proba(x)[:, 1]}
LOGGED = {"SVM_calibrated": (.945, .054), "DistilBERT": (.969, .058), "zero-shot": (.906, .045), "LoRA": (.945, .054)}
ALL3 = []
for name, f in SCORERS.items():
    t0 = time.time(); S = {k: np.asarray(f(m2(d["dialogue"]))) for k, d in SETS.items()}
    cut = float(np.quantile(S["calibration_3100_sms"], 0.95)); msg = []
    for k, d in SETS.items():
        fl = S[k] > cut; msg.append(f"{k} {int(fl.sum())}/{len(fl)} = {fl.mean():.3f}")
        ALL3.append(pd.DataFrame(dict(model=name, set=k, source=d.source.values, type=d.type.values, label=d.label.values, text_masked=m2(d["dialogue"]),
                                      score=S[k], score_kind="probability of scam", cutoff=cut, flagged=fl.astype(int))))
    print(f"{name}: cut-off {cut:.4f} | " + " | ".join(msg) + f" | {time.time() - t0:.0f} s")
    print(f"     logged masked v2 seed 42: real spam caught {LOGGED[name][0]}, fresh normal flagged {LOGGED[name][1]}")
    pd.concat(ALL3[-4:]).to_csv(f"{OUT3}/scores_{name}_masked_v2_seed42.csv", index=False)
pd.concat(ALL3).to_csv(f"{OUT3}/scores_gpu4_masked_v2_seed42.csv", index=False)
print(sorted(os.listdir(OUT3)), "rows", sum(len(a) for a in ALL3))
print("DONE step 3")

{'calibration_3100_sms': 3100, 'fresh_normal_sms_big_test': 904, 'real_spam_test': 128, 'dialogue_scam_test': 707}
DistilBERT retrained in 43 s
SVM_calibrated: cut-off 0.6154 | calibration_3100_sms 155/3100 = 0.050 | fresh_normal_sms_big_test 49/904 = 0.054 | real_spam_test 121/128 = 0.945 | dialogue_scam_test 697/707 = 0.986 | 0 s
     logged masked v2 seed 42: real spam caught 0.945, fresh normal flagged 0.054
DistilBERT: cut-off 0.3788 | calibration_3100_sms 155/3100 = 0.050 | fresh_normal_sms_big_test 52/904 = 0.058 | real_spam_test 124/128 = 0.969 | dialogue_scam_test 698/707 = 0.987 | 3 s
     logged masked v2 seed 42: real spam caught 0.969, fresh normal flagged 0.058
zero-shot: cut-off 0.5662 | calibration_3100_sms 155/3100 = 0.050 | fresh_normal_sms_big_test 42/904 = 0.046 | real_spam_test 116/128 = 0.906 | dialogue_scam_test 342/707 = 0.484 | 134 s
     logged masked v2 seed 42: real spam caught 0.906, fresh normal flagged 0.045
LoRA: cut-off 0.5585 | calibration_3100_sms 155

Builds the ensembles from saved scores and reports catch rates at 1%, 5% and 10% false-alarm limits.

In [ ]:
# Step 4 (ensembles from saved scores) + Step 5 (catch rate at 1%, 5%, 10% false-alarm limits)
# Reads only the per-message score files on Drive. No model is trained or changed.
import os, numpy as np, pandas as pd
DRV = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42"
rd = lambda p: pd.read_csv(p, keep_default_na=False)
g4 = rd(f"{DRV}/gpu4_seed42/scores_gpu4_masked_v2_seed42.csv")
q6 = rd(f"{DRV}/quick6_seed42/scores_quick6_masked_v2_seed42.csv")
oc = rd(f"{DRV}/oneclass_seed42/scores_oneclass_seed42.csv")
print("rows read: gpu4", len(g4), "| quick6", len(q6), "| one-class", len(oc))
SETN = ["calibration_3100_sms", "fresh_normal_sms_big_test", "real_spam_test", "dialogue_scam_test"]
def part(df, m, s): return df[(df.model == m) & (df.set == s)].reset_index(drop=True)
# ---- alignment check: every model must have the same messages in the same order ----
for s in SETN:
    ref = part(g4, "LoRA", s).text_masked
    same = [m for m in g4.model.unique() if part(g4, m, s).text_masked.equals(ref)] + [m for m in q6.model.unique() if part(q6, m, s).text_masked.equals(ref)]
    print(f"  {s}: n={len(ref)}, models with identical message order: {len(same)}/10")

# Step 4
ENS = {"E1 SVM+LoRA": ["SVM_calibrated", "LoRA"], "E2 SVM+LoRA+zero-shot": ["SVM_calibrated", "LoRA", "zero-shot"],
       "E3 LoRA+zero-shot": ["LoRA", "zero-shot"], "E4 all four": ["SVM_calibrated", "DistilBERT", "LoRA", "zero-shot"]}
LOGGED_E = {"E1 SVM+LoRA": (.977, .055), "E2 SVM+LoRA+zero-shot": (.961, .049), "E3 LoRA+zero-shot": (.953, .049), "E4 all four": (.977, .046)}
print("\nmember score ranges (all are probabilities of scam, 0 to 1):")
for m in ["SVM_calibrated", "DistilBERT", "LoRA", "zero-shot"]:
    v = g4[g4.model == m].score; print(f"  {m}: min {v.min():.4f} max {v.max():.4f}")
EN_ROWS = []
for en, mem in ENS.items():
    S = {s: np.mean([part(g4, m, s).score.values for m in mem], axis=0) for s in SETN}
    cut = float(np.quantile(S["calibration_3100_sms"], 0.95)); msg = []
    for s in SETN:
        fl = S[s] > cut; msg.append(f"{s} {int(fl.sum())}/{len(fl)} = {fl.mean():.3f}")
        EN_ROWS.append(part(g4, "LoRA", s)[["set", "source", "type", "label", "text_masked"]].assign(model=en, members="+".join(mem), score=S[s], score_kind="plain mean of member probabilities", cutoff=cut, flagged=fl.astype(int)))
    ok = (round(float((S["real_spam_test"] > cut).mean()), 3) == LOGGED_E[en][0], round(float((S["fresh_normal_sms_big_test"] > cut).mean()), 3) == LOGGED_E[en][1])
    print(f"{en}: cut-off {cut:.4f} | " + " | ".join(msg))
    print(f"     logged: real spam caught {LOGGED_E[en][0]}, fresh normal flagged {LOGGED_E[en][1]} | match: {ok}")
ENDF = pd.concat(EN_ROWS); os.makedirs(f"{DRV}/ensembles_seed42", exist_ok=True)
ENDF.to_csv(f"{DRV}/ensembles_seed42/scores_ensembles_masked_v2_seed42.csv", index=False); print("saved ensemble rows:", len(ENDF))

# Step 5
SC = {}
for m in ["NB", "LR", "SVM", "XGB", "CNN", "BiLSTM"]: SC[m] = {s: part(q6, m, s).score.values for s in SETN}
for m in ["DistilBERT", "zero-shot", "LoRA"]: SC[m] = {s: part(g4, m, s).score.values for s in SETN}
for en in ENS: SC[en] = {s: ENDF[(ENDF.model == en) & (ENDF.set == s)].score.values for s in SETN}
SC["SVM_calibrated (extra: ensemble member)"] = {s: part(g4, "SVM_calibrated", s).score.values for s in SETN}
for m in ["IsolationForest", "OneClassSVM"]:
    o = oc[(oc.model == m) & (oc.context == "main")]; t = o[o.set == "test_te42"]
    SC[m] = {"calibration_3100_sms": o[o.set == "calibration_cut42"].score.values, "fresh_normal_sms_big_test": o[o.set == "bigtest_big42"].score.values,
             "real_spam_test": t[(t.source == "real") & (t.label == 1)].score.values, "dialogue_scam_test": t[(t.source == "synthetic") & (t.label == 1)].score.values}
print("\nset sizes per model:", {m: [len(v[s]) for s in SETN] for m, v in list(SC.items())[:1]}, "| all equal:", len({tuple(len(v[s]) for s in SETN) for v in SC.values()}) == 1)
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
T = []
for m, v in SC.items():
    for a in [0.01, 0.05, 0.10]:
        cut = float(np.quantile(v["calibration_3100_sms"], 1 - a)); r = dict(model=m, limit=a, cutoff=cut)
        for s, tag in [("calibration_3100_sms", "calib"), ("fresh_normal_sms_big_test", "fresh_FA"), ("real_spam_test", "real_spam"), ("dialogue_scam_test", "dialogue_scam")]:
            fl = v[s] > cut; k, n = int(fl.sum()), len(fl); lo, hi = wilson(k, n)
            r.update({f"{tag}_k": k, f"{tag}_n": n, f"{tag}_rate": round(k / n, 4), f"{tag}_lo95": round(lo, 4), f"{tag}_hi95": round(hi, 4)})
        T.append(r)
T = pd.DataFrame(T)
f = lambda r, t: f"{r[t+'_k']}/{r[t+'_n']}={r[t+'_rate']:.3f} [{r[t+'_lo95']:.3f},{r[t+'_hi95']:.3f}]"
show = pd.DataFrame({"model": T.model, "limit": T.limit, "calib": T.calib_k, "fresh FA": T.apply(lambda r: f(r, "fresh_FA"), axis=1),
                     "real spam": T.apply(lambda r: f(r, "real_spam"), axis=1), "dialogue scam": T.apply(lambda r: f(r, "dialogue_scam"), axis=1)})
pd.set_option("display.width", 250); print(show.to_string(index=False))
odd = T[(T.calib_k - (T.limit * 3100).round()).abs() > 1]; print("\ncalibration counts off target by more than 1 (ties):", odd[["model", "limit", "calib_k"]].values.tolist())
T.to_csv(f"{DRV}/catch_rate_at_FA_limits_seed42.csv", index=False); T.to_excel(f"{DRV}/catch_rate_at_FA_limits_seed42.xlsx", index=False)
print("saved", len(T), "rows to Drive/ScamGuard_masked_v2_seed42/catch_rate_at_FA_limits_seed42 (.csv and .xlsx)")
print("DONE steps 4 and 5")

rows read: gpu4 19356 | quick6 29034 | one-class 28396
  calibration_3100_sms: n=3100, models with identical message order: 10/10
  fresh_normal_sms_big_test: n=904, models with identical message order: 10/10
  real_spam_test: n=128, models with identical message order: 10/10
  dialogue_scam_test: n=707, models with identical message order: 10/10

member score ranges (all are probabilities of scam, 0 to 1):
  SVM_calibrated: min 0.0040 max 0.9999
  DistilBERT: min 0.0024 max 0.9990
  LoRA: min 0.0000 max 0.9999
  zero-shot: min 0.0000 max 0.9985
E1 SVM+LoRA: cut-off 0.4682 | calibration_3100_sms 155/3100 = 0.050 | fresh_normal_sms_big_test 50/904 = 0.055 | real_spam_test 125/128 = 0.977 | dialogue_scam_test 703/707 = 0.994
     logged: real spam caught 0.977, fresh normal flagged 0.055 | match: (True, True)
E2 SVM+LoRA+zero-shot: cut-off 0.4355 | calibration_3100_sms 155/3100 = 0.050 | fresh_normal_sms_big_test 44/904 = 0.049 | real_spam_test 123/128 = 0.961 | dialogue_scam_test 698/70

Three checks comparing the plain LinearSVC with the calibrated SVM: a tie check, a ranking check and three seeds.

In [ ]:
# Calibrated SVM tie check (reads the saved scores only, no model loaded or trained)
from google.colab import drive; drive.mount("/content/drive")
import numpy as np, pandas as pd
P = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/gpu4_seed42/scores_SVM_calibrated_masked_v2_seed42.csv"
d = pd.read_csv(P, keep_default_na=False)
cal = d[d.set == "calibration_3100_sms"]; rs = d[d.set == "real_spam_test"]
c, s = cal.score.values, rs.score.values
print(f"calibration n {len(c)} | distinct scores {len(np.unique(c))} | distinct after rounding to 6 dp {len(np.unique(np.round(c, 6)))} | range {c.min():.6f} to {c.max():.6f}")
vc = pd.Series(np.round(c, 6)).value_counts().head(5); print("most repeated calibration scores (6 dp):", vc.to_dict())
CUT = {}
for a in [0.01, 0.05, 0.10]:
    cut = float(np.quantile(c, 1 - a)); CUT[a] = cut
    print(f"limit {a:.0%}: cut-off {cut:.10f} | calib exactly equal {int((c == cut).sum())} | calib within 1e-9 {int((np.abs(c - cut) < 1e-9).sum())} | calib strictly above {int((c > cut).sum())} (target {round(a * 3100)}) | real spam strictly above {int((s > cut).sum())}/128, exactly equal {int((s == cut).sum())}")
print("\n40 highest calibration scores:", np.round(np.sort(c)[::-1][:40], 6).tolist())
ss = np.sort(s)
print("\nreal spam test, 20 lowest scores:", np.round(ss[:20], 6).tolist())
print("real spam test, all 128 sorted:", np.round(ss, 4).tolist())
print(f"\n1% cut-off {CUT[0.01]:.6f} | 5% cut-off {CUT[0.05]:.6f}")
print("real spam scoring between the 5% and 1% cut-offs:", int(((s > CUT[0.05]) & (s <= CUT[0.01])).sum()), "| above the 1% cut-off:", int((s > CUT[0.01]).sum()))
pd.set_option("display.max_colwidth", 250)
print("\n10 highest-scoring calibration SMS (all labelled normal):")
for sc, t in zip(cal.sort_values("score", ascending=False).head(10).score, cal.sort_values("score", ascending=False).head(10).text_masked):
    print(f"  {sc:.6f} | {t}")
print("DONE tie check")

Mounted at /content/drive
calibration n 3100 | distinct scores 3071 | distinct after rounding to 6 dp 3057 | range 0.004034 to 0.966749
most repeated calibration scores (6 dp): {0.231095: 13, 0.169562: 3, 0.389368: 2, 0.31076: 2, 0.228157: 2}
limit 1%: cut-off 0.8539457047 | calib exactly equal 0 | calib within 1e-9 0 | calib strictly above 31 (target 31) | real spam strictly above 51/128, exactly equal 0
limit 5%: cut-off 0.6153828434 | calib exactly equal 0 | calib within 1e-9 0 | calib strictly above 155 (target 155) | real spam strictly above 121/128, exactly equal 0
limit 10%: cut-off 0.4274741552 | calib exactly equal 0 | calib within 1e-9 0 | calib strictly above 310 (target 310) | real spam strictly above 124/128, exactly equal 0

40 highest calibration scores: [0.966749, 0.963085, 0.9511, 0.948122, 0.947707, 0.931293, 0.92898, 0.928847, 0.924185, 0.919878, 0.919557, 0.9097, 0.909601, 0.909141, 0.908789, 0.901641, 0.895269, 0.889938, 0.886906, 0.884147, 0.883918, 0.881599, 0.87

In [ ]:
# Plain LinearSVC vs calibrated SVM ranking check, masked v2, seed 42
# Retrains both SVMs with the existing code (needs setup + masking_model), then checks them against the saved Drive scores.
import os, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
if "MASK_V2" not in globals():
    all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
    MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
tr42, te42, s_tr42, s_te42 = build_data(42)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]; big42 = s_te42[s_te42.label == 0]
X, y = m2(tr42["dialogue"]), tr42["label"].values
plain = make_pipeline(tfidf(), LinearSVC(random_state=42)).fit(X, y)                                   # Model 3 (SIX["SVM"])
calib = make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=42), cv=3)).fit(X, y)     # ensemble member / website model
P = lambda T: plain.decision_function(m2(T)); C = lambda T: calib.predict_proba(m2(T))[:, 1]
pc, cc = P(cut42["dialogue"]), C(cut42["dialogue"])
# ---- check against cut-offs recorded in catch_rate_at_FA_limits_seed42 (saved table) ----
REC_CUT = {"plain 1%": 0.3443135215683249, "plain 5%": 0.023036774456674326, "calibrated 1%": 0.8539457047140352, "calibrated 5%": 0.6153828434070271}
NOW_CUT = {"plain 1%": np.quantile(pc, 0.99), "plain 5%": np.quantile(pc, 0.95), "calibrated 1%": np.quantile(cc, 0.99), "calibrated 5%": np.quantile(cc, 0.95)}
for k in REC_CUT: print(f"  {k}: recorded {REC_CUT[k]:.10f} | now {NOW_CUT[k]:.10f} | equal: {abs(REC_CUT[k] - NOW_CUT[k]) < 1e-12}")
print("calibration method:", calib[-1].method, "| fold models averaged:", len(calib[-1].calibrated_classifiers_))
# ---- AUC on the real SMS test set ----
rt = te42[te42.source == "real"]; yr = rt.label.values
print("real SMS test:", {"spam": int(yr.sum()), "normal": int((yr == 0).sum())})
pr, cr = P(rt["dialogue"]), C(rt["dialogue"])
a_p, a_c = roc_auc_score(yr, pr), roc_auc_score(yr, cr)
rng = np.random.default_rng(42); diffs = []
for _ in range(2000):
    i = rng.integers(0, len(yr), len(yr))
    if yr[i].min() == yr[i].max(): continue
    diffs.append(roc_auc_score(yr[i], pr[i]) - roc_auc_score(yr[i], cr[i]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"AUC real SMS test (128 spam vs 128 normal): plain {a_p:.4f} | calibrated {a_c:.4f} | plain minus calibrated {a_p - a_c:+.4f}, bootstrap 95% [{lo:+.4f}, {hi:+.4f}] ({len(diffs)} resamples)")
rs = rt[rt.label == 1]["dialogue"]; pb, cb = P(big42["dialogue"]), C(big42["dialogue"])
print(f"AUC real spam (128) vs fresh normal (904): plain {roc_auc_score(np.r_[np.ones(128), np.zeros(len(pb))], np.r_[P(rs), pb]):.4f} | calibrated {roc_auc_score(np.r_[np.ones(128), np.zeros(len(cb))], np.r_[C(rs), cb]):.4f}")
# ---- calibration SMS: rank agreement and the top 1% ----
rho, pv = spearmanr(pc, cc); print(f"\nSpearman between the two models on the 3,100 calibration SMS: {rho:.4f} (p {pv:.1e})")
k1p, k1c = np.quantile(pc, 0.99), np.quantile(cc, 0.99)
top_p, top_c = set(np.where(pc > k1p)[0]), set(np.where(cc > k1c)[0])
print(f"1% cut-offs: plain {k1p:.4f} ({len(top_p)} above) | calibrated {k1c:.4f} ({len(top_c)} above) | in both top sets: {len(top_p & top_c)} of {len(top_c)}")
rk = pd.Series(pc).rank(ascending=False).values
print("plain-SVM rank (1 = highest of 3,100) of each calibrated top-31 SMS:", sorted(int(rk[i]) for i in top_c))
print(f"real spam above own 1% cut-off: plain {int((P(rs) > k1p).sum())}/128 | calibrated {int((C(rs) > k1c).sum())}/128")
rho_s, _ = spearmanr(P(rs), C(rs)); print(f"Spearman between the two models on the 128 real spam: {rho_s:.4f}")
print("DONE ranking check")

  plain 1%: recorded 0.3443135216 | now 0.3443135216 | equal: True
  plain 5%: recorded 0.0230367745 | now 0.0230367745 | equal: True
  calibrated 1%: recorded 0.8539457047 | now 0.8539457047 | equal: True
  calibrated 5%: recorded 0.6153828434 | now 0.6153828434 | equal: True
calibration method: sigmoid | fold models averaged: 3
real SMS test: {'spam': 128, 'normal': 128}
AUC real SMS test (128 spam vs 128 normal): plain 0.9890 | calibrated 0.9708 | plain minus calibrated +0.0182, bootstrap 95% [+0.0052, +0.0350] (2000 resamples)
AUC real spam (128) vs fresh normal (904): plain 0.9894 | calibrated 0.9708

Spearman between the two models on the 3,100 calibration SMS: 0.7537 (p 0.0e+00)
1% cut-offs: plain 0.3443 (31 above) | calibrated 0.8539 (31 above) | in both top sets: 24 of 31
plain-SVM rank (1 = highest of 3,100) of each calibrated top-31 SMS: [2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 23, 24, 26, 27, 29, 31, 41, 42, 46, 51, 54, 60, 61]
real spam above own 1

In [ ]:
# Plain LinearSVC vs calibrated SVM over seeds 42, 7, 2026 (masked v2, existing code, nothing tuned)
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
if "MASK_V2" not in globals():
    all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
    MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
AUC_ROWS, CATCH_ROWS = [], []
for seed in [42, 7, 2026]:
    tr, te, s_tr, s_te = build_data(seed)
    cut = s_tr[(s_tr.label == 0) & ~s_tr.dialogue.isin(tr.dialogue)]; big = s_te[s_te.label == 0]
    rt = te[te.source == "real"]; yr = rt.label.values
    X, y = m2(tr["dialogue"]), tr["label"].values
    M = {"plain LinearSVC": make_pipeline(tfidf(), LinearSVC(random_state=seed)).fit(X, y),
         "calibrated SVM": make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=seed), cv=3)).fit(X, y)}
    sc = lambda m, T: m.decision_function(m2(T)) if not hasattr(m, "predict_proba") else m.predict_proba(m2(T))[:, 1]
    S = {k: {"cal": sc(m, cut["dialogue"]), "rt": sc(m, rt["dialogue"]), "big": sc(m, big["dialogue"])} for k, m in M.items()}
    print(f"\nseed {seed}: calibration SMS {len(cut)} | real SMS test spam {int(yr.sum())}, normal {int((yr == 0).sum())} | fresh normal {len(big)}")
    a = {k: roc_auc_score(yr, v["rt"]) for k, v in S.items()}
    rng = np.random.default_rng(seed); d = []
    for _ in range(2000):
        i = rng.integers(0, len(yr), len(yr))
        if yr[i].min() == yr[i].max(): continue
        d.append(roc_auc_score(yr[i], S["plain LinearSVC"]["rt"][i]) - roc_auc_score(yr[i], S["calibrated SVM"]["rt"][i]))
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"  AUC real SMS test: plain {a['plain LinearSVC']:.4f} | calibrated {a['calibrated SVM']:.4f} | difference {a['plain LinearSVC'] - a['calibrated SVM']:+.4f}, paired bootstrap 95% [{lo:+.4f}, {hi:+.4f}] ({len(d)} resamples)")
    AUC_ROWS.append(dict(seed=seed, auc_plain=round(a["plain LinearSVC"], 4), auc_calibrated=round(a["calibrated SVM"], 4),
                         diff=round(a["plain LinearSVC"] - a["calibrated SVM"], 4), diff_lo95=round(lo, 4), diff_hi95=round(hi, 4), resamples=len(d)))
    spam = yr == 1
    for k, v in S.items():
        for lim in [0.01, 0.05, 0.10]:
            c = float(np.quantile(v["cal"], 1 - lim)); fs = v["rt"][spam] > c; fb = v["big"] > c; lo2, hi2 = wilson(int(fs.sum()), len(fs))
            print(f"  {k:16s} {lim:.0%}: cut-off {c:.4f} | calib flagged {int((v['cal'] > c).sum())} | fresh flagged {int(fb.sum())}/{len(fb)} = {fb.mean():.3f} | real spam caught {int(fs.sum())}/{len(fs)} = {fs.mean():.3f} [{lo2:.3f}, {hi2:.3f}]")
            CATCH_ROWS.append(dict(seed=seed, model=k, limit=lim, cutoff=c, calib_flagged=int((v["cal"] > c).sum()), fresh_flagged=int(fb.sum()), fresh_n=len(fb),
                                   fresh_rate=round(fb.mean(), 4), spam_caught=int(fs.sum()), spam_n=len(fs), spam_rate=round(fs.mean(), 4), spam_lo95=round(lo2, 4), spam_hi95=round(hi2, 4)))
A, Cc = pd.DataFrame(AUC_ROWS), pd.DataFrame(CATCH_ROWS)
print("\n", A.to_string(index=False))
print("\nmean over 3 seeds:", A[["auc_plain", "auc_calibrated", "diff"]].mean().round(4).to_dict())
os.makedirs("svm_plain_vs_calibrated", exist_ok=True)
A.to_csv("svm_plain_vs_calibrated/auc_by_seed.csv", index=False); Cc.to_csv("svm_plain_vs_calibrated/catch_by_seed.csv", index=False)
print("saved locally in svm_plain_vs_calibrated/")
try:
    from google.colab import drive; drive.mount("/content/drive")
    import shutil; shutil.copytree("svm_plain_vs_calibrated", "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/svm_plain_vs_calibrated", dirs_exist_ok=True)
    print("copied to Drive: ScamGuard_masked_v2_seed42/svm_plain_vs_calibrated", os.listdir("/content/drive/MyDrive/ScamGuard_masked_v2_seed42/svm_plain_vs_calibrated"))
except Exception as e:
    print("Drive copy FAILED:", repr(e))
print("DONE seeds check")


seed 42: calibration SMS 3100 | real SMS test spam 128, normal 128 | fresh normal 904
  AUC real SMS test: plain 0.9890 | calibrated 0.9708 | difference +0.0182, paired bootstrap 95% [+0.0052, +0.0350] (2000 resamples)
  plain LinearSVC  1%: cut-off 0.3443 | calib flagged 31 | fresh flagged 11/904 = 0.012 | real spam caught 111/128 = 0.867 [0.798, 0.915]
  plain LinearSVC  5%: cut-off 0.0230 | calib flagged 155 | fresh flagged 53/904 = 0.059 | real spam caught 121/128 = 0.945 [0.891, 0.973]
  plain LinearSVC  10%: cut-off -0.1327 | calib flagged 310 | fresh flagged 96/904 = 0.106 | real spam caught 124/128 = 0.969 [0.922, 0.988]
  calibrated SVM   1%: cut-off 0.8539 | calib flagged 31 | fresh flagged 12/904 = 0.013 | real spam caught 51/128 = 0.398 [0.318, 0.485]
  calibrated SVM   5%: cut-off 0.6154 | calib flagged 155 | fresh flagged 49/904 = 0.054 | real spam caught 121/128 = 0.945 [0.891, 0.973]
  calibrated SVM   10%: cut-off 0.4275 | calib flagged 310 | fresh flagged 96/904 = 0.

Saves the plain LinearSVC, the model used by the website.

In [ ]:
# Save the plain LinearSVC (Model 3 pipeline), masked v2, seed 42, for the website
import sys, os, json, joblib, numpy as np, sklearn, spacy
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
DRV = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42"; print("Drive mounted:", os.path.isdir(DRV))
tr42, te42, s_tr42, s_te42 = build_data(42)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]
print("training pieces", len(tr42), "| calibration SMS", len(cut42))
svm_plain = make_pipeline(tfidf(), LinearSVC(random_state=42)).fit(m2(tr42["dialogue"]), tr42["label"].values)   # same as SIX["SVM"]
s_cal = svm_plain.decision_function(m2(cut42["dialogue"]))
CUTS = {f"{int(a * 100)}%": float(np.quantile(s_cal, 1 - a)) for a in [0.01, 0.05, 0.10]}
ABOVE = {k: int((s_cal > v).sum()) for k, v in CUTS.items()}
for k in CUTS: print(f"  {k}: cut-off {CUTS[k]:.10f} | calibration SMS above {ABOVE[k]} of {len(s_cal)}")
print("5% cut-off rounds to 0.0230:", round(CUTS["5%"], 4) == 0.0230, "| equals the recorded 0.023036774456674326:", abs(CUTS["5%"] - 0.023036774456674326) < 1e-12)
joblib.dump(svm_plain, f"{DRV}/svm_plain_masked_seed42.joblib")
info = {"model": "plain LinearSVC (TF-IDF word 1-2 grams, min_df=2) trained on masked v2 text, seed-42 training split",
        "score": "decision_function margin (not a probability)",
        "rule": "flag as scam if decision_function(mask_texts([text]))[0] > cutoff for the chosen limit",
        "cutoffs": CUTS, "calibration_sms_above": ABOVE, "calibration_sms_n": int(len(s_cal)),
        "limits_meaning": "share of the 3,100 unseen real normal training SMS flagged at each cut-off",
        "seed": 42, "masking": "masking_model v2",
        "python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "spacy": spacy.__version__, "numpy": np.__version__, "joblib": joblib.__version__}
try:
    import en_core_web_sm; info["en_core_web_sm"] = en_core_web_sm.__version__
except Exception:
    info["en_core_web_sm"] = spacy.load("en_core_web_sm").meta["version"]
json.dump(info, open(f"{DRV}/cutoff_plain.json", "w"), indent=2); print(json.dumps(info, indent=2))
back = joblib.load(f"{DRV}/svm_plain_masked_seed42.joblib")
s_back = back.decision_function(m2(cut42["dialogue"]))
print("reloaded from Drive: identical calibration scores:", np.array_equal(s_back, s_cal), "| max diff", float(np.abs(s_back - s_cal).max()))
te_back = back.decision_function(m2(te42["dialogue"])); te_now = svm_plain.decision_function(m2(te42["dialogue"]))
print("reloaded from Drive: identical test-set scores:", np.array_equal(te_back, te_now), f"({len(te_now)} pieces)")
print({f: os.path.getsize(f"{DRV}/{f}") for f in ["svm_plain_masked_seed42.joblib", "cutoff_plain.json"]})
print("DONE plain SVM save")

# 10. Hidden scam topics at the 5% cut-off
The main test of the study. Each scam topic (reward, refund, SSN, support) is hidden from training in turn, the model is retrained, and recall on the hidden topic is measured at a cut-off that flags 5% of normal messages.

Model 1 of 3: plain LinearSVC. This cell also defines `run_topic()`, used by the next two cells.

In [ ]:
# Hidden topics at the 5% cut-off, model 1 of 3: plain LinearSVC (training split only, masked v2, seed 42)
# Defines the shared helper run_topic() used by the DistilBERT and LoRA hidden-topic cells. Needs setup + masking_model.
import os, gc, json, time, numpy as np, pandas as pd
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
DRV = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42"
if not os.path.isdir(DRV):
    from google.colab import drive; drive.mount("/content/drive")
HT = f"{DRV}/hidden_topics_5pct"; os.makedirs(HT, exist_ok=True); print("Drive folder ready:", os.path.isdir(HT))
if "MASK_V2" not in globals():
    all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
    MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
tr42, te42, s_tr42, s_te42 = build_data(42)
full42 = pd.concat([tr42.assign(split="train"), te42.assign(split="test")], ignore_index=True)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]; big42 = s_te42[s_te42.label == 0]
SCAM_T, NORMAL_T = ["refund", "reward", "ssn", "support"], ["delivery", "insurance", "telemarketing", "wrong"]
print("training pieces", len(tr42), "| calibration SMS", len(cut42), "| fresh normal SMS", len(big42))
print("topic labels in data:", {t: sorted(full42[full42.type == t].label.unique().tolist()) for t in SCAM_T + NORMAL_T})
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
def run_topic(model_name, fit_fn, t, score_kind):
    """fit_fn(train_df) -> scorer(texts) -> scores. Retrain without topic t, cut-off on the 3,100 calibration SMS, score all pieces of t."""
    t0 = time.time(); trn = tr42[tr42.type != t]; lab = int(full42[full42.type == t].label.iloc[0])
    print(f"[{model_name}] {t} ({'scam' if lab else 'normal'}): training pieces {len(tr42)} -> {len(trn)} (removed {len(tr42) - len(trn)})")
    scorer = fit_fn(trn)
    s_cal = np.asarray(scorer(m2(cut42["dialogue"]))); cut = float(np.quantile(s_cal, 0.95))
    d = full42[full42.type == t]; s = np.asarray(scorer(m2(d["dialogue"]))); fl = s > cut
    s_big = np.asarray(scorer(m2(big42["dialogue"]))); fb = s_big > cut
    k, n = int(fl.sum()), len(fl); lo, hi = wilson(k, n)
    row = dict(model=model_name, topic=t, kind="scam" if lab else "normal", train_before=len(tr42), train_after=len(trn), pieces=n,
               pieces_train_split=int((d.split == "train").sum()), pieces_test_split=int((d.split == "test").sum()), cutoff=cut, calib_flagged=int((s_cal > cut).sum()),
               hit=k, rate=round(k / n, 4), lo95=round(lo, 4), hi95=round(hi, 4), fresh_flagged=int(fb.sum()), fresh_n=len(fb), fresh_rate=round(fb.mean(), 4),
               score_kind=score_kind, minutes=round((time.time() - t0) / 60, 2))
    os.makedirs(f"{HT}/{model_name}", exist_ok=True)
    pd.concat([d.assign(set=f"hidden_topic_{t}")[["set", "split", "source", "type", "label"]].assign(text_masked=m2(d["dialogue"]), score=s, flagged=fl.astype(int)),
               cut42.assign(set="calibration_3100_sms", split="train")[["set", "split", "source", "type", "label"]].assign(text_masked=m2(cut42["dialogue"]), score=s_cal, flagged=(s_cal > cut).astype(int)),
               big42.assign(set="fresh_normal_sms_big_test", split="test")[["set", "split", "source", "type", "label"]].assign(text_masked=m2(big42["dialogue"]), score=s_big, flagged=fb.astype(int))]
             ).assign(model=model_name, cutoff=cut, score_kind=score_kind).to_csv(f"{HT}/{model_name}/scores_{t}.csv", index=False)
    json.dump(row, open(f"{HT}/{model_name}/summary_{t}.json", "w"), indent=2)
    print(f"     cut-off {cut:.4f} (calib flagged {row['calib_flagged']}) | {'caught' if lab else 'flagged'} {k}/{n} = {k / n:.3f} [{lo:.3f}, {hi:.3f}] | fresh flagged {int(fb.sum())}/{len(fb)} = {fb.mean():.3f} | {row['minutes']} min | saved")
    return row
def show(rows, name):
    T = pd.DataFrame(rows); T.to_csv(f"{HT}/{name}/summary_all_topics.csv", index=False)
    print("\n", T[["topic", "kind", "train_after", "pieces", "cutoff", "hit", "rate", "lo95", "hi95", "fresh_flagged", "fresh_n", "fresh_rate"]].to_string(index=False))
    return T
HT_ROWS = {}
HT_ROWS["SVM_plain"] = [run_topic("SVM_plain", lambda trn: make_pipeline(tfidf(), LinearSVC(random_state=42)).fit(m2(trn["dialogue"]), trn["label"].values).decision_function,
                                   t, "decision_function margin") for t in SCAM_T + NORMAL_T]
show(HT_ROWS["SVM_plain"], "SVM_plain")
print("DONE hidden topics: plain LinearSVC")

Drive folder ready: True
training pieces 6844 | calibration SMS 3100 | fresh normal SMS 904
topic labels in data: {'refund': [1], 'reward': [1], 'ssn': [1], 'support': [1], 'delivery': [0], 'insurance': [0], 'telemarketing': [0], 'wrong': [0]}
[SVM_plain] refund (scam): training pieces 6844 -> 6081 (removed 763)
     cut-off -0.0090 (calib flagged 155) | caught 809/954 = 0.848 [0.824, 0.869] | fresh flagged 44/904 = 0.049 | 0.02 min | saved
[SVM_plain] reward (scam): training pieces 6844 -> 6176 (removed 668)
     cut-off -0.0231 (calib flagged 155) | caught 723/822 = 0.880 [0.856, 0.900] | fresh flagged 50/904 = 0.055 | 0.05 min | saved
[SVM_plain] ssn (scam): training pieces 6844 -> 6073 (removed 771)
     cut-off 0.0011 (calib flagged 155) | caught 932/962 = 0.969 [0.956, 0.978] | fresh flagged 54/904 = 0.060 | 0.02 min | saved
[SVM_plain] support (scam): training pieces 6844 -> 6138 (removed 706)
     cut-off -0.0252 (calib flagged 155) | caught 791/877 = 0.902 [0.880, 0.920] | fre

Model 2 of 3: DistilBERT.

In [ ]:
# Hidden topics at the 5% cut-off, model 2 of 3: DistilBERT (training split only, masked v2, seed 42)
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
DEVICE = "cuda"; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])


def fit_bert(trn):
    global _bm
    if "_bm" in globals():
        del _bm; gc.collect(); torch.cuda.empty_cache()
    _bm = DistilBERTClassifier(seed=42).fit(m2(trn["dialogue"]), trn["label"].values)
    return lambda x: _bm.predict_proba(x)[:, 1]
HT_ROWS["DistilBERT"] = [run_topic("DistilBERT", fit_bert, t, "probability of scam") for t in SCAM_T + NORMAL_T]
show(HT_ROWS["DistilBERT"], "DistilBERT")
print("DONE hidden topics: DistilBERT")

GPU: Tesla T4


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[DistilBERT] refund (scam): training pieces 6844 -> 6081 (removed 763)


... (22 outputs left out)

     cut-off 0.3459 (calib flagged 155) | flagged 293/1155 = 0.254 [0.229, 0.280] | fresh flagged 58/904 = 0.064 | 0.48 min | saved
[DistilBERT] wrong (normal): training pieces 6844 -> 6493 (removed 351)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


     cut-off 0.3728 (calib flagged 155) | flagged 251/437 = 0.574 [0.528, 0.620] | fresh flagged 42/904 = 0.046 | 0.52 min | saved

         topic   kind  train_after  pieces   cutoff  hit   rate   lo95   hi95  fresh_flagged  fresh_n  fresh_rate
       refund   scam         6081     954 0.237298  791 0.8291 0.8039 0.8517             52      904      0.0575
       reward   scam         6176     822 0.477572  571 0.6946 0.6623 0.7252             53      904      0.0586
          ssn   scam         6073     962 0.545041  932 0.9688 0.9558 0.9781             38      904      0.0420
      support   scam         6138     877 0.525254  757 0.8632 0.8388 0.8843             49      904      0.0542
     delivery normal         6188     814 0.212739  622 0.7641 0.7338 0.7920             49      904      0.0542
    insurance normal         5877    1209 0.179649   64 0.0529 0.0417 0.0670             52      904      0.0575
telemarketing normal         5910    1155 0.345890  293 0.2537 0.2294 0.2796

Model 3 of 3: LoRA.

In [ ]:
# Hidden topics at the 5% cut-off, model 3 of 3: LoRA, part 1 = four SCAM topics (training split only, masked v2, seed 42)
# Fix: load the base model ONCE, and for each run add a fresh LoRA adapter, train it, then remove it (unload).
# Needs only the GPU RESTART SETUP cell above.
!pip -q install peft
!pip -q uninstall -y torchao
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc, psutil
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); disable_progress_bars()
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

BASE = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
BASE.gradient_checkpointing_enable(); BASE.enable_input_require_grads()

class LoRALLM:
    _active = None
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4):
        self.seed, self.r, self.epochs, self.batch, self.lr = seed, r, epochs, batch, lr
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()
    def fit(self, X, y):
        global BASE
        if LoRALLM._active is not None:                     # remove the previous adapter, keep the base
            BASE = LoRALLM._active.model.unload(); LoRALLM._active.model = None
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(BASE, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        self.model.eval(); LoRALLM._active = self
        return self
    def predict(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return (out > 0.5).astype(int)


# Model 9 (LoRA): 3 seeds + big false-alarm test (same as seed_eval, without the hide-one-type part)
# Hide-one-type would add 27 training runs (about 5.5 hours), so those stay single-seed for LoRA.
# Each seed: new SMS split, new balancing sample, new LoRA seed; cut-off chosen on ~3,100 leftover real normal TRAIN SMS.
def _lora_pp(self, X, batch=32):
    texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(texts), batch):
            ids = order[b:b + batch]
            enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
            out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
    return np.column_stack([1 - out, out])
LoRALLM.predict_proba = _lora_pp


# Model 10: ensembles (average of member scores)
# Members, retrained on each seed's data: SVM (calibrated to give probabilities), DistilBERT, LoRA LLM, and the
# zero-shot LLM (same Qwen weights with the LoRA adapter switched off, so no second copy in memory).
# Each ensemble score = plain average of its members' 0-to-1 scores. Same harness: 3 seeds, big false-alarm test, 5% cut-offs.
# Needs: GPU RESTART SETUP cell + the LoRA cell above (LoRALLM, BASE, build, llm_tok) in this runtime.
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])

ZS_CACHE = {}
def zs_scores(texts, batch=32):
    """Zero-shot scores from the base Qwen model (LoRA adapter switched off). Cached, so each text is scored once."""
    texts = [str(t) for t in texts]; todo = sorted(set(t for t in texts if t not in ZS_CACHE), key=len)
    act = LoRALLM._active
    with act.model.disable_adapter(), torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(todo), batch):
            chunk = todo[b:b + batch]
            enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
            ZS_CACHE.update(zip(chunk, torch.softmax(act._last_logits(enc), dim=1)[:, 1].cpu().numpy()))
    return np.array([ZS_CACHE[t] for t in texts])

ENSEMBLES = {"E1 SVM+LoRA": ["SVM", "LoRA"], "E2 SVM+LoRA+zero-shot": ["SVM", "LoRA", "zero-shot"],
             "E3 LoRA+zero-shot": ["LoRA", "zero-shot"], "E4 all four": ["SVM", "DistilBERT", "LoRA", "zero-shot"]}


# ---------- new code below ----------
def fit_lora(trn):
    global _lm
    _lm = LoRALLM(seed=42).fit(m2(trn["dialogue"]), trn["label"].values)   # fit() removes the previous adapter and keeps BASE
    return lambda x: _lm.predict_proba(x)[:, 1]
print("GPU memory before LoRA runs:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
HT_ROWS["LoRA"] = HT_ROWS.get("LoRA", [])
for t in SCAM_T:
    HT_ROWS["LoRA"].append(run_topic("LoRA", fit_lora, t, "probability of scam"))
show(HT_ROWS["LoRA"], "LoRA")
print("DONE hidden topics: LoRA scam topics")

GPU memory before LoRA runs: 3.66 GB
[LoRA] refund (scam): training pieces 6844 -> 6081 (removed 763)
     cut-off 0.5549 (calib flagged 155) | caught 819/954 = 0.858 [0.835, 0.879] | fresh flagged 57/904 = 0.063 | 10.62 min | saved
[LoRA] reward (scam): training pieces 6844 -> 6176 (removed 668)
     cut-off 0.5156 (calib flagged 152) | caught 562/822 = 0.684 [0.651, 0.715] | fresh flagged 36/904 = 0.040 | 11.25 min | saved
[LoRA] ssn (scam): training pieces 6844 -> 6073 (removed 771)
     cut-off 0.6188 (calib flagged 154) | caught 934/962 = 0.971 [0.958, 0.980] | fresh flagged 48/904 = 0.053 | 11.05 min | saved
[LoRA] support (scam): training pieces 6844 -> 6138 (removed 706)
     cut-off 0.4811 (calib flagged 155) | caught 750/877 = 0.855 [0.830, 0.877] | fresh flagged 52/904 = 0.058 | 11.16 min | saved

   topic kind  train_after  pieces   cutoff  hit   rate   lo95   hi95  fresh_flagged  fresh_n  fresh_rate
 refund scam         6081     954 0.554856  819 0.8585 0.8349 0.8792      

**Extra check:** hide the scam topic and all real spam SMS from training.

In [ ]:
# Plain LinearSVC, scam topic removed AND all real spam SMS removed from training (training split only, masked v2, seed 42)
import os, json, numpy as np, pandas as pd
HT = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/hidden_topics_5pct" if os.path.isdir("/content/drive/MyDrive") else "hidden_topics_5pct_local"   # Drive not mounted: save locally
if "MASK_V2" not in globals():
    all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
    MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
tr42, te42, s_tr42, s_te42 = build_data(42)
full42 = pd.concat([tr42.assign(split="train"), te42.assign(split="test")], ignore_index=True)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]; big42 = s_te42[s_te42.label == 0]
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
EARLIER = {"refund": (809, 954, .848, .824, .869), "reward": (723, 822, .880, .856, .900), "ssn": (932, 962, .969, .956, .978), "support": (791, 877, .902, .880, .920)}   # from the plain LinearSVC hidden-topic run (printed table)
print("saving to:", HT)
print("real spam SMS in the seed-42 training split:", int((tr42.type == "sms_spam").sum()), "| real normal SMS kept:", int((tr42.type == "sms_ham").sum()))
rs_te = te42[(te42.source == "real") & (te42.label == 1)]
OUT = f"{HT}/SVM_plain_no_real_spam"; os.makedirs(OUT, exist_ok=True); ROWS = []
for t in ["reward", "refund", "ssn", "support"]:
    trn = tr42[(tr42.type != t) & (tr42.type != "sms_spam")]
    print(f"\n{t}: training pieces {len(tr42)} -> {len(trn)} (removed {int((tr42.type == t).sum())} {t} pieces + {int((tr42.type == 'sms_spam').sum())} real spam SMS) | labels left: {trn.label.value_counts().to_dict()}")
    m = make_pipeline(tfidf(), LinearSVC(random_state=42)).fit(m2(trn["dialogue"]), trn["label"].values)
    s_cal = m.decision_function(m2(cut42["dialogue"])); cut = float(np.quantile(s_cal, 0.95))
    d = full42[full42.type == t]; s = m.decision_function(m2(d["dialogue"])); fl = s > cut
    fb = m.decision_function(m2(big42["dialogue"])) > cut; frs = m.decision_function(m2(rs_te["dialogue"])) > cut
    k, n = int(fl.sum()), len(fl); lo, hi = wilson(k, n); e = EARLIER[t]
    print(f"   cut-off {cut:.4f} (calib flagged {int((s_cal > cut).sum())}) | {t} caught {k}/{n} = {k / n:.3f} [{lo:.3f}, {hi:.3f}] | earlier (real spam kept) {e[0]}/{e[1]} = {e[2]:.3f} [{e[3]:.3f}, {e[4]:.3f}]")
    print(f"   fresh normal flagged {int(fb.sum())}/{len(fb)} = {fb.mean():.3f} | real spam test caught {int(frs.sum())}/{len(frs)} = {frs.mean():.3f}")
    ROWS.append(dict(topic=t, train_after=len(trn), pieces=n, cutoff=cut, calib_flagged=int((s_cal > cut).sum()), hit=k, rate=round(k / n, 4), lo95=round(lo, 4), hi95=round(hi, 4),
                     earlier_rate=e[2], earlier_lo95=e[3], earlier_hi95=e[4], fresh_flagged=int(fb.sum()), fresh_n=len(fb), real_spam_test_caught=int(frs.sum()), real_spam_test_n=len(frs)))
    d.assign(text_masked=m2(d["dialogue"]), score=s, flagged=fl.astype(int), cutoff=cut)[["split", "source", "type", "label", "text_masked", "score", "flagged", "cutoff"]].to_csv(f"{OUT}/scores_{t}.csv", index=False)
T = pd.DataFrame(ROWS); T.to_csv(f"{OUT}/summary_all_topics.csv", index=False)
print("\n", T[["topic", "train_after", "pieces", "cutoff", "hit", "rate", "lo95", "hi95", "earlier_rate", "fresh_flagged", "real_spam_test_caught"]].to_string(index=False))
print("saved to Drive:", sorted(os.listdir(OUT)))
print("DONE no-real-spam check")

saving to: hidden_topics_5pct_local
real spam SMS in the seed-42 training split: 514 | real normal SMS kept: 514

reward: training pieces 6844 -> 5662 (removed 668 reward pieces + 514 real spam SMS) | labels left: {0: 3422, 1: 2240}
   cut-off -0.3543 (calib flagged 155) | reward caught 782/822 = 0.951 [0.934, 0.964] | earlier (real spam kept) 723/822 = 0.880 [0.856, 0.900]
   fresh normal flagged 64/904 = 0.071 | real spam test caught 14/128 = 0.109

refund: training pieces 6844 -> 5567 (removed 763 refund pieces + 514 real spam SMS) | labels left: {0: 3422, 1: 2145}
   cut-off -0.3376 (calib flagged 155) | refund caught 895/954 = 0.938 [0.921, 0.952] | earlier (real spam kept) 809/954 = 0.848 [0.824, 0.869]
   fresh normal flagged 56/904 = 0.062 | real spam test caught 21/128 = 0.164

ssn: training pieces 6844 -> 5559 (removed 771 ssn pieces + 514 real spam SMS) | labels left: {0: 3422, 1: 2137}
   cut-off -0.3254 (calib flagged 155) | ssn caught 957/962 = 0.995 [0.988, 0.998] | earl

In [ ]:
# Mount Drive and copy the no-real-spam SVM results into hidden_topics_5pct
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
SRC = "/content/hidden_topics_5pct_local/SVM_plain_no_real_spam"
DST = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/hidden_topics_5pct/SVM_plain_no_real_spam"
print("local folder exists:", os.path.isdir(SRC), "| files:", sorted(os.listdir(SRC)) if os.path.isdir(SRC) else None)
if os.path.isdir(SRC):
    shutil.copytree(SRC, DST, dirs_exist_ok=True)
    for f in sorted(os.listdir(DST)):
        print(f"  {f}: Drive {os.path.getsize(os.path.join(DST, f))} bytes | local {os.path.getsize(os.path.join(SRC, f)) if os.path.exists(os.path.join(SRC, f)) else 'missing'} bytes")
    print("COPIED")
else:
    print("NEEDS RERUN")

Mounted at /content/drive
local folder exists: True | files: ['scores_refund.csv', 'scores_reward.csv', 'scores_ssn.csv', 'scores_support.csv', 'summary_all_topics.csv']
  scores_refund.csv: Drive 301189 bytes | local 301189 bytes
  scores_reward.csv: Drive 258002 bytes | local 258002 bytes
  scores_ssn.csv: Drive 304164 bytes | local 304164 bytes
  scores_support.csv: Drive 276928 bytes | local 276928 bytes
  summary_all_topics.csv: Drive 548 bytes | local 548 bytes
COPIED


# 11. Chat cut-offs and the call-level calibration split
Normal chat pieces score differently from normal SMS, so one SMS cut-off does not fit chats. These cells set a separate 5% cut-off for chats using held-out normal calls, split by call so pieces of one call never sit on both sides.

First: how normal chat pieces score compared with normal SMS.

In [ ]:
# Where do normal dialogue pieces score compared with real normal SMS? plain LinearSVC, reward hidden, with and without real spam
import numpy as np, pandas as pd
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
if "MASK_V2" not in globals():
    all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
    MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
tr42, te42, s_tr42, s_te42 = build_data(42)
full42 = pd.concat([tr42.assign(split="train"), te42.assign(split="test")], ignore_index=True)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]
SETS = {"real normal SMS (3,100 calibration)": cut42,
        "normal dialogue, test split (707)": te42[(te42.source == "synthetic") & (te42.label == 0)],
        "scam dialogue, test split, not reward": te42[(te42.source == "synthetic") & (te42.label == 1) & (te42.type != "reward")],
        "reward pieces, both splits (hidden)": full42[full42.type == "reward"],
        "real spam test (128)": te42[(te42.source == "real") & (te42.label == 1)]}
for name, trn in [("WITH real spam (earlier, .880)", tr42[tr42.type != "reward"]), ("WITHOUT real spam (new, .951)", tr42[(tr42.type != "reward") & (tr42.type != "sms_spam")])]:
    m = make_pipeline(tfidf(), LinearSVC(random_state=42)).fit(m2(trn["dialogue"]), trn["label"].values)
    sc = {k: m.decision_function(m2(d["dialogue"])) for k, d in SETS.items()}
    cut = float(np.quantile(sc["real normal SMS (3,100 calibration)"], 0.95))
    print(f"\n=== {name}: trained on {len(trn)} pieces | 5% cut-off (from real normal SMS) {cut:.4f}")
    print(f"  {'set':42s} {'n':>5s} {'p5':>8s} {'median':>8s} {'p95':>8s} {'share above cut-off':>20s}")
    for k, v in sc.items():
        print(f"  {k:42s} {len(v):5d} {np.quantile(v, .05):8.3f} {np.median(v):8.3f} {np.quantile(v, .95):8.3f} {int((v > cut).sum()):>8d} = {np.mean(v > cut):.3f}")
print("\nDONE score-location check")


=== WITH real spam (earlier, .880): trained on 6176 pieces | 5% cut-off (from real normal SMS) -0.0231
  set                                            n       p5   median      p95  share above cut-off
  real normal SMS (3,100 calibration)         3100   -1.349   -0.672   -0.023      155 = 0.050
  normal dialogue, test split (707)            707   -1.645   -1.150   -0.385       10 = 0.014
  scam dialogue, test split, not reward        553    0.331    1.154    1.869      549 = 0.993
  reward pieces, both splits (hidden)          822   -0.264    0.515    1.280      723 = 0.880
  real spam test (128)                         128    0.024    0.879    1.513      122 = 0.953

=== WITHOUT real spam (new, .951): trained on 5662 pieces | 5% cut-off (from real normal SMS) -0.3543
  set                                            n       p5   median      p95  share above cut-off
  real normal SMS (3,100 calibration)         3100   -1.355   -0.877   -0.354      155 = 0.050
  normal dialogue, test s

What a chat-based cut-off does to the SMS false-alarm rate.

In [ ]:
# Dialogue-source cut-off: 5% of normal DIALOGUE flagged, then what it does to the 3,100 calibration SMS (plain LinearSVC, reward hidden)
import numpy as np
big42 = s_te42[s_te42.label == 0]
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
DIA_TE = te42[(te42.source == "synthetic") & (te42.label == 0)]
EVAL = {"calibration SMS": cut42, "fresh SMS": big42, "normal dialogue test": DIA_TE, "reward (hidden)": full42[full42.type == "reward"],
        "real spam test": te42[(te42.source == "real") & (te42.label == 1)]}
for name, trn in [("WITH real spam", tr42[tr42.type != "reward"]), ("WITHOUT real spam", tr42[(tr42.type != "reward") & (tr42.type != "sms_spam")])]:
    m = make_pipeline(tfidf(), LinearSVC(random_state=42)).fit(m2(trn["dialogue"]), trn["label"].values)
    S = {k: m.decision_function(m2(d["dialogue"])) for k, d in EVAL.items()}
    dia_tr = trn[(trn.source == "synthetic") & (trn.label == 0)]; s_dtr = m.decision_function(m2(dia_tr["dialogue"]))
    CUTS = {"SMS-source (95th pct of 3,100 calibration SMS)": np.quantile(S["calibration SMS"], 0.95),
            f"dialogue-source A (95th pct of {len(DIA_TE)} test-split normal dialogue)": np.quantile(S["normal dialogue test"], 0.95),
            f"dialogue-source B (95th pct of {len(dia_tr)} training normal dialogue)": np.quantile(s_dtr, 0.95)}
    print(f"\n=== {name} (trained on {len(trn)} pieces)")
    for cn, c in CUTS.items():
        out = []
        for k, v in S.items():
            f = v > c; lo, hi = wilson(int(f.sum()), len(f)); out.append(f"{k} {int(f.sum())}/{len(f)} = {f.mean():.3f} [{lo:.3f}, {hi:.3f}]")
        print(f"  {cn}: cut-off {c:.4f}\n     " + "\n     ".join(out))
print("\nDONE dialogue-source cut-off")


=== WITH real spam (trained on 6176 pieces)
  SMS-source (95th pct of 3,100 calibration SMS): cut-off -0.0231
     calibration SMS 155/3100 = 0.050 [0.043, 0.058]
     fresh SMS 50/904 = 0.055 [0.042, 0.072]
     normal dialogue test 10/707 = 0.014 [0.008, 0.026]
     reward (hidden) 723/822 = 0.880 [0.856, 0.900]
     real spam test 122/128 = 0.953 [0.902, 0.978]
  dialogue-source A (95th pct of 707 test-split normal dialogue): cut-off -0.3846
     calibration SMS 686/3100 = 0.221 [0.207, 0.236]
     fresh SMS 198/904 = 0.219 [0.193, 0.247]
     normal dialogue test 36/707 = 0.051 [0.037, 0.070]
     reward (hidden) 792/822 = 0.964 [0.948, 0.974]
     real spam test 126/128 = 0.984 [0.945, 0.996]
  dialogue-source B (95th pct of 2908 training normal dialogue): cut-off -0.7526
     calibration SMS 1821/3100 = 0.587 [0.570, 0.605]
     fresh SMS 548/904 = 0.606 [0.574, 0.638]
     normal dialogue test 105/707 = 0.149 [0.124, 0.177]
     reward (hidden) 818/822 = 0.995 [0.988, 0.998]
  

Builds the call-level calibration split and the two cut-offs (SMS and chat) for the plain LinearSVC.

In [ ]:
# Call-level dialogue calibration split + two 5% cut-offs (SMS-source, dialogue-source), plain LinearSVC, seed 42, masked v2
import os, numpy as np, pandas as pd
from collections import defaultdict
from sklearn.metrics import roc_auc_score
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
big42 = s_te42[s_te42.label == 0]
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
# ---- 1. piece -> call id, same chunking rule as make_pieces (max 45 words, keep chunks of 10+ words) ----
raw_tr = ds["train"].to_pandas(); CALLS = defaultdict(set)
for cid, (text, typ, lab) in enumerate(zip(raw_tr["dialogue"], raw_tr["type"], raw_tr["label"])):
    w = text.split()
    for i in range(0, len(w), 45):
        ch = w[i:i + 45]
        if len(ch) >= 10: CALLS[" ".join(ch)].add(cid)
nd = tr42[(tr42.source == "synthetic") & (tr42.label == 0)].copy()
nd["call"] = [min(CALLS[t]) if t in CALLS else -1 for t in nd["dialogue"]]
amb = sum(len(CALLS[t]) > 1 for t in nd["dialogue"] if t in CALLS)
print(f"normal dialogue pieces in training split: {len(nd)} | from {nd.call.nunique()} calls | not matched to a call: {int((nd.call == -1).sum())} | text found in more than one call: {amb}")
# ---- 2. split CALLS 80/20 within each normal topic (seed 42) ----
rng = np.random.default_rng(42); cal_calls = set()
for typ, g in nd.groupby("type"):
    calls = np.array(sorted(g.call.unique())); rng.shuffle(calls); k = int(round(0.2 * len(calls))); cal_calls |= set(calls[:k].tolist())
    print(f"  {typ}: {len(calls)} calls -> {k} calibration calls")
cal_dia = nd[nd.call.isin(cal_calls)]; trn_dia = nd[~nd.call.isin(cal_calls)]
print(f"dialogue calibration: {len(cal_dia)} pieces from {cal_dia.call.nunique()} calls | dialogue normal kept for training: {len(trn_dia)} pieces from {trn_dia.call.nunique()} calls")
print("calls in both parts:", len(set(cal_dia.call) & set(trn_dia.call)), "| calibration texts also in kept training text:", len(set(cal_dia.dialogue) & set(trn_dia.dialogue)))
print("calibration pieces by topic:", cal_dia.type.value_counts().to_dict())
base = tr42[~tr42.dialogue.isin(cal_dia.dialogue)]     # training split minus the calibration-dialogue pieces
print(f"training split {len(tr42)} -> {len(base)} after removing the dialogue calibration pieces")
# ---- 3. evaluation sets (test sets unchanged) ----
EV = {"calibration_sms_3100": cut42, "calibration_dialogue": cal_dia, "fresh_sms_904": big42,
      "dialogue_normal_test": te42[(te42.source == "synthetic") & (te42.label == 0)], "dialogue_scam_test": te42[(te42.source == "synthetic") & (te42.label == 1)],
      "real_sms_test": te42[te42.source == "real"]}
for t in ["reward", "refund", "ssn", "support"]: EV[f"topic_{t}"] = full42[full42.type == t]
print({k: len(v) for k, v in EV.items()})
OUTD = "dialogue_calibration_split"; os.makedirs(OUTD, exist_ok=True); ROWS, SCORE_FILES = [], []
for variant in ["with real spam", "without real spam"]:
    for hid in [None, "reward", "refund", "ssn", "support"]:
        trn = base.copy()
        if variant == "without real spam": trn = trn[trn.type != "sms_spam"]
        if hid: trn = trn[trn.type != hid]
        m = make_pipeline(tfidf(), LinearSVC(random_state=42)).fit(m2(trn["dialogue"]), trn["label"].values)
        S = {k: m.decision_function(m2(d["dialogue"])) for k, d in EV.items()}
        CUT = {"SMS-source": float(np.quantile(S["calibration_sms_3100"], 0.95)), "dialogue-source": float(np.quantile(S["calibration_dialogue"], 0.95))}
        rs = EV["real_sms_test"]; ds_ = pd.concat([EV["dialogue_normal_test"], EV["dialogue_scam_test"]])
        auc_sms = roc_auc_score(rs.label, S["real_sms_test"]); auc_dia = roc_auc_score(ds_.label, np.r_[S["dialogue_normal_test"], S["dialogue_scam_test"]])
        print(f"\n=== {variant} | hidden: {hid or 'none'} | trained on {len(trn)} pieces | AUC SMS alone {auc_sms:.4f} | AUC dialogue alone {auc_dia:.4f}")
        for rule, c in CUT.items():
            r = dict(variant=variant, hidden=hid or "none", train_n=len(trn), rule=rule, cutoff=c, auc_sms=round(auc_sms, 4), auc_dialogue=round(auc_dia, 4))
            msg = []
            for k, lab in [("calibration_sms_3100", None), ("calibration_dialogue", None), ("fresh_sms_904", None), ("dialogue_normal_test", None)] + ([(f"topic_{hid}", None)] if hid else []) + [("real_sms_test", 1)]:
                v = S[k] if lab is None else S[k][(EV[k].label == lab).values]; f = v > c; kk, n = int(f.sum()), len(f); lo, hi = wilson(kk, n)
                name = "real_spam_test_128" if k == "real_sms_test" else ("hidden_topic" if k.startswith("topic_") else k)
                r.update({f"{name}_k": kk, f"{name}_n": n, f"{name}_rate": round(kk / n, 4), f"{name}_lo95": round(lo, 4), f"{name}_hi95": round(hi, 4)})
                msg.append(f"{name} {kk}/{n} = {kk / n:.3f} [{lo:.3f}, {hi:.3f}]")
            print(f"  {rule:15s} cut-off {c:.4f}\n     " + "\n     ".join(msg)); ROWS.append(r)
        fn = f"{OUTD}/scores_{variant.replace(' ', '_')}_hidden_{hid or 'none'}.csv"
        pd.concat([d.assign(set=k)[["set", "source", "type", "label"]].assign(text_masked=m2(d["dialogue"]), score=S[k]) for k, d in EV.items()]).assign(
            variant=variant, hidden=hid or "none", cutoff_sms_source=CUT["SMS-source"], cutoff_dialogue_source=CUT["dialogue-source"]).to_csv(fn, index=False); SCORE_FILES.append(fn)
cal_dia.assign(text_masked=m2(cal_dia["dialogue"]))[["call", "type", "text_masked"]].to_csv(f"{OUTD}/dialogue_calibration_pieces.csv", index=False)
T = pd.DataFrame(ROWS); T.to_csv(f"{OUTD}/summary.csv", index=False)
print("\nsaved locally:", len(os.listdir(OUTD)), "files")
try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
    import shutil; DST = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/dialogue_calibration_split"; shutil.copytree(OUTD, DST, dirs_exist_ok=True)
    print("copied to Drive:", {f: os.path.getsize(f"{DST}/{f}") for f in sorted(os.listdir(DST))})
except Exception as e:
    print("Drive copy FAILED:", repr(e))
print("DONE call-level calibration split")

normal dialogue pieces in training split: 2908 | from 639 calls | not matched to a call: 0 | text found in more than one call: 89
  delivery: 160 calls -> 32 calibration calls
  insurance: 160 calls -> 32 calibration calls
  telemarketing: 163 calls -> 33 calibration calls
  wrong: 159 calls -> 32 calibration calls
dialogue calibration: 577 pieces from 129 calls | dialogue normal kept for training: 2331 pieces from 510 calls
calls in both parts: 0 | calibration texts also in kept training text: 0
calibration pieces by topic: {'insurance': 195, 'telemarketing': 179, 'delivery': 136, 'wrong': 67}
training split 6844 -> 6267 after removing the dialogue calibration pieces
{'calibration_sms_3100': 3100, 'calibration_dialogue': 577, 'fresh_sms_904': 904, 'dialogue_normal_test': 707, 'dialogue_scam_test': 707, 'real_sms_test': 256, 'topic_reward': 822, 'topic_refund': 954, 'topic_ssn': 962, 'topic_support': 877}

=== with real spam | hidden: none | trained on 6267 pieces | AUC SMS alone 0.988

The same two cut-offs for the six quick models, then DistilBERT and the zero-shot model.

In [ ]:
# Two cut-offs (SMS-source, dialogue-source) on the standard call-level split: six quick models, real spam kept, seed 42
from xgboost import XGBClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
# Model 5: CNN for text (PyTorch, runs on the GPU)
# Words -> numbers -> embedding (128 numbers per word) -> filters that look at 3, 4 and 5 words at a time
# -> keep each filter's strongest match anywhere in the message -> one scam score.
# Same fit/predict/predict_proba interface as the sklearn models, so evaluate() and seed_eval() work unchanged.
import re, torch, torch.nn as nn
from collections import Counter
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
TOKEN = re.compile(r"[a-z0-9£$]+|[!?]")
def tokenize(t): return TOKEN.findall(str(t).lower())

class TextCNN(nn.Module):
    def __init__(self, vocab, dim=128, filters=100, sizes=(3, 4, 5), drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(dim, filters, k) for k in sizes])
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(filters * len(sizes), 1)
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        h = torch.cat([torch.relu(c(e)).max(dim=2).values for c in self.convs], dim=1)
        return self.out(self.drop(h)).squeeze(1)

class CNNClassifier:
    def __init__(self, seed=42, max_len=64, max_vocab=20000, epochs=8, patience=2, batch=64, lr=1e-3, verbose=False):
        self.seed, self.max_len, self.max_vocab = seed, max_len, max_vocab
        self.epochs, self.patience, self.batch, self.lr, self.verbose = epochs, patience, batch, lr, verbose
    def _encode(self, texts):
        X = np.zeros((len(texts), self.max_len), dtype=np.int64)
        for i, t in enumerate(texts):
            ids = [self.vocab.get(w, 1) for w in tokenize(t)][:self.max_len]
            X[i, :len(ids)] = ids
        return torch.tensor(X)
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = np.asarray(labels, dtype=np.float32)
        # 10% of the training rows are held back to decide when to stop training (early stopping)
        tr_i, va_i = train_test_split(np.arange(len(texts)), test_size=0.1, stratify=y, random_state=self.seed)
        cnt = Counter(w for i in tr_i for w in tokenize(texts[i]))
        words = [w for w, c in cnt.most_common(self.max_vocab - 2) if c >= 2]
        self.vocab = {w: i + 2 for i, w in enumerate(words)}          # 0 = padding, 1 = unknown word
        X, Y = self._encode(texts), torch.tensor(y)
        self.model = TextCNN(len(self.vocab) + 2).to(DEVICE)
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.BCEWithLogitsLoss()
        g = torch.Generator().manual_seed(self.seed)
        Xva, Yva = X[va_i].to(DEVICE), Y[va_i].to(DEVICE)
        best, best_state, bad = 1e9, None, 0
        for ep in range(self.epochs):
            self.model.train()
            perm = torch.tensor(tr_i)[torch.randperm(len(tr_i), generator=g)]
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                opt.zero_grad()
                loss = loss_fn(self.model(X[idx].to(DEVICE)), Y[idx].to(DEVICE))
                loss.backward(); opt.step()
            self.model.eval()
            with torch.no_grad():
                vl = loss_fn(self.model(Xva), Yva).item()
            if self.verbose: print(f"    epoch {ep + 1}: validation loss {vl:.4f}")
            if vl < best - 1e-4:
                best, bad = vl, 0
                best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            else:
                bad += 1
                if bad >= self.patience: break
        self.model.load_state_dict(best_state)
        self.epochs_run = ep + 1
        return self
    def predict_proba(self, texts):
        self.model.eval(); X = self._encode(list(texts)); out = []
        with torch.no_grad():
            for b in range(0, len(X), 512):
                out.append(torch.sigmoid(self.model(X[b:b + 512].to(DEVICE))).cpu().numpy())
        p = np.concatenate(out)
        return np.column_stack([1 - p, p])
    def predict(self, texts):
        return (self.predict_proba(texts)[:, 1] > 0.5).astype(int)


# Model 6: BiLSTM (PyTorch, GPU)
# Reads the message word by word, left to right AND right to left, carrying a memory along the way.
# The final memory of each direction is joined and turned into one scam score.
# Same tokens, vocabulary rule, early stopping and interface as the CNN, so the only change is the network.
class BiLSTMNet(nn.Module):
    def __init__(self, vocab, dim=128, hidden=128, drop=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.lstm = nn.LSTM(dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(2 * hidden, 1)
    def forward(self, x):
        lengths = (x != 0).sum(1).clamp(min=1).cpu()        # real length of each message, so padding is skipped
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(x), lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.out(self.drop(torch.cat([h[0], h[1]], dim=1))).squeeze(1)

class BiLSTMClassifier(CNNClassifier):
    """Same training loop as CNNClassifier; only the network is swapped."""
    def fit(self, texts, labels):
        global TextCNN
        saved, TextCNN = TextCNN, BiLSTMNet      # CNNClassifier.fit builds TextCNN(vocab); point it at the BiLSTM for this call
        try:
            return super().fit(texts, labels)
        finally:
            TextCNN = saved



# MASKED RERUN, step 4a: seed 42, six models, both training ways, raw AND masked
# Same splits, seed, cut-off rule and tests as seed_eval / evaluate; only the text fed to the model changes.
# The raw run is a check: it should reproduce the earlier seed-42 numbers. Hand-written count = right of 10.
from xgboost import XGBClassifier

class RepeatRealSet:
    """Repeats real-SMS rows k times before training (same as RepeatReal), using the real-SMS set of the text version in use."""
    def __init__(self, base, k, real_set):
        self.base, self.k, self.real_set = base, k, real_set
    def fit(self, X, y):
        X = pd.Series(list(X)); y = pd.Series(np.asarray(y)); real = X.isin(self.real_set).values
        self.base.fit(pd.concat([X] + [X[real]] * (self.k - 1), ignore_index=True),
                      pd.concat([y] + [y[real]] * (self.k - 1), ignore_index=True))
        return self
    def predict(self, X): return self.base.predict(X)
    def __getattr__(self, name): return getattr(self.base, name)

SIX = {
    "NB":     lambda: make_pipeline(tfidf(), MultinomialNB()),
    "LR":     lambda: make_pipeline(tfidf(), LogisticRegression(max_iter=1000, random_state=42)),
    "SVM":    lambda: make_pipeline(tfidf(), LinearSVC(random_state=42)),
    "XGB":    lambda: make_pipeline(tfidf(), XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist", device="cuda", random_state=42)),
    "CNN":    lambda: CNNClassifier(seed=42),
    "BiLSTM": lambda: BiLSTMClassifier(seed=42),
}

# ---------- new code: shared helper for the two-cut-off evaluation (standard split in dialogue_calibration_split) ----------
import os, gc, time, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
DCS = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/dialogue_calibration_split"; MD = f"{DCS}/models"
need = [v for v in ["base", "cal_dia", "EV", "cut42", "big42", "te42", "full42", "m2", "wilson"] if v not in globals()]
assert not need, f"missing from memory, rerun the call-level split cell first: {need}"
assert os.path.isdir(DCS), "Drive not mounted"
saved = pd.read_csv(f"{DCS}/dialogue_calibration_pieces.csv", keep_default_na=False)
print("standard split check: calibration pieces in memory", len(cal_dia), "| on Drive", len(saved), "| identical texts:", list(m2(cal_dia["dialogue"])) == list(saved.text_masked))
print("training pieces (real spam kept, nothing hidden):", len(base))
HIDS = [None, "reward", "refund", "ssn", "support"]; ALLROWS = globals().get("ALLROWS", [])
def two_cut_eval(name, fit_fn, retrain=True):
    os.makedirs(f"{MD}/{name}", exist_ok=True); rows, files = [], []; S = None
    for hid in HIDS:
        t0 = time.time()
        if retrain or S is None:
            trn = base if hid is None else base[base.type != hid]
            if retrain: print(f"[{name}] hidden {hid or 'none'}: training pieces {len(base)} -> {len(trn)}")
            scorer = fit_fn(trn); S = {k: np.asarray(scorer(m2(d["dialogue"]))) for k, d in EV.items()}
        c_sms, c_dia = float(np.quantile(S["calibration_sms_3100"], 0.95)), float(np.quantile(S["calibration_dialogue"], 0.95))
        rs = EV["real_sms_test"]; spam = (rs.label == 1).values
        dia_lab = np.r_[np.zeros(len(EV["dialogue_normal_test"])), np.ones(len(EV["dialogue_scam_test"]))]
        r = dict(model=name, hidden=hid or "none", retrained=bool(retrain), cut_sms=c_sms, cut_dialogue=c_dia,
                 auc_sms=round(roc_auc_score(rs.label, S["real_sms_test"]), 4), auc_dialogue=round(roc_auc_score(dia_lab, np.r_[S["dialogue_normal_test"], S["dialogue_scam_test"]]), 4))
        def add(tag, v, c):
            f = v > c; k, n = int(f.sum()), len(f); lo, hi = wilson(k, n); r.update({f"{tag}_k": k, f"{tag}_n": n, f"{tag}_rate": round(k / n, 4), f"{tag}_lo95": round(lo, 4), f"{tag}_hi95": round(hi, 4)})
        add("calsms_at_diacut", S["calibration_sms_3100"], c_dia); add("fresh_at_diacut", S["fresh_sms_904"], c_dia)
        add("calsms_at_smscut", S["calibration_sms_3100"], c_sms); add("fresh_at_smscut", S["fresh_sms_904"], c_sms)
        add("caldia_at_diacut", S["calibration_dialogue"], c_dia); add("dianorm_at_smscut", S["dialogue_normal_test"], c_sms); add("dianorm_at_diacut", S["dialogue_normal_test"], c_dia)
        add("spam_at_smscut", S["real_sms_test"][spam], c_sms); add("spam_at_diacut", S["real_sms_test"][spam], c_dia)
        if hid: add("topic_at_smscut", S[f"topic_{hid}"], c_sms); add("topic_at_diacut", S[f"topic_{hid}"], c_dia)
        rows.append(r)
        msg = (f"  cut SMS {c_sms:.4f} | cut dia {c_dia:.4f} | AUC SMS {r['auc_sms']:.4f} dia {r['auc_dialogue']:.4f} | calSMS@dia {r['calsms_at_diacut_k']}/3100={r['calsms_at_diacut_rate']:.3f} | "
               f"fresh@dia {r['fresh_at_diacut_k']}/904={r['fresh_at_diacut_rate']:.3f} | dianorm@SMS {r['dianorm_at_smscut_k']}/707={r['dianorm_at_smscut_rate']:.3f} | spam@SMS {r['spam_at_smscut_k']}/128")
        if hid: msg += f" | {hid} @SMS {r['topic_at_smscut_k']}/{r['topic_at_smscut_n']}={r['topic_at_smscut_rate']:.3f} @dia {r['topic_at_diacut_k']}/{r['topic_at_diacut_n']}={r['topic_at_diacut_rate']:.3f} [{r['topic_at_diacut_lo95']:.3f}, {r['topic_at_diacut_hi95']:.3f}]"
        print(f"[{name}] hidden {hid or 'none'}{'' if retrain else ' (no training: same scores)'}\n" + msg + f" | {time.time() - t0:.0f} s")
        if retrain or hid is None:
            fn = f"{MD}/{name}/scores_hidden_{hid or 'none'}.csv"
            pd.concat([d.assign(set=k)[["set", "source", "type", "label"]].assign(text_masked=m2(d["dialogue"]), score=S[k]) for k, d in EV.items()]).assign(
                model=name, hidden=hid or "none", cut_sms=c_sms, cut_dialogue=c_dia).to_csv(fn, index=False); files.append(fn)
    pd.DataFrame(rows).to_csv(f"{MD}/{name}/summary_{name}.csv", index=False); files.append(f"{MD}/{name}/summary_{name}.csv")
    ALLROWS.extend(rows); pd.DataFrame(ALLROWS).to_csv(f"{MD}/summary_all_models.csv", index=False)
    print(f"[{name}] SAVED:", [(f.replace(DCS + "/", ""), os.path.getsize(f)) for f in files])
    return rows

for nm, mk in SIX.items():
    two_cut_eval(nm, lambda trn, mk=mk: (lambda mod: (lambda x: scores(mod, x)))(mk().fit(m2(trn["dialogue"]), trn["label"].values)))
print("DONE six quick models")

standard split check: calibration pieces in memory 577 | on Drive 577 | identical texts: True
training pieces (real spam kept, nothing hidden): 6267
[NB] hidden none: training pieces 6267 -> 6267
[NB] hidden none
  cut SMS 0.7114 | cut dia 0.3804 | AUC SMS 0.9743 dia 0.9977 | calSMS@dia 1245/3100=0.402 | fresh@dia 368/904=0.407 | dianorm@SMS 16/707=0.023 | spam@SMS 116/128 | 2 s
[NB] hidden reward: training pieces 6267 -> 5599
[NB] hidden reward
  cut SMS 0.6511 | cut dia 0.2591 | AUC SMS 0.9759 dia 0.9962 | calSMS@dia 1685/3100=0.543 | fresh@dia 499/904=0.552 | dianorm@SMS 14/707=0.020 | spam@SMS 116/128 | reward @SMS 630/822=0.766 @dia 765/822=0.931 [0.911, 0.946] | 3 s
[NB] hidden refund: training pieces 6267 -> 5504
[NB] hidden refund
  cut SMS 0.6313 | cut dia 0.1981 | AUC SMS 0.9775 dia 0.9954 | calSMS@dia 2061/3100=0.665 | fresh@dia 606/904=0.670 | dianorm@SMS 12/707=0.017 | spam@SMS 117/128 | refund @SMS 736/954=0.771 @dia 894/954=0.937 [0.920, 0.951] | 2 s
[NB] hidden ssn: tra

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [10:53:11] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[XGB] hidden none
  cut SMS 0.3866 | cut dia 0.3096 | AUC SMS 0.9889 dia 0.9952 | calSMS@dia 242/3100=0.078 | fresh@dia 72/904=0.080 | dianorm@SMS 40/707=0.057 | spam@SMS 123/128 | 5 s
[XGB] hidden reward: training pieces 6267 -> 5599
[XGB] hidden reward
  cut SMS 0.3587 | cut dia 0.2418 | AUC SMS 0.9904 dia 0.9918 | calSMS@dia 282/3100=0.091 | fresh@dia 84/904=0.093 | dianorm@SMS 39/707=0.055 | spam@SMS 124/128 | reward @SMS 703/822=0.855 @dia 736/822=0.895 [0.873, 0.914] | 4 s
[XGB] hidden refund: training pieces 6267 -> 5504
[XGB] hidden refund
  cut SMS 0.3811 | cut dia 0.1917 | AUC SMS 0.9898 dia 0.9922 | calSMS@dia 422/3100=0.136 | fresh@dia 125/904=0.138 | dianorm@SMS 31/707=0.044 | spam@SMS 122/128 | refund @SMS 815/954=0.854 @dia 880/954=0.922 [0.904, 0.938] | 4 s
[XGB] hidden ssn: training pieces 6267 -> 5496
[XGB] hidden ssn
  cut SMS 0.3378 | cut dia 0.2584 | AUC SMS 0.9933 dia 0.9929 | calSMS@dia 242/3100=0.078 | fresh@dia 75/904=0.083 | dianorm@SMS 38/707=0.054 | spam@SMS

In [ ]:
# Two cut-offs on the standard call-level split: DistilBERT, real spam kept, seed 42
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
DEVICE = "cuda"; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])



def fit_bert2(trn):
    global _bm2
    if "_bm2" in globals():
        del _bm2; gc.collect(); torch.cuda.empty_cache()
    _bm2 = DistilBERTClassifier(seed=42).fit(m2(trn["dialogue"]), trn["label"].values)
    return lambda x: _bm2.predict_proba(x)[:, 1]
two_cut_eval("DistilBERT", fit_bert2)
print("DONE DistilBERT")

GPU: Tesla T4


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[DistilBERT] hidden none: training pieces 6267 -> 6267


... (13 outputs left out)

[DistilBERT] hidden ssn
  cut SMS 0.2782 | cut dia 0.0443 | AUC SMS 0.9929 dia 0.9961 | calSMS@dia 404/3100=0.130 | fresh@dia 135/904=0.149 | dianorm@SMS 22/707=0.031 | spam@SMS 123/128 | ssn @SMS 940/962=0.977 @dia 953/962=0.991 [0.982, 0.995] | 31 s
[DistilBERT] hidden support: training pieces 6267 -> 5561


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[DistilBERT] hidden support
  cut SMS 0.3094 | cut dia 0.0792 | AUC SMS 0.9926 dia 0.9958 | calSMS@dia 387/3100=0.125 | fresh@dia 122/904=0.135 | dianorm@SMS 14/707=0.020 | spam@SMS 125/128 | support @SMS 797/877=0.909 @dia 851/877=0.970 [0.957, 0.980] | 30 s
[DistilBERT] SAVED: [('models/DistilBERT/scores_hidden_none.csv', 2692561), ('models/DistilBERT/scores_hidden_reward.csv', 2733785), ('models/DistilBERT/scores_hidden_refund.csv', 2733694), ('models/DistilBERT/scores_hidden_ssn.csv', 2703337), ('models/DistilBERT/scores_hidden_support.csv', 2732489), ('models/DistilBERT/summary_DistilBERT.csv', 3099)]
DONE DistilBERT


In [ ]:
# Two cut-offs on the standard call-level split: zero-shot Qwen2.5-1.5B (no training), seed 42
# Fix: load the base model ONCE, and for each run add a fresh LoRA adapter, train it, then remove it (unload).
# Needs only the GPU RESTART SETUP cell above.
!pip -q install peft
!pip -q uninstall -y torchao
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc, psutil
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); disable_progress_bars()
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

BASE = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
BASE.gradient_checkpointing_enable(); BASE.enable_input_require_grads()

class LoRALLM:
    _active = None
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4):
        self.seed, self.r, self.epochs, self.batch, self.lr = seed, r, epochs, batch, lr
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()
    def fit(self, X, y):
        global BASE
        if LoRALLM._active is not None:                     # remove the previous adapter, keep the base
            BASE = LoRALLM._active.model.unload(); LoRALLM._active.model = None
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(BASE, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        self.model.eval(); LoRALLM._active = self
        return self
    def predict(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return (out > 0.5).astype(int)


# Model 9 (LoRA): 3 seeds + big false-alarm test (same as seed_eval, without the hide-one-type part)
# Hide-one-type would add 27 training runs (about 5.5 hours), so those stay single-seed for LoRA.
# Each seed: new SMS split, new balancing sample, new LoRA seed; cut-off chosen on ~3,100 leftover real normal TRAIN SMS.
def _lora_pp(self, X, batch=32):
    texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(texts), batch):
            ids = order[b:b + batch]
            enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
            out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
    return np.column_stack([1 - out, out])
LoRALLM.predict_proba = _lora_pp


# Model 10: ensembles (average of member scores)
# Members, retrained on each seed's data: SVM (calibrated to give probabilities), DistilBERT, LoRA LLM, and the
# zero-shot LLM (same Qwen weights with the LoRA adapter switched off, so no second copy in memory).
# Each ensemble score = plain average of its members' 0-to-1 scores. Same harness: 3 seeds, big false-alarm test, 5% cut-offs.
# Needs: GPU RESTART SETUP cell + the LoRA cell above (LoRALLM, BASE, build, llm_tok) in this runtime.
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])

ZS_CACHE = {}
def zs_scores(texts, batch=32):
    """Zero-shot scores from the base Qwen model (LoRA adapter switched off). Cached, so each text is scored once."""
    texts = [str(t) for t in texts]; todo = sorted(set(t for t in texts if t not in ZS_CACHE), key=len)
    act = LoRALLM._active
    with act.model.disable_adapter(), torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(todo), batch):
            chunk = todo[b:b + batch]
            enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
            ZS_CACHE.update(zip(chunk, torch.softmax(act._last_logits(enc), dim=1)[:, 1].cpu().numpy()))
    return np.array([ZS_CACHE[t] for t in texts])

ENSEMBLES = {"E1 SVM+LoRA": ["SVM", "LoRA"], "E2 SVM+LoRA+zero-shot": ["SVM", "LoRA", "zero-shot"],
             "E3 LoRA+zero-shot": ["LoRA", "zero-shot"], "E4 all four": ["SVM", "DistilBERT", "LoRA", "zero-shot"]}

# ---------- new code: zero-shot scores from BASE directly (no LoRA adapter exists yet, so zs_scores' disable_adapter() cannot be used);
# same prompt, same batching by length, same P(Yes)/(P(Yes)+P(No)) as zs_scores ----------
_zs_act = LoRALLM(); _zs_act.model = BASE; BASE.eval(); ZS2 = {}
def zs_base(texts, batch=32):
    texts = [str(t) for t in texts]; todo = sorted(set(t for t in texts if t not in ZS2), key=len)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(todo), batch):
            chunk = todo[b:b + batch]
            enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
            ZS2.update(zip(chunk, torch.softmax(_zs_act._last_logits(enc), dim=1)[:, 1].cpu().numpy()))
    return np.array([ZS2[t] for t in texts])
two_cut_eval("zero-shot", lambda trn: zs_base, retrain=False)
print("DONE zero-shot")

[zero-shot] hidden none (no training: same scores)
  cut SMS 0.5662 | cut dia 0.7724 | AUC SMS 0.9823 dia 0.7104 | calSMS@dia 76/3100=0.025 | fresh@dia 16/904=0.018 | dianorm@SMS 129/707=0.182 | spam@SMS 116/128 | 184 s
[zero-shot] hidden reward (no training: same scores)
  cut SMS 0.5662 | cut dia 0.7724 | AUC SMS 0.9823 dia 0.7104 | calSMS@dia 76/3100=0.025 | fresh@dia 16/904=0.018 | dianorm@SMS 129/707=0.182 | spam@SMS 116/128 | reward @SMS 394/822=0.479 @dia 227/822=0.276 [0.247, 0.308] | 0 s
[zero-shot] hidden refund (no training: same scores)
  cut SMS 0.5662 | cut dia 0.7724 | AUC SMS 0.9823 dia 0.7104 | calSMS@dia 76/3100=0.025 | fresh@dia 16/904=0.018 | dianorm@SMS 129/707=0.182 | spam@SMS 116/128 | refund @SMS 365/954=0.383 @dia 160/954=0.168 [0.145, 0.193] | 0 s
[zero-shot] hidden ssn (no training: same scores)
  cut SMS 0.5662 | cut dia 0.7724 | AUC SMS 0.9823 dia 0.7104 | calSMS@dia 76/3100=0.025 | fresh@dia 16/904=0.018 | dianorm@SMS 129/707=0.182 | spam@SMS 116/128 | ssn

Reloads the saved split from Drive and checks it. The second cell fixes the check by using call ids.

In [ ]:
# Reload the standard call-level split from Drive (dialogue_calibration_pieces.csv) and check it; trains nothing
import os, numpy as np, pandas as pd
from collections import defaultdict
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
DCS = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/dialogue_calibration_split"
all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
MASK_V2 = dict(zip(all_raw, mask_texts(all_raw)))
def m2(x): return [MASK_V2[str(t)] for t in x]
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
tr42, te42, s_tr42, s_te42 = build_data(42)
full42 = pd.concat([tr42.assign(split="train"), te42.assign(split="test")], ignore_index=True)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]; big42 = s_te42[s_te42.label == 0]
saved = pd.read_csv(f"{DCS}/dialogue_calibration_pieces.csv", keep_default_na=False)
print("saved calibration pieces:", len(saved), "| calls:", saved.call.nunique(), "| by topic:", saved.type.value_counts().to_dict())
nd = tr42[(tr42.source == "synthetic") & (tr42.label == 0)].copy(); nd["text_masked"] = m2(nd["dialogue"])
coll = nd.groupby("text_masked").dialogue.nunique(); print("masked texts shared by more than one raw piece (normal dialogue, training split):", int((coll > 1).sum()))
cal_dia = nd[nd.text_masked.isin(set(saved.text_masked))].copy()
same_order = list(cal_dia.text_masked) == list(saved.text_masked)
same_set = sorted(cal_dia.text_masked) == sorted(saved.text_masked)
cal_dia["call"] = saved.call.values if same_order else np.nan
print(f"rebuilt calibration pieces: {len(cal_dia)} | texts identical in the same order: {same_order} | same texts ignoring order: {same_set} | topics match: {cal_dia.type.tolist() == saved.type.tolist() if same_order else 'n/a'}")
# independent check: recompute each piece's call with the make_pieces chunking rule and compare with the saved call ids
raw_tr = ds["train"].to_pandas(); CALLS = defaultdict(set)
for cid, text in enumerate(raw_tr["dialogue"]):
    w = text.split()
    for i in range(0, len(w), 45):
        ch = w[i:i + 45]
        if len(ch) >= 10: CALLS[" ".join(ch)].add(cid)
re_calls = [min(CALLS[t]) for t in cal_dia["dialogue"]]
print("call ids recomputed from the raw dialogues match the saved ones:", re_calls == list(saved.call))
base = tr42[~tr42.dialogue.isin(cal_dia["dialogue"])]
print(f"training pieces: {len(tr42)} -> {len(base)} after holding out the calibration pieces (earlier run: 6267)")
print("calibration texts also in the kept training text:", len(set(cal_dia.dialogue) & set(base.dialogue)))
EV = {"calibration_sms_3100": cut42, "calibration_dialogue": cal_dia, "fresh_sms_904": big42,
      "dialogue_normal_test": te42[(te42.source == "synthetic") & (te42.label == 0)], "dialogue_scam_test": te42[(te42.source == "synthetic") & (te42.label == 1)],
      "real_sms_test": te42[te42.source == "real"]}
for t in ["reward", "refund", "ssn", "support"]: EV[f"topic_{t}"] = full42[full42.type == t]
print("evaluation sets:", {k: len(v) for k, v in EV.items()})
print("DONE reload: waiting for go")

Mounted at /content/drive
saved calibration pieces: 577 | calls: 129 | by topic: {'insurance': 195, 'telemarketing': 179, 'delivery': 136, 'wrong': 67}
masked texts shared by more than one raw piece (normal dialogue, training split): 8
rebuilt calibration pieces: 580 | texts identical in the same order: False | same texts ignoring order: False | topics match: n/a
call ids recomputed from the raw dialogues match the saved ones: False
training pieces: 6844 -> 6264 after holding out the calibration pieces (earlier run: 6267)
calibration texts also in the kept training text: 0
evaluation sets: {'calibration_sms_3100': 3100, 'calibration_dialogue': 580, 'fresh_sms_904': 904, 'dialogue_normal_test': 707, 'dialogue_scam_test': 707, 'real_sms_test': 256, 'topic_reward': 822, 'topic_refund': 954, 'topic_ssn': 962, 'topic_support': 877}
DONE reload: waiting for go


In [ ]:
# Reload check, fixed: rebuild the calibration set by the saved CALL ids (as the split was defined), not by masked text
raw_tr = ds["train"].to_pandas(); CALLS = defaultdict(set)
for cid, text in enumerate(raw_tr["dialogue"]):
    w = text.split()
    for i in range(0, len(w), 45):
        ch = w[i:i + 45]
        if len(ch) >= 10: CALLS[" ".join(ch)].add(cid)
nd["call"] = [min(CALLS[t]) if t in CALLS else -1 for t in nd["dialogue"]]
print("normal dialogue pieces:", len(nd), "| from calls:", nd.call.nunique(), "| unmatched:", int((nd.call == -1).sum()))
cal_dia = nd[nd.call.isin(set(saved.call))].copy()
print(f"rebuilt calibration pieces: {len(cal_dia)} from {cal_dia.call.nunique()} calls (saved: {len(saved)} from {saved.call.nunique()})")
print("masked texts identical in the same order:", list(cal_dia.text_masked) == list(saved.text_masked))
print("call ids identical in the same order:", list(cal_dia.call) == list(saved.call), "| topics identical:", list(cal_dia.type) == list(saved.type))
base = tr42[~tr42.dialogue.isin(cal_dia["dialogue"])]
print(f"training pieces: {len(tr42)} -> {len(base)} (earlier run: 6267) | calibration texts also in kept training text: {len(set(cal_dia.dialogue) & set(base.dialogue))}")
EV["calibration_dialogue"] = cal_dia
print("evaluation sets:", {k: len(v) for k, v in EV.items()})
print("DONE reload (fixed): waiting for go")

normal dialogue pieces: 2908 | from calls: 639 | unmatched: 0
rebuilt calibration pieces: 577 from 129 calls (saved: 577 from 129)
masked texts identical in the same order: True
call ids identical in the same order: True | topics identical: True
training pieces: 6844 -> 6267 (earlier run: 6267) | calibration texts also in kept training text: 0
evaluation sets: {'calibration_sms_3100': 3100, 'calibration_dialogue': 577, 'fresh_sms_904': 904, 'dialogue_normal_test': 707, 'dialogue_scam_test': 707, 'real_sms_test': 256, 'topic_reward': 822, 'topic_refund': 954, 'topic_ssn': 962, 'topic_support': 877}
DONE reload (fixed): waiting for go


False alarms on normal chats at the chat cut-off, and how they differ from the summary table.

In [ ]:
# Step 1 (no training): normal dialogue false alarms at the dialogue-source cut-off, from the saved score files
import os, numpy as np, pandas as pd
MD = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/dialogue_calibration_split/models"
OLD = pd.read_csv(f"{MD}/summary_all_models.csv")
ROWS1 = []
for m in ["NB", "LR", "SVM", "XGB", "CNN", "BiLSTM", "DistilBERT", "zero-shot"]:
    for hid in ["none", "reward", "refund", "ssn", "support"]:
        fn = f"{MD}/{m}/scores_hidden_{hid if m != 'zero-shot' else 'none'}.csv"
        d = pd.read_csv(fn, keep_default_na=False)
        cal = d[d.set == "calibration_dialogue"].score.values; dn = d[d.set == "dialogue_normal_test"].score.values
        c = float(np.quantile(cal, 0.95)); f = dn > c; k, n = int(f.sum()), len(f); lo, hi = wilson(k, n)
        r = dict(model=m, hidden=hid, score_file=os.path.basename(fn), cut_dialogue=c, calib_dialogue_flagged=int((cal > c).sum()), calib_dialogue_n=len(cal),
                 dianorm_k=k, dianorm_n=n, dianorm_rate=round(k / n, 4), dianorm_lo95=round(lo, 4), dianorm_hi95=round(hi, 4))
        if hid != "none":
            ts = d[d.set == f"topic_{hid}"].score.values; ft = ts > c; kt, nt = int(ft.sum()), len(ft); lo2, hi2 = wilson(kt, nt)
            r.update(topic_k=kt, topic_n=nt, topic_rate=round(kt / nt, 4), topic_lo95=round(lo2, 4), topic_hi95=round(hi2, 4))
        o = OLD[(OLD.model == m) & (OLD.hidden == hid)].iloc[0]
        r["matches_summary"] = bool(abs(o.cut_dialogue - c) < 1e-12 and int(o.dianorm_at_diacut_k) == k)
        ROWS1.append(r)
T1 = pd.DataFrame(ROWS1)
OUT1 = f"{MD}/dialogue_false_alarms_at_dialogue_cutoff.csv"; T1.to_csv(OUT1, index=False)
pd.set_option("display.width", 250)
print(T1[["model", "hidden", "cut_dialogue", "calib_dialogue_flagged", "dianorm_k", "dianorm_n", "dianorm_rate", "dianorm_lo95", "dianorm_hi95", "topic_k", "topic_n", "topic_rate", "matches_summary"]].to_string(index=False))
print("\nall rows match summary_all_models.csv:", bool(T1.matches_summary.all()))
print("saved:", OUT1, os.path.getsize(OUT1), "bytes")
print("DONE step 1")

     model  hidden  cut_dialogue  calib_dialogue_flagged  dianorm_k  dianorm_n  dianorm_rate  dianorm_lo95  dianorm_hi95  topic_k  topic_n  topic_rate  matches_summary
        NB    none      0.380370                      29         35        707        0.0495        0.0358        0.0681      NaN      NaN         NaN             True
        NB  reward      0.259112                      29         34        707        0.0481        0.0346        0.0665    765.0    822.0      0.9307             True
        NB  refund      0.198052                      29         44        707        0.0622        0.0467        0.0825    894.0    954.0      0.9371             True
        NB     ssn      0.266458                      29         41        707        0.0580        0.0430        0.0777    956.0    962.0      0.9938             True
        NB support      0.282205                      29         36        707        0.0509        0.0370        0.0697    860.0    877.0      0.9806          

In [ ]:
# Which Step 1 rows differ from summary_all_models.csv, and by how much
M = T1.merge(OLD[["model", "hidden", "cut_dialogue", "dianorm_at_diacut_k", "topic_at_diacut_k"]].rename(columns={"cut_dialogue": "old_cut", "dianorm_at_diacut_k": "old_dianorm_k", "topic_at_diacut_k": "old_topic_k"}), on=["model", "hidden"])
M["cut_diff"] = (M.cut_dialogue - M.old_cut).abs()
print("rows:", len(M), "| cut-off differs by more than 1e-12:", int((M.cut_diff > 1e-12).sum()), "| largest cut-off difference:", f"{M.cut_diff.max():.2e}")
print("dialogue-normal count differs:", int((M.dianorm_k != M.old_dianorm_k).sum()), "| topic count differs:", int(((M.topic_k != M.old_topic_k) & M.topic_k.notna()).sum()))
print(M[~M.matches_summary][["model", "hidden", "cut_dialogue", "old_cut", "cut_diff", "dianorm_k", "old_dianorm_k", "topic_k", "old_topic_k"]].to_string(index=False))

rows: 40 | cut-off differs by more than 1e-12: 25 | largest cut-off difference: 4.33e-07
dialogue-normal count differs: 0 | topic count differs: 0
     model  hidden  cut_dialogue  old_cut     cut_diff  dianorm_k  old_dianorm_k  topic_k  old_topic_k
       XGB    none      0.309599 0.309599 1.533690e-07         52             52      NaN          NaN
       XGB  reward      0.241776 0.241776 1.355557e-07         47             47    736.0        736.0
       XGB  refund      0.191745 0.191745 1.477768e-08         48             48    880.0        880.0
       XGB     ssn      0.258401 0.258401 4.334950e-07         44             44    940.0        940.0
       XGB support      0.281973 0.281973 1.453880e-08         39             39    781.0        781.0
       CNN    none      0.361004 0.361005 2.427905e-07         38             38      NaN          NaN
       CNN  reward      0.121794 0.121794 2.195784e-07         46             46    782.0        782.0
       CNN  refund      0.118

LoRA, then the calibrated SVM and ensembles E1 to E4, on the standard split with two cut-offs.

In [ ]:
# Step 2: LoRA on the standard call-level split, two 5% cut-offs, five retrains (main + 4 hidden scam topics), resumable
# Fix: load the base model ONCE, and for each run add a fresh LoRA adapter, train it, then remove it (unload).
# Needs only the GPU RESTART SETUP cell above.
!pip -q install peft
!pip -q uninstall -y torchao
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
import torch.nn.functional as F, gc, psutil
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar(); disable_progress_bars()
DEVICE = "cuda"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
YES = llm_tok.encode("Yes", add_special_tokens=False)[0]
NO  = llm_tok.encode("No",  add_special_tokens=False)[0]
SYSTEM = "You are a careful assistant that checks text messages and call transcripts for fraud."
QUESTION = ("Is the following message a scam or spam, meaning it tries to trick the reader into giving money or "
            "personal information, or pushes an unwanted offer? Answer with only Yes or No.\n\nMessage: \"{msg}\"")
def build(t):
    return llm_tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": QUESTION.format(msg=t)}],
                                       tokenize=False, add_generation_prompt=True)

BASE = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE)
BASE.gradient_checkpointing_enable(); BASE.enable_input_require_grads()

class LoRALLM:
    _active = None
    def __init__(self, seed=42, r=8, epochs=1, batch=16, lr=2e-4):
        self.seed, self.r, self.epochs, self.batch, self.lr = seed, r, epochs, batch, lr
    def _last_logits(self, enc):
        try:
            out = self.model(**enc, logits_to_keep=1).logits[:, -1, :]
        except TypeError:
            out = self.model(**enc, num_logits_to_keep=1).logits[:, -1, :]
        return torch.stack([out[:, NO], out[:, YES]], dim=1).float()
    def fit(self, X, y):
        global BASE
        if LoRALLM._active is not None:                     # remove the previous adapter, keep the base
            BASE = LoRALLM._active.model.unload(); LoRALLM._active.model = None
        gc.collect(); torch.cuda.empty_cache()
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        cfg = LoraConfig(r=self.r, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        self.model = get_peft_model(BASE, cfg)
        for p in self.model.parameters():
            if p.requires_grad: p.data = p.data.float()
        prompts = [build(t) for t in X]; yy = torch.tensor(np.asarray(y), dtype=torch.long)
        opt = torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad], lr=self.lr)
        steps = self.epochs * ((len(prompts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        self.model.train()
        for ep in range(self.epochs):
            perm = torch.randperm(len(prompts), generator=g).tolist()
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                enc = llm_tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.cross_entropy(self._last_logits(enc), yy[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        self.model.eval(); LoRALLM._active = self
        return self
    def predict(self, X, batch=32):
        texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, len(texts), batch):
                ids = order[b:b + batch]
                enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
                out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
        return (out > 0.5).astype(int)


# Model 9 (LoRA): 3 seeds + big false-alarm test (same as seed_eval, without the hide-one-type part)
# Hide-one-type would add 27 training runs (about 5.5 hours), so those stay single-seed for LoRA.
# Each seed: new SMS split, new balancing sample, new LoRA seed; cut-off chosen on ~3,100 leftover real normal TRAIN SMS.
def _lora_pp(self, X, batch=32):
    texts = [str(t) for t in X]; out = np.zeros(len(texts)); order = np.argsort([len(t) for t in texts])
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(texts), batch):
            ids = order[b:b + batch]
            enc = llm_tok([build(texts[i]) for i in ids], return_tensors="pt", padding=True).to(DEVICE)
            out[ids] = torch.softmax(self._last_logits(enc), dim=1)[:, 1].cpu().numpy()
    return np.column_stack([1 - out, out])
LoRALLM.predict_proba = _lora_pp


# Model 10: ensembles (average of member scores)
# Members, retrained on each seed's data: SVM (calibrated to give probabilities), DistilBERT, LoRA LLM, and the
# zero-shot LLM (same Qwen weights with the LoRA adapter switched off, so no second copy in memory).
# Each ensemble score = plain average of its members' 0-to-1 scores. Same harness: 3 seeds, big false-alarm test, 5% cut-offs.
# Needs: GPU RESTART SETUP cell + the LoRA cell above (LoRALLM, BASE, build, llm_tok) in this runtime.
from transformers import AutoModelForSequenceClassification
from sklearn.calibration import CalibratedClassifierCV
import warnings; warnings.filterwarnings("ignore")
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

class DistilBERTClassifier:                       # same settings as Model 7
    def __init__(self, seed=42, max_len=64, epochs=2, batch=32, lr=5e-5):
        self.seed, self.max_len, self.epochs, self.batch, self.lr = seed, max_len, epochs, batch, lr
    def _enc(self, texts):
        return bert_tok([str(t) for t in texts], truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
    def fit(self, texts, labels):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        texts = list(texts); y = torch.tensor(np.asarray(labels), dtype=torch.long); enc = self._enc(texts)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * ((len(texts) + self.batch - 1) // self.batch)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps); scaler = torch.amp.GradScaler("cuda")
        g = torch.Generator().manual_seed(self.seed)
        for ep in range(self.epochs):
            self.model.train(); perm = torch.randperm(len(texts), generator=g)
            for b in range(0, len(perm), self.batch):
                idx = perm[b:b + self.batch]
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(input_ids=enc["input_ids"][idx].to(DEVICE), attention_mask=enc["attention_mask"][idx].to(DEVICE), labels=y[idx].to(DEVICE))
                opt.zero_grad(); scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        return self
    def predict_proba(self, texts):
        self.model.eval(); enc = self._enc(list(texts)); out = []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in range(0, enc["input_ids"].shape[0], 256):
                lg = self.model(input_ids=enc["input_ids"][b:b + 256].to(DEVICE), attention_mask=enc["attention_mask"][b:b + 256].to(DEVICE)).logits.float()
                out.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        p = np.concatenate(out); return np.column_stack([1 - p, p])

ZS_CACHE = {}
def zs_scores(texts, batch=32):
    """Zero-shot scores from the base Qwen model (LoRA adapter switched off). Cached, so each text is scored once."""
    texts = [str(t) for t in texts]; todo = sorted(set(t for t in texts if t not in ZS_CACHE), key=len)
    act = LoRALLM._active
    with act.model.disable_adapter(), torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(todo), batch):
            chunk = todo[b:b + batch]
            enc = llm_tok([build(t) for t in chunk], return_tensors="pt", padding=True).to(DEVICE)
            ZS_CACHE.update(zip(chunk, torch.softmax(act._last_logits(enc), dim=1)[:, 1].cpu().numpy()))
    return np.array([ZS_CACHE[t] for t in texts])

ENSEMBLES = {"E1 SVM+LoRA": ["SVM", "LoRA"], "E2 SVM+LoRA+zero-shot": ["SVM", "LoRA", "zero-shot"],
             "E3 LoRA+zero-shot": ["LoRA", "zero-shot"], "E4 all four": ["SVM", "DistilBERT", "LoRA", "zero-shot"]}

# ---------- new code: shared helper for the two-cut-off evaluation (standard split in dialogue_calibration_split) ----------
import os, gc, time, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
DCS = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42/dialogue_calibration_split"; MD = f"{DCS}/models"
need = [v for v in ["base", "cal_dia", "EV", "cut42", "big42", "te42", "full42", "m2", "wilson"] if v not in globals()]
assert not need, f"missing from memory, rerun the call-level split cell first: {need}"
assert os.path.isdir(DCS), "Drive not mounted"
saved = pd.read_csv(f"{DCS}/dialogue_calibration_pieces.csv", keep_default_na=False)
print("standard split check: calibration pieces in memory", len(cal_dia), "| on Drive", len(saved), "| identical texts:", list(m2(cal_dia["dialogue"])) == list(saved.text_masked))
print("training pieces (real spam kept, nothing hidden):", len(base))
HIDS = [None, "reward", "refund", "ssn", "support"]; ALLROWS = globals().get("ALLROWS", [])
def two_cut_eval(name, fit_fn, retrain=True):
    os.makedirs(f"{MD}/{name}", exist_ok=True); rows, files = [], []; S = None
    for hid in HIDS:
        t0 = time.time()
        if retrain or S is None:
            trn = base if hid is None else base[base.type != hid]
            if retrain: print(f"[{name}] hidden {hid or 'none'}: training pieces {len(base)} -> {len(trn)}")
            scorer = fit_fn(trn); S = {k: np.asarray(scorer(m2(d["dialogue"]))) for k, d in EV.items()}
        c_sms, c_dia = float(np.quantile(S["calibration_sms_3100"], 0.95)), float(np.quantile(S["calibration_dialogue"], 0.95))
        rs = EV["real_sms_test"]; spam = (rs.label == 1).values
        dia_lab = np.r_[np.zeros(len(EV["dialogue_normal_test"])), np.ones(len(EV["dialogue_scam_test"]))]
        r = dict(model=name, hidden=hid or "none", retrained=bool(retrain), cut_sms=c_sms, cut_dialogue=c_dia,
                 auc_sms=round(roc_auc_score(rs.label, S["real_sms_test"]), 4), auc_dialogue=round(roc_auc_score(dia_lab, np.r_[S["dialogue_normal_test"], S["dialogue_scam_test"]]), 4))
        def add(tag, v, c):
            f = v > c; k, n = int(f.sum()), len(f); lo, hi = wilson(k, n); r.update({f"{tag}_k": k, f"{tag}_n": n, f"{tag}_rate": round(k / n, 4), f"{tag}_lo95": round(lo, 4), f"{tag}_hi95": round(hi, 4)})
        add("calsms_at_diacut", S["calibration_sms_3100"], c_dia); add("fresh_at_diacut", S["fresh_sms_904"], c_dia)
        add("calsms_at_smscut", S["calibration_sms_3100"], c_sms); add("fresh_at_smscut", S["fresh_sms_904"], c_sms)
        add("caldia_at_diacut", S["calibration_dialogue"], c_dia); add("dianorm_at_smscut", S["dialogue_normal_test"], c_sms); add("dianorm_at_diacut", S["dialogue_normal_test"], c_dia)
        add("spam_at_smscut", S["real_sms_test"][spam], c_sms); add("spam_at_diacut", S["real_sms_test"][spam], c_dia)
        if hid: add("topic_at_smscut", S[f"topic_{hid}"], c_sms); add("topic_at_diacut", S[f"topic_{hid}"], c_dia)
        rows.append(r)
        msg = (f"  cut SMS {c_sms:.4f} | cut dia {c_dia:.4f} | AUC SMS {r['auc_sms']:.4f} dia {r['auc_dialogue']:.4f} | calSMS@dia {r['calsms_at_diacut_k']}/3100={r['calsms_at_diacut_rate']:.3f} | "
               f"fresh@dia {r['fresh_at_diacut_k']}/904={r['fresh_at_diacut_rate']:.3f} | dianorm@SMS {r['dianorm_at_smscut_k']}/707={r['dianorm_at_smscut_rate']:.3f} | spam@SMS {r['spam_at_smscut_k']}/128")
        if hid: msg += f" | {hid} @SMS {r['topic_at_smscut_k']}/{r['topic_at_smscut_n']}={r['topic_at_smscut_rate']:.3f} @dia {r['topic_at_diacut_k']}/{r['topic_at_diacut_n']}={r['topic_at_diacut_rate']:.3f} [{r['topic_at_diacut_lo95']:.3f}, {r['topic_at_diacut_hi95']:.3f}]"
        print(f"[{name}] hidden {hid or 'none'}{'' if retrain else ' (no training: same scores)'}\n" + msg + f" | {time.time() - t0:.0f} s")
        if retrain or hid is None:
            fn = f"{MD}/{name}/scores_hidden_{hid or 'none'}.csv"
            pd.concat([d.assign(set=k)[["set", "source", "type", "label"]].assign(text_masked=m2(d["dialogue"]), score=S[k]) for k, d in EV.items()]).assign(
                model=name, hidden=hid or "none", cut_sms=c_sms, cut_dialogue=c_dia).to_csv(fn, index=False); files.append(fn)
    pd.DataFrame(rows).to_csv(f"{MD}/{name}/summary_{name}.csv", index=False); files.append(f"{MD}/{name}/summary_{name}.csv")
    ALLROWS.extend(rows); pd.DataFrame(ALLROWS).to_csv(f"{MD}/summary_all_models.csv", index=False)
    print(f"[{name}] SAVED:", [(f.replace(DCS + "/", ""), os.path.getsize(f)) for f in files])
    return rows


# ---------- new code: resumable LoRA loop ----------
LDIR = f"{MD}/LoRA"; os.makedirs(LDIR, exist_ok=True)
ALL_H = [None, "reward", "refund", "ssn", "support"]
tag = lambda h: h or "none"
done = [h for h in ALL_H if os.path.exists(f"{LDIR}/scores_hidden_{tag(h)}.csv") and os.path.exists(f"{LDIR}/row_hidden_{tag(h)}.csv")]
todo = [h for h in ALL_H if h not in done]
print("already saved on Drive:", [tag(h) for h in done], "| to run now:", [tag(h) for h in todo])
prev = pd.read_csv(f"{MD}/summary_all_models.csv")
ALLROWS = prev[prev.model != "LoRA"].to_dict("records") + sum([pd.read_csv(f"{LDIR}/row_hidden_{tag(h)}.csv").to_dict("records") for h in done], [])
print("rows kept from other models in summary_all_models.csv:", int((prev.model != "LoRA").sum()))
def fit_lora2(trn):
    global _lm2
    _lm2 = LoRALLM(seed=42).fit(m2(trn["dialogue"]), trn["label"].values)   # fit() removes the previous adapter and keeps BASE
    return lambda x: _lm2.predict_proba(x)[:, 1]
print("GPU:", torch.cuda.get_device_name(0), "| memory in use:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
for h in todo:
    HIDS = [h]; t0 = time.time()
    r = two_cut_eval("LoRA", fit_lora2)[0]
    pd.DataFrame([r]).to_csv(f"{LDIR}/row_hidden_{tag(h)}.csv", index=False)
    line = (f"ONE-LINE LoRA {tag(h)}: cut SMS {r['cut_sms']:.4f}, cut dia {r['cut_dialogue']:.4f}, AUC SMS {r['auc_sms']:.4f}, AUC dia {r['auc_dialogue']:.4f}, "
            f"calSMS@dia {r['calsms_at_diacut_k']}/3100, fresh@dia {r['fresh_at_diacut_k']}/904, dianorm@SMS {r['dianorm_at_smscut_k']}/707, dianorm@dia {r['dianorm_at_diacut_k']}/707, spam@SMS {r['spam_at_smscut_k']}/128")
    if h: line += f", {h} @SMS {r['topic_at_smscut_k']}/{r['topic_at_smscut_n']} @dia {r['topic_at_diacut_k']}/{r['topic_at_diacut_n']} = {r['topic_at_diacut_rate']:.3f}"
    print(line + f" | {(time.time() - t0) / 60:.1f} min | saved scores_hidden_{tag(h)}.csv")
got = [h for h in ALL_H if os.path.exists(f"{LDIR}/row_hidden_{tag(h)}.csv")]
pd.concat([pd.read_csv(f"{LDIR}/row_hidden_{tag(h)}.csv") for h in got]).to_csv(f"{LDIR}/summary_LoRA.csv", index=False)
print("LoRA files on Drive:", sorted((f, os.path.getsize(f"{LDIR}/{f}")) for f in os.listdir(LDIR)))
print("DONE LoRA")

standard split check: calibration pieces in memory 577 | on Drive 577 | identical texts: True
training pieces (real spam kept, nothing hidden): 6267
already saved on Drive: [] | to run now: ['none', 'reward', 'refund', 'ssn', 'support']
rows kept from other models in summary_all_models.csv: 40
GPU: Tesla T4 | memory in use: 3.09 GB
[LoRA] hidden none: training pieces 6267 -> 6267
[LoRA] hidden none
  cut SMS 0.4926 | cut dia 0.2315 | AUC SMS 0.9909 dia 0.9962 | calSMS@dia 298/3100=0.096 | fresh@dia 90/904=0.100 | dianorm@SMS 24/707=0.034 | spam@SMS 123/128 | 759 s
[LoRA] SAVED: [('models/LoRA/scores_hidden_none.csv', 2709199), ('models/LoRA/summary_LoRA.csv', 1297)]
ONE-LINE LoRA none: cut SMS 0.4926, cut dia 0.2315, AUC SMS 0.9909, AUC dia 0.9962, calSMS@dia 298/3100, fresh@dia 90/904, dianorm@SMS 24/707, dianorm@dia 41/707, spam@SMS 123/128 | 12.6 min | saved scores_hidden_none.csv
[LoRA] hidden reward: training pieces 6267 -> 5599
[LoRA] hidden reward
  cut SMS 0.4924 | cut dia 0.21

In [ ]:
# Calibrated SVM member (retrained per topic) + ensembles E1-E4 on the standard split, two 5% cut-offs
# Uses two_cut_eval, EV, base, m2, MD from memory (LoRA cell). Member scores are probabilities (0-1); ensemble = plain mean, as in the logged runs.
import os, numpy as np, pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
HIDS = [None, "reward", "refund", "ssn", "support"]
print("rows in summary_all_models before:", len(ALLROWS), "| models:", sorted(set(r["model"] for r in ALLROWS)))
# ---- 1. calibrated SVM member, retrained per topic ----
def fit_svm_cal(trn):
    m = make_pipeline(tfidf(), CalibratedClassifierCV(LinearSVC(random_state=42), cv=3)).fit(m2(trn["dialogue"]), trn["label"].values)
    return lambda x: m.predict_proba(x)[:, 1]
two_cut_eval("SVM_calibrated", fit_svm_cal)
# ---- 2. load member scores (text_masked -> score) for every hidden topic ----
FOLD = {"SVM": "SVM_calibrated", "DistilBERT": "DistilBERT", "LoRA": "LoRA", "zero-shot": "zero-shot"}
LOOK = {}
for mem, fold in FOLD.items():
    for h in HIDS:
        fn = f"{MD}/{fold}/scores_hidden_{'none' if (mem == 'zero-shot' or h is None) else h}.csv"
        d = pd.read_csv(fn, keep_default_na=False)
        g = d.groupby("text_masked").score.agg(["min", "max"]); spread = float((g["max"] - g["min"]).max())
        LOOK[(mem, h)] = dict(zip(g.index, g["min"]))
        if h is None or mem != "zero-shot": print(f"  {mem:10s} hidden {h or 'none':8s}: {len(g)} distinct texts | largest score spread for one masked text: {spread:.2e} | min {d.score.min():.4f} max {d.score.max():.4f}")
need = set(t for k, df in EV.items() for t in m2(df["dialogue"]))
for key, dct in LOOK.items():
    miss = len(need - set(dct)); assert miss == 0, f"{key}: {miss} texts missing"
print("every evaluation text found in every member file: True")
ENS = {"E1": ["SVM", "LoRA"], "E2": ["SVM", "LoRA", "zero-shot"], "E3": ["LoRA", "zero-shot"], "E4": ["SVM", "DistilBERT", "LoRA", "zero-shot"]}
def hid_of(trn):
    gone = [t for t in ["reward", "refund", "ssn", "support"] if t not in set(trn.type)]
    return gone[0] if gone else None
for en, mem in ENS.items():
    def fit_ens(trn, mem=mem):
        h = hid_of(trn); print(f"  [{en}] members {mem}, hidden {h or 'none'}: plain mean of member probabilities")
        return lambda xs: np.mean([[LOOK[(m, h)][t] for t in xs] for m in mem], axis=0)
    two_cut_eval(en, fit_ens)
print("rows in summary_all_models after:", len(ALLROWS))
print("DONE ensembles")

rows in summary_all_models before: 45 | models: ['BiLSTM', 'CNN', 'DistilBERT', 'LR', 'LoRA', 'NB', 'SVM', 'XGB', 'zero-shot']
[SVM_calibrated] hidden none: training pieces 6267 -> 6267
[SVM_calibrated] hidden none
  cut SMS 0.5953 | cut dia 0.2251 | AUC SMS 0.9742 dia 0.9987 | calSMS@dia 1095/3100=0.353 | fresh@dia 300/904=0.332 | dianorm@SMS 21/707=0.030 | spam@SMS 122/128 | 2 s
[SVM_calibrated] hidden reward: training pieces 6267 -> 5599
[SVM_calibrated] hidden reward
  cut SMS 0.5857 | cut dia 0.2075 | AUC SMS 0.9712 dia 0.9977 | calSMS@dia 1218/3100=0.393 | fresh@dia 351/904=0.388 | dianorm@SMS 17/707=0.024 | spam@SMS 123/128 | reward @SMS 720/822=0.876 @dia 795/822=0.967 [0.953, 0.977] | 2 s
[SVM_calibrated] hidden refund: training pieces 6267 -> 5504
[SVM_calibrated] hidden refund
  cut SMS 0.5983 | cut dia 0.2015 | AUC SMS 0.9755 dia 0.9962 | calSMS@dia 1276/3100=0.412 | fresh@dia 373/904=0.413 | dianorm@SMS 16/707=0.023 | spam@SMS 123/128 | refund @SMS 810/954=0.849 @dia 902/9

In [ ]:
# Compact view of the new rows (SVM_calibrated, E1-E4)
A = pd.DataFrame(ALLROWS); A = A[A.model.isin(["SVM_calibrated", "E1", "E2", "E3", "E4", "LoRA", "SVM"])]
for _, r in A.iterrows():
    t = f" | topic {int(r.topic_at_smscut_k)}={r.topic_at_smscut_rate:.3f} -> {int(r.topic_at_diacut_k)}={r.topic_at_diacut_rate:.3f} [{r.topic_at_diacut_lo95:.3f},{r.topic_at_diacut_hi95:.3f}]" if r.hidden != "none" else ""
    print(f"{r.model:14s} {r.hidden:7s} AUC {r.auc_sms:.4f}/{r.auc_dialogue:.4f} | calSMS@D {int(r.calsms_at_diacut_k)} fresh@D {int(r.fresh_at_diacut_k)} fresh@S {int(r.fresh_at_smscut_k)} | dia@S {int(r.dianorm_at_smscut_k)} dia@D {int(r.dianorm_at_diacut_k)} | spam@S {int(r.spam_at_smscut_k)}" + t)
M = A[A.hidden != "none"].groupby("model", sort=False)[["topic_at_smscut_rate", "topic_at_diacut_rate"]].mean().round(3); print(M.to_string())

SVM            none    AUC 0.9885/0.9987 | calSMS@D 625 fresh@D 170 fresh@S 54 | dia@S 22 dia@D 44 | spam@S 122
SVM            reward  AUC 0.9899/0.9979 | calSMS@D 710 fresh@D 201 fresh@S 49 | dia@S 16 dia@D 41 | spam@S 123 | topic 735=0.894 -> 799=0.972 [0.958,0.981]
SVM            refund  AUC 0.9894/0.9963 | calSMS@D 650 fresh@D 180 fresh@S 45 | dia@S 17 dia@D 37 | spam@S 122 | topic 820=0.860 -> 907=0.951 [0.935,0.963]
SVM            ssn     AUC 0.9916/0.9981 | calSMS@D 590 fresh@D 164 fresh@S 51 | dia@S 20 dia@D 43 | spam@S 123 | topic 936=0.973 -> 957=0.995 [0.988,0.998]
SVM            support AUC 0.9907/0.9967 | calSMS@D 639 fresh@D 176 fresh@S 54 | dia@S 20 dia@D 42 | spam@S 121 | topic 804=0.917 -> 866=0.988 [0.978,0.993]
LoRA           none    AUC 0.9909/0.9962 | calSMS@D 298 fresh@D 90 fresh@S 44 | dia@S 24 dia@D 41 | spam@S 123
LoRA           reward  AUC 0.9911/0.9909 | calSMS@D 355 fresh@D 126 fresh@S 46 | dia@S 19 dia@D 45 | spam@S 123 | topic 570=0.693 -> 680=0.827 [0.800

# 12. Website export
Writes the masking and chunking code to files and trains the final plain LinearSVC on the standard split, so the website gives exactly the same scores as this notebook.

`masking.py`: the same masking used in the experiments.

In [ ]:
%%writefile /content/masking.py
# Masking_model
import re, spacy

EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
LINK = re.compile(r"(?:https?://|www\.)\S+|\b[\w-]+\.(?:com|net|org|co|io|ly|me|rw|ke|ug|tz|ng|gh|za)(?:/\S*)?\b", re.IGNORECASE)
CURRENCY = r"(?:rwf|frw|kes|ksh|ugx|tzs|ngn|ghs|zar|usd|shillings?|francs?|naira|cedis?|rand|dollars?|birr)"
AMOUNT = re.compile(rf"\b{CURRENCY}\.?\s?\d[\d,.]*|[$€£]\s?\d[\d,.]*|\d[\d,.]*\s?{CURRENCY}\b", re.IGNORECASE)
NUMBER = re.compile(r"\+?\d[\d\s\-()]{5,}\d")

GREETING = r"(?i:hi|hello|hey|dear|thanks|thank you)"
TITLE = r"(?i:mr|mrs|ms|miss|dr|prof)"
# A space only (no comma or full stop) between the greeting and the name: "Hi Chidi", "Dear Dr. Habimana"
AFTER_GREETING = re.compile(rf"\b{GREETING}\s+(?:{TITLE}\.?\s+)*([A-Z][a-z\-]{{2,}})")
AFTER_TITLE = re.compile(rf"\b{TITLE}\.?\s+([A-Z][a-z\-]{{2,}})")
NOT_NAMES = {"customer", "client", "user", "friend", "winner", "beneficiary", "sir", "madam",
             "everyone", "all", "team", "there", "what", "when", "where", "who", "why", "how",
             "this", "that", "please", "just", "yes", "well", "now", "today", "and", "but",
             "so", "the", "you", "your", "our", "for", "from", "with", "hmm"}

# Filters for spaCy's PERSON tags
INTERJECTIONS = {"hmm", "hm", "ya", "yeah", "yep", "wat", "ok", "okay", "lol", "haha", "oh", "ah",
                 "wow", "hey", "hi", "hello", "thanks", "pls", "plz", "sorry", "yes", "no"}
LEADING = {"this", "the", "a", "an", "my", "your", "our", "his", "her", "their", "dear", "hello",
           "hi", "hey", "thanks", "mr", "mrs", "ms", "miss", "dr", "prof"}
BAD_WORDS = {"st", "street", "rd", "road", "ave", "avenue", "usa", "uk", "ltd", "inc", "limited", "delivery", "bank"}

TOK = dict(name="nametoken", email="emailtoken", link="linktoken", amount="amounttoken", number="numtoken")
DISPLAY = {v: f"<{k.upper()}>" for k, v in TOK.items()}

_nlp = None
def _get_nlp():
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("en_core_web_sm")
    return _nlp

def _person_span(ent):
    toks = list(ent)
    while toks and toks[0].lower_.rstrip(".") in LEADING:   # "this John Smith" -> "John Smith"
        toks = toks[1:]
    if not toks or len(toks) > 3:
        return None
    for t in toks:
        w = t.text
        if not w.replace("-", "").replace("'", "").isalpha() or not w[0].isupper() or len(w) < 2 or w.isupper():
            return None
        if w.lower() in INTERJECTIONS or w.lower() in BAD_WORDS:
            return None
    return toks[0].idx, toks[-1].idx + len(toks[-1])

def _greeted(m):
    if m.group(1).lower() in NOT_NAMES:
        return m.group(0)
    return m.group(0)[: m.start(1) - m.start(0)] + TOK["name"]

def _rest(text, known_names):
    for n in known_names:
        text = re.sub(rf"(?<!\w){re.escape(n)}(?!\w)", TOK["name"], text, flags=re.IGNORECASE)
    for pattern in (AFTER_GREETING, AFTER_TITLE):
        text = pattern.sub(_greeted, text)
    text = EMAIL.sub(TOK["email"], text)
    text = LINK.sub(TOK["link"], text)
    text = AMOUNT.sub(TOK["amount"], text)   # amounts before long numbers
    text = NUMBER.sub(TOK["number"], text)
    return text

def mask_texts(texts, known_names=()):
    texts = [str(t) for t in texts]
    out = []
    for doc, text in zip(_get_nlp().pipe(texts, batch_size=64), texts):
        spans = [s for s in (_person_span(e) for e in doc.ents if e.label_ == "PERSON") if s]
        for start, end in reversed(spans):
            text = text[:start] + TOK["name"] + text[end:]
        out.append(_rest(text, known_names))
    return out

def to_display(text):
    for tok, shown in DISPLAY.items():
        text = text.replace(tok, shown)
    return text

Writing /content/masking.py


`chunking.py`: splits chats into 45-word pieces.

In [ ]:
%%writefile /content/chunking.py
import pandas as pd


def make_pieces(df, max_words=45, min_words=10):
    rows = []
    for text, typ, lab in zip(df["dialogue"], df["type"], df["label"]):
        words = text.split()
        for i in range(0, len(words), max_words):
            chunk = words[i:i + max_words]
            if len(chunk) >= min_words:
                rows.append({"dialogue": " ".join(chunk), "type": typ, "label": lab})
    return pd.DataFrame(rows)


Writing /content/chunking.py


Checks before export (trains nothing).

In [ ]:
# Website_export: step A, checks only (trains nothing)
import importlib.util, sys, numpy as np, pandas as pd
need = [v for v in ["base","EV","m2","MASK_V2","make_pieces","mask_texts","syn_te","sms_df"] if v not in globals()]
print("missing globals:", need)
print("base pieces:", len(base), "| real spam in base:", int(((base.source=="real")&(base.label==1)).sum()), "| types:", sorted(base.type.unique()))
print({k: len(d) for k, d in EV.items()})
def load(p, n):
    s = importlib.util.spec_from_file_location(n, p); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
mk = load("/content/masking.py", "masking_x"); ch = load("/content/chunking.py", "chunking_x")
import inspect; print("mask_texts", inspect.signature(mk.mask_texts), "| to_display", inspect.signature(mk.to_display), "| make_pieces", inspect.signature(ch.make_pieces))
rng = np.random.RandomState(0); samp = list(rng.choice(list(MASK_V2.keys()), 400, replace=False))
a = mk.mask_texts(samp); print("masking.py == MASK_V2 on 400 random texts:", sum(x == MASK_V2[t] for t, x in zip(samp, a)), "/ 400")
one = [mk.mask_texts([t])[0] for t in samp[:100]]; print("single-call == batch on 100:", sum(x == y for x, y in zip(one, a[:100])), "/ 100")
p1 = ch.make_pieces(ds["test"].to_pandas()); p2 = make_pieces(ds["test"].to_pandas()); print("chunking.py == notebook make_pieces on ds test:", p1.equals(p2), len(p1))
print("SMS word counts in sms_df: min", sms_df.dialogue.str.split().str.len().min(), "max", sms_df.dialogue.str.split().str.len().max(), "| >45 words:", int((sms_df.dialogue.str.split().str.len()>45).sum()))
print("sample dialogue piece:", repr(syn_te.dialogue.iloc[0][:300]))

missing globals: []
base pieces: 6267 | real spam in base: 514 | types: ['delivery', 'insurance', 'refund', 'reward', 'sms_ham', 'sms_spam', 'ssn', 'support', 'telemarketing', 'wrong']
{'calibration_sms_3100': 3100, 'calibration_dialogue': 577, 'fresh_sms_904': 904, 'dialogue_normal_test': 707, 'dialogue_scam_test': 707, 'real_sms_test': 256, 'topic_reward': 822, 'topic_refund': 954, 'topic_ssn': 962, 'topic_support': 877}
mask_texts (texts, known_names=()) | to_display (text) | make_pieces (df, max_words=45, min_words=10)
masking.py == MASK_V2 on 400 random texts: 400 / 400
single-call == batch on 100: 100 / 100
chunking.py == notebook make_pieces on ds test: True 1817
SMS word counts in sms_df: min 1 max 171 | >45 words: 65
sample dialogue piece: "caller: Hello, this is Officer Johnson from the Social Security Administration. We've had a report of suspicious activity on your social security number and I need to verify some information to secure your account. receiver: Oh no, what kin

Trains and saves the deployed model, then reloads it to check the scores match.

In [ ]:
# Website_export: step B, train plain LinearSVC on the standard split, save files
import os, json, sys, shutil, joblib, numpy as np, sklearn, spacy
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
OUT = "/content/website_export_local"; os.makedirs(OUT, exist_ok=True)
shutil.copy("/content/masking.py", f"{OUT}/masking.py"); shutil.copy("/content/chunking.py", f"{OUT}/chunking.py")

svm = make_pipeline(tfidf(), LinearSVC(random_state=42))
svm.fit(m2(base["dialogue"]), base["label"].values)
print("pipeline steps:", [n for n, _ in svm.steps], "| vocab size:", len(svm.steps[0][1].vocabulary_), "| train pieces:", len(base), "| labels:", base.label.value_counts().to_dict())
joblib.dump(svm, f"{OUT}/svm_plain.joblib")

s_sms = svm.decision_function(m2(EV["calibration_sms_3100"]["dialogue"]))
s_dia = svm.decision_function(m2(EV["calibration_dialogue"]["dialogue"]))
cut = lambda s: {f"fpr_{k:02d}": float(np.quantile(s, 1 - k / 100)) for k in (1, 5, 10)}
C = {"model_version": "svm_plain_masked_v2_seed42_standard_split",
     "trained_on": "6,267 masked v2 pieces, standard call-level split, seed 42, real spam kept, no topic hidden",
     "sms": cut(s_sms), "dialogue": cut(s_dia)}
json.dump(C, open(f"{OUT}/cutoffs.json", "w"), indent=2); print(json.dumps(C, indent=2))
for src, s in (("sms", s_sms), ("dialogue", s_dia)):
    print(src, "n", len(s), {k: int((s > v).sum()) for k, v in C[src].items()}, "flagged above each cut-off")

SMS = ["Congratulations! You have won RWF 500,000 in the MTN promo. Call +250 788 123 456 now to claim your prize before midnight.",
 "Dear customer, your account is suspended. Verify your details at www.secure-bank-check.com within 24 hours.",
 "URGENT: Your parcel is held at customs. Pay a fee of $25 at parcel-release.net/pay to release it today.",
 "Hi Grace, I am stuck in Nairobi and lost my wallet. Please send KES 5,000 to 0722 555 019, I will pay you back Monday.",
 "FREE entry! Text WIN to 80888 to get a chance to win a brand new iPhone. T&Cs apply.",
 "Hey, are we still meeting at the cafe at 6? Let me know if you are running late.",
 "Mum says dinner is at 7 tonight, can you pick up bread on the way home?",
 "Thanks for the notes from class today, they really helped with the assignment.",
 "Running 10 minutes late, the bus is stuck in traffic near the market.",
 "Hi Eric, the meeting moved to Thursday at 2pm in room 4. See you there."]
CHAT = ["caller: Hello, this is David from the tax office. You owe a penalty of $800 and police will come unless you pay today with gift cards. receiver: What? I have never had a tax problem.",
 "caller: Hi, I am calling from your bank security team. We saw a strange login. Please read me the code we just sent to your phone so we can block it. receiver: Okay, one second.",
 "caller: Good morning, this is Microsoft support. Your computer has a virus. Install the remote app from fix-my-pc.com so I can repair it for you. receiver: How did you get my number?",
 "caller: Congratulations, you were selected for a cash reward of $1,000. To receive it we only need your social security number and date of birth. receiver: Is this real?",
 "caller: Hello, I am from the refund department. We overcharged you $300 and need your card number to send the money back today. receiver: Which company are you with?",
 "caller: Hi, this is Sarah from Dr. Mugisha's clinic calling to confirm your appointment tomorrow at 10am. receiver: Yes, I will be there, thank you for the reminder.",
 "caller: Hello, I am calling about the delivery of your sofa. The driver will arrive between 2 and 4 this afternoon. receiver: Great, I will be home all day.",
 "caller: Hi, this is Peter from the library. The book you reserved is ready for pickup at the front desk. receiver: Perfect, I will come by on Saturday.",
 "caller: Good afternoon, I think I have the wrong number. I was looking for Alice's bakery. receiver: No problem, this is a private number. Have a good day.",
 "caller: Hello, this is the school office. Classes are cancelled tomorrow because of the power cut. receiver: Thanks for letting us know, we will keep the kids at home."]
mk_raw = SMS + CHAT
masked = mk.mask_texts(mk_raw)
sc = svm.decision_function(masked)
verdict = lambda s, src: {k: ("scam" if s > v else "not_scam") for k, v in C[src].items()}
G = []
for i, (t, mt, s) in enumerate(zip(mk_raw, masked, sc)):
    kind = "sms_like" if i < 10 else "chat_like"
    n_pieces = len(ch.make_pieces(pd.DataFrame({"dialogue": [t], "type": ["x"], "label": [0]})))
    G.append({"id": i + 1, "kind": kind, "raw": t, "words": len(t.split()), "pieces_from_make_pieces": n_pieces,
              "expected_masked": mt, "expected_score": float(s),
              "expected_verdict": {"sms_rule": verdict(s, "sms"), "dialogue_rule": verdict(s, "dialogue")}})
    print(f"{i+1:2d} {kind:9s} w={len(t.split()):2d} pcs={n_pieces} score={s:+.4f} sms@5={G[-1]['expected_verdict']['sms_rule']['fpr_05']:8s} dia@5={G[-1]['expected_verdict']['dialogue_rule']['fpr_05']:8s} | {mt[:70]}")
json.dump({"model_version": C["model_version"], "verdict_rule": "scam if decision_function score > cut-off", "masking": "masking.mask_texts([raw])[0], scored on that masked text directly", "cases": G},
          open(f"{OUT}/golden_cases.json", "w"), indent=2, ensure_ascii=False)

# reload check
svm2 = joblib.load(f"{OUT}/svm_plain.joblib"); print("reloaded model, max score diff on golden:", float(np.abs(svm2.decision_function(masked) - sc).max()))
nlp = mk._get_nlp()
V = [f"python {sys.version}", f"scikit-learn {sklearn.__version__}", f"spacy {spacy.__version__}",
     f"en_core_web_sm {nlp.meta['version']}", f"numpy {np.__version__}", f"joblib {joblib.__version__}"]
open(f"{OUT}/versions.txt", "w").write("\n".join(V) + "\n"); print("\n".join(V))

if not os.path.isdir("/content/drive/MyDrive"): print("DRIVE NOT MOUNTED")
else:
    D = "/content/drive/MyDrive/website_export"; os.makedirs(D, exist_ok=True)
    for f in sorted(os.listdir(OUT)): shutil.copy(f"{OUT}/{f}", f"{D}/{f}")
    for f in sorted(os.listdir(D)): print(f"{f:20s} {os.path.getsize(D + '/' + f):>9,d} bytes")
print("DONE export")

pipeline steps: ['tfidfvectorizer', 'linearsvc'] | vocab size: 17764 | train pieces: 6267 | labels: {1: 3422, 0: 2845}
{
  "model_version": "svm_plain_masked_v2_seed42_standard_split",
  "trained_on": "6,267 masked v2 pieces, standard call-level split, seed 42, real spam kept, no topic hidden",
  "sms": {
    "fpr_01": 0.34717345663209526,
    "fpr_05": 0.028324431809978554,
    "fpr_10": -0.12959568844328123
  },
  "dialogue": {
    "fpr_01": 0.10133805334216302,
    "fpr_05": -0.30250515598192995,
    "fpr_10": -0.545883060358889
  }
}
sms n 3100 {'fpr_01': 31, 'fpr_05': 155, 'fpr_10': 310} flagged above each cut-off
dialogue n 577 {'fpr_01': 6, 'fpr_05': 29, 'fpr_10': 58} flagged above each cut-off
 1 sms_like  w=22 pcs=1 score=+1.9564 sms@5=scam     dia@5=scam     | Congratulations! You have won amounttoken in the MTN promo. Call numto
 2 sms_like  w=14 pcs=1 score=+0.7177 sms@5=scam     dia@5=scam     | Dear customer, your account is suspended. Verify your details at linkt
 3 sms_

# 13. Final results
These cells only read saved results; nothing is retrained. The numbers in the report come from here.

In [ ]:
# RESULTS step 1: inventory of saved results (reads only, trains nothing)
import os, glob, pandas as pd
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
R = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42"
paths = [f"{R}/dialogue_calibration_split/models/summary_all_models.csv", f"{R}/catch_rate_at_FA_limits_seed42.csv",
         f"{R}/svm_plain_vs_calibrated/auc_by_seed.csv", f"{R}/svm_plain_vs_calibrated/catch_by_seed.csv"]
paths += sorted(glob.glob(f"{R}/oneclass_seed42/**/*.csv", recursive=True))
print("oneclass_seed42 contents:", sorted(os.listdir(f"{R}/oneclass_seed42")) if os.path.isdir(f"{R}/oneclass_seed42") else "MISSING")
for p in paths:
    if not os.path.exists(p): print("MISSING:", p); continue
    d = pd.read_csv(p); print(f"\n{p.replace(R + '/', '')} | rows {len(d)}\n  columns: {list(d.columns)}")

Mounted at /content/drive
oneclass_seed42 contents: ['near_copy_check_seed42.csv', 'scores_lora_te42_seed42.csv', 'scores_oneclass_seed42.csv', 'summary_oneclass_seed42.json']

dialogue_calibration_split/models/summary_all_models.csv | rows 70
  columns: ['model', 'hidden', 'retrained', 'cut_sms', 'cut_dialogue', 'auc_sms', 'auc_dialogue', 'calsms_at_diacut_k', 'calsms_at_diacut_n', 'calsms_at_diacut_rate', 'calsms_at_diacut_lo95', 'calsms_at_diacut_hi95', 'fresh_at_diacut_k', 'fresh_at_diacut_n', 'fresh_at_diacut_rate', 'fresh_at_diacut_lo95', 'fresh_at_diacut_hi95', 'calsms_at_smscut_k', 'calsms_at_smscut_n', 'calsms_at_smscut_rate', 'calsms_at_smscut_lo95', 'calsms_at_smscut_hi95', 'fresh_at_smscut_k', 'fresh_at_smscut_n', 'fresh_at_smscut_rate', 'fresh_at_smscut_lo95', 'fresh_at_smscut_hi95', 'caldia_at_diacut_k', 'caldia_at_diacut_n', 'caldia_at_diacut_rate', 'caldia_at_diacut_lo95', 'caldia_at_diacut_hi95', 'dianorm_at_smscut_k', 'dianorm_at_smscut_n', 'dianorm_at_smscut_rate', '

In [ ]:
# RESULTS step 2: summary_all_models.csv (reads only)
import numpy as np
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
S = pd.read_csv(f"{R}/dialogue_calibration_split/models/summary_all_models.csv")
ORDER = ["SVM", "LR", "NB", "CNN", "DistilBERT", "LoRA", "XGB", "BiLSTM", "zero-shot", "SVM_calibrated", "E1", "E2", "E3", "E4"]
print("models in file:", sorted(S.model.unique()), "| hidden values:", sorted(S.hidden.unique()))
print("rows per model:", S.model.value_counts().to_dict())
missing = [m for m in ORDER if m not in set(S.model)]; print("models missing from file:", missing)
T = S[S.hidden != "none"]

print("\n(a) Mean catch rate on the 4 hidden scam topics (5% false-alarm cut-offs)")
a = T.groupby("model").agg(sms_cut=("topic_at_smscut_rate", "mean"), dia_cut=("topic_at_diacut_rate", "mean"),
                           caught_sms=("topic_at_smscut_k", "sum"), caught_dia=("topic_at_diacut_k", "sum"), n=("topic_at_diacut_n", "sum")).reindex(ORDER)
a["pooled_sms"] = a.caught_sms / a.n; a["pooled_dia"] = a.caught_dia / a.n
print(a.round(3).to_string())
print("note: 'sms_cut'/'dia_cut' = plain mean of the 4 topic rates; 'pooled' = all caught / all pieces (topics have different sizes)")

print("\n(b) Catch per hidden topic, k/n and rate, SMS cut-off -> dialogue cut-off")
b = T.assign(cell=lambda d: d.topic_at_smscut_k.astype(int).astype(str) + "/" + d.topic_at_smscut_n.astype(int).astype(str)
             + " (" + d.topic_at_smscut_rate.round(3).astype(str) + ") -> " + d.topic_at_diacut_k.astype(int).astype(str)
             + " (" + d.topic_at_diacut_rate.round(3).astype(str) + ")")
print(b.pivot(index="model", columns="hidden", values="cell").reindex(ORDER)[["reward", "refund", "ssn", "support"]].to_string())

print("\n(c) Main run, nothing hidden: rate [Wilson 95%] (k/n)")
N = S[S.hidden == "none"].set_index("model").reindex(ORDER)
def f(t): return N.apply(lambda r: f"{r[t+'_rate']:.3f} [{r[t+'_lo95']:.3f}-{r[t+'_hi95']:.3f}] ({int(r[t+'_k'])}/{int(r[t+'_n'])})", axis=1)
c = pd.DataFrame({"AUC_sms": N.auc_sms.round(4), "AUC_dia": N.auc_dialogue.round(4), "cut_sms": N.cut_sms.round(4), "cut_dia": N.cut_dialogue.round(4)})
print(c.to_string())
for t, lab in [("spam_at_smscut", "real spam caught @ SMS cut"), ("fresh_at_smscut", "fresh normal SMS flagged @ SMS cut"),
               ("fresh_at_diacut", "fresh normal SMS flagged @ dialogue cut"), ("dianorm_at_smscut", "normal test dialogue flagged @ SMS cut"),
               ("dianorm_at_diacut", "normal test dialogue flagged @ dialogue cut"), ("calsms_at_diacut", "calibration SMS flagged @ dialogue cut")]:
    print(f"\n{lab}"); print(f(t).to_string())

models in file: ['BiLSTM', 'CNN', 'DistilBERT', 'E1', 'E2', 'E3', 'E4', 'LR', 'LoRA', 'NB', 'SVM', 'SVM_calibrated', 'XGB', 'zero-shot'] | hidden values: ['none', 'refund', 'reward', 'ssn', 'support']
rows per model: {'NB': 5, 'LR': 5, 'SVM': 5, 'XGB': 5, 'CNN': 5, 'BiLSTM': 5, 'DistilBERT': 5, 'zero-shot': 5, 'LoRA': 5, 'SVM_calibrated': 5, 'E1': 5, 'E2': 5, 'E3': 5, 'E4': 5}
models missing from file: []

(a) Mean catch rate on the 4 hidden scam topics (5% false-alarm cut-offs)
                sms_cut  dia_cut  caught_sms  caught_dia       n  pooled_sms  pooled_dia
model                                                                                   
SVM               0.911    0.976      3295.0      3529.0  3615.0       0.911       0.976
LR                0.856    0.962      3103.0      3478.0  3615.0       0.858       0.962
NB                0.837    0.961      3036.0      3475.0  3615.0       0.840       0.961
CNN               0.836    0.957      3035.0      3460.0  3615.0       

In [ ]:
# RESULTS steps 3-6 (reads only)
import json
print("STEP 3. Catch at 1% / 5% / 10% false-alarm limits (single cut-off from calibration SMS, nothing hidden)")
F = pd.read_csv(f"{R}/catch_rate_at_FA_limits_seed42.csv")
print("models:", list(F.model.unique()), "| limits:", list(F.limit.unique()))
for t, lab in [("real_spam", "real spam caught"), ("dialogue_scam", "dialogue scams caught"), ("fresh_FA", "fresh normal SMS flagged (false alarms)")]:
    F["cell"] = F[t+"_rate"].map("{:.3f}".format) + " [" + F[t+"_lo95"].map("{:.3f}".format) + "-" + F[t+"_hi95"].map("{:.3f}".format) + "]"
    print(f"\n{lab}"); print(F.pivot(index="model", columns="limit", values="cell").reindex(F.model.unique()).to_string())

print("\n\nSTEP 4. One-class models")
J = json.load(open(f"{R}/oneclass_seed42/summary_oneclass_seed42.json")); print(json.dumps(J, indent=1)[:3000])
O = pd.read_csv(f"{R}/oneclass_seed42/scores_oneclass_seed42.csv")
print("\nmodels:", list(O.model.unique()), "| contexts:", list(O.context.unique()), "| sets:", list(O.set.unique()))
g = O.groupby(["model", "context", "set", "label"]).flagged.agg(k="sum", n="size").reset_index()
g["rate"] = (g.k / g.n).round(3); g["wilson95"] = [f"{wilson(k, n)[0]:.3f}-{wilson(k, n)[1]:.3f}" if "wilson" in globals() else "" for k, n in zip(g.k, g.n)]
print("flagged share by model / context / set / label (label 1 = scam, so rate = catch; label 0 = normal, so rate = false alarm)")
print(g.to_string(index=False))

print("\n\nSTEP 5. Plain vs calibrated SVM across seeds")
A = pd.read_csv(f"{R}/svm_plain_vs_calibrated/auc_by_seed.csv"); print("AUC on real SMS test:"); print(A.round(4).to_string(index=False))
Cb = pd.read_csv(f"{R}/svm_plain_vs_calibrated/catch_by_seed.csv"); print("\nreal spam caught by seed and limit:")
print(Cb[["seed", "model", "limit", "calib_flagged", "fresh_rate", "spam_caught", "spam_n", "spam_rate", "spam_lo95", "spam_hi95"]].round(3).to_string(index=False))

print("\n\nSTEP 6. Summary (computed from the tables above)")
singles = [m for m in ORDER if not m.startswith("E")]; ens = ["E1", "E2", "E3", "E4"]
best = a.loc[singles, "dia_cut"].idxmax(); worst = a.loc[singles, "dia_cut"].idxmin(); bens = a.loc[ens, "dia_cut"].idxmax()
print(f"- Best single model on unseen topics (dialogue cut-off): {best} {a.loc[best,'dia_cut']:.3f}")
print(f"- Weakest: {worst} {a.loc[worst,'dia_cut']:.3f} (it also drops from {a.loc[worst,'sms_cut']:.3f} at the SMS cut-off)")
print(f"- Best ensemble: {bens} {a.loc[bens,'dia_cut']:.3f}, {100*(a.loc[bens,'dia_cut']-a.loc[best,'dia_cut']):+.1f} points vs {best}")
fa = N.loc[singles, "dianorm_at_diacut_rate"]; fe = N.loc[ens, "dianorm_at_diacut_rate"]
print(f"- Normal dialogue false alarms at the dialogue cut-off: singles {fa.min():.3f} ({fa.idxmin()}) to {fa.max():.3f} ({fa.idxmax()}); ensembles {fe.min():.3f} to {fe.max():.3f}")

STEP 3. Catch at 1% / 5% / 10% false-alarm limits (single cut-off from calibration SMS, nothing hidden)
models: ['NB', 'LR', 'SVM', 'XGB', 'CNN', 'BiLSTM', 'DistilBERT', 'zero-shot', 'LoRA', 'E1 SVM+LoRA', 'E2 SVM+LoRA+zero-shot', 'E3 LoRA+zero-shot', 'E4 all four', 'SVM_calibrated (extra: ensemble member)', 'IsolationForest', 'OneClassSVM'] | limits: [np.float64(0.01), np.float64(0.05), np.float64(0.1)]

real spam caught
limit                                                   0.01                 0.05                 0.10
model                                                                                                 
NB                                       0.820 [0.745-0.877]  0.914 [0.853-0.951]  0.930 [0.872-0.963]
LR                                       0.789 [0.711-0.851]  0.938 [0.881-0.968]  0.969 [0.922-0.988]
SVM                                      0.867 [0.798-0.915]  0.945 [0.891-0.973]  0.969 [0.922-0.988]
XGB                                      0.898 [0.834-0.940

In [ ]:
# RESULTS step 7: save the final results tables to one Excel file on Drive (reads only, trains nothing)
import os, json, numpy as np, pandas as pd
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
R = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42"; OUTD = f"{R}/results_tables"; os.makedirs(OUTD, exist_ok=True)
def wil(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d; return c - h, c + h
ORDER = ["SVM", "LR", "NB", "CNN", "DistilBERT", "LoRA", "XGB", "BiLSTM", "zero-shot", "SVM_calibrated", "E1", "E2", "E3", "E4"]
S = pd.read_csv(f"{R}/dialogue_calibration_split/models/summary_all_models.csv"); T = S[S.hidden != "none"]; N = S[S.hidden == "none"].set_index("model").reindex(ORDER)
print("summary rows:", len(S), "| models:", S.model.nunique())

a = T.groupby("model").agg(catch_sms_cut=("topic_at_smscut_rate", "mean"), catch_dialogue_cut=("topic_at_diacut_rate", "mean"),
                           caught_sms_cut=("topic_at_smscut_k", "sum"), caught_dialogue_cut=("topic_at_diacut_k", "sum"), pieces=("topic_at_diacut_n", "sum")).reindex(ORDER)
a["normal_dialogue_flagged_dialogue_cut"] = N.dianorm_at_diacut_rate; a["auc_sms"] = N.auc_sms; a["auc_dialogue"] = N.auc_dialogue
a = a.round(4).reset_index()

b = T[["model", "hidden", "topic_at_smscut_k", "topic_at_smscut_n", "topic_at_smscut_rate", "topic_at_smscut_lo95", "topic_at_smscut_hi95",
       "topic_at_diacut_k", "topic_at_diacut_rate", "topic_at_diacut_lo95", "topic_at_diacut_hi95"]].copy()
b["model"] = pd.Categorical(b.model, ORDER); b = b.sort_values(["model", "hidden"]).round(4)

cols = ["cut_sms", "cut_dialogue", "auc_sms", "auc_dialogue"]
for t in ["spam_at_smscut", "fresh_at_smscut", "fresh_at_diacut", "dianorm_at_smscut", "dianorm_at_diacut", "calsms_at_diacut"]:
    cols += [f"{t}_k", f"{t}_n", f"{t}_rate", f"{t}_lo95", f"{t}_hi95"]
c = N[cols].round(4).reset_index()

F = pd.read_csv(f"{R}/catch_rate_at_FA_limits_seed42.csv").round(4)

O = pd.read_csv(f"{R}/oneclass_seed42/scores_oneclass_seed42.csv")
g = O.groupby(["model", "context", "set", "label"]).flagged.agg(k="sum", n="size").reset_index()
g["rate"] = g.k / g.n; g["lo95"] = [wil(k, n)[0] for k, n in zip(g.k, g.n)]; g["hi95"] = [wil(k, n)[1] for k, n in zip(g.k, g.n)]; g = g.round(4)

A = pd.read_csv(f"{R}/svm_plain_vs_calibrated/auc_by_seed.csv").round(4); Cb = pd.read_csv(f"{R}/svm_plain_vs_calibrated/catch_by_seed.csv").round(4)

notes = pd.DataFrame({"note": [
 "All results: masked v2 text, seed 42, standard call-level split (6,267 training pieces), unless stated.",
 "Cut-offs set so 5% of normal calibration messages are flagged: SMS cut-off from 3,100 normal SMS, dialogue cut-off from 577 normal dialogue pieces.",
 "unseen_topics: each scam topic (reward, refund, ssn, support) removed from training, then tested on that topic. Mean = plain mean of the 4 topic rates.",
 "A message is flagged as scam if its score is above the cut-off. lo95/hi95 = Wilson 95% interval.",
 "FA_limits sheet comes from the earlier setup (full training set, single SMS cut-off, nothing hidden), so its numbers do not match the other sheets.",
 "plain_vs_cal sheets are the only results run on 3 seeds (42, 7, 2026).",
 "Dialogue data is synthetic (BothBosu scam-dialogue); real SMS from the UCI SMS Spam Collection."]})

P = f"{OUTD}/final_results_seed42.xlsx"
with pd.ExcelWriter(P) as w:
    notes.to_excel(w, sheet_name="notes", index=False); a.to_excel(w, sheet_name="unseen_topics_mean", index=False); b.to_excel(w, sheet_name="unseen_topics_per_topic", index=False)
    c.to_excel(w, sheet_name="main_run_nothing_hidden", index=False); F.to_excel(w, sheet_name="FA_limits_earlier_setup", index=False)
    g.to_excel(w, sheet_name="one_class", index=False); A.to_excel(w, sheet_name="plain_vs_cal_auc", index=False); Cb.to_excel(w, sheet_name="plain_vs_cal_catch", index=False)
print("saved:", P, "|", os.path.getsize(P), "bytes")
for s in pd.ExcelFile(P).sheet_names: print(f"  {s}: {len(pd.read_excel(P, sheet_name=s))} rows")
print(a[["model", "catch_sms_cut", "catch_dialogue_cut", "normal_dialogue_flagged_dialogue_cut", "auc_dialogue"]].to_string(index=False))

Mounted at /content/drive
summary rows: 70 | models: 14
saved: /content/drive/MyDrive/ScamGuard_masked_v2_seed42/results_tables/final_results_seed42.xlsx | 24807 bytes
  notes: 7 rows
  unseen_topics_mean: 14 rows
  unseen_topics_per_topic: 56 rows
  main_run_nothing_hidden: 14 rows
  FA_limits_earlier_setup: 48 rows
  one_class: 32 rows
  plain_vs_cal_auc: 3 rows
  plain_vs_cal_catch: 18 rows
         model  catch_sms_cut  catch_dialogue_cut  normal_dialogue_flagged_dialogue_cut  auc_dialogue
           SVM         0.9109              0.9762                                0.0622        0.9987
            LR         0.8559              0.9620                                0.0481        0.9976
            NB         0.8373              0.9606                                0.0495        0.9977
           CNN         0.8360              0.9566                                0.0537        0.9965
    DistilBERT         0.8756              0.9307                                0.0438      

In [ ]:
# Precision, recall, F1 and macro-F1 for all models (reads saved scores only, trains nothing)
import os, numpy as np, pandas as pd
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
R  = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42"
MD = f"{R}/dialogue_calibration_split/models"
ORDER = ["SVM","LR","NB","XGB","CNN","BiLSTM","DistilBERT","LoRA","zero-shot","SVM_calibrated","E1","E2","E3","E4"]
TOPICS = ["reward","refund","ssn","support"]

def prf(y, flag):
    y = np.asarray(y).astype(int); f = np.asarray(flag).astype(bool)
    tp = int(((y==1)&f).sum()); fp = int(((y==0)&f).sum()); fn = int(((y==1)&~f).sum()); tn = int(((y==0)&~f).sum())
    p  = tp/(tp+fp) if tp+fp else np.nan;  r  = tp/(tp+fn) if tp+fn else np.nan
    f1 = 2*p*r/(p+r) if (p+r) else np.nan
    pn = tn/(tn+fn) if tn+fn else np.nan;  rn = tn/(tn+fp) if tn+fp else np.nan
    f1n = 2*pn*rn/(pn+rn) if (pn+rn) else np.nan
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=p, recall=r, f1=f1, macro_f1=(f1+f1n)/2)

rows = []
for m in ORDER:
    base = f"{MD}/{m}/scores_hidden_none.csv"
    if not os.path.exists(base): print("MISSING:", base); continue
    d = pd.read_csv(base); cs, cd = d.cut_sms.iloc[0], d.cut_dialogue.iloc[0]
    if m == "SVM": print("sets in file:", d.set.value_counts().to_dict())
    sms = d[d.set == "real_sms_test"]
    dia = d[d.set.isin(["dialogue_scam_test", "dialogue_normal_test"])]
    a = prf(sms.label, sms.score > cs); b = prf(dia.label, dia.score > cd)
    un, src = [], []
    for t in TOPICS:
        fn = f"{MD}/{m}/scores_hidden_{t}.csv"
        h = pd.read_csv(fn) if os.path.exists(fn) else d          # zero-shot has no retrained runs
        src.append("retrained" if os.path.exists(fn) else "main run")
        x = h[h.set.isin([f"topic_{t}", "dialogue_normal_test"])]
        un.append(prf(x.label, x.score > h.cut_dialogue.iloc[0]))
    u = pd.DataFrame(un).mean(numeric_only=True)
    rows.append(dict(model=m,
        sms_precision=a["precision"], sms_recall=a["recall"], sms_f1=a["f1"], sms_macro_f1=a["macro_f1"],
        dia_precision=b["precision"], dia_recall=b["recall"], dia_f1=b["f1"], dia_macro_f1=b["macro_f1"],
        unseen_precision=u["precision"], unseen_recall=u["recall"], unseen_f1=u["f1"], unseen_macro_f1=u["macro_f1"],
        sms_counts=f"tp{a['tp']} fp{a['fp']} fn{a['fn']} tn{a['tn']}", dia_counts=f"tp{b['tp']} fp{b['fp']} fn{b['fn']} tn{b['tn']}",
        unseen_scores_from=",".join(sorted(set(src)))))

T = pd.DataFrame(rows).set_index("model")
pd.set_option("display.width", 250)
print("\nKnown scam types (main run, 5% cut-offs)")
print(T[["sms_precision","sms_recall","sms_f1","sms_macro_f1","dia_precision","dia_recall","dia_f1","dia_macro_f1"]].round(3).to_string())
print("\nUnseen scam types (mean of 4 hidden topics + 707 normal dialogue, dialogue cut-off)")
print(T[["unseen_precision","unseen_recall","unseen_f1","unseen_macro_f1","unseen_scores_from"]].round(3).to_string())
print("\nCounts (check: SVM SMS should be tp122, SVM dialogue fp44)")
print(T[["sms_counts","dia_counts"]].to_string())

O = pd.read_csv(f"{R}/oneclass_seed42/scores_oneclass_seed42.csv")
oc = O[(O.context == "main") & (O.set == "test_te42")]
print("\nOne-class models (earlier setup, test_te42, 5% SMS cut-off)")
for m, g in oc.groupby("model"):
    r = prf(g.label, g.flagged); print(f"  {m}: precision {r['precision']:.3f} recall {r['recall']:.3f} f1 {r['f1']:.3f} macro-F1 {r['macro_f1']:.3f} (tp{r['tp']} fp{r['fp']})")

out = f"{R}/results_tables/precision_recall_f1_seed42.csv"; T.round(4).to_csv(out); print("\nsaved:", out)

sets in file: {'calibration_sms_3100': 3100, 'topic_ssn': 962, 'topic_refund': 954, 'fresh_sms_904': 904, 'topic_support': 877, 'topic_reward': 822, 'dialogue_scam_test': 707, 'dialogue_normal_test': 707, 'calibration_dialogue': 577, 'real_sms_test': 256}

Known scam types (main run, 5% cut-offs)
                sms_precision  sms_recall  sms_f1  sms_macro_f1  dia_precision  dia_recall  dia_f1  dia_macro_f1
model                                                                                                           
SVM                     0.938       0.953   0.946         0.945          0.941       1.000   0.970         0.969
LR                      0.945       0.938   0.941         0.941          0.954       0.994   0.974         0.973
NB                      0.967       0.906   0.935         0.937          0.953       0.993   0.972         0.972
XGB                     0.961       0.961   0.961         0.961          0.931       0.993   0.961         0.960
CNN                     

### Whole-chat evaluation
A whole chat is scored by its highest-scoring piece.

In [ ]:
# Whole-chat evaluation of the deployed plain SVM (masked v2, seed 42)
import numpy as np, pandas as pd, time
from collections import defaultdict
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

# 1. masked texts + seed-42 splits (copied from the existing cells)
if "MASK_V2" not in globals():
    all_raw = list(dict.fromkeys(list(syn_tr["dialogue"]) + list(syn_te["dialogue"]) + list(sms_df["dialogue"]) + sum(HAND.values(), [])))
    t0 = time.time(); MASK_V2 = dict(zip(all_raw, mask_texts(all_raw))); print(f"masked {len(MASK_V2)} texts in {time.time()-t0:.0f} s")
def m2(x): return [MASK_V2[str(t)] if str(t) in MASK_V2 else mask_texts([str(t)])[0] for t in x]
tr42, te42, s_tr42, s_te42 = build_data(42)
cut42 = s_tr42[(s_tr42.label == 0) & ~s_tr42.dialogue.isin(tr42.dialogue)]

# 2. call-level calibration split (copied from the call-level cell)
def chunks(text):
    w = text.split(); return [" ".join(w[i:i+45]) for i in range(0, len(w), 45) if len(w[i:i+45]) >= 10]
raw_tr, raw_te = ds["train"].to_pandas(), ds["test"].to_pandas()
CALLS = defaultdict(set)
for cid, text in enumerate(raw_tr["dialogue"]):
    for ch in chunks(text): CALLS[ch].add(cid)
nd = tr42[(tr42.source == "synthetic") & (tr42.label == 0)].copy()
nd["call"] = [min(CALLS[t]) if t in CALLS else -1 for t in nd["dialogue"]]
rng = np.random.default_rng(42); cal_calls = set()
for typ, g in nd.groupby("type"):
    calls = np.array(sorted(g.call.unique())); rng.shuffle(calls); k = int(round(0.2*len(calls))); cal_calls |= set(calls[:k].tolist())
cal_dia = nd[nd.call.isin(cal_calls)]
base = tr42[~tr42.dialogue.isin(cal_dia.dialogue)]
print(f"calibration dialogue: {len(cal_dia)} pieces from {cal_dia.call.nunique()} calls | base: {len(base)} pieces  (expect 577, 129, 6267)")

# 3. deployed model + cut-offs
def fit_svm(df): return make_pipeline(tfidf(), LinearSVC(random_state=42)).fit(m2(df["dialogue"]), df["label"].values)
def cutoffs(m):
    q = lambda s: {k: float(np.quantile(s, 1 - k/100)) for k in (1, 5, 10)}
    return q(m.decision_function(m2(cut42["dialogue"]))), q(m.decision_function(m2(cal_dia["dialogue"])))
svm = fit_svm(base); CUT_SMS, CUT_DIA = cutoffs(svm)
print("LinearSVC params:", {k: v for k, v in svm.steps[-1][1].get_params().items() if k in ("C", "loss", "penalty", "class_weight", "max_iter")})
print("chat cut-offs:", {k: round(v, 4) for k, v in CUT_DIA.items()}, " (expect 5% about -0.3025)")
dn = te42[(te42.source == "synthetic") & (te42.label == 0)]; dsc = te42[(te42.source == "synthetic") & (te42.label == 1)]
tp = int((svm.decision_function(m2(dsc.dialogue)) > CUT_DIA[5]).sum()); fp = int((svm.decision_function(m2(dn.dialogue)) > CUT_DIA[5]).sum())
print(f"SANITY piece level @5%: tp {tp}/{len(dsc)}, fp {fp}/{len(dn)}  (expect 707 and 44)")

# 4. whole-chat: score every piece of each test call, keep the highest
def chat_scores(m, calls_df):
    out = []
    for cid, (text, typ, lab) in enumerate(zip(calls_df["dialogue"], calls_df["type"], calls_df["label"])):
        ps = chunks(text); s = m.decision_function(m2(ps))
        out.append({"call": cid, "type": typ, "label": lab, "pieces": len(ps), "max_score": float(s.max()), "text": text})
    return pd.DataFrame(out)
CH = chat_scores(svm, raw_te)
print("\ntest calls:", CH.label.value_counts().to_dict(), "| pieces per call (min/median/max):",
      {lab: (int(g.pieces.min()), int(g.pieces.median()), int(g.pieces.max())) for lab, g in CH.groupby("label")})
ROWS = []
for k in (1, 5, 10):
    c = CUT_DIA[k]; sc, nm = CH[CH.label == 1], CH[CH.label == 0]
    rk, fk = int((sc.max_score > c).sum()), int((nm.max_score > c).sum())
    piece_fp = int((svm.decision_function(m2(dn.dialogue)) > c).sum())
    ROWS.append({"cut-off": f"{k}%", "value": round(c, 4),
                 "whole-chat recall": f"{rk}/{len(sc)} = {rk/len(sc):.3f} [{wilson(rk, len(sc))[0]:.3f}, {wilson(rk, len(sc))[1]:.3f}]",
                 "whole-chat false alarms": f"{fk}/{len(nm)} = {fk/len(nm):.3f} [{wilson(fk, len(nm))[0]:.3f}, {wilson(fk, len(nm))[1]:.3f}]",
                 "piece false alarms": f"{piece_fp}/{len(dn)} = {piece_fp/len(dn):.3f}"})
print(pd.DataFrame(ROWS).to_string(index=False))
print("\nTop 5 normal calls by max score (possible false alarms):")
for _, r in CH[CH.label == 0].nlargest(5, "max_score").iterrows():
    print(f"  {r['type']:13s} {r['max_score']:+.3f} | {m2([r['text']])[0][:200]}")

# 5. whole-chat withheld topics: retrain without each scam topic, all calls of that topic (train + test)
raw_all = pd.concat([raw_tr, raw_te], ignore_index=True); TOP = []
for t in ["reward", "refund", "ssn", "support"]:
    m = fit_svm(base[base.type != t]); _, cd = cutoffs(m)
    hid = chat_scores(m, raw_all[raw_all.type == t]); nrm = chat_scores(m, raw_te[raw_te.label == 0])
    rk, fk = int((hid.max_score > cd[5]).sum()), int((nrm.max_score > cd[5]).sum())
    TOP.append({"hidden topic": t, "whole-chat recall": f"{rk}/{len(hid)} = {rk/len(hid):.3f}", "normal calls flagged": f"{fk}/{len(nrm)} = {fk/len(nrm):.3f}", "_r": rk/len(hid)})
    print("done", t)
top = pd.DataFrame(TOP); print(top.drop(columns="_r").to_string(index=False)); print(f"mean whole-chat withheld recall: {top._r.mean():.3f}  (piece level was 0.976)")
pd.DataFrame(ROWS).to_csv("/content/wholechat_results_seed42.csv", index=False); top.to_csv("/content/wholechat_topics_seed42.csv", index=False)
print("saved locally: /content/wholechat_results_seed42.csv, /content/wholechat_topics_seed42.csv")

masked 14184 texts in 130 s
calibration dialogue: 577 pieces from 129 calls | base: 6267 pieces  (expect 577, 129, 6267)
LinearSVC params: {'C': 1.0, 'class_weight': None, 'loss': 'squared_hinge', 'max_iter': 1000, 'penalty': 'l2'}
chat cut-offs: {1: 0.1013, 5: -0.3025, 10: -0.5459}  (expect 5% about -0.3025)
SANITY piece level @5%: tp 707/707, fp 44/707  (expect 707 and 44)

test calls: {1: 160, 0: 160} | pieces per call (min/median/max): {0: (2, 5, 8), 1: (3, 7, 9)}
cut-off   value              whole-chat recall       whole-chat false alarms piece false alarms
     1%  0.1013 160/160 = 1.000 [0.977, 1.000] 16/160 = 0.100 [0.062, 0.156]     17/707 = 0.024
     5% -0.3025 160/160 = 1.000 [0.977, 1.000] 34/160 = 0.212 [0.156, 0.282]     44/707 = 0.062
    10% -0.5459 160/160 = 1.000 [0.977, 1.000] 51/160 = 0.319 [0.252, 0.394]     73/707 = 0.103

Top 5 normal calls by max score (possible false alarms):
  wrong         +0.905 | caller: Hi, is this nametoken? receiver: No, this is nametok

Chat-level cut-offs set on whole normal calibration calls.

In [ ]:
# Chat-level cut-offs set on whole calibration calls (max piece score), plain SVM, seed 42
def call_max(m, call_ids):
    return np.array([m.decision_function(m2(chunks(raw_tr["dialogue"].iloc[c]))).max() for c in sorted(call_ids)])
def chat_cuts(m):
    s = call_max(m, cal_calls)
    return {k: float(np.quantile(s, 1 - k/100)) for k in (1, 5, 10)}, s
CCUT, cal_max = chat_cuts(svm)
print(f"calibration calls: {len(cal_max)} | max-score range {cal_max.min():+.3f} to {cal_max.max():+.3f}")
print("chat-level cut-offs:", {k: round(v, 4) for k, v in CCUT.items()}, "| piece-level were:", {k: round(v, 4) for k, v in CUT_DIA.items()})
R2 = []
for k in (1, 5, 10):
    c = CCUT[k]; sc, nm = CH[CH.label == 1], CH[CH.label == 0]
    rk, fk = int((sc.max_score > c).sum()), int((nm.max_score > c).sum())
    R2.append({"chat cut-off": f"{k}%", "value": round(c, 4),
               "whole-chat recall": f"{rk}/{len(sc)} = {rk/len(sc):.3f} [{wilson(rk, len(sc))[0]:.3f}, {wilson(rk, len(sc))[1]:.3f}]",
               "whole-chat false alarms": f"{fk}/{len(nm)} = {fk/len(nm):.3f} [{wilson(fk, len(nm))[0]:.3f}, {wilson(fk, len(nm))[1]:.3f}]"})
print(pd.DataFrame(R2).to_string(index=False))
print("\nLowest-scoring scam calls (closest to being missed):")
for _, r in CH[CH.label == 1].nsmallest(3, "max_score").iterrows():
    print(f"  {r['type']:8s} {r['max_score']:+.3f}")

TOP2 = []
for t in ["reward", "refund", "ssn", "support"]:
    m = fit_svm(base[base.type != t]); cc, _ = chat_cuts(m)
    hid = chat_scores(m, raw_all[raw_all.type == t]); nrm = chat_scores(m, raw_te[raw_te.label == 0])
    rk, fk = int((hid.max_score > cc[5]).sum()), int((nrm.max_score > cc[5]).sum())
    TOP2.append({"hidden topic": t, "chat cut-off 5%": round(cc[5], 4), "whole-chat recall": f"{rk}/{len(hid)} = {rk/len(hid):.3f}",
                 "normal calls flagged": f"{fk}/{len(nrm)} = {fk/len(nrm):.3f}", "_r": rk/len(hid), "_f": fk/len(nrm)})
    print("done", t)
t2 = pd.DataFrame(TOP2); print(t2.drop(columns=["_r", "_f"]).to_string(index=False))
print(f"mean whole-chat withheld recall {t2._r.mean():.3f} | mean normal calls flagged {t2._f.mean():.3f}")
pd.DataFrame(R2).to_csv("/content/wholechat_chatcutoff_seed42.csv", index=False); t2.to_csv("/content/wholechat_chatcutoff_topics_seed42.csv", index=False)
print("saved locally: /content/wholechat_chatcutoff_seed42.csv, /content/wholechat_chatcutoff_topics_seed42.csv")

calibration calls: 129 | max-score range -1.341 to +0.339
chat-level cut-offs: {1: 0.2848, 5: 0.0894, 10: -0.0277} | piece-level were: {1: 0.1013, 5: -0.3025, 10: -0.5459}
chat cut-off   value              whole-chat recall       whole-chat false alarms
          1%  0.2848 160/160 = 1.000 [0.977, 1.000]  9/160 = 0.056 [0.030, 0.103]
          5%  0.0894 160/160 = 1.000 [0.977, 1.000] 16/160 = 0.100 [0.062, 0.156]
         10% -0.0277 160/160 = 1.000 [0.977, 1.000] 21/160 = 0.131 [0.087, 0.192]

Lowest-scoring scam calls (closest to being missed):
  support  +1.120
  refund   +1.244
  refund   +1.276
done reward
done refund
done ssn
done support
hidden topic  chat cut-off 5% whole-chat recall normal calls flagged
      reward           0.0161   200/200 = 1.000       11/160 = 0.069
      refund           0.0121   200/200 = 1.000       15/160 = 0.094
         ssn           0.0518   200/200 = 1.000       17/160 = 0.106
     support           0.0035   200/200 = 1.000       16/160 = 0.100
m

All models and all metrics in one table.

In [ ]:
# All models and all metrics in one table (reads saved results only, trains nothing)
import os, pandas as pd
from IPython.display import display
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
R = '/content/drive/MyDrive/ScamGuard_masked_v2_seed42/results_tables'
X = pd.ExcelFile(f'{R}/final_results_seed42.xlsx')
ORDER = ['SVM','LR','NB','XGB','CNN','BiLSTM','DistilBERT','LoRA','zero-shot','SVM_calibrated','E1','E2','E3','E4']
main = pd.read_excel(X, 'main_run_nothing_hidden').set_index('model')
un = pd.read_excel(X, 'unseen_topics_mean').set_index('model')
prf = pd.read_csv(f'{R}/precision_recall_f1_seed42.csv').set_index('model')
lim = pd.read_excel(X, 'FA_limits_earlier_setup')
print('models missing from a sheet:', {n: [m for m in ORDER if m not in d.index] for n, d in [('main', main), ('unseen', un), ('prf', prf)]})
ci = lambda d, p: d[f'{p}_rate'].map('{:.3f}'.format) + ' [' + d[f'{p}_lo95'].map('{:.3f}'.format) + ', ' + d[f'{p}_hi95'].map('{:.3f}'.format) + ']'
f3 = lambda s: s.map('{:.3f}'.format)
import re
def exact(col):
    c = prf[col].astype(str).apply(lambda s: dict((k, int(v)) for k, v in re.findall(r'(tp|fp|fn|tn)\s*(\d+)', s)))
    tp, fp, fn, tn = [c.map(lambda d: d[k]) for k in ('tp', 'fp', 'fn', 'tn')]
    p, r = tp / (tp + fp), tp / (tp + fn); f1 = 2 * p * r / (p + r)
    pn, rn = tn / (tn + fn), tn / (tn + fp); f1n = 2 * pn * rn / (pn + rn)
    return p, r, f1, (f1 + f1n) / 2
sp, sr, sf, sm = exact('sms_counts'); dp, dr, df_, dm = exact('dia_counts')
print('SVM sms_counts:', prf.loc['SVM', 'sms_counts'], '| E3 dia_counts:', prf.loc['E3', 'dia_counts'])

A = pd.DataFrame({'AUC SMS': f3(main['auc_sms']), 'AUC chat': f3(main['auc_dialogue']),
    'SMS precision': f3(sp), 'SMS recall': f3(sr), 'SMS F1': f3(sf), 'SMS macro-F1': f3(sm),
    'Chat precision': f3(dp), 'Chat recall': f3(dr), 'Chat F1': f3(df_), 'Chat macro-F1': f3(dm)}).reindex(ORDER)
B = pd.DataFrame({'False alarms, 904 fresh normal SMS [95% Wilson]': ci(main, 'fresh_at_smscut'),
    'False alarms, 707 normal chat pieces [95% Wilson]': ci(main, 'dianorm_at_diacut'),
    'Withheld-topic recall (SMS cut-off)': f3(un['catch_sms_cut']), 'Withheld-topic recall (chat cut-off)': f3(un['catch_dialogue_cut']),
    'Withheld precision': f3(prf['unseen_precision']), 'Withheld F1': f3(prf['unseen_f1']), 'Withheld macro-F1': f3(prf['unseen_macro_f1'])}).reindex(ORDER)
C = lim.pivot_table(index='model', columns='limit', values=['real_spam_rate', 'dialogue_scam_rate'])
C.columns = [('Real SMS spam recall' if a == 'real_spam_rate' else 'Chat scam recall') + f' at {b:.0%} false alarms' for a, b in C.columns]
C = C.reindex([m for m in ORDER if m in C.index]).apply(f3)

print('ScamGuard model comparison, seed 42, masked text, cut-offs at 5% false alarms unless stated')
for title, t in [('A. Ranking, precision, recall, F1 and macro-F1 on known scam types', A),
                 ('B. False-alarm rates (95% Wilson intervals) and withheld scam topics', B),
                 ('C. Recall at fixed false-alarm limits of 1%, 5% and 10% (single models, earlier setup; cut-offs set on calibration data only)', C)]:
    display(t.style.set_caption(title).set_table_styles([{'selector': 'caption', 'props': 'font-weight: bold; font-size: 13px; text-align: left'}]))

models missing from a sheet: {'main': [], 'unseen': [], 'prf': []}
SVM sms_counts: tp122 fp8 fn6 tn120 | E3 dia_counts: tp682 fp47 fn25 tn660
ScamGuard model comparison, seed 42, masked text, cut-offs at 5% false alarms unless stated


,AUC SMS,AUC chat,SMS precision,SMS recall,SMS F1,SMS macro-F1,Chat precision,Chat recall,Chat F1,Chat macro-F1
model,,,,,,,,,,
SVM,0.989,0.999,0.938,0.953,0.946,0.945,0.941,1.000,0.970,0.969
LR,0.982,0.998,0.945,0.938,0.941,0.941,0.954,0.994,0.974,0.973
NB,0.974,0.998,0.967,0.906,0.935,0.937,0.953,0.993,0.972,0.972
XGB,0.989,0.995,0.961,0.961,0.961,0.961,0.931,0.993,0.961,0.960
CNN,0.972,0.997,0.967,0.906,0.935,0.937,0.948,0.984,0.966,0.965
BiLSTM,0.971,0.996,0.943,0.906,0.924,0.926,0.941,0.987,0.963,0.962
DistilBERT,0.995,0.999,1.000,0.977,0.988,0.988,0.958,0.993,0.975,0.975
LoRA,0.991,0.996,0.976,0.961,0.969,0.969,0.945,0.989,0.966,0.965
zero-shot,0.982,0.710,0.951,0.906,0.928,0.930,0.812,0.263,0.397,0.550


,"False alarms, 904 fresh normal SMS [95% Wilson]","False alarms, 707 normal chat pieces [95% Wilson]",Withheld-topic recall (SMS cut-off),Withheld-topic recall (chat cut-off),Withheld precision,Withheld F1,Withheld macro-F1
model,,,,,,,
SVM,"0.060 [0.046, 0.077]","0.062 [0.047, 0.083]",0.911,0.976,0.956,0.966,0.961
LR,"0.054 [0.041, 0.071]","0.048 [0.035, 0.067]",0.856,0.962,0.960,0.961,0.956
NB,"0.050 [0.037, 0.066]","0.050 [0.036, 0.068]",0.837,0.961,0.957,0.959,0.953
XGB,"0.052 [0.039, 0.069]","0.074 [0.057, 0.095]",0.882,0.921,0.949,0.935,0.928
CNN,"0.051 [0.038, 0.067]","0.054 [0.039, 0.073]",0.836,0.957,0.956,0.956,0.951
BiLSTM,"0.048 [0.035, 0.064]","0.062 [0.047, 0.083]",0.769,0.867,0.947,0.904,0.898
DistilBERT,"0.058 [0.044, 0.075]","0.044 [0.031, 0.062]",0.876,0.931,0.956,0.943,0.938
LoRA,"0.049 [0.036, 0.065]","0.058 [0.043, 0.078]",0.861,0.921,0.955,0.937,0.932
zero-shot,"0.045 [0.034, 0.061]","0.061 [0.045, 0.081]",0.442,0.243,0.831,0.374,0.511


,Chat scam recall at 1% false alarms,Chat scam recall at 5% false alarms,Chat scam recall at 10% false alarms,Real SMS spam recall at 1% false alarms,Real SMS spam recall at 5% false alarms,Real SMS spam recall at 10% false alarms
model,,,,,,
SVM,0.962,0.990,0.999,0.867,0.945,0.969
LR,0.931,0.976,0.989,0.789,0.938,0.969
NB,0.942,0.969,0.982,0.820,0.914,0.930
XGB,0.956,0.986,0.994,0.898,0.961,0.969
CNN,0.936,0.982,0.989,0.805,0.891,0.930
BiLSTM,0.812,0.915,0.951,0.766,0.875,0.914
DistilBERT,0.943,0.987,0.993,0.922,0.969,0.984
LoRA,0.902,0.975,0.989,0.859,0.945,0.969
zero-shot,0.141,0.484,0.721,0.688,0.906,0.953


### Words the model relies on
The words and word pairs with the largest scam and normal weights in the deployed SVM.

In [ ]:
# Which words does the deployed SVM rely on? (reads the saved model; no retraining)
import os, joblib, numpy as np, pandas as pd
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
svm = joblib.load("/content/drive/MyDrive/ScamGuard_masked_v2_seed42/svm_plain_masked_seed42.joblib")
vec, clf = svm.steps[0][1], svm.steps[-1][1]
words = vec.get_feature_names_out()
w = clf.coef_.ravel()
order = np.argsort(w)
scam = order[::-1][:20]
norm = order[:20]
print(f"Vocabulary: {len(words):,} words and word pairs | bias b = {clf.intercept_[0]:.3f}")
top = pd.DataFrame({"Pushes towards SCAM": words[scam], "weight": w[scam].round(3),
                    "Pushes towards NORMAL": words[norm], "weight ": w[norm].round(3)},
                   index=range(1, 21))
top

ValueError: mount failed

# 14. Comparison with existing systems
ScamGuard compared with (a) a public pretrained SMS spam model, bert-tiny, on the same hidden-topic test, and (b) Bitdefender Scamio, tested by hand on messages picked here.

First: find the saved test pieces and their masked text.

In [ ]:
# Comparison with an existing public model (bert-tiny), step 1: find the saved data
import os, pandas as pd
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
R  = "/content/drive/MyDrive/ScamGuard_masked_v2_seed42"
MD = f"{R}/dialogue_calibration_split/models"
print("Model folders:", sorted(os.listdir(MD)))
print("SVM files:", sorted(os.listdir(f"{MD}/SVM")))
for t in ["none", "reward"]:
    d = pd.read_csv(f"{MD}/SVM/scores_hidden_{t}.csv")
    print(f"\n--- SVM scores_hidden_{t}.csv: {d.shape} ---")
    print("Columns:", list(d.columns))
    print(d["set"].value_counts().to_string())
    print(d.head(3).to_string()[:1500])

Mounted at /content/drive
Model folders: ['BiLSTM', 'CNN', 'DistilBERT', 'E1', 'E2', 'E3', 'E4', 'LR', 'LoRA', 'NB', 'SVM', 'SVM_calibrated', 'XGB', 'dialogue_false_alarms_at_dialogue_cutoff.csv', 'summary_all_models.csv', 'zero-shot']
SVM files: ['scores_hidden_none.csv', 'scores_hidden_refund.csv', 'scores_hidden_reward.csv', 'scores_hidden_ssn.csv', 'scores_hidden_support.csv', 'summary_SVM.csv']

--- SVM scores_hidden_none.csv: (9866, 10) ---
Columns: ['set', 'source', 'type', 'label', 'text_masked', 'score', 'model', 'hidden', 'cut_sms', 'cut_dialogue']
set
calibration_sms_3100    3100
topic_ssn                962
topic_refund             954
fresh_sms_904            904
topic_support            877
topic_reward             822
dialogue_scam_test       707
dialogue_normal_test     707
calibration_dialogue     577
real_sms_test            256
                    set source     type  label                                                          text_masked     score model hidden   

Scores the same pieces with bert-tiny and compares at the same 5% cut-off. **Result:** ScamGuard 0.976 recall on new scam types, bert-tiny 0.050.

In [ ]:
# Comparison with an existing public model (bert-tiny), step 2: score and compare
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
TOPICS = ["reward", "refund", "ssn", "support"]
D = pd.read_csv(f"{MD}/SVM/scores_hidden_none.csv")      # same masked pieces every model was tested on

# --- my Linear SVM: withheld-topic recall, read from the saved retrained runs (no retraining) ---
svm_rec, svm_fa = {}, []
for t in TOPICS:
    h = pd.read_csv(f"{MD}/SVM/scores_hidden_{t}.csv")
    cut = h.cut_dialogue.iloc[0]
    svm_rec[t] = float((h[h.set == f"topic_{t}"].score > cut).mean())
    svm_fa.append(float((h[h.set == "dialogue_normal_test"].score > cut).mean()))
print("Linear SVM recall per hidden topic:", {k: round(v, 3) for k, v in svm_rec.items()},
      "| mean", round(np.mean(list(svm_rec.values())), 3), "| false alarms on 707 normal pieces", round(np.mean(svm_fa), 3))

# --- public model ---
NAME = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
tok = AutoTokenizer.from_pretrained(NAME)
bert = AutoModelForSequenceClassification.from_pretrained(NAME).to("cuda" if torch.cuda.is_available() else "cpu").eval()
print("\nLabel mapping from config:", bert.config.id2label)

def spam_prob(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i+bs]), padding=True, truncation=True, max_length=128, return_tensors="pt").to(bert.device)
        with torch.no_grad():
            out.append(torch.softmax(bert(**enc).logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

checks = ["URGENT! Your account is blocked. Send your PIN to 0788123456 now to unlock it.",
          "Hey, are we still meeting at 5 for the group project?",
          "Thanks, I got home safely. Talk tomorrow."]
for s, p in zip(checks, spam_prob(checks)): print(f"  spam prob {p:.3f} | {s}")

# --- score the same pieces ---
D["bert"] = spam_prob(D.text_masked.astype(str).tolist())
cal = D[D.set == "calibration_dialogue"].bert.values
cut5 = float(np.quantile(cal, 0.95))
print(f"\nbert-tiny 5% cut-off (from 577 normal calibration pieces): {cut5:.4f}")

rows = []
for tag, c in [("default 0.5", 0.5), ("5% cut-off", cut5)]:
    rec = {t: float((D[D.set == f"topic_{t}"].bert > c).mean()) for t in TOPICS}
    fa = float((D[D.set == "dialogue_normal_test"].bert > c).mean())
    print(f"bert-tiny, {tag}: recall per topic {({k: round(v, 3) for k, v in rec.items()})} | mean {np.mean(list(rec.values())):.3f} | false alarms on 707 normal pieces {fa:.3f}")
    rows.append((tag, np.mean(list(rec.values())), fa))

final = pd.DataFrame([
    {"Model": "ScamGuard: Linear SVM (mine)", "Recall on new scam types (mean)": np.mean(list(svm_rec.values())), "False alarms at 5% cut-off": np.mean(svm_fa)},
    {"Model": "bert-tiny SMS spam (public)", "Recall on new scam types (mean)": rows[1][1], "False alarms at 5% cut-off": rows[1][2]},
]).round(3)
print("\n" + final.to_string(index=False))

Linear SVM recall per hidden topic: {'reward': 0.972, 'refund': 0.951, 'ssn': 0.995, 'support': 0.987} | mean 0.976 | false alarms on 707 normal pieces 0.058


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/324 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 17.6MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]


Label mapping from config: {0: 'LABEL_0', 1: 'LABEL_1'}
  spam prob 0.887 | URGENT! Your account is blocked. Send your PIN to 0788123456 now to unlock it.
  spam prob 0.063 | Hey, are we still meeting at 5 for the group project?
  spam prob 0.062 | Thanks, I got home safely. Talk tomorrow.

bert-tiny 5% cut-off (from 577 normal calibration pieces): 0.1040
bert-tiny, default 0.5: recall per topic {'reward': 0.004, 'refund': 0.003, 'ssn': 0.011, 'support': 0.006} | mean 0.006 | false alarms on 707 normal pieces 0.010
bert-tiny, 5% cut-off: recall per topic {'reward': 0.068, 'refund': 0.017, 'ssn': 0.045, 'support': 0.071} | mean 0.050 | false alarms on 707 normal pieces 0.064

                       Model  Recall on new scam types (mean)  False alarms at 5% cut-off
ScamGuard: Linear SVM (mine)                            0.976                       0.058
 bert-tiny SMS spam (public)                            0.050                       0.064


Picks 30 test messages (15 scam, 15 normal) for a manual test, with ScamGuard's verdicts.

In [ ]:
# Comparison with an existing product (manual test): pick 30 messages from my test data
# 15 scams: 3 from each hidden chat topic (scored by the SVM retrained WITHOUT that topic) + 3 real SMS spam
# 15 normal: 8 normal chat pieces + 7 real normal SMS. ScamGuard's verdict uses the same 5% cut-offs as the report.
rng = 7
picks = []
for t in TOPICS:
    h = pd.read_csv(f"{MD}/SVM/scores_hidden_{t}.csv")
    x = h[h.set == f"topic_{t}"].sample(3, random_state=rng).copy()
    x["scamguard"] = np.where(x.score > x.cut_dialogue, "SCAM", "not scam"); x["group"] = f"scam chat ({t})"
    picks.append(x)
sms_test = D[D.set == "real_sms_test"]
for lab, n, g in [(1, 3, "scam SMS"), (0, 7, "normal SMS")]:
    x = sms_test[sms_test.label == lab].sample(n, random_state=rng).copy()
    x["scamguard"] = np.where(x.score > x.cut_sms, "SCAM", "not scam"); x["group"] = g
    picks.append(x)
x = D[D.set == "dialogue_normal_test"].sample(8, random_state=rng).copy()
x["scamguard"] = np.where(x.score > x.cut_dialogue, "SCAM", "not scam"); x["group"] = "normal chat"
picks.append(x)
M = pd.concat(picks, ignore_index=True)[["group", "label", "scamguard", "text_masked"]]
M.insert(0, "no", range(1, len(M) + 1))
M["truth"] = np.where(M.label == 1, "SCAM", "not scam")
M["scamio"] = ""
M.to_csv(f"{R}/manual_test_30_messages.csv", index=False)
print("Saved to Drive:", f"{R}/manual_test_30_messages.csv")
print("ScamGuard on these 30: scams caught", int(((M.label == 1) & (M.scamguard == "SCAM")).sum()), "of 15 | normal wrongly flagged",
      int(((M.label == 0) & (M.scamguard == "SCAM")).sum()), "of 15\n")
for r in M.itertuples():
    print(f"[{r.no}] {r.group} | truth {r.truth} | ScamGuard {r.scamguard}\n    {r.text_masked}\n")

Saved to Drive: /content/drive/MyDrive/ScamGuard_masked_v2_seed42/manual_test_30_messages.csv
ScamGuard on these 30: scams caught 14 of 15 | normal wrongly flagged 2 of 15

[1] scam chat (reward) | truth SCAM | ScamGuard SCAM
    the retail store again? caller: It's Walmart. receiver: Okay, so how do I get this gift card? caller: We just need to verify some information to ensure it's sent to the right person. Can you please provide your social security number? receiver: Why do you

[2] scam chat (reward) | truth SCAM | ScamGuard SCAM
    to process the reward. receiver: I see. Well, can you tell me more about how I was selected? Was it based on my purchase history or something? caller: Uh, yeah... it was based on your purchase history. You're one of our valued customers. receiver: That's

[3] scam chat (reward) | truth SCAM | ScamGuard SCAM
    to verify your information to ensure you're the correct recipient. Can you please confirm your full name and address? receiver: Hold on, let me

Finds the scam pieces that look most harmless (lowest SVM scores).

In [ ]:
# Scam chats that don't look like scams: lowest-scoring real scam pieces, one model trained WITHOUT that topic
hard = []
for t in TOPICS:
    h = pd.read_csv(f"{MD}/SVM/scores_hidden_{t}.csv")
    x = h[(h.set == f"topic_{t}") & (h.text_masked.str.split().str.len() >= 40)].copy()
    x["topic"] = t; x["scamguard"] = np.where(x.score > x.cut_dialogue, "SCAM", "not scam")
    hard.append(x.nsmallest(3, "score"))
H = pd.concat(hard, ignore_index=True)[["topic", "score", "scamguard", "text_masked"]]
for i, r in enumerate(H.itertuples(), 1):
    print(f"[{i}] {r.topic} | score {r.score:.3f} | ScamGuard {r.scamguard}\n    {r.text_masked}\n")

[1] reward | score -1.026 | ScamGuard not scam
    agencies to regulate consumer rewards programs. We're a reputable organization with an A+ rating from the Better Business Bureau. receiver: That sounds good. But how did you get my phone number? Did I sign up for something online? caller: Ah, good question! We obtained your

[2] reward | score -0.728 | ScamGuard not scam
    sounds too good to be true. What's the name of your company and can I look it up online? caller: Our company is called "Lucky Winners Inc." and yes, you can definitely look us up. But please note that our website is currently under maintenance,

[3] reward | score -0.726 | ScamGuard not scam
    database and we want to reward you with this gift card. receiver: That sounds too good to be true. How did you get my number? caller: Like I said, it was randomly selected from our database. But let me ask you, are you interested in

[4] refund | score -1.331 | ScamGuard not scam
    me check on that. Okay, it says here tha

Loads the full chats behind those pieces. They come from the training split, so they were not used for the head-to-head.

In [ ]:
# Full scam chats behind the 5 hardest pieces (Scamio needs the whole conversation)
from datasets import load_dataset
bb = load_dataset("BothBosu/scam-dialogue")
allc = pd.concat([bb["train"].to_pandas().assign(split="train"), bb["test"].to_pandas().assign(split="test")], ignore_index=True)
keys = ["A+ rating from the Better Business Bureau", "order #1234. Does that sound right", "it seems we have the wrong account information",
        "scammers based in Nigeria", "Windows 97"]
FULL = []
for k in keys:
    m = allc[(allc.label == 1) & allc.dialogue.str.contains(k, regex=False)]
    print(f"'{k}': {len(m)} matching scam chat(s)")
    if len(m): FULL.append(m.iloc[0])
FULL = pd.DataFrame(FULL)
FULL.to_csv(f"{R}/manual_test_5_full_scam_chats.csv", index=False)
for i, r in enumerate(FULL.itertuples(), 1):
    print(f"\n===== [{i}] {r.type} | {r.split} | {len(r.dialogue.split())} words =====\n{r.dialogue}")

README.md:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

scam-dialogue_train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

scam-dialogue_test.csv:   0%|          | 0.00/453k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1280 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/320 [00:00<?, ? examples/s]

'A+ rating from the Better Business Bureau': 1 matching scam chat(s)
'order #1234. Does that sound right': 1 matching scam chat(s)
'it seems we have the wrong account information': 1 matching scam chat(s)
'scammers based in Nigeria': 1 matching scam chat(s)
'Windows 97': 1 matching scam chat(s)

===== [1] reward | train | 371 words =====
caller: Hello, is this Mrs. Johnson? receiver: Yes, that's me. Who's calling? caller: My name is Rachel and I'm calling from the Consumer Rewards Center. How are you today? receiver: I'm doing great, thanks. What can I do for you, Rachel? caller: Well, congratulations are in order! You've been selected to receive a free $1,000 gift card from one of our major retailers. receiver: That sounds amazing! How did I win it? caller: It's part of a promotional campaign we're running. You were chosen based on your shopping habits and loyalty to our partner brands. receiver: That's terrific! What do I need to do to claim my prize? caller: All I need is some basic

Picks 10 full chats from the test split for the head-to-head with Scamio. **Result:** both caught 5 of 5 scams; Scamio had no false alarms and ScamGuard had 1 (medium risk).

In [ ]:
# Fair head-to-head: 10 FULL chats from the TEST split (never used to train ScamGuard)
te = bb['test'].to_pandas()
te['words'] = te.dialogue.str.split().str.len()
te = te[(te.words >= 120) & (te.words <= 260)]
scam_types = ['reward', 'refund', 'ssn', 'support', 'reward']
norm_types = ['delivery', 'insurance', 'telemarketing', 'wrong', 'delivery']
rows = []
for lab, types in [(1, scam_types), (0, norm_types)]:
    used = set()
    for t in types:
        pool = te[(te.label == lab) & (te.type == t) & ~te.index.isin(used)]
        r = pool.sample(1, random_state=11).iloc[0]; used.add(r.name); rows.append(r)
H2H = pd.DataFrame(rows).sample(frac=1, random_state=3).reset_index(drop=True)
H2H.insert(0, 'no', range(1, 11))
H2H['truth'] = np.where(H2H.label == 1, 'SCAM', 'not scam'); H2H['scamguard'] = ''; H2H['scamio'] = ''
H2H.to_csv(f'{R}/head_to_head_10_test_chats.csv', index=False)
print('Saved:', f'{R}/head_to_head_10_test_chats.csv')
print(H2H[['no', 'type', 'truth', 'words']].to_string(index=False))
for r in H2H.itertuples():
    print(f"\n===== CHAT {r.no} ({r.words} words) =====\n{r.dialogue}")

Saved: /content/drive/MyDrive/ScamGuard_masked_v2_seed42/head_to_head_10_test_chats.csv
 no          type    truth  words
  1      delivery not scam    152
  2        reward     SCAM    205
  3        refund     SCAM    170
  4           ssn     SCAM    192
  5      delivery not scam    140
  6     insurance not scam    225
  7 telemarketing not scam    258
  8        reward     SCAM    250
  9       support     SCAM    229
 10         wrong not scam    121

===== CHAT 1 (152 words) =====
caller: Hi, I'm calling from Speedy Delivery. I just wanted to confirm that your package will be delivered today. receiver: Oh, really? What's the package about? caller: Let me check. Okay, it says here that you ordered a new laptop from BestBuy. receiver: That's correct. But how did you get my number? caller: It was provided by BestBuy when we received the shipping request. receiver: I see. Can you tell me more about the laptop I ordered? caller: Sure thing. You got a Dell Inspiron with 16GB of RAM a